In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 1 — Reload final CosMx Step4 + Step5/9E outputs
#
# Purpose:
#   This cell prepares the CosMx data so the Xenium-style downstream analyses
#   can be run on CosMx.
#
# It loads:
#   1. Corrected Step4 denoised AnnData
#   2. Raw matrix from denoised_adata.layers["raw"]
#   3. Step4 denoised matrix from denoised_adata.X
#   4. was_corrected mask
#   5. Step4 config
#   6. Observed molecule table after Step5 coordinate normalization
#   7. Learned 9E imputed molecule records
#   8. Three empirical baseline imputed molecule records
#   9. Imputation targets / reconciliation / summaries
#
# CosMx-safe:
#   - Keeps cell_id as string, e.g. "1_1"
#   - Uses denoised_adata.obs_names as authoritative cell order
#   - Uses denoised_adata.var_names as authoritative gene order
#   - Does NOT convert cell IDs to int
#
# Notes:
#   - The completed molecule table is very large. By default, this cell does not
#     load it into memory. It keeps the path available as COMPLETED_MOL_PATH.
#   - If you really need to load the completed table, set LOAD_COMPLETED_TABLE=True.
# ==============================================================================

import os
import gc
import json
import pickle
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import sparse

try:
    import anndata as ad
    import scanpy as sc
except Exception as e:
    print(f"AnnData/Scanpy import failed: {e}")
    print("Installing required packages...")
    !pip install -q anndata scanpy
    import anndata as ad
    import scanpy as sc

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=" * 100)
print("COSMX DOWNSTREAM CELL 1: Reloading final Step4 + Step5/9E outputs")
print("=" * 100)

# ------------------------------------------------------------------------------
# 1. Main paths
# ------------------------------------------------------------------------------

STEP4_EXPORT_DIR = Path("/content/drive/MyDrive/diffusion/step4_cosmx/step4_exports")
IMPUTATION_DIR   = Path("/content/drive/MyDrive/diffusion/step4_cosmx/imputation")

# New CosMx downstream output/checkpoint folder.
# Save all new downstream figures, CSVs, reports, and intermediate outputs here.
DOWNSTREAM_DIR   = Path("/content/drive/MyDrive/diffusion/step4_cosmx/downstream")
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

print(f"STEP4_EXPORT_DIR : {STEP4_EXPORT_DIR}")
print(f"IMPUTATION_DIR   : {IMPUTATION_DIR}")
print(f"DOWNSTREAM_DIR   : {DOWNSTREAM_DIR}")
print(f"RUN_NAME         : {RUN_NAME}")

if not STEP4_EXPORT_DIR.exists():
    raise FileNotFoundError(f"Missing Step4 export folder: {STEP4_EXPORT_DIR}")

if not IMPUTATION_DIR.exists():
    raise FileNotFoundError(f"Missing imputation folder: {IMPUTATION_DIR}")

if not DOWNSTREAM_DIR.exists():
    raise FileNotFoundError(f"Could not create downstream folder: {DOWNSTREAM_DIR}")

# ------------------------------------------------------------------------------
# 2. Required file paths
# ------------------------------------------------------------------------------

STEP4_CONFIG_PATH   = STEP4_EXPORT_DIR / "step4_config.json"
DENOISED_ADATA_PATH = STEP4_EXPORT_DIR / "denoised_adata.h5ad"
WAS_CORRECTED_PATH  = STEP4_EXPORT_DIR / "was_corrected.npy"
STEP4_MOLECULES_PATH = STEP4_EXPORT_DIR / "molecules.parquet"
CELL_DATA_PATH      = STEP4_EXPORT_DIR / "cell_data.npz"

OBSERVED_MOL_PATH = IMPUTATION_DIR / "step5_mol_processed.parquet"

LEARNED_IMPUTED_PATH = IMPUTATION_DIR / f"{RUN_NAME}_imputed_records.parquet"
COMPLETED_MOL_PATH   = IMPUTATION_DIR / f"{RUN_NAME}_completed_molecule_table.parquet"

GENE_EMP_IMPUTED_PATH = IMPUTATION_DIR / f"{RUN_NAME}_gene_emp_imputed_records.parquet"
CT_GENE_EMP_IMPUTED_PATH = IMPUTATION_DIR / f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet"
SPATIAL_KNN_EMP_IMPUTED_PATH = IMPUTATION_DIR / f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet"

IMPUTATION_TARGETS_PATH = IMPUTATION_DIR / f"{RUN_NAME}_imputation_targets.csv"
COUNT_RECONCILIATION_PATH = IMPUTATION_DIR / f"{RUN_NAME}_count_reconciliation.csv"

IMPUTATION_SUMMARY_CSV_PATH = IMPUTATION_DIR / f"{RUN_NAME}_imputation_summary.csv"
IMPUTATION_SUMMARY_JSON_PATH = IMPUTATION_DIR / f"{RUN_NAME}_imputation_summary.json"

BASELINE_SUMMARY_CSV_PATH = IMPUTATION_DIR / f"{RUN_NAME}_baseline_imputation_summary.csv"
BASELINE_SUMMARY_JSON_PATH = IMPUTATION_DIR / f"{RUN_NAME}_baseline_imputation_summary.json"

FINAL_RECOVERY_METRICS_CSV_PATH = IMPUTATION_DIR / f"{RUN_NAME}_final_large_recovery_metrics.csv"
FINAL_RECOVERY_REPORT_PATH = IMPUTATION_DIR / f"{RUN_NAME}_final_large_recovery_report.txt"

DOWNSTREAM_COUNT_METRICS_PATH = IMPUTATION_DIR / f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv"
DOWNSTREAM_CLUSTERING_METRICS_PATH = IMPUTATION_DIR / f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv"
DOWNSTREAM_REPORT_PATH = IMPUTATION_DIR / f"{RUN_NAME}_downstream_validation_report_step4matched.txt"
DOWNSTREAM_SAMPLE_IDX_PATH = IMPUTATION_DIR / f"{RUN_NAME}_downstream_step4matched_sample_idx.npy"

# Optional CosMx summary geometry cache.
COSMX_GEOM_CACHE_PATH = IMPUTATION_DIR / f"{RUN_NAME}_cosmx_summary_geometry_cache.pkl"

# ------------------------------------------------------------------------------
# 3. Check required files
# ------------------------------------------------------------------------------

required_files = {
    "Step4 config": STEP4_CONFIG_PATH,
    "Step4 denoised AnnData": DENOISED_ADATA_PATH,
    "was_corrected mask": WAS_CORRECTED_PATH,
    "Step4 molecule table": STEP4_MOLECULES_PATH,
    "Observed Step5 molecule table": OBSERVED_MOL_PATH,
    "Learned 9E imputed records": LEARNED_IMPUTED_PATH,
    "Completed learned 9E molecule table": COMPLETED_MOL_PATH,
    "Gene empirical imputed records": GENE_EMP_IMPUTED_PATH,
    "Cell-type gene empirical imputed records": CT_GENE_EMP_IMPUTED_PATH,
    "Spatial-kNN empirical imputed records": SPATIAL_KNN_EMP_IMPUTED_PATH,
    "Imputation targets": IMPUTATION_TARGETS_PATH,
    "Count reconciliation": COUNT_RECONCILIATION_PATH,
    "Imputation summary CSV": IMPUTATION_SUMMARY_CSV_PATH,
    "Imputation summary JSON": IMPUTATION_SUMMARY_JSON_PATH,
    "Baseline summary CSV": BASELINE_SUMMARY_CSV_PATH,
    "Baseline summary JSON": BASELINE_SUMMARY_JSON_PATH,
}

optional_files = {
    "cell_data.npz": CELL_DATA_PATH,
    "final large recovery metrics CSV": FINAL_RECOVERY_METRICS_CSV_PATH,
    "final large recovery report": FINAL_RECOVERY_REPORT_PATH,
    "downstream count metrics": DOWNSTREAM_COUNT_METRICS_PATH,
    "downstream clustering metrics": DOWNSTREAM_CLUSTERING_METRICS_PATH,
    "downstream validation report": DOWNSTREAM_REPORT_PATH,
    "downstream sample index": DOWNSTREAM_SAMPLE_IDX_PATH,
    "CosMx summary geometry cache": COSMX_GEOM_CACHE_PATH,
}

print("\n" + "=" * 100)
print("Checking required files")
print("=" * 100)

missing_required = []

for label, path in required_files.items():
    if path.exists():
        print(f"  ✓ {label:45s} {path.name:85s} {path.stat().st_size / (1024**2):10.2f} MB")
    else:
        print(f"  ✗ MISSING: {label:45s} {path}")
        missing_required.append((label, path))

if missing_required:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join([f"{label}: {path}" for label, path in missing_required])
    )

print("\n" + "=" * 100)
print("Checking optional files")
print("=" * 100)

for label, path in optional_files.items():
    if path.exists():
        print(f"  ✓ {label:45s} {path.name:85s} {path.stat().st_size / (1024**2):10.2f} MB")
    else:
        print(f"  - optional missing: {label:45s} {path.name}")

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_cell_type_column(adata):
    candidate_cols = [
        "cell_type",
        "Final_CosMx_Cell_Type",
        "Assigned_Xenium_Cell_Type",
        "Reference_Cell_Type",
        "celltype",
        "Cell_Type",
    ]

    for col in candidate_cols:
        if col in adata.obs.columns:
            return col

    raise KeyError(
        "No recognized cell-type column found in denoised_adata.obs. "
        f"Available obs columns: {list(adata.obs.columns)}"
    )


def summarize_molecule_table(df, name):
    print("\n" + "-" * 100)
    print(f"{name}")
    print("-" * 100)
    print(f"Rows        : {len(df):,}")
    print(f"Columns     : {list(df.columns)}")

    if "cell_id" in df.columns:
        print(f"Unique cells: {df['cell_id'].astype(str).nunique():,}")

    if "gene_id" in df.columns:
        print(f"Unique genes: {df['gene_id'].astype(str).nunique():,}")

    if "is_imputed" in df.columns:
        print("is_imputed counts:")
        print(df["is_imputed"].value_counts(dropna=False).to_string())

    if "status" in df.columns:
        print("status counts:")
        print(df["status"].value_counts(dropna=False).to_string())

    coord_cols = [c for c in ["x", "y", "z", "r_norm", "theta", "z_rel", "p_nuclear"] if c in df.columns]
    if coord_cols:
        print("\nCoordinate/localization summary:")
        display(df[coord_cols].describe().T)

# ------------------------------------------------------------------------------
# 5. Load Step4 config
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading Step4 config")
print("=" * 100)

with open(STEP4_CONFIG_PATH, "r") as f:
    step4_config = json.load(f)

print("Step4 config keys:")
print(list(step4_config.keys()))

# Prefer config values if present, but later use denoised_adata as authoritative.
config_shared_genes = [str(g) for g in step4_config.get("shared_genes", [])]
config_cell_type_col = step4_config.get("cell_type_column", None)

print(f"Shared genes in config: {len(config_shared_genes):,}")
print(f"Cell-type column in config: {config_cell_type_col}")

# ------------------------------------------------------------------------------
# 6. Load denoised AnnData
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading denoised_adata.h5ad")
print("=" * 100)

denoised_adata = ad.read_h5ad(DENOISED_ADATA_PATH)

print(denoised_adata)
print(f"Shape: {denoised_adata.shape}")
print(f"Layers: {list(denoised_adata.layers.keys())}")
print(f"obs columns: {list(denoised_adata.obs.columns)}")
print(f"var columns: {list(denoised_adata.var.columns)}")

if "raw" not in denoised_adata.layers:
    raise KeyError(
        "denoised_adata.layers['raw'] is missing. "
        "Downstream needs this as the raw observed count matrix."
    )

ct_col = get_cell_type_column(denoised_adata)

print(f"\nUsing cell-type column: {ct_col}")
print("Cell-type counts:")
display(denoised_adata.obs[ct_col].astype(str).value_counts().to_frame("n_cells"))

# Authoritative cell/gene order.
cell_ids_step4 = np.array([str(c) for c in denoised_adata.obs_names], dtype=str)
shared_genes = [str(g) for g in denoised_adata.var_names]

cell_idx_map = {cid: i for i, cid in enumerate(cell_ids_step4)}
gene_idx_map = {g: j for j, g in enumerate(shared_genes)}

print(f"\nAuthoritative cells: {len(cell_ids_step4):,}")
print(f"Authoritative genes: {len(shared_genes):,}")
print("First 10 cells:")
print(list(cell_ids_step4[:10]))
print("First 10 genes:")
print(shared_genes[:10])

# ------------------------------------------------------------------------------
# 7. Load raw and Step4 matrices
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Preparing raw and Step4 count matrices")
print("=" * 100)

X_raw_counts = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

if X_raw_counts.shape != denoised_adata.shape:
    raise ValueError(
        f"X_raw_counts shape {X_raw_counts.shape} does not match denoised_adata shape {denoised_adata.shape}"
    )

if X_denoised.shape != denoised_adata.shape:
    raise ValueError(
        f"X_denoised shape {X_denoised.shape} does not match denoised_adata shape {denoised_adata.shape}"
    )

delta_step4 = X_denoised - X_raw_counts

print(f"X_raw_counts shape       : {X_raw_counts.shape}")
print(f"X_denoised shape         : {X_denoised.shape}")
print(f"Raw total counts         : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Step4 denoised total     : {X_denoised.sum(dtype=np.float64):,.2f}")
print(f"Step4 added total        : {delta_step4.sum(dtype=np.float64):,.2f}")
print(f"Negative delta entries   : {int((delta_step4 < -1e-6).sum()):,}")
print(f"Positive delta entries   : {int((delta_step4 > 1e-6).sum()):,}")

# Backward-compatible aliases for later downstream cells.
X_raw_full = X_raw_counts
X_step4_full = X_denoised

# ------------------------------------------------------------------------------
# 8. Load was_corrected
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading was_corrected mask")
print("=" * 100)

was_corrected = np.load(WAS_CORRECTED_PATH)

if was_corrected.shape != denoised_adata.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match denoised_adata shape {denoised_adata.shape}"
    )

was_corrected = was_corrected.astype(bool)

print(f"was_corrected shape      : {was_corrected.shape}")
print(f"Corrected cell-gene pairs: {was_corrected.sum():,}")
print(f"Correction rate          : {100 * was_corrected.sum() / was_corrected.size:.4f}%")

# ------------------------------------------------------------------------------
# 9. Load molecule tables
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading molecule tables")
print("=" * 100)

# Load observed Step5 processed molecule table.
# This is the preferred observed molecule table for downstream because it has
# r_norm/theta/z_rel/p_nuclear/status/weight/is_imputed.
mol_observed = pd.read_parquet(OBSERVED_MOL_PATH)

# CosMx-safe type handling.
if "cell_id" in mol_observed.columns:
    mol_observed["cell_id"] = mol_observed["cell_id"].astype(str)
if "gene_id" in mol_observed.columns:
    mol_observed["gene_id"] = mol_observed["gene_id"].astype(str)

summarize_molecule_table(mol_observed, "Observed Step5 processed molecules: mol_observed")

# Load learned 9E imputed records.
mol_learned_9E = pd.read_parquet(LEARNED_IMPUTED_PATH)

if "cell_id" in mol_learned_9E.columns:
    mol_learned_9E["cell_id"] = mol_learned_9E["cell_id"].astype(str)
if "gene_id" in mol_learned_9E.columns:
    mol_learned_9E["gene_id"] = mol_learned_9E["gene_id"].astype(str)

summarize_molecule_table(mol_learned_9E, "Learned 9E imputed molecules: mol_learned_9E")

# Load empirical baseline imputed records.
mol_gene_emp = pd.read_parquet(GENE_EMP_IMPUTED_PATH)
mol_ct_gene_emp = pd.read_parquet(CT_GENE_EMP_IMPUTED_PATH)
mol_spatial_knn_emp = pd.read_parquet(SPATIAL_KNN_EMP_IMPUTED_PATH)

for df_name, df in [
    ("mol_gene_emp", mol_gene_emp),
    ("mol_ct_gene_emp", mol_ct_gene_emp),
    ("mol_spatial_knn_emp", mol_spatial_knn_emp),
]:
    if "cell_id" in df.columns:
        df["cell_id"] = df["cell_id"].astype(str)
    if "gene_id" in df.columns:
        df["gene_id"] = df["gene_id"].astype(str)

summarize_molecule_table(mol_gene_emp, "Gene empirical imputed molecules: mol_gene_emp")
summarize_molecule_table(mol_ct_gene_emp, "Cell-type gene empirical imputed molecules: mol_ct_gene_emp")
summarize_molecule_table(mol_spatial_knn_emp, "Spatial-kNN empirical imputed molecules: mol_spatial_knn_emp")

# ------------------------------------------------------------------------------
# 10. Optional: load completed molecule table
# ------------------------------------------------------------------------------

LOAD_COMPLETED_TABLE = False

print("\n" + "=" * 100)
print("Completed molecule table")
print("=" * 100)

print(f"Completed molecule table path:")
print(COMPLETED_MOL_PATH)
print(f"Size: {COMPLETED_MOL_PATH.stat().st_size / (1024**3):.3f} GB")

if LOAD_COMPLETED_TABLE:
    print("LOAD_COMPLETED_TABLE=True, loading completed molecule table into memory...")
    mol_completed_9E = pd.read_parquet(COMPLETED_MOL_PATH)

    if "cell_id" in mol_completed_9E.columns:
        mol_completed_9E["cell_id"] = mol_completed_9E["cell_id"].astype(str)
    if "gene_id" in mol_completed_9E.columns:
        mol_completed_9E["gene_id"] = mol_completed_9E["gene_id"].astype(str)

    summarize_molecule_table(mol_completed_9E, "Completed learned 9E molecule table: mol_completed_9E")
else:
    mol_completed_9E = None
    print(
        "Not loading completed molecule table to save memory. "
        "Use COMPLETED_MOL_PATH if a later cell needs it."
    )

# ------------------------------------------------------------------------------
# 11. Load targets, reconciliation, summaries
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading imputation targets, reconciliation, and summaries")
print("=" * 100)

imputation_targets = pd.read_csv(IMPUTATION_TARGETS_PATH)
count_reconciliation = pd.read_csv(COUNT_RECONCILIATION_PATH)

imputation_summary_df = pd.read_csv(IMPUTATION_SUMMARY_CSV_PATH)
baseline_imputation_summary_df = pd.read_csv(BASELINE_SUMMARY_CSV_PATH)

with open(IMPUTATION_SUMMARY_JSON_PATH, "r") as f:
    imputation_summary_json = json.load(f)

with open(BASELINE_SUMMARY_JSON_PATH, "r") as f:
    baseline_imputation_summary_json = json.load(f)

# CosMx-safe ID casting.
for df in [imputation_targets, count_reconciliation]:
    if "cell_id" in df.columns:
        df["cell_id"] = df["cell_id"].astype(str)
    if "gene_id" in df.columns:
        df["gene_id"] = df["gene_id"].astype(str)

print(f"imputation_targets shape        : {imputation_targets.shape}")
print(f"count_reconciliation shape      : {count_reconciliation.shape}")
print(f"imputation_summary_df shape     : {imputation_summary_df.shape}")
print(f"baseline_summary_df shape       : {baseline_imputation_summary_df.shape}")

print("\nImputation summary:")
display(imputation_summary_df)

print("\nBaseline imputation summary:")
display(baseline_imputation_summary_df)

print("\nCount reconciliation preview:")
display(count_reconciliation.head())

# ------------------------------------------------------------------------------
# 12. Optional: load previously saved validation summaries if they exist
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading previously saved downstream/recovery summaries if available")
print("=" * 100)

final_large_recovery_metrics_df = None
downstream_count_metrics_df = None
downstream_clustering_metrics_df = None

if FINAL_RECOVERY_METRICS_CSV_PATH.exists():
    final_large_recovery_metrics_df = pd.read_csv(FINAL_RECOVERY_METRICS_CSV_PATH)
    print("Loaded final large recovery metrics:")
    display(final_large_recovery_metrics_df)
else:
    print("Final large recovery metrics CSV not found.")

if DOWNSTREAM_COUNT_METRICS_PATH.exists():
    downstream_count_metrics_df = pd.read_csv(DOWNSTREAM_COUNT_METRICS_PATH)
    print("\nLoaded previous downstream count metrics:")
    display(downstream_count_metrics_df)
else:
    print("Previous downstream count metrics not found.")

if DOWNSTREAM_CLUSTERING_METRICS_PATH.exists():
    downstream_clustering_metrics_df = pd.read_csv(DOWNSTREAM_CLUSTERING_METRICS_PATH)
    print("\nLoaded previous downstream clustering metrics:")
    display(downstream_clustering_metrics_df)
else:
    print("Previous downstream clustering metrics not found.")

if DOWNSTREAM_SAMPLE_IDX_PATH.exists():
    downstream_sample_idx = np.load(DOWNSTREAM_SAMPLE_IDX_PATH)
    print(f"\nLoaded previous downstream sample index: {len(downstream_sample_idx):,} cells")
else:
    downstream_sample_idx = None
    print("\nPrevious downstream sample index not found.")

# ------------------------------------------------------------------------------
# 13. Optional geometry loading
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading optional CosMx geometry/cache information")
print("=" * 100)

cell_data = None
cosmx_geometry_cache = None

if CELL_DATA_PATH.exists():
    try:
        cell_data = np.load(CELL_DATA_PATH, allow_pickle=True)
        print("Loaded cell_data.npz")
        print(f"Available arrays: {list(cell_data.keys())}")
    except Exception as e:
        print(f"Could not load cell_data.npz: {e}")

if COSMX_GEOM_CACHE_PATH.exists():
    try:
        with open(COSMX_GEOM_CACHE_PATH, "rb") as f:
            cosmx_geometry_cache = pickle.load(f)

        print("Loaded CosMx summary geometry cache.")
        if isinstance(cosmx_geometry_cache, dict):
            print(f"Geometry cache keys: {list(cosmx_geometry_cache.keys())}")
    except Exception as e:
        print(f"Could not load CosMx geometry cache: {e}")
else:
    print("CosMx geometry cache not found.")

print(
    "\nImportant CosMx geometry note:\n"
    "  Exact Xenium-style cell/nucleus polygons are not available in this Step5 setup.\n"
    "  Downstream cells that require polygon geometry must be adapted to use\n"
    "  centroid/radius/r_norm/p_nuclear-based approximations."
)

# ------------------------------------------------------------------------------
# 14. Build convenient method registry for later downstream cells
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Building method registry")
print("=" * 100)

method_molecule_tables = {
    "Raw observed": mol_observed,
    "Learned 9E": mol_learned_9E,
    "Gene empirical": mol_gene_emp,
    "Cell-type gene empirical": mol_ct_gene_emp,
    "Spatial-kNN empirical": mol_spatial_knn_emp,
}

method_imputed_tables = {
    "Learned 9E": mol_learned_9E,
    "Gene empirical": mol_gene_emp,
    "Cell-type gene empirical": mol_ct_gene_emp,
    "Spatial-kNN empirical": mol_spatial_knn_emp,
}

method_paths = {
    "Raw observed": OBSERVED_MOL_PATH,
    "Learned 9E imputed": LEARNED_IMPUTED_PATH,
    "Learned 9E completed": COMPLETED_MOL_PATH,
    "Gene empirical imputed": GENE_EMP_IMPUTED_PATH,
    "Cell-type gene empirical imputed": CT_GENE_EMP_IMPUTED_PATH,
    "Spatial-kNN empirical imputed": SPATIAL_KNN_EMP_IMPUTED_PATH,
}

print("Available molecule tables:")
for name, df in method_molecule_tables.items():
    print(f"  {name:25s}: {len(df):,} rows")

print("\nAvailable paths:")
for name, path in method_paths.items():
    print(f"  {name:35s}: {path}")

# ------------------------------------------------------------------------------
# 15. Final sanity checks
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Final sanity checks")
print("=" * 100)

# Expected CosMx rough values from your run.
print(f"Cells                      : {denoised_adata.n_obs:,}")
print(f"Genes                      : {denoised_adata.n_vars:,}")
print(f"Observed molecules          : {len(mol_observed):,}")
print(f"Learned 9E imputed molecules: {len(mol_learned_9E):,}")
print(f"Gene empirical molecules    : {len(mol_gene_emp):,}")
print(f"CT-gene empirical molecules : {len(mol_ct_gene_emp):,}")
print(f"Spatial-kNN empirical mols  : {len(mol_spatial_knn_emp):,}")

expected_completed_rows = len(mol_observed) + len(mol_learned_9E)
print(f"Expected completed rows      : {expected_completed_rows:,}")

if COMPLETED_MOL_PATH.exists():
    print(
        "Completed table exists. It is not loaded by default, but expected rows should be "
        f"observed + learned imputed = {expected_completed_rows:,}."
    )

# Verify imputed cells/genes map to Step4 object.
for name, df in method_imputed_tables.items():
    unmapped_cells = (~df["cell_id"].astype(str).isin(cell_idx_map)).sum() if "cell_id" in df.columns else np.nan
    unmapped_genes = (~df["gene_id"].astype(str).isin(gene_idx_map)).sum() if "gene_id" in df.columns else np.nan

    print(f"\n{name}:")
    print(f"  unmapped cells: {unmapped_cells:,}")
    print(f"  unmapped genes: {unmapped_genes:,}")

    if unmapped_cells != 0 or unmapped_genes != 0:
        print("  WARNING: Some imputed molecules do not map to denoised_adata cell/gene order.")

gc.collect()

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 1 COMPLETE")
print("=" * 100)

print("Main variables now available:")
print("""
denoised_adata
X_raw_counts
X_denoised
X_raw_full
X_step4_full
was_corrected
shared_genes
cell_ids_step4
cell_idx_map
gene_idx_map
ct_col

mol_observed
mol_learned_9E
mol_gene_emp
mol_ct_gene_emp
mol_spatial_knn_emp
mol_completed_9E  # None unless LOAD_COMPLETED_TABLE=True

imputation_targets
count_reconciliation
imputation_summary_df
baseline_imputation_summary_df
final_large_recovery_metrics_df
downstream_count_metrics_df
downstream_clustering_metrics_df
downstream_sample_idx

method_molecule_tables
method_imputed_tables
method_paths

STEP4_EXPORT_DIR
IMPUTATION_DIR
DOWNSTREAM_DIR

COMPLETED_MOL_PATH
OBSERVED_MOL_PATH
LEARNED_IMPUTED_PATH
GENE_EMP_IMPUTED_PATH
CT_GENE_EMP_IMPUTED_PATH
SPATIAL_KNN_EMP_IMPUTED_PATH
""")

===========COUNT-LEVEL VALIDATION============

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 2 — Statistical power / analyzable cell-gene pair gain
#
# Biological question:
#   Does 9E imputation make more cell-gene pairs usable for downstream analyses?
#
# Main comparison:
#   Raw observed counts vs Learned 9E completed counts
#
# Same algorithm as Xenium Cell 2:
#   1. Build completed count matrix = raw counts + learned 9E imputed molecule counts
#   2. Count analyzable cell-gene pairs at thresholds [1, 2, 3, 5, 8, 10]
#   3. Summarize overall gain
#   4. Summarize gain by gene
#   5. Summarize gain by cell type
#
# CosMx-safe:
#   - Keeps cell_id as string, e.g. "1_1"
#   - Uses denoised_adata.obs_names / var_names order from Cell 1
#   - Saves outputs to DOWNSTREAM_DIR
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
from scipy import sparse

print("=" * 100)
print("COSMX DOWNSTREAM CELL 2 — Statistical power / analyzable cell-gene pair gain")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables from Cell 1
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "X_raw_counts",
    "mol_learned_9E",
    "shared_genes",
    "cell_ids_step4",
    "cell_idx_map",
    "gene_idx_map",
    "denoised_adata",
    "ct_col",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s) from Cell 1:\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

# Make sure output folder exists.
DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

POWER_SUMMARY_PATH = DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_power_gain_summary.csv"

POWER_BY_GENE_PATH = DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_power_gain_by_gene.csv"

POWER_BY_CELLTYPE_PATH = DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_power_gain_by_celltype.csv"

print("\nOutputs will be saved to:")
print(f"  Overall summary : {POWER_SUMMARY_PATH}")
print(f"  By gene         : {POWER_BY_GENE_PATH}")
print(f"  By cell type    : {POWER_BY_CELLTYPE_PATH}")

# ------------------------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def build_completed_count_matrix_from_imputed_cosmx(X_raw, imputed_df):
    """
    Builds:
        X_completed = X_raw + imputed molecule counts per (cell_id, gene_id)

    CosMx-safe:
      - cell_id is always kept as string
      - gene_id is kept as string
      - mapping uses cell_idx_map and gene_idx_map from Cell 1
    """

    X_completed = np.asarray(X_raw, dtype=np.float32).copy()

    required_cols = ["cell_id", "gene_id"]
    missing_cols = [c for c in required_cols if c not in imputed_df.columns]

    if missing_cols:
        raise KeyError(f"mol_learned_9E missing required columns: {missing_cols}")

    counts = (
        imputed_df[["cell_id", "gene_id"]]
        .copy()
        .assign(
            cell_id=lambda d: d["cell_id"].astype(str),
            gene_id=lambda d: d["gene_id"].astype(str),
        )
        .groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name="n_imputed")
    )

    rr = counts["cell_id"].map(cell_idx_map)
    cc = counts["gene_id"].map(gene_idx_map)

    ok = rr.notna() & cc.notna()

    n_unmapped = len(counts) - int(ok.sum())
    if n_unmapped > 0:
        print(f"WARNING: dropped {n_unmapped:,} unmapped imputed cell-gene rows.")

        print("\nExample unmapped rows:")
        display(counts.loc[~ok].head())

    X_completed[
        rr[ok].astype(int).to_numpy(),
        cc[ok].astype(int).to_numpy(),
    ] += counts.loc[ok, "n_imputed"].to_numpy(dtype=np.float32)

    return X_completed, counts


def analyzable_summary(X, thresholds, label):
    """
    Count how many cell-gene entries are analyzable at each threshold.
    """
    rows = []

    total_pairs = int(X.shape[0] * X.shape[1])

    for t in thresholds:
        n_pairs = int((X >= t).sum())
        frac_pairs = n_pairs / total_pairs

        rows.append({
            "dataset": label,
            "threshold_min_count": int(t),
            "analyzable_cell_gene_pairs": n_pairs,
            "fraction_of_all_cell_gene_pairs": frac_pairs,
            "total_cell_gene_pairs": total_pairs,
        })

    return rows


def analyzable_by_gene(X_raw, X_completed, thresholds, gene_names):
    """
    For each gene, count how many cells become analyzable.
    """
    rows = []
    n_cells = X_raw.shape[0]

    for j, gene in enumerate(gene_names):
        raw_gene_counts = X_raw[:, j]
        comp_gene_counts = X_completed[:, j]

        for t in thresholds:
            raw_n = int((raw_gene_counts >= t).sum())
            comp_n = int((comp_gene_counts >= t).sum())
            gain = comp_n - raw_n

            rows.append({
                "gene_id": str(gene),
                "threshold_min_count": int(t),
                "raw_analyzable_cells": raw_n,
                "completed_analyzable_cells": comp_n,
                "gain_cells": gain,
                "gain_fraction_of_all_cells": gain / n_cells,
                "raw_fraction_cells": raw_n / n_cells,
                "completed_fraction_cells": comp_n / n_cells,
            })

    return pd.DataFrame(rows)


def analyzable_by_celltype(X_raw, X_completed, thresholds, cell_type_labels):
    """
    For each cell type, count how many cell-gene pairs become analyzable.
    """
    rows = []

    ct_series = pd.Series(cell_type_labels.astype(str), index=np.arange(len(cell_type_labels)))
    unique_cts = sorted(ct_series.unique())

    for ct in unique_cts:
        mask = (ct_series.values == ct)
        n_cells_ct = int(mask.sum())

        Xr = X_raw[mask, :]
        Xc = X_completed[mask, :]

        total_pairs_ct = int(Xr.shape[0] * Xr.shape[1])

        for t in thresholds:
            raw_n = int((Xr >= t).sum())
            comp_n = int((Xc >= t).sum())
            gain = comp_n - raw_n

            rows.append({
                "cell_type": str(ct),
                "n_cells": n_cells_ct,
                "threshold_min_count": int(t),
                "raw_analyzable_cell_gene_pairs": raw_n,
                "completed_analyzable_cell_gene_pairs": comp_n,
                "gain_cell_gene_pairs": gain,
                "gain_fraction_of_celltype_pairs": gain / total_pairs_ct if total_pairs_ct > 0 else np.nan,
                "raw_fraction_celltype_pairs": raw_n / total_pairs_ct if total_pairs_ct > 0 else np.nan,
                "completed_fraction_celltype_pairs": comp_n / total_pairs_ct if total_pairs_ct > 0 else np.nan,
            })

    return pd.DataFrame(rows)


# ------------------------------------------------------------------------------
# 3. Prepare raw and learned 9E completed count matrices
# ------------------------------------------------------------------------------

print("\nBuilding raw and learned 9E completed count matrices...")

X_raw_counts_cell2 = ensure_dense(X_raw_counts).astype(np.float32)

# Use Cell 1's authoritative gene/cell order.
if X_raw_counts_cell2.shape != (len(cell_ids_step4), len(shared_genes)):
    raise ValueError(
        "X_raw_counts shape does not match authoritative Cell 1 cell/gene order:\n"
        f"  X_raw_counts shape: {X_raw_counts_cell2.shape}\n"
        f"  expected: {(len(cell_ids_step4), len(shared_genes))}"
    )

X_completed_9E, imputed_pair_counts_df = build_completed_count_matrix_from_imputed_cosmx(
    X_raw=X_raw_counts_cell2,
    imputed_df=mol_learned_9E,
)

print(f"X_raw_counts shape       : {X_raw_counts_cell2.shape}")
print(f"X_completed_9E shape     : {X_completed_9E.shape}")
print(f"Raw total counts         : {X_raw_counts_cell2.sum(dtype=np.float64):,.0f}")
print(f"Completed total counts   : {X_completed_9E.sum(dtype=np.float64):,.0f}")
print(f"Added counts             : {(X_completed_9E - X_raw_counts_cell2).sum(dtype=np.float64):,.0f}")
print(f"Unique imputed pairs     : {len(imputed_pair_counts_df):,}")

# Expected from your current CosMx run:
# Raw total counts       : 30,129,866
# Added counts           : 4,990,120
# Completed total counts : 35,119,986

# Backward-compatible variable for later cells if needed.
X_completed_9E_counts = X_completed_9E

# ------------------------------------------------------------------------------
# 4. Overall analyzable pair gain
# ------------------------------------------------------------------------------

thresholds = [1, 2, 3, 5, 8, 10]

print("\nComputing overall analyzable pair gain...")

summary_rows = []
summary_rows.extend(
    analyzable_summary(
        X_raw_counts_cell2,
        thresholds,
        "Raw observed counts",
    )
)

summary_rows.extend(
    analyzable_summary(
        X_completed_9E,
        thresholds,
        "Learned 9E completed counts",
    )
)

power_summary_long_df = pd.DataFrame(summary_rows)

gain_rows = []

for t in thresholds:
    raw_row = power_summary_long_df[
        (power_summary_long_df["dataset"] == "Raw observed counts")
        & (power_summary_long_df["threshold_min_count"] == t)
    ].iloc[0]

    comp_row = power_summary_long_df[
        (power_summary_long_df["dataset"] == "Learned 9E completed counts")
        & (power_summary_long_df["threshold_min_count"] == t)
    ].iloc[0]

    raw_n = int(raw_row["analyzable_cell_gene_pairs"])
    comp_n = int(comp_row["analyzable_cell_gene_pairs"])
    gain = comp_n - raw_n

    gain_rows.append({
        "threshold_min_count": int(t),
        "raw_analyzable_pairs": raw_n,
        "completed_analyzable_pairs": comp_n,
        "gain_pairs": gain,
        "relative_gain_percent": 100.0 * gain / max(raw_n, 1),
        "raw_fraction_all_pairs": float(raw_row["fraction_of_all_cell_gene_pairs"]),
        "completed_fraction_all_pairs": float(comp_row["fraction_of_all_cell_gene_pairs"]),
        "total_cell_gene_pairs": int(raw_row["total_cell_gene_pairs"]),
    })

power_gain_summary_df = pd.DataFrame(gain_rows)

print("\nOverall analyzable pair gain:")
display(power_gain_summary_df)

# ------------------------------------------------------------------------------
# 5. By-gene analyzable gain
# ------------------------------------------------------------------------------

print("\nComputing by-gene analyzable gain...")

power_by_gene_df = analyzable_by_gene(
    X_raw=X_raw_counts_cell2,
    X_completed=X_completed_9E,
    thresholds=thresholds,
    gene_names=shared_genes,
)

print("\nTop genes by gain at threshold >=5:")
display(
    power_by_gene_df[power_by_gene_df["threshold_min_count"] == 5]
    .sort_values("gain_cells", ascending=False)
    .head(20)
)

# ------------------------------------------------------------------------------
# 6. By-cell-type analyzable gain
# ------------------------------------------------------------------------------

print("\nComputing by-cell-type analyzable gain...")

cell_type_labels = denoised_adata.obs[ct_col].astype(str).values

if len(cell_type_labels) != X_raw_counts_cell2.shape[0]:
    raise ValueError(
        f"cell_type_labels length {len(cell_type_labels)} does not match "
        f"number of cells {X_raw_counts_cell2.shape[0]}"
    )

power_by_celltype_df = analyzable_by_celltype(
    X_raw=X_raw_counts_cell2,
    X_completed=X_completed_9E,
    thresholds=thresholds,
    cell_type_labels=cell_type_labels,
)

print("\nCell-type gains at threshold >=5:")
display(
    power_by_celltype_df[power_by_celltype_df["threshold_min_count"] == 5]
    .sort_values("gain_cell_gene_pairs", ascending=False)
)

# ------------------------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------------------------

power_gain_summary_df.to_csv(POWER_SUMMARY_PATH, index=False)
power_by_gene_df.to_csv(POWER_BY_GENE_PATH, index=False)
power_by_celltype_df.to_csv(POWER_BY_CELLTYPE_PATH, index=False)

print("\nSaved statistical power / analyzable-pair outputs:")
print(f"  Overall summary : {POWER_SUMMARY_PATH}")
print(f"  By gene         : {POWER_BY_GENE_PATH}")
print(f"  By cell type    : {POWER_BY_CELLTYPE_PATH}")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 2 COMPLETE — statistical power / analyzable pair gain")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 3 — Marker-based cell-type signal strengthening
#
# Biological question:
#   Do known marker genes become clearer in their expected cell types after
#   learned 9E imputation?
#
# Main comparison:
#   Raw observed counts vs Learned 9E completed counts
#
# Same algorithm as Xenium Cell 3:
#   1. Normalize raw and learned 9E completed count matrices
#   2. For each expected cell type and marker gene:
#        - compare marker expression in expected cell type vs background cells
#        - compute mean expression, detection rate, AUROC, Cohen's d
#   3. Compare raw vs completed values
#   4. Save marker gain table and summary
#
# CosMx-safe:
#   - Keeps cell_id as string, e.g. "1_1"
#   - Uses CosMx cell-type labels from denoised_adata.obs[ct_col]
#   - Uses shared_genes / gene_idx_map from Cell 1
#   - Uses X_completed_9E from Cell 2 if available, otherwise rebuilds it
#   - Saves outputs to DOWNSTREAM_DIR
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import roc_auc_score

print("=" * 100)
print("COSMX DOWNSTREAM CELL 3 — Marker-based cell-type signal strengthening")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables from Cell 1 / Cell 2
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "denoised_adata",
    "ct_col",
    "X_raw_counts",
    "shared_genes",
    "cell_ids_step4",
    "cell_idx_map",
    "gene_idx_map",
    "mol_learned_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first. "
        "If X_completed_9E is missing, this cell can rebuild it from mol_learned_9E."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

MARKER_SIGNAL_PATH = DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_signal_strengthening.csv"

MARKER_SIGNAL_SUMMARY_PATH = DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_signal_strengthening_summary.csv"

MARKER_FILTERED_SET_PATH = DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_sets_used.csv"

print("\nOutputs will be saved to:")
print(f"  Marker gain table : {MARKER_SIGNAL_PATH}")
print(f"  Summary table     : {MARKER_SIGNAL_SUMMARY_PATH}")
print(f"  Marker sets used  : {MARKER_FILTERED_SET_PATH}")

# ------------------------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def build_completed_count_matrix_from_imputed_cosmx(X_raw, imputed_df):
    """
    Builds:
        X_completed = X_raw + imputed molecule counts per (cell_id, gene_id)

    CosMx-safe:
      - cell_id remains string
      - gene_id remains string
      - mapping uses cell_idx_map and gene_idx_map from Cell 1
    """
    X_completed = np.asarray(X_raw, dtype=np.float32).copy()

    required_cols = ["cell_id", "gene_id"]
    missing_cols = [c for c in required_cols if c not in imputed_df.columns]
    if missing_cols:
        raise KeyError(f"mol_learned_9E missing required columns: {missing_cols}")

    counts = (
        imputed_df[["cell_id", "gene_id"]]
        .copy()
        .assign(
            cell_id=lambda d: d["cell_id"].astype(str),
            gene_id=lambda d: d["gene_id"].astype(str),
        )
        .groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name="n_imputed")
    )

    rr = counts["cell_id"].map(cell_idx_map)
    cc = counts["gene_id"].map(gene_idx_map)

    ok = rr.notna() & cc.notna()

    n_unmapped = len(counts) - int(ok.sum())
    if n_unmapped > 0:
        print(f"WARNING: dropped {n_unmapped:,} unmapped imputed cell-gene rows.")
        display(counts.loc[~ok].head())

    X_completed[
        rr[ok].astype(int).to_numpy(),
        cc[ok].astype(int).to_numpy(),
    ] += counts.loc[ok, "n_imputed"].to_numpy(dtype=np.float32)

    return X_completed


def log1p_normalize_counts(X):
    """
    Library-size normalize then log1p transform.
    This makes marker expression more comparable across cells.
    """
    X = np.asarray(X, dtype=np.float32)

    lib = X.sum(axis=1, keepdims=True)
    lib = np.maximum(lib, 1.0)

    X_norm = X / lib * 1e4
    X_log = np.log1p(X_norm)

    return X_log.astype(np.float32)


def safe_auroc(y_true, scores):
    """
    AUROC can fail if y_true has only one class.
    """
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, scores))
    except Exception:
        return np.nan


def cohens_d(x_pos, x_neg):
    """
    Cohen's d effect size.
    """
    x_pos = np.asarray(x_pos, dtype=np.float32)
    x_neg = np.asarray(x_neg, dtype=np.float32)

    n1 = len(x_pos)
    n0 = len(x_neg)

    if n1 < 2 or n0 < 2:
        return np.nan

    m1 = np.mean(x_pos)
    m0 = np.mean(x_neg)

    v1 = np.var(x_pos, ddof=1)
    v0 = np.var(x_neg, ddof=1)

    pooled = np.sqrt(((n1 - 1) * v1 + (n0 - 1) * v0) / max(n1 + n0 - 2, 1))

    if pooled <= 1e-8:
        return np.nan

    return float((m1 - m0) / pooled)


# ------------------------------------------------------------------------------
# 3. Prepare matrices and labels
# ------------------------------------------------------------------------------

print("\nPreparing raw and completed matrices...")

X_raw_counts_cell3 = ensure_dense(X_raw_counts).astype(np.float32)

# If Cell 2 was already run, it created X_completed_9E or X_completed_9E_counts.
if "X_completed_9E" in globals():
    X_completed_9E_cell3 = ensure_dense(X_completed_9E).astype(np.float32)
    print("Using existing X_completed_9E from previous cell.")
elif "X_completed_9E_counts" in globals():
    X_completed_9E_cell3 = ensure_dense(X_completed_9E_counts).astype(np.float32)
    print("Using existing X_completed_9E_counts from previous cell.")
else:
    print("X_completed_9E not found; rebuilding completed matrix from mol_learned_9E.")
    X_completed_9E_cell3 = build_completed_count_matrix_from_imputed_cosmx(
        X_raw=X_raw_counts_cell3,
        imputed_df=mol_learned_9E,
    )

if X_raw_counts_cell3.shape != X_completed_9E_cell3.shape:
    raise ValueError(
        f"Shape mismatch: raw {X_raw_counts_cell3.shape}, "
        f"completed {X_completed_9E_cell3.shape}"
    )

expected_shape = (len(cell_ids_step4), len(shared_genes))
if X_raw_counts_cell3.shape != expected_shape:
    raise ValueError(
        f"Matrix shape {X_raw_counts_cell3.shape} does not match expected "
        f"{expected_shape} from Cell 1."
    )

cell_type_labels = denoised_adata.obs[ct_col].astype(str).values
unique_cell_types = sorted(pd.Series(cell_type_labels).unique())

if len(cell_type_labels) != X_raw_counts_cell3.shape[0]:
    raise ValueError(
        f"cell_type_labels length {len(cell_type_labels)} does not match "
        f"number of cells {X_raw_counts_cell3.shape[0]}"
    )

print(f"Raw matrix shape        : {X_raw_counts_cell3.shape}")
print(f"Completed matrix shape  : {X_completed_9E_cell3.shape}")
print(f"Raw total counts        : {X_raw_counts_cell3.sum(dtype=np.float64):,.0f}")
print(f"Completed total counts  : {X_completed_9E_cell3.sum(dtype=np.float64):,.0f}")
print(f"Added counts            : {(X_completed_9E_cell3 - X_raw_counts_cell3).sum(dtype=np.float64):,.0f}")
print(f"Cell-type column        : {ct_col}")
print(f"Unique cell types       : {unique_cell_types}")

# Backward-compatible alias for later cells.
X_completed_9E = X_completed_9E_cell3

# ------------------------------------------------------------------------------
# 4. CosMx marker dictionary
# ------------------------------------------------------------------------------

# This dictionary is intentionally broad.
# The code automatically keeps only:
#   - cell types present in your CosMx data
#   - marker genes present in your 951-gene CosMx shared panel
#
# IMPORTANT:
#   These are broad marker sets for the CosMx cell types currently present:
#   Epithelial cells, T lymphocytes, B lymphocytes, Myeloid cells, NK cells,
#   Fibroblasts, Endothelial cells, MAST cells, Oligodendrocytes, Unlabeled.
#
#   "Unlabeled" is intentionally excluded because it is not a biological marker class.

marker_sets = {
    "Epithelial cells": [
        "EPCAM", "KRT8", "KRT18", "KRT19", "KRT7", "KRT5", "KRT13",
        "KRT14", "KRT17", "MUC1", "CEACAM5", "CEACAM6", "TACSTD2",
        "SFTPA1", "SFTPA2", "SFTPB", "SFTPC", "SCGB1A1", "SCGB3A1",
        "AGER", "FOXJ1", "MKI67", "TOP2A"
    ],

    "T lymphocytes": [
        "CD3D", "CD3E", "CD3G", "TRAC", "TRBC1", "TRBC2",
        "CD4", "CD8A", "CD8B", "IL7R", "CCR7", "TCF7",
        "LTB", "GZMB", "NKG7", "PRF1", "CXCR4"
    ],

    "B lymphocytes": [
        "MS4A1", "CD79A", "CD79B", "BANK1", "CD19", "CD22",
        "CD74", "HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DPB1",
        "IGHM", "IGKC", "JCHAIN", "TNFRSF17", "MZB1", "XBP1"
    ],

    "Myeloid cells": [
        "LYZ", "LST1", "TYROBP", "AIF1", "FCGR3A", "FCER1G",
        "CD68", "CD163", "MSR1", "MARCO", "C1QA", "C1QB", "C1QC",
        "S100A8", "S100A9", "IL1B", "CTSS", "HLA-DRA", "HLA-DPA1",
        "HLA-DPB1", "CSF1R", "ITGAM", "ITGAX"
    ],

    "NK cells": [
        "NKG7", "GNLY", "PRF1", "GZMB", "GZMA", "KLRD1",
        "KLRF1", "KLRB1", "NCAM1", "FCGR3A", "TYROBP"
    ],

    "Fibroblasts": [
        "COL1A1", "COL1A2", "COL3A1", "COL6A1", "COL6A2", "COL6A3",
        "DCN", "LUM", "DPT", "FBLN1", "FBLN2", "PDGFRA",
        "PDGFRB", "ACTA2", "TAGLN", "POSTN", "THY1", "CXCL12",
        "MMP2", "MMP11", "IGFBP7"
    ],

    "Endothelial cells": [
        "PECAM1", "VWF", "KDR", "FLT1", "CLDN5", "RAMP2",
        "ENG", "ESAM", "EMCN", "PLVAP", "CDH5", "CAV1",
        "ACKR1", "SELE", "ICAM1"
    ],

    "MAST cells": [
        "TPSAB1", "TPSB2", "CPA3", "KIT", "MS4A2", "HDC",
        "GATA2", "HPGDS", "FCER1A"
    ],

    "Oligodendrocytes": [
        "MBP", "MOG", "PLP1", "MAG", "MOBP", "OLIG1", "OLIG2",
        "SOX10", "CLDN11"
    ],
}

available_genes = set(str(g) for g in shared_genes)
available_cell_types = set(str(ct) for ct in unique_cell_types)

filtered_marker_sets = {}

for ct, genes in marker_sets.items():
    if ct not in available_cell_types:
        continue

    genes_present = [str(g) for g in genes if str(g) in available_genes]

    if len(genes_present) > 0:
        filtered_marker_sets[ct] = genes_present

print("\nFiltered marker sets present in this CosMx dataset:")
for ct, genes in filtered_marker_sets.items():
    print(f"  {ct:25s}: {genes}")

if len(filtered_marker_sets) == 0:
    raise RuntimeError(
        "No marker genes from marker_sets were found in this CosMx dataset. "
        "Check cell-type names and shared gene names."
    )

# Save the filtered marker set used for reproducibility.
marker_set_rows = []
for ct, genes in filtered_marker_sets.items():
    for g in genes:
        marker_set_rows.append({
            "cell_type": ct,
            "gene_id": g,
        })

marker_sets_used_df = pd.DataFrame(marker_set_rows)
marker_sets_used_df.to_csv(MARKER_FILTERED_SET_PATH, index=False)

print(f"\nSaved marker sets actually used:")
print(f"  {MARKER_FILTERED_SET_PATH}")

# ------------------------------------------------------------------------------
# 5. Marker signal metric function
# ------------------------------------------------------------------------------

def marker_signal_for_matrix(X_log, X_counts_for_detection, dataset_name):
    """
    Compute marker signal metrics for one matrix.

    X_log:
      normalized log expression matrix

    X_counts_for_detection:
      unnormalized count matrix used for detection rate
    """
    rows = []

    ct_labels = np.asarray(cell_type_labels).astype(str)

    for expected_ct, marker_genes in filtered_marker_sets.items():
        expected_mask = (ct_labels == expected_ct)
        background_mask = ~expected_mask

        n_expected = int(expected_mask.sum())
        n_background = int(background_mask.sum())

        if n_expected == 0 or n_background == 0:
            continue

        y_true = expected_mask.astype(int)

        for gene in marker_genes:
            if gene not in gene_idx_map:
                continue

            j = gene_idx_map[gene]

            expr = X_log[:, j]
            counts_gene = X_counts_for_detection[:, j]

            expected_expr = expr[expected_mask]
            background_expr = expr[background_mask]

            expected_counts = counts_gene[expected_mask]
            background_counts = counts_gene[background_mask]

            mean_expected = float(np.mean(expected_expr))
            mean_background = float(np.mean(background_expr))

            detection_expected = float((expected_counts > 0).mean())
            detection_background = float((background_counts > 0).mean())

            signal_diff = mean_expected - mean_background
            signal_ratio = (mean_expected + 1e-6) / (mean_background + 1e-6)

            log2_fc_like = float(np.log2((mean_expected + 1e-6) / (mean_background + 1e-6)))

            auc = safe_auroc(y_true, expr)
            d = cohens_d(expected_expr, background_expr)

            rows.append({
                "dataset": dataset_name,
                "expected_cell_type": expected_ct,
                "gene_id": gene,
                "n_expected_cells": n_expected,
                "n_background_cells": n_background,

                "mean_expr_expected": mean_expected,
                "mean_expr_background": mean_background,
                "signal_diff_expected_minus_background": signal_diff,
                "signal_ratio_expected_over_background": signal_ratio,
                "log2fc_like_expected_vs_background": log2_fc_like,

                "detection_rate_expected": detection_expected,
                "detection_rate_background": detection_background,
                "detection_diff_expected_minus_background": detection_expected - detection_background,

                "AUROC_expected_vs_background": auc,
                "cohens_d_expected_vs_background": d,
            })

    return pd.DataFrame(rows)

# ------------------------------------------------------------------------------
# 6. Normalize raw and completed matrices
# ------------------------------------------------------------------------------

print("\nNormalizing raw and completed count matrices...")

X_raw_log = log1p_normalize_counts(X_raw_counts_cell3)
X_completed_log = log1p_normalize_counts(X_completed_9E_cell3)

print(f"X_raw_log shape       : {X_raw_log.shape}")
print(f"X_completed_log shape : {X_completed_log.shape}")

# ------------------------------------------------------------------------------
# 7. Compute marker signal metrics
# ------------------------------------------------------------------------------

print("\nComputing marker signal metrics...")

raw_marker_df = marker_signal_for_matrix(
    X_log=X_raw_log,
    X_counts_for_detection=X_raw_counts_cell3,
    dataset_name="Raw observed counts",
)

completed_marker_df = marker_signal_for_matrix(
    X_log=X_completed_log,
    X_counts_for_detection=X_completed_9E_cell3,
    dataset_name="Learned 9E completed counts",
)

marker_signal_df = pd.concat([raw_marker_df, completed_marker_df], ignore_index=True)

print(f"Raw marker rows      : {len(raw_marker_df):,}")
print(f"Completed marker rows: {len(completed_marker_df):,}")

print("\nMarker signal preview:")
display(marker_signal_df.head(20))

if len(raw_marker_df) == 0 or len(completed_marker_df) == 0:
    raise RuntimeError(
        "Marker signal table is empty. Check filtered_marker_sets, cell-type labels, and gene names."
    )

# ------------------------------------------------------------------------------
# 8. Build raw-vs-completed gain table
# ------------------------------------------------------------------------------

print("\nComputing raw-vs-completed marker signal gains...")

merge_keys = ["expected_cell_type", "gene_id"]

raw_renamed = raw_marker_df.rename(columns={
    c: f"raw_{c}" for c in raw_marker_df.columns if c not in merge_keys
})

completed_renamed = completed_marker_df.rename(columns={
    c: f"completed_{c}" for c in completed_marker_df.columns if c not in merge_keys
})

marker_gain_df = raw_renamed.merge(
    completed_renamed,
    on=merge_keys,
    how="inner",
)

metric_pairs = [
    "mean_expr_expected",
    "mean_expr_background",
    "signal_diff_expected_minus_background",
    "signal_ratio_expected_over_background",
    "log2fc_like_expected_vs_background",
    "detection_rate_expected",
    "detection_rate_background",
    "detection_diff_expected_minus_background",
    "AUROC_expected_vs_background",
    "cohens_d_expected_vs_background",
]

for m in metric_pairs:
    raw_col = f"raw_{m}"
    comp_col = f"completed_{m}"

    if raw_col in marker_gain_df.columns and comp_col in marker_gain_df.columns:
        marker_gain_df[f"delta_{m}"] = marker_gain_df[comp_col] - marker_gain_df[raw_col]

marker_gain_df["improved_AUROC"] = marker_gain_df["delta_AUROC_expected_vs_background"] > 0
marker_gain_df["improved_signal_diff"] = marker_gain_df["delta_signal_diff_expected_minus_background"] > 0
marker_gain_df["improved_detection_diff"] = marker_gain_df["delta_detection_diff_expected_minus_background"] > 0

print("\nTop marker improvements by AUROC gain:")
display(
    marker_gain_df
    .sort_values("delta_AUROC_expected_vs_background", ascending=False)
    .head(20)
)

print("\nTop marker improvements by signal-difference gain:")
display(
    marker_gain_df
    .sort_values("delta_signal_diff_expected_minus_background", ascending=False)
    .head(20)
)

print("\nLargest marker decreases by AUROC:")
display(
    marker_gain_df
    .sort_values("delta_AUROC_expected_vs_background", ascending=True)
    .head(20)
)

# ------------------------------------------------------------------------------
# 9. Summary statistics
# ------------------------------------------------------------------------------

summary = {
    "n_marker_tests": int(len(marker_gain_df)),

    "mean_delta_AUROC": float(marker_gain_df["delta_AUROC_expected_vs_background"].mean()),
    "median_delta_AUROC": float(marker_gain_df["delta_AUROC_expected_vs_background"].median()),
    "fraction_markers_AUROC_improved": float(marker_gain_df["improved_AUROC"].mean()),

    "mean_delta_signal_diff": float(marker_gain_df["delta_signal_diff_expected_minus_background"].mean()),
    "median_delta_signal_diff": float(marker_gain_df["delta_signal_diff_expected_minus_background"].median()),
    "fraction_markers_signal_diff_improved": float(marker_gain_df["improved_signal_diff"].mean()),

    "mean_delta_detection_diff": float(marker_gain_df["delta_detection_diff_expected_minus_background"].mean()),
    "median_delta_detection_diff": float(marker_gain_df["delta_detection_diff_expected_minus_background"].median()),
    "fraction_markers_detection_diff_improved": float(marker_gain_df["improved_detection_diff"].mean()),
}

marker_signal_summary_df = pd.DataFrame([summary])

print("\nMarker signal strengthening summary:")
display(marker_signal_summary_df)

# ------------------------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------------------------

marker_gain_df.to_csv(MARKER_SIGNAL_PATH, index=False)
marker_signal_summary_df.to_csv(MARKER_SIGNAL_SUMMARY_PATH, index=False)

print("\nSaved marker-based signal strengthening outputs:")
print(f"  Marker gain table : {MARKER_SIGNAL_PATH}")
print(f"  Summary table     : {MARKER_SIGNAL_SUMMARY_PATH}")
print(f"  Marker sets used  : {MARKER_FILTERED_SET_PATH}")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 3 COMPLETE — marker-based cell-type signal strengthening")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 3B — Marker signal gain heatmaps
#
# Biological question:
#   Which marker genes/cell types show stronger biological signal after 9E
#   imputation compared with raw observed counts?
#
# Heatmaps:
#   1. AUROC gain:
#        AUROC_completed - AUROC_raw
#   2. Signal-difference gain:
#        signal_diff_completed - signal_diff_raw
#
# Rows    = marker genes
# Columns = expected cell types
# Color   = improvement after imputation
#
# CosMx-safe:
#   - Reads marker gain table from DOWNSTREAM_DIR
#   - Saves heatmaps to DOWNSTREAM_DIR
#   - Uses Cell 3 output names
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("COSMX DOWNSTREAM CELL 3B — Marker signal gain heatmaps")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables / paths
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s) from previous cells:\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 and Cell 3 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

MARKER_SIGNAL_PATH = DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_signal_strengthening.csv"

HEATMAP_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_marker_signal_heatmaps"
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

AUROC_HEATMAP_PNG = HEATMAP_DIR / f"{RUN_NAME}_marker_AUROC_gain_heatmap.png"

SIGNAL_DIFF_HEATMAP_PNG = HEATMAP_DIR / f"{RUN_NAME}_marker_signal_diff_gain_heatmap.png"

AUROC_MATRIX_CSV = HEATMAP_DIR / f"{RUN_NAME}_marker_AUROC_gain_matrix.csv"

SIGNAL_DIFF_MATRIX_CSV = HEATMAP_DIR / f"{RUN_NAME}_marker_signal_diff_gain_matrix.csv"

HEATMAP_SUMMARY_CSV = HEATMAP_DIR / f"{RUN_NAME}_marker_signal_heatmap_summary.csv"

TOP_AUROC_IMPROVED_PATH = HEATMAP_DIR / f"{RUN_NAME}_top_AUROC_marker_improvements.csv"

TOP_AUROC_WEAKENED_PATH = HEATMAP_DIR / f"{RUN_NAME}_top_AUROC_marker_weakened.csv"

print(f"DOWNSTREAM_DIR: {DOWNSTREAM_DIR}")
print(f"HEATMAP_DIR   : {HEATMAP_DIR}")

# ------------------------------------------------------------------------------
# 1. Load marker_gain_df if not already in memory
# ------------------------------------------------------------------------------

if "marker_gain_df" not in globals():
    if not MARKER_SIGNAL_PATH.exists():
        raise FileNotFoundError(
            f"marker_gain_df not in memory and marker signal CSV not found:\n"
            f"{MARKER_SIGNAL_PATH}\n\n"
            "Run COSMX DOWNSTREAM CELL 3 first."
        )

    print("Loading marker_gain_df from:")
    print(f"  {MARKER_SIGNAL_PATH}")

    marker_gain_df = pd.read_csv(MARKER_SIGNAL_PATH)

else:
    print("Using marker_gain_df from memory.")

print(f"marker_gain_df shape: {marker_gain_df.shape}")
print("Columns:")
print(list(marker_gain_df.columns))

required_cols = [
    "expected_cell_type",
    "gene_id",
    "delta_AUROC_expected_vs_background",
    "delta_signal_diff_expected_minus_background",
]

missing_cols = [c for c in required_cols if c not in marker_gain_df.columns]

if missing_cols:
    raise KeyError(f"marker_gain_df is missing required columns: {missing_cols}")

if len(marker_gain_df) == 0:
    raise RuntimeError("marker_gain_df is empty. Run Cell 3 again and check marker sets.")

# ------------------------------------------------------------------------------
# 2. Prepare heatmap matrices
# ------------------------------------------------------------------------------

df = marker_gain_df.copy()

df["expected_cell_type"] = df["expected_cell_type"].astype(str)
df["gene_id"] = df["gene_id"].astype(str)

# Force numeric values in case CSV reload stores them as object.
for col in [
    "delta_AUROC_expected_vs_background",
    "delta_signal_diff_expected_minus_background",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# If the same gene/cell-type pair appears more than once, average it.
auroc_matrix = (
    df
    .pivot_table(
        index="gene_id",
        columns="expected_cell_type",
        values="delta_AUROC_expected_vs_background",
        aggfunc="mean"
    )
)

signal_diff_matrix = (
    df
    .pivot_table(
        index="gene_id",
        columns="expected_cell_type",
        values="delta_signal_diff_expected_minus_background",
        aggfunc="mean"
    )
)

if auroc_matrix.empty:
    raise RuntimeError("AUROC heatmap matrix is empty.")
if signal_diff_matrix.empty:
    raise RuntimeError("Signal-difference heatmap matrix is empty.")

# Order genes by strongest absolute AUROC gain, so most informative genes appear first.
gene_order = (
    auroc_matrix
    .abs()
    .max(axis=1)
    .sort_values(ascending=False)
    .index
)

# Keep cell types in a biologically readable order when present.
preferred_celltype_order = [
    "Epithelial cells",
    "Fibroblasts",
    "Endothelial cells",
    "Myeloid cells",
    "T lymphocytes",
    "B lymphocytes",
    "NK cells",
    "MAST cells",
    "Oligodendrocytes",
    "Unlabeled",
]

available_cols = auroc_matrix.columns.tolist()
celltype_order = [ct for ct in preferred_celltype_order if ct in available_cols]
celltype_order += sorted([ct for ct in available_cols if ct not in celltype_order])

auroc_matrix = auroc_matrix.loc[gene_order, celltype_order]
signal_diff_matrix = signal_diff_matrix.reindex(index=gene_order, columns=celltype_order)

print("\nAUROC gain matrix shape:")
print(auroc_matrix.shape)

print("\nSignal-difference gain matrix shape:")
print(signal_diff_matrix.shape)

auroc_matrix.to_csv(AUROC_MATRIX_CSV)
signal_diff_matrix.to_csv(SIGNAL_DIFF_MATRIX_CSV)

print("\nSaved heatmap matrices:")
print(f"  AUROC gain matrix       : {AUROC_MATRIX_CSV}")
print(f"  Signal-diff gain matrix : {SIGNAL_DIFF_MATRIX_CSV}")

# ------------------------------------------------------------------------------
# 3. Heatmap plotting helper
# ------------------------------------------------------------------------------

def plot_gain_heatmap(matrix, title, colorbar_label, out_path, annotate=True):
    """
    Plot a diverging heatmap centered at 0.

    Positive value:
      marker signal improved after imputation.

    Negative value:
      marker signal weakened after imputation.
    """

    data = matrix.copy()

    n_rows, n_cols = data.shape

    if n_rows == 0 or n_cols == 0:
        raise ValueError("Cannot plot empty heatmap matrix.")

    # Adaptive figure size.
    fig_w = max(10, 0.85 * n_cols + 4)
    fig_h = max(8, 0.35 * n_rows + 3)

    arr = data.values.astype(float)

    finite_vals = arr[np.isfinite(arr)]

    if len(finite_vals) == 0:
        raise ValueError("No finite values available for heatmap.")

    # Symmetric color range around zero.
    vmax = np.nanpercentile(np.abs(finite_vals), 95)

    if vmax <= 0 or not np.isfinite(vmax):
        vmax = np.nanmax(np.abs(finite_vals))

    if vmax <= 0 or not np.isfinite(vmax):
        vmax = 1.0

    vmin = -vmax

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        arr,
        aspect="auto",
        cmap="coolwarm",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(title, fontsize=14, pad=16)
    ax.set_xlabel("Expected cell type", fontsize=12)
    ax.set_ylabel("Marker gene", fontsize=12)

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(data.columns, rotation=45, ha="right", fontsize=9)

    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(data.index, fontsize=8)

    # Grid lines.
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Optional numeric annotations.
    # For your current Cell 3 output, this should be manageable:
    # about 99 marker rows × 8 cell-type columns.
    if annotate and n_rows <= 120 and n_cols <= 20:
        for i in range(n_rows):
            for j in range(n_cols):
                val = arr[i, j]

                if not np.isfinite(val):
                    continue

                txt_color = "black" if abs(val) < 0.65 * vmax else "white"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=5.5,
                    color=txt_color,
                )

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label(colorbar_label, fontsize=11)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved heatmap: {out_path}")

# ------------------------------------------------------------------------------
# 4. Plot AUROC gain heatmap
# ------------------------------------------------------------------------------

plot_gain_heatmap(
    matrix=auroc_matrix,
    title="CosMx marker signal gain after 9E imputation: AUROC completed − AUROC raw",
    colorbar_label="ΔAUROC",
    out_path=AUROC_HEATMAP_PNG,
    annotate=True,
)

# ------------------------------------------------------------------------------
# 5. Plot signal-difference gain heatmap
# ------------------------------------------------------------------------------

plot_gain_heatmap(
    matrix=signal_diff_matrix,
    title="CosMx marker signal gain after 9E imputation: signal difference completed − raw",
    colorbar_label="Δ signal difference",
    out_path=SIGNAL_DIFF_HEATMAP_PNG,
    annotate=True,
)

# ------------------------------------------------------------------------------
# 6. Summary table: how many marker/cell-type pairs improved?
# ------------------------------------------------------------------------------

summary_rows = []

for metric_name, matrix in [
    ("delta_AUROC_expected_vs_background", auroc_matrix),
    ("delta_signal_diff_expected_minus_background", signal_diff_matrix),
]:
    vals = matrix.values.astype(float)
    finite_vals = vals[np.isfinite(vals)]

    n_total = len(finite_vals)
    n_improved = int((finite_vals > 0).sum())
    n_weakened = int((finite_vals < 0).sum())
    n_unchanged = int((finite_vals == 0).sum())

    summary_rows.append({
        "metric": metric_name,
        "n_marker_celltype_pairs": n_total,
        "n_improved_positive_gain": n_improved,
        "n_weakened_negative_gain": n_weakened,
        "n_unchanged_zero_gain": n_unchanged,
        "fraction_improved": n_improved / n_total if n_total > 0 else np.nan,
        "mean_gain": float(np.mean(finite_vals)) if n_total > 0 else np.nan,
        "median_gain": float(np.median(finite_vals)) if n_total > 0 else np.nan,
        "max_gain": float(np.max(finite_vals)) if n_total > 0 else np.nan,
        "min_gain": float(np.min(finite_vals)) if n_total > 0 else np.nan,
    })

heatmap_summary_df = pd.DataFrame(summary_rows)
heatmap_summary_df.to_csv(HEATMAP_SUMMARY_CSV, index=False)

print("\nMarker heatmap summary:")
display(heatmap_summary_df)

print("\nSaved heatmap summary:")
print(f"  {HEATMAP_SUMMARY_CSV}")

# ------------------------------------------------------------------------------
# 7. Top improved and weakened marker/cell-type pairs
# ------------------------------------------------------------------------------

top_auroc_improved = (
    df
    .sort_values("delta_AUROC_expected_vs_background", ascending=False)
    [[
        "expected_cell_type",
        "gene_id",
        "delta_AUROC_expected_vs_background",
        "raw_AUROC_expected_vs_background",
        "completed_AUROC_expected_vs_background",
    ]]
    .head(20)
)

top_auroc_weakened = (
    df
    .sort_values("delta_AUROC_expected_vs_background", ascending=True)
    [[
        "expected_cell_type",
        "gene_id",
        "delta_AUROC_expected_vs_background",
        "raw_AUROC_expected_vs_background",
        "completed_AUROC_expected_vs_background",
    ]]
    .head(20)
)

print("\nTop 20 marker/cell-type pairs improved by AUROC:")
display(top_auroc_improved)

print("\nTop 20 marker/cell-type pairs weakened by AUROC:")
display(top_auroc_weakened)

top_auroc_improved.to_csv(TOP_AUROC_IMPROVED_PATH, index=False)
top_auroc_weakened.to_csv(TOP_AUROC_WEAKENED_PATH, index=False)

print("\nSaved top improvement/weakened tables:")
print(f"  Improved: {TOP_AUROC_IMPROVED_PATH}")
print(f"  Weakened: {TOP_AUROC_WEAKENED_PATH}")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 3B COMPLETE — Marker signal gain heatmaps")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 4 — Marker-only cell-type classification
#
# Biological question:
#   Do known marker genes classify cell types better after learned 9E imputation?
#
# Main comparison:
#   Raw observed counts vs Learned 9E completed counts
#
# Optional comparison:
#   Step4 denoised counts, if X_denoised is available.
#
# Same algorithm as Xenium Cell 4:
#   1. Use only known marker genes present in the measured CosMx shared-gene panel.
#   2. Normalize counts using library-size normalization + log1p.
#   3. Train/test split cells using the same split for all datasets.
#   4. Train a simple logistic-regression classifier.
#   5. Compare accuracy, balanced accuracy, macro-F1, weighted-F1.
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR for outputs.
#   - Uses CosMx cell-type labels from denoised_adata.obs[ct_col].
#   - Uses CosMx marker sets matched to current broad cell-type labels.
#   - Uses shared_genes / gene_idx_map from Cell 1.
#   - Uses X_completed_9E from Cell 2/3 if available.
#   - Does not convert CosMx cell IDs to integer.
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("=" * 100)
print("COSMX DOWNSTREAM CELL 4 — Marker-only cell-type classification")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "denoised_adata",
    "ct_col",
    "shared_genes",
    "gene_idx_map",
    "X_raw_counts",
    "X_completed_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 and Cell 2/3 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

MARKER_CLASSIFICATION_SUMMARY_PATH = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_only_classification_summary.csv"
)

MARKER_CLASSIFICATION_REPORT_PATH = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_only_classification_report.txt"
)

MARKER_CLASSIFICATION_CONFUSION_PATH = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_only_classification_confusion_matrices.csv"
)

MARKER_GENE_LIST_PATH = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_only_genes_used.csv"
)

MARKER_CLASSIFICATION_PER_CLASS_PATH = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_only_classification_per_class_report.csv"
)

MARKER_CLASSIFICATION_IMPROVEMENT_PATH = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_countlevel_marker_only_classification_improvement.csv"
)

print("\nOutputs will be saved to:")
print(f"  Summary            : {MARKER_CLASSIFICATION_SUMMARY_PATH}")
print(f"  Text report        : {MARKER_CLASSIFICATION_REPORT_PATH}")
print(f"  Confusion matrices : {MARKER_CLASSIFICATION_CONFUSION_PATH}")
print(f"  Marker genes used  : {MARKER_GENE_LIST_PATH}")

# ------------------------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def normalize_log_marker_matrix(X, marker_indices):
    """
    Library-size normalize using the whole count matrix, then select marker genes.
    This avoids normalizing only by marker genes, which could distort cells where
    marker-gene counts are low.
    """
    X = ensure_dense(X).astype(np.float32)

    lib = X.sum(axis=1, keepdims=True)
    lib = np.maximum(lib, 1.0)

    X_norm = X / lib * 1e4
    X_log = np.log1p(X_norm)

    return X_log[:, marker_indices].astype(np.float32)


def evaluate_marker_classifier(dataset_name, X_counts, y_labels, train_idx, test_idx):
    """
    Train and evaluate marker-only logistic regression classifier.
    """

    print("\n" + "-" * 100)
    print(f"Evaluating marker-only classifier: {dataset_name}")
    print("-" * 100)

    X_marker = normalize_log_marker_matrix(X_counts, marker_gene_indices)

    X_train = X_marker[train_idx]
    X_test = X_marker[test_idx]

    y_train = y_labels[train_idx]
    y_test = y_labels[test_idx]

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs",
            multi_class="auto",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy          : {acc:.4f}")
    print(f"Balanced accuracy : {bal_acc:.4f}")
    print(f"Macro-F1          : {macro_f1:.4f}")
    print(f"Weighted-F1       : {weighted_f1:.4f}")

    report_dict = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0,
    )

    report_df = (
        pd.DataFrame(report_dict)
        .T
        .reset_index()
        .rename(columns={"index": "label"})
    )
    report_df.insert(0, "dataset", dataset_name)

    labels_sorted = sorted(pd.unique(y_labels).tolist())

    cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in labels_sorted],
        columns=[f"pred_{x}" for x in labels_sorted],
    )

    cm_long = (
        cm_df
        .reset_index()
        .melt(id_vars="index", var_name="predicted_label", value_name="n_cells")
        .rename(columns={"index": "true_label"})
    )
    cm_long.insert(0, "dataset", dataset_name)

    summary = {
        "dataset": dataset_name,
        "n_cells_total": int(len(y_labels)),
        "n_train_cells": int(len(train_idx)),
        "n_test_cells": int(len(test_idx)),
        "n_marker_genes": int(len(marker_genes_used)),
        "n_cell_types": int(len(np.unique(y_labels))),
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
    }

    del X_marker, X_train, X_test
    gc.collect()

    return summary, report_df, cm_long

# ------------------------------------------------------------------------------
# 3. Prepare matrices and labels
# ------------------------------------------------------------------------------

print("\nPreparing matrices and cell-type labels...")

X_raw_counts_cell4 = ensure_dense(X_raw_counts).astype(np.float32)
X_completed_9E_cell4 = ensure_dense(X_completed_9E).astype(np.float32)

if X_raw_counts_cell4.shape != X_completed_9E_cell4.shape:
    raise ValueError(
        f"Shape mismatch: raw {X_raw_counts_cell4.shape}, "
        f"completed {X_completed_9E_cell4.shape}"
    )

if "X_denoised" in globals():
    X_denoised_cell4 = ensure_dense(X_denoised).astype(np.float32)
    if X_denoised_cell4.shape != X_raw_counts_cell4.shape:
        raise ValueError(
            f"X_denoised shape {X_denoised_cell4.shape} does not match raw shape {X_raw_counts_cell4.shape}"
        )
else:
    X_denoised_cell4 = None

cell_type_labels = denoised_adata.obs[ct_col].astype(str).values
unique_cell_types = sorted(pd.Series(cell_type_labels).unique())

if len(cell_type_labels) != X_raw_counts_cell4.shape[0]:
    raise ValueError(
        f"cell_type_labels length {len(cell_type_labels)} does not match "
        f"number of cells {X_raw_counts_cell4.shape[0]}"
    )

print(f"Raw matrix shape       : {X_raw_counts_cell4.shape}")
print(f"Completed matrix shape : {X_completed_9E_cell4.shape}")
if X_denoised_cell4 is not None:
    print(f"Step4 matrix shape     : {X_denoised_cell4.shape}")
print(f"Cell-type column       : {ct_col}")
print(f"Unique cell types      : {unique_cell_types}")

# ------------------------------------------------------------------------------
# 4. CosMx marker dictionary
# ------------------------------------------------------------------------------

# Broad CosMx marker dictionary matched to current cell-type labels.
# The code automatically keeps only:
#   - marker genes present in shared_genes
#   - cell types present in denoised_adata.obs[ct_col]
#
# "Unlabeled" is intentionally excluded because it is not a marker-defined class.

marker_sets = {
    "Epithelial cells": [
        "EPCAM", "KRT8", "KRT18", "KRT19", "KRT7", "KRT5", "KRT13",
        "KRT14", "KRT17", "MUC1", "CEACAM5", "CEACAM6", "TACSTD2",
        "SFTPA1", "SFTPA2", "SFTPB", "SFTPC", "SCGB1A1", "SCGB3A1",
        "AGER", "FOXJ1", "MKI67", "TOP2A"
    ],

    "T lymphocytes": [
        "CD3D", "CD3E", "CD3G", "TRAC", "TRBC1", "TRBC2",
        "CD4", "CD8A", "CD8B", "IL7R", "CCR7", "TCF7",
        "LTB", "GZMB", "NKG7", "PRF1", "CXCR4"
    ],

    "B lymphocytes": [
        "MS4A1", "CD79A", "CD79B", "BANK1", "CD19", "CD22",
        "CD74", "HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DPB1",
        "IGHM", "IGKC", "JCHAIN", "TNFRSF17", "MZB1", "XBP1"
    ],

    "Myeloid cells": [
        "LYZ", "LST1", "TYROBP", "AIF1", "FCGR3A", "FCER1G",
        "CD68", "CD163", "MSR1", "MARCO", "C1QA", "C1QB", "C1QC",
        "S100A8", "S100A9", "IL1B", "CTSS", "HLA-DRA", "HLA-DPA1",
        "HLA-DPB1", "CSF1R", "ITGAM", "ITGAX"
    ],

    "NK cells": [
        "NKG7", "GNLY", "PRF1", "GZMB", "GZMA", "KLRD1",
        "KLRF1", "KLRB1", "NCAM1", "FCGR3A", "TYROBP"
    ],

    "Fibroblasts": [
        "COL1A1", "COL1A2", "COL3A1", "COL6A1", "COL6A2", "COL6A3",
        "DCN", "LUM", "DPT", "FBLN1", "FBLN2", "PDGFRA",
        "PDGFRB", "ACTA2", "TAGLN", "POSTN", "THY1", "CXCL12",
        "MMP2", "MMP11", "IGFBP7"
    ],

    "Endothelial cells": [
        "PECAM1", "VWF", "KDR", "FLT1", "CLDN5", "RAMP2",
        "ENG", "ESAM", "EMCN", "PLVAP", "CDH5", "CAV1",
        "ACKR1", "SELE", "ICAM1"
    ],

    "MAST cells": [
        "TPSAB1", "TPSB2", "CPA3", "KIT", "MS4A2", "HDC",
        "GATA2", "HPGDS", "FCER1A"
    ],

    "Oligodendrocytes": [
        "MBP", "MOG", "PLP1", "MAG", "MOBP", "OLIG1", "OLIG2",
        "SOX10", "CLDN11"
    ],
}

available_genes = set(str(g) for g in shared_genes)
available_cell_types = set(str(ct) for ct in unique_cell_types)

filtered_marker_sets = {}

for ct, genes in marker_sets.items():
    if ct not in available_cell_types:
        continue

    genes_present = [str(g) for g in genes if str(g) in available_genes]

    if len(genes_present) > 0:
        filtered_marker_sets[ct] = genes_present

marker_genes_used = sorted(set(g for genes in filtered_marker_sets.values() for g in genes))
marker_gene_indices = [gene_idx_map[g] for g in marker_genes_used]

print("\nMarker sets used:")
for ct, genes in filtered_marker_sets.items():
    print(f"  {ct:25s}: {genes}")

print(f"\nTotal unique marker genes used: {len(marker_genes_used)}")
print(marker_genes_used)

if len(marker_genes_used) < 5:
    raise RuntimeError(
        "Too few marker genes found in this CosMx panel for marker-only classification."
    )

marker_gene_df = pd.DataFrame({
    "gene_id": marker_genes_used,
    "gene_col": marker_gene_indices,
})

# Also record which cell type each marker was assigned to.
marker_set_rows = []
for ct, genes in filtered_marker_sets.items():
    for g in genes:
        marker_set_rows.append({
            "cell_type": ct,
            "gene_id": g,
            "gene_col": gene_idx_map[g],
        })

marker_set_df = pd.DataFrame(marker_set_rows)

marker_gene_df.to_csv(MARKER_GENE_LIST_PATH, index=False)
marker_set_df.to_csv(
    str(MARKER_GENE_LIST_PATH).replace("_genes_used.csv", "_marker_sets_used.csv"),
    index=False,
)

print("\nSaved marker gene list:")
print(f"  {MARKER_GENE_LIST_PATH}")

# ------------------------------------------------------------------------------
# 5. Train/test split
# ------------------------------------------------------------------------------

RANDOM_STATE = 42
TEST_SIZE = 0.25

y_labels = np.asarray(cell_type_labels).astype(str)

# Keep only cell types with enough cells for stratified train/test split.
cell_type_counts = pd.Series(y_labels).value_counts()
eligible_cell_types = cell_type_counts[cell_type_counts >= 10].index.tolist()

eligible_mask = np.isin(y_labels, eligible_cell_types)

eligible_indices = np.where(eligible_mask)[0]
eligible_labels = y_labels[eligible_indices]

print("\nCell type counts:")
display(
    cell_type_counts
    .reset_index()
    .rename(columns={"index": "cell_type", "count": "n_cells"})
)

print(f"\nEligible cells for classification: {len(eligible_indices):,} / {len(y_labels):,}")
print(f"Eligible cell types: {len(eligible_cell_types):,}")

train_local_idx, test_local_idx = train_test_split(
    np.arange(len(eligible_indices)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=eligible_labels,
)

train_idx = eligible_indices[train_local_idx]
test_idx = eligible_indices[test_local_idx]

print(f"Train cells: {len(train_idx):,}")
print(f"Test cells : {len(test_idx):,}")

# ------------------------------------------------------------------------------
# 6. Evaluate raw / Step4 / completed
# ------------------------------------------------------------------------------

classification_summaries = []
classification_reports = []
confusion_matrices = []

datasets_to_evaluate = {
    "Raw observed counts": X_raw_counts_cell4,
}

if X_denoised_cell4 is not None:
    datasets_to_evaluate["Step4 denoised counts"] = X_denoised_cell4

datasets_to_evaluate["Learned 9E completed counts"] = X_completed_9E_cell4

for dataset_name, X_counts in datasets_to_evaluate.items():
    summary, report_df, cm_long = evaluate_marker_classifier(
        dataset_name=dataset_name,
        X_counts=X_counts,
        y_labels=y_labels,
        train_idx=train_idx,
        test_idx=test_idx,
    )

    classification_summaries.append(summary)
    classification_reports.append(report_df)
    confusion_matrices.append(cm_long)

classification_summary_df = pd.DataFrame(classification_summaries)
classification_report_df = pd.concat(classification_reports, ignore_index=True)
classification_confusion_df = pd.concat(confusion_matrices, ignore_index=True)

dataset_order = [
    "Raw observed counts",
    "Step4 denoised counts",
    "Learned 9E completed counts",
]

classification_summary_df["dataset"] = pd.Categorical(
    classification_summary_df["dataset"],
    categories=dataset_order,
    ordered=True,
)

classification_summary_df = (
    classification_summary_df
    .sort_values("dataset")
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("MARKER-ONLY CLASSIFICATION SUMMARY")
print("=" * 100)
display(classification_summary_df)

# ------------------------------------------------------------------------------
# 7. Raw-vs-completed improvement table
# ------------------------------------------------------------------------------

raw_row = classification_summary_df[
    classification_summary_df["dataset"].astype(str) == "Raw observed counts"
]

completed_row = classification_summary_df[
    classification_summary_df["dataset"].astype(str) == "Learned 9E completed counts"
]

if len(raw_row) == 1 and len(completed_row) == 1:
    raw_row = raw_row.iloc[0]
    completed_row = completed_row.iloc[0]

    improvement_summary = {
        "comparison": "Learned 9E completed counts - Raw observed counts",
        "delta_accuracy": float(completed_row["accuracy"] - raw_row["accuracy"]),
        "delta_balanced_accuracy": float(completed_row["balanced_accuracy"] - raw_row["balanced_accuracy"]),
        "delta_macro_f1": float(completed_row["macro_f1"] - raw_row["macro_f1"]),
        "delta_weighted_f1": float(completed_row["weighted_f1"] - raw_row["weighted_f1"]),
        "relative_accuracy_gain_percent": float(
            100.0 * (completed_row["accuracy"] - raw_row["accuracy"]) / max(raw_row["accuracy"], 1e-8)
        ),
        "relative_balanced_accuracy_gain_percent": float(
            100.0 * (completed_row["balanced_accuracy"] - raw_row["balanced_accuracy"]) / max(raw_row["balanced_accuracy"], 1e-8)
        ),
        "relative_macro_f1_gain_percent": float(
            100.0 * (completed_row["macro_f1"] - raw_row["macro_f1"]) / max(raw_row["macro_f1"], 1e-8)
        ),
        "relative_weighted_f1_gain_percent": float(
            100.0 * (completed_row["weighted_f1"] - raw_row["weighted_f1"]) / max(raw_row["weighted_f1"], 1e-8)
        ),
    }

    marker_classification_improvement_df = pd.DataFrame([improvement_summary])

    print("\nRaw vs learned 9E completed improvement:")
    display(marker_classification_improvement_df)

else:
    marker_classification_improvement_df = pd.DataFrame()
    print("WARNING: Could not compute Raw-vs-Completed improvement table.")

# Optional Step4-vs-raw and learned-vs-Step4 comparisons.
extra_improvement_rows = []

def add_pairwise_improvement(label_a, label_b, comparison_name):
    a = classification_summary_df[
        classification_summary_df["dataset"].astype(str) == label_a
    ]
    b = classification_summary_df[
        classification_summary_df["dataset"].astype(str) == label_b
    ]

    if len(a) == 1 and len(b) == 1:
        a = a.iloc[0]
        b = b.iloc[0]

        extra_improvement_rows.append({
            "comparison": comparison_name,
            "delta_accuracy": float(b["accuracy"] - a["accuracy"]),
            "delta_balanced_accuracy": float(b["balanced_accuracy"] - a["balanced_accuracy"]),
            "delta_macro_f1": float(b["macro_f1"] - a["macro_f1"]),
            "delta_weighted_f1": float(b["weighted_f1"] - a["weighted_f1"]),
        })

add_pairwise_improvement(
    "Raw observed counts",
    "Step4 denoised counts",
    "Step4 denoised counts - Raw observed counts",
)

add_pairwise_improvement(
    "Step4 denoised counts",
    "Learned 9E completed counts",
    "Learned 9E completed counts - Step4 denoised counts",
)

pairwise_improvement_df = pd.DataFrame(extra_improvement_rows)

if len(pairwise_improvement_df) > 0:
    print("\nAdditional pairwise improvements:")
    display(pairwise_improvement_df)

# ------------------------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------------------------

classification_summary_df.to_csv(MARKER_CLASSIFICATION_SUMMARY_PATH, index=False)
classification_report_df.to_csv(MARKER_CLASSIFICATION_PER_CLASS_PATH, index=False)
classification_confusion_df.to_csv(MARKER_CLASSIFICATION_CONFUSION_PATH, index=False)

if len(marker_classification_improvement_df) > 0:
    marker_classification_improvement_df.to_csv(
        MARKER_CLASSIFICATION_IMPROVEMENT_PATH,
        index=False,
    )

if len(pairwise_improvement_df) > 0:
    pairwise_improvement_df.to_csv(
        str(MARKER_CLASSIFICATION_IMPROVEMENT_PATH).replace(
            "_improvement.csv",
            "_pairwise_improvements.csv",
        ),
        index=False,
    )

# Save human-readable report.
with open(MARKER_CLASSIFICATION_REPORT_PATH, "w") as f:
    f.write("COSMX MARKER-ONLY CELL-TYPE CLASSIFICATION REPORT\n")
    f.write("=" * 90 + "\n\n")

    f.write(f"RUN_NAME: {RUN_NAME}\n")
    f.write(f"RANDOM_STATE: {RANDOM_STATE}\n")
    f.write(f"TEST_SIZE: {TEST_SIZE}\n")
    f.write(f"Cell-type column: {ct_col}\n")
    f.write(f"n_marker_genes: {len(marker_genes_used)}\n")
    f.write(f"marker_genes_used: {marker_genes_used}\n\n")

    f.write("MARKER SETS USED\n")
    f.write("-" * 90 + "\n")
    for ct, genes in filtered_marker_sets.items():
        f.write(f"{ct}: {genes}\n")
    f.write("\n")

    f.write("CLASSIFICATION SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(classification_summary_df.to_string(index=False))
    f.write("\n\n")

    if len(marker_classification_improvement_df) > 0:
        f.write("RAW VS LEARNED 9E COMPLETED IMPROVEMENT\n")
        f.write("-" * 90 + "\n")
        f.write(marker_classification_improvement_df.to_string(index=False))
        f.write("\n\n")

    if len(pairwise_improvement_df) > 0:
        f.write("PAIRWISE IMPROVEMENTS\n")
        f.write("-" * 90 + "\n")
        f.write(pairwise_improvement_df.to_string(index=False))
        f.write("\n\n")

    f.write("INTERPRETATION GUIDE\n")
    f.write("-" * 90 + "\n")
    f.write("Good sign: Learned 9E completed counts improve balanced accuracy and macro-F1 over raw counts.\n")
    f.write("Balanced accuracy and macro-F1 are especially important because cell types are imbalanced.\n")
    f.write("This validates that known marker genes carry stronger cell-type signal after imputation.\n")

print("\nSaved marker-only classification outputs:")
print(f"  Summary             : {MARKER_CLASSIFICATION_SUMMARY_PATH}")
print(f"  Per-class report    : {MARKER_CLASSIFICATION_PER_CLASS_PATH}")
print(f"  Confusion matrices  : {MARKER_CLASSIFICATION_CONFUSION_PATH}")
if len(marker_classification_improvement_df) > 0:
    print(f"  Improvement table   : {MARKER_CLASSIFICATION_IMPROVEMENT_PATH}")
if len(pairwise_improvement_df) > 0:
    print(
        "  Pairwise improvements: "
        + str(MARKER_CLASSIFICATION_IMPROVEMENT_PATH).replace(
            "_improvement.csv",
            "_pairwise_improvements.csv",
        )
    )
print(f"  Marker genes used   : {MARKER_GENE_LIST_PATH}")
print(f"  Text report         : {MARKER_CLASSIFICATION_REPORT_PATH}")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 4 COMPLETE — marker-only cell-type classification")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 5 — PCA/UMAP visualization of Raw vs Step4 vs Learned 9E
#
# Biological/statistical question:
#   Does cell-type separation visually improve from raw counts to Step4 and 9E?
#
# Main comparison:
#   Raw observed counts
#   Step4 denoised counts
#   Learned 9E completed counts
#
# Outputs:
#   1. PCA scatter plots
#   2. UMAP scatter plots
#   3. Embedding CSVs for reproducibility
#
# Same configs as Xenium Cell 5:
#   SAMPLE_SIZE   = 50,000
#   RANDOM_STATE  = 42
#   N_PCS         = 30
#   N_NEIGHBORS   = 15
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR for outputs
#   - Uses denoised_adata.obs[ct_col] for cell-type labels
#   - Keeps CosMx cell IDs as strings, e.g. "1_1"
#   - Uses denoised_adata.var_names / obs_names order from Cell 1
#
# GPU not needed.
# ==============================================================================

import os
import gc
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

print("=" * 100)
print("COSMX DOWNSTREAM CELL 5 — PCA/UMAP visualization of Raw vs Step4 vs Learned 9E")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "denoised_adata",
    "ct_col",
    "cell_ids_step4",
    "shared_genes",
    "X_raw_counts",
    "X_denoised",
    "X_completed_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s) from previous cells:\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 and Cell 2 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Imports / install Scanpy if needed
# ------------------------------------------------------------------------------

try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print(f"Scanpy import failed: {e}")
    print("Installing scanpy/leiden dependencies...")
    !pip install -q scanpy leidenalg igraph anndata
    import scanpy as sc
    import anndata as ad

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

VIS_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_visualizations_pca_umap"
VIS_DIR.mkdir(parents=True, exist_ok=True)

PCA_FIG_PATH = VIS_DIR / f"{RUN_NAME}_raw_step4_9E_PCA_celltypes.png"
UMAP_FIG_PATH = VIS_DIR / f"{RUN_NAME}_raw_step4_9E_UMAP_celltypes.png"
EMBEDDINGS_CSV_PATH = VIS_DIR / f"{RUN_NAME}_raw_step4_9E_embeddings.csv"
SUMMARY_CSV_PATH = VIS_DIR / f"{RUN_NAME}_raw_step4_9E_visualization_summary.csv"
SAMPLE_IDX_PATH = VIS_DIR / f"{RUN_NAME}_raw_step4_9E_visualization_sample_idx.npy"

print(f"VIS_DIR: {VIS_DIR}")

# ------------------------------------------------------------------------------
# 3. Config
# ------------------------------------------------------------------------------

SAMPLE_SIZE = 50_000
RANDOM_STATE = 42
N_PCS = 30
N_NEIGHBORS = 15

print(f"SAMPLE_SIZE : {SAMPLE_SIZE:,}")
print(f"RANDOM_STATE: {RANDOM_STATE}")
print(f"N_PCS       : {N_PCS}")
print(f"N_NEIGHBORS : {N_NEIGHBORS}")

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def make_embedding_for_matrix(X_full, dataset_name, sample_idx):
    """
    Build AnnData for one matrix, normalize/log1p, compute PCA and UMAP.
    """

    print("\n" + "-" * 100)
    print(f"Computing PCA/UMAP for: {dataset_name}")
    print("-" * 100)

    X_full = ensure_dense(X_full).astype(np.float32)

    if X_full.shape != denoised_adata.shape:
        raise ValueError(
            f"{dataset_name} shape {X_full.shape} does not match denoised_adata shape {denoised_adata.shape}"
        )

    obs_sub = denoised_adata.obs.iloc[sample_idx].copy()
    var_sub = denoised_adata.var.copy()

    adata_tmp = ad.AnnData(
        X=X_full[sample_idx, :].copy(),
        obs=obs_sub,
        var=var_sub,
    )

    # Critical CosMx-safe fix:
    # use authoritative gene names directly from denoised_adata.
    adata_tmp.var_names = pd.Index(shared_genes).astype(str)
    adata_tmp.var_names_make_unique()

    n_pcs_use = min(N_PCS, adata_tmp.n_vars - 1)

    print(f"  AnnData shape        : {adata_tmp.shape}")
    print(f"  Total sampled counts : {adata_tmp.X.sum(dtype=np.float64):,.2f}")
    print(f"  n_pcs_use            : {n_pcs_use}")

    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    sc.pp.pca(
        adata_tmp,
        n_comps=n_pcs_use,
        random_state=RANDOM_STATE,
    )

    sc.pp.neighbors(
        adata_tmp,
        n_neighbors=N_NEIGHBORS,
        n_pcs=n_pcs_use,
    )

    sc.tl.umap(
        adata_tmp,
        random_state=RANDOM_STATE,
    )

    pca = adata_tmp.obsm["X_pca"][:, :2]
    umap = adata_tmp.obsm["X_umap"][:, :2]

    # Critical CosMx-safe fix:
    # keep cell IDs as strings; do NOT convert to int.
    labels = denoised_adata.obs[ct_col].astype(str).values[sample_idx]
    cids = np.asarray(cell_ids_step4).astype(str)[sample_idx]

    emb_df = pd.DataFrame({
        "dataset": dataset_name,
        "cell_id": cids,
        "cell_type": labels,
        "PCA1": pca[:, 0],
        "PCA2": pca[:, 1],
        "UMAP1": umap[:, 0],
        "UMAP2": umap[:, 1],
    })

    summary = {
        "dataset": dataset_name,
        "n_cells": int(adata_tmp.n_obs),
        "n_genes": int(adata_tmp.n_vars),
        "n_pcs": int(n_pcs_use),
        "n_neighbors": int(N_NEIGHBORS),
        "total_sampled_counts_before_normalization": float(
            X_full[sample_idx, :].sum(dtype=np.float64)
        ),
    }

    del adata_tmp
    gc.collect()

    return emb_df, summary


def plot_embedding_grid(embedding_df, x_col, y_col, out_path, title):
    """
    Plot Raw / Step4 / 9E side by side, colored by cell type.
    """

    datasets = [
        "Raw observed counts",
        "Step4 denoised counts",
        "Learned 9E completed counts",
    ]

    # Use stable readable CosMx cell-type order.
    preferred_celltype_order = [
        "Epithelial cells",
        "Fibroblasts",
        "Endothelial cells",
        "Myeloid cells",
        "T lymphocytes",
        "B lymphocytes",
        "NK cells",
        "MAST cells",
        "Oligodendrocytes",
        "Unlabeled",
    ]

    available_cell_types = embedding_df["cell_type"].astype(str).unique().tolist()
    cell_types = [ct for ct in preferred_celltype_order if ct in available_cell_types]
    cell_types += sorted([ct for ct in available_cell_types if ct not in cell_types])

    cmap = plt.get_cmap("tab20")
    color_map = {
        ct: cmap(i % 20)
        for i, ct in enumerate(cell_types)
    }

    fig, axes = plt.subplots(1, 3, figsize=(21, 6), constrained_layout=True)

    for ax, dataset in zip(axes, datasets):
        sub = embedding_df[embedding_df["dataset"] == dataset].copy()

        for ct in cell_types:
            s = sub[sub["cell_type"] == ct]
            if len(s) == 0:
                continue

            ax.scatter(
                s[x_col],
                s[y_col],
                s=2,
                alpha=0.45,
                label=ct,
                color=color_map[ct],
                linewidths=0,
            )

        ax.set_title(dataset, fontsize=13)
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_xticks([])
        ax.set_yticks([])

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        fontsize=8,
        markerscale=4,
        frameon=False,
    )

    fig.suptitle(title, fontsize=16)
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {out_path}")

# ------------------------------------------------------------------------------
# 5. Prepare and validate matrices
# ------------------------------------------------------------------------------

print("\nPreparing matrices...")

X_raw_counts_cell5 = ensure_dense(X_raw_counts).astype(np.float32)
X_denoised_cell5 = ensure_dense(X_denoised).astype(np.float32)
X_completed_9E_cell5 = ensure_dense(X_completed_9E).astype(np.float32)

expected_shape = denoised_adata.shape

for name, X in [
    ("Raw observed counts", X_raw_counts_cell5),
    ("Step4 denoised counts", X_denoised_cell5),
    ("Learned 9E completed counts", X_completed_9E_cell5),
]:
    if X.shape != expected_shape:
        raise ValueError(
            f"{name} matrix shape {X.shape} does not match expected shape {expected_shape}"
        )

print(f"Raw matrix shape       : {X_raw_counts_cell5.shape}")
print(f"Step4 matrix shape     : {X_denoised_cell5.shape}")
print(f"Completed matrix shape : {X_completed_9E_cell5.shape}")

print(f"Raw total counts       : {X_raw_counts_cell5.sum(dtype=np.float64):,.0f}")
print(f"Step4 total counts     : {X_denoised_cell5.sum(dtype=np.float64):,.2f}")
print(f"Completed total counts : {X_completed_9E_cell5.sum(dtype=np.float64):,.0f}")

# ------------------------------------------------------------------------------
# 6. Fixed sample
# ------------------------------------------------------------------------------

print("\nCreating fixed sampled cell set...")

n_cells = len(cell_ids_step4)
n_sample = min(SAMPLE_SIZE, n_cells)

rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(n_cells, size=n_sample, replace=False)

np.save(SAMPLE_IDX_PATH, sample_idx)

print(f"Sampled cells: {n_sample:,} / {n_cells:,}")
print(f"Saved visualization sample index: {SAMPLE_IDX_PATH}")

# ------------------------------------------------------------------------------
# 7. Compute embeddings
# ------------------------------------------------------------------------------

embedding_dfs = []
summary_rows = []

datasets = {
    "Raw observed counts": X_raw_counts_cell5,
    "Step4 denoised counts": X_denoised_cell5,
    "Learned 9E completed counts": X_completed_9E_cell5,
}

for dataset_name, X in datasets.items():
    emb_df, summary = make_embedding_for_matrix(
        X_full=X,
        dataset_name=dataset_name,
        sample_idx=sample_idx,
    )

    embedding_dfs.append(emb_df)
    summary_rows.append(summary)

embedding_df = pd.concat(embedding_dfs, ignore_index=True)
visual_summary_df = pd.DataFrame(summary_rows)

print("\nEmbedding dataframe preview:")
display(embedding_df.head())

print("\nVisualization summary:")
display(visual_summary_df)

# ------------------------------------------------------------------------------
# 8. Save embeddings
# ------------------------------------------------------------------------------

embedding_df.to_csv(EMBEDDINGS_CSV_PATH, index=False)
visual_summary_df.to_csv(SUMMARY_CSV_PATH, index=False)

print("\nSaved embeddings:")
print(f"  {EMBEDDINGS_CSV_PATH}")

print("Saved visualization summary:")
print(f"  {SUMMARY_CSV_PATH}")

# ------------------------------------------------------------------------------
# 9. Plot PCA and UMAP
# ------------------------------------------------------------------------------

plot_embedding_grid(
    embedding_df=embedding_df,
    x_col="PCA1",
    y_col="PCA2",
    out_path=PCA_FIG_PATH,
    title="CosMx PCA visualization: Raw vs Step4 vs Learned 9E",
)

plot_embedding_grid(
    embedding_df=embedding_df,
    x_col="UMAP1",
    y_col="UMAP2",
    out_path=UMAP_FIG_PATH,
    title="CosMx UMAP visualization: Raw vs Step4 vs Learned 9E",
)

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 5 COMPLETE — PCA/UMAP visualization")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 6 — Spatial map visualization for selected genes/cells
#
# Biological/visual question:
#   Where are observed and imputed molecules placed inside real cells?
#
# Main comparison:
#   Observed molecules vs Learned 9E imputed molecules vs 3 empirical baselines
#
# Outputs:
#   Per selected gene/cell figure panels showing molecule dots and approximate
#   CosMx cell/nucleus shape.
#
# Same algorithm/task as Xenium Cell 6:
#   1. Select informative gene-cell pairs with both observed and learned-imputed molecules.
#   2. For each selected pair, plot:
#        Observed
#        Learned 9E
#        Gene empirical
#        Cell-type gene empirical
#        Spatial-kNN empirical
#   3. Save one PNG/PDF figure per selected cell-gene pair.
#   4. Save selected-pair table, figure manifest, and per-panel summary.
#
# CosMx-safe:
#   - Keeps cell_id as string, e.g. "1_1"
#   - Uses DOWNSTREAM_DIR for outputs
#   - Uses mol_learned_9E / mol_gene_emp / mol_ct_gene_emp / mol_spatial_knn_emp
#   - Does not convert cell IDs to int
#   - Uses approximate cell/nucleus ellipses from CosMx centroid/width/height/area
#     when exact polygons are unavailable.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

print("=" * 100)
print("COSMX DOWNSTREAM CELL 6 — Spatial map visualization for selected genes/cells")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "IMPUTATION_DIR",
    "denoised_adata",
    "cell_idx_map",
    "mol_observed",
    "mol_learned_9E",
    "mol_gene_emp",
    "mol_ct_gene_emp",
    "mol_spatial_knn_emp",
    "shared_genes",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s) from previous cells:\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
IMPUTATION_DIR = Path(IMPUTATION_DIR)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

SPATIAL_VIS_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_spatial_molecule_maps"
SPATIAL_VIS_DIR.mkdir(parents=True, exist_ok=True)

SELECTION_CSV_PATH = SPATIAL_VIS_DIR / f"{RUN_NAME}_selected_gene_cell_pairs_for_spatial_maps.csv"

SUMMARY_CSV_PATH = SPATIAL_VIS_DIR / f"{RUN_NAME}_spatial_map_visualization_summary.csv"

FIGURE_MANIFEST_PATH = SPATIAL_VIS_DIR / f"{RUN_NAME}_spatial_map_figure_manifest.csv"

print(f"SPATIAL_VIS_DIR: {SPATIAL_VIS_DIR}")

# ------------------------------------------------------------------------------
# 2. Try loading exact polygons if available
# ------------------------------------------------------------------------------

CELL_POLYGONS_PATH = IMPUTATION_DIR / "cell_polygons.pkl"
NUC_POLYGONS_PATH = IMPUTATION_DIR / "nuc_polygons.pkl"

cell_polygons = {}
nuc_polygons = {}

if CELL_POLYGONS_PATH.exists() and CELL_POLYGONS_PATH.stat().st_size > 10:
    try:
        print(f"Loading cell polygons: {CELL_POLYGONS_PATH}")
        with open(CELL_POLYGONS_PATH, "rb") as f:
            cell_polygons = pickle.load(f)
    except Exception as e:
        print(f"Could not load cell polygons: {e}")
        cell_polygons = {}
else:
    print("Exact cell polygons are missing/empty. Will use approximate CosMx cell ellipses.")

if NUC_POLYGONS_PATH.exists() and NUC_POLYGONS_PATH.stat().st_size > 10:
    try:
        print(f"Loading nucleus polygons: {NUC_POLYGONS_PATH}")
        with open(NUC_POLYGONS_PATH, "rb") as f:
            nuc_polygons = pickle.load(f)
    except Exception as e:
        print(f"Could not load nucleus polygons: {e}")
        nuc_polygons = {}
else:
    print("Exact nucleus polygons are missing/empty. Will use approximate nucleus ellipses.")

print(f"cell_polygons entries: {len(cell_polygons):,}")
print(f"nuc_polygons entries : {len(nuc_polygons):,}")

print(
    "\nCosMx geometry note:\n"
    "  Exact Xenium-style polygons are not available in this Step5 setup.\n"
    "  Figures will show molecule dots with approximate cell/nucleus ellipses\n"
    "  based on CosMx centroid, width/height, and nuclear-area metadata."
)

# ------------------------------------------------------------------------------
# 3. Configuration: selected genes
# ------------------------------------------------------------------------------

candidate_genes = [
    # Epithelial / tumor-like
    "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "KRT17", "KRT13",
    "CEACAM6", "TACSTD2", "MKI67", "TOP2A",

    # Stromal / ECM
    "LUM", "DCN", "COL1A1", "COL1A2", "COL6A1", "COL6A2",
    "IGFBP7", "MMP2", "TAGLN", "ACTA2",

    # Immune / myeloid / lymphoid
    "CD3D", "CD3E", "CD79A", "MS4A1", "CD74",
    "LYZ", "TYROBP", "FCER1G", "MARCO", "S100A8", "S100A9",
    "NKG7", "GZMB", "PRF1", "GNLY",

    # Endothelial / vascular
    "PECAM1", "VWF", "RAMP2", "CAV1", "CDH5", "ENG",

    # Mast
    "TPSAB1", "TPSB2", "CPA3", "HPGDS",
]

available_genes = set(str(g) for g in shared_genes)
selected_genes = [g for g in candidate_genes if g in available_genes]

print(f"\nSelected candidate genes present in CosMx panel: {selected_genes}")

if len(selected_genes) == 0:
    raise RuntimeError("None of the candidate genes are present in shared_genes.")

MAX_GENE_CELL_PAIRS = 12

# Same selection thresholds as original Cell 6.
MIN_OBSERVED_FOR_PAIR = 2
MIN_IMPUTED_FOR_PAIR = 2

# Same plotting controls as original Cell 6.
MAX_POINTS_PER_PANEL = 300
POINT_SIZE = 16
ALPHA_POINTS = 0.75

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def safe_cell_id(x):
    """
    CosMx-safe cell ID conversion.
    """
    return str(x)


def safe_gene_id(x):
    return str(x)


def safe_filename_token(x):
    """
    Make a cell/gene ID safe for filename use.
    """
    s = str(x)
    for old, new in [
        ("/", "_"),
        ("\\", "_"),
        (" ", "_"),
        ("+", "plus"),
        ("|", "_"),
        (":", "_"),
    ]:
        s = s.replace(old, new)
    return s


def _plot_polygon_boundary(ax, geom, linestyle="-", linewidth=1.0, alpha=0.9):
    """
    Plot shapely Polygon or MultiPolygon boundary if available.
    """
    if geom is None:
        return

    try:
        if geom.geom_type == "Polygon":
            x, y = geom.exterior.xy
            ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)

        elif geom.geom_type == "MultiPolygon":
            for poly in geom.geoms:
                x, y = poly.exterior.xy
                ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)

    except Exception:
        pass


def get_cell_obs_row(cid):
    """
    Return denoised_adata.obs row for a CosMx cell ID.
    """
    cid = safe_cell_id(cid)

    if cid not in cell_idx_map:
        return None

    idx = cell_idx_map[cid]
    return denoised_adata.obs.iloc[idx]


def get_cell_center_width_height(cid):
    """
    Get approximate CosMx cell center and size from obs metadata.
    Returns:
      center_x, center_y, width, height
    in global pixel coordinates.
    """
    row = get_cell_obs_row(cid)
    if row is None:
        return None

    # Center columns, in priority order.
    x_candidates = [
        "center_x_global_px",
        "CenterX_global_px",
        "x_centroid",
        "center_x_um",
    ]

    y_candidates = [
        "center_y_global_px",
        "CenterY_global_px",
        "y_centroid",
        "center_y_um",
    ]

    cx = None
    cy = None

    for c in x_candidates:
        if c in row.index and pd.notna(row[c]):
            cx = float(row[c])
            break

    for c in y_candidates:
        if c in row.index and pd.notna(row[c]):
            cy = float(row[c])
            break

    if cx is None or cy is None:
        return None

    # Width / height columns, in priority order.
    w_candidates = ["width_px", "Width"]
    h_candidates = ["height_px", "Height"]

    width = None
    height = None

    for c in w_candidates:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            width = float(row[c])
            break

    for c in h_candidates:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            height = float(row[c])
            break

    # Fallback from area if width/height not available.
    if width is None or height is None:
        area_candidates = ["cell_area_px", "label_cell_area_px", "Area"]
        area = None

        for c in area_candidates:
            if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
                area = float(row[c])
                break

        if area is not None:
            radius = np.sqrt(area / np.pi)
            width = 2.0 * radius
            height = 2.0 * radius
        else:
            width = 20.0
            height = 20.0

    return cx, cy, width, height


def get_approx_nucleus_width_height(cid, cell_width, cell_height):
    """
    Approximate nucleus ellipse from nuclear area if available.
    """
    row = get_cell_obs_row(cid)

    if row is None:
        return 0.55 * cell_width, 0.55 * cell_height

    nuc_area = None
    for c in ["label_nuclear_area_px", "nuc_area_px", "nuclear_area_px"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            nuc_area = float(row[c])
            break

    cell_area = None
    for c in ["label_cell_area_px", "cell_area_px", "Area"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            cell_area = float(row[c])
            break

    if nuc_area is not None and cell_area is not None and cell_area > 0:
        scale = np.sqrt(np.clip(nuc_area / cell_area, 0.05, 0.95))
        return cell_width * scale, cell_height * scale

    return 0.55 * cell_width, 0.55 * cell_height


def plot_approx_cell_and_nucleus(ax, cid):
    """
    Draw approximate CosMx cell and nucleus ellipses.
    """
    geom = get_cell_center_width_height(cid)

    if geom is None:
        return None

    cx, cy, width, height = geom

    cell_ellipse = Ellipse(
        xy=(cx, cy),
        width=width,
        height=height,
        angle=0,
        fill=False,
        linestyle="-",
        linewidth=1.3,
        alpha=0.9,
        edgecolor="black",
    )

    nw, nh = get_approx_nucleus_width_height(cid, width, height)

    nucleus_ellipse = Ellipse(
        xy=(cx, cy),
        width=nw,
        height=nh,
        angle=0,
        fill=False,
        linestyle="--",
        linewidth=1.1,
        alpha=0.9,
        edgecolor="gray",
    )

    ax.add_patch(cell_ellipse)
    ax.add_patch(nucleus_ellipse)

    return cx, cy, width, height


def get_geom_bounds(cid, fallback_df=None, padding=5.0):
    """
    Get plot bounds from:
      1. Exact cell polygon if available.
      2. Approximate CosMx cell center/width/height.
      3. Molecule coordinates fallback.
    """
    cid = safe_cell_id(cid)

    if cid in cell_polygons:
        try:
            minx, miny, maxx, maxy = cell_polygons[cid].bounds
            return minx - padding, maxx + padding, miny - padding, maxy + padding
        except Exception:
            pass

    # Sometimes polygon keys may be non-string.
    for key in [cid, str(cid)]:
        if key in cell_polygons:
            try:
                minx, miny, maxx, maxy = cell_polygons[key].bounds
                return minx - padding, maxx + padding, miny - padding, maxy + padding
            except Exception:
                pass

    geom = get_cell_center_width_height(cid)
    if geom is not None:
        cx, cy, width, height = geom
        half_w = max(width / 2.0, 5.0)
        half_h = max(height / 2.0, 5.0)
        return (
            cx - half_w - padding,
            cx + half_w + padding,
            cy - half_h - padding,
            cy + half_h + padding,
        )

    if fallback_df is not None and len(fallback_df) > 0:
        x = pd.to_numeric(fallback_df["x"], errors="coerce")
        y = pd.to_numeric(fallback_df["y"], errors="coerce")

        if x.notna().sum() > 0 and y.notna().sum() > 0:
            return (
                float(x.min()) - padding,
                float(x.max()) + padding,
                float(y.min()) - padding,
                float(y.max()) + padding,
            )

    return None


def get_pair_counts(df, label):
    """
    Count molecules per (cell_id, gene_id).
    CosMx-safe: cell_id stays string.
    """
    required_cols = ["cell_id", "gene_id"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in molecule table: {missing}")

    d = df[["cell_id", "gene_id"]].copy()
    d["cell_id"] = d["cell_id"].astype(str)
    d["gene_id"] = d["gene_id"].astype(str)

    out = (
        d.groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name=label)
    )

    return out


def select_gene_cell_pairs():
    """
    Select gene/cell pairs that have both observed and learned-imputed molecules.
    These pairs make the clearest visualization panels.
    """
    print("\nSelecting gene/cell pairs for spatial visualization...")

    observed_sub = mol_observed[
        mol_observed["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    learned_sub = mol_learned_9E[
        mol_learned_9E["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    observed_counts = get_pair_counts(observed_sub, "n_observed")
    learned_counts = get_pair_counts(learned_sub, "n_learned_imputed")

    merged = observed_counts.merge(
        learned_counts,
        on=["cell_id", "gene_id"],
        how="inner",
    )

    merged = merged[
        (merged["n_observed"] >= MIN_OBSERVED_FOR_PAIR)
        & (merged["n_learned_imputed"] >= MIN_IMPUTED_FOR_PAIR)
    ].copy()

    if len(merged) == 0:
        print("No pairs found with strict thresholds.")
        print("Relaxing thresholds to at least one observed and one learned-imputed molecule.")

        merged = observed_counts.merge(
            learned_counts,
            on=["cell_id", "gene_id"],
            how="inner",
        )

        merged = merged[
            (merged["n_observed"] >= 1)
            & (merged["n_learned_imputed"] >= 1)
        ].copy()

    if len(merged) == 0:
        raise RuntimeError(
            "Could not find any gene/cell pair with both observed and learned-imputed molecules."
        )

    merged["total_pair_molecules"] = merged["n_observed"] + merged["n_learned_imputed"]

    selected_rows = []

    # Pick one strong example per gene when possible.
    for gene in selected_genes:
        sub = merged[merged["gene_id"] == gene].copy()

        if len(sub) == 0:
            continue

        sub = sub.sort_values(
            ["n_learned_imputed", "n_observed", "total_pair_molecules"],
            ascending=False,
        )

        selected_rows.append(sub.iloc[0])

        if len(selected_rows) >= MAX_GENE_CELL_PAIRS:
            break

    selected = pd.DataFrame(selected_rows)

    if len(selected) < min(MAX_GENE_CELL_PAIRS, len(merged)):
        remaining = merged.merge(
            selected[["cell_id", "gene_id"]],
            on=["cell_id", "gene_id"],
            how="left",
            indicator=True,
        )

        remaining = remaining[remaining["_merge"] == "left_only"].drop(columns=["_merge"])
        remaining = remaining.sort_values(
            ["n_learned_imputed", "n_observed", "total_pair_molecules"],
            ascending=False,
        )

        needed = MAX_GENE_CELL_PAIRS - len(selected)
        selected = pd.concat([selected, remaining.head(needed)], ignore_index=True)

    selected = selected.head(MAX_GENE_CELL_PAIRS).reset_index(drop=True)

    return selected


def get_pair_molecules(df, cid, gid):
    """
    Get molecules for one cell-gene pair.
    CosMx-safe: cell_id is string.
    """
    cid = safe_cell_id(cid)
    gid = safe_gene_id(gid)

    sub = df[
        (df["cell_id"].astype(str) == cid)
        & (df["gene_id"].astype(str) == gid)
    ].copy()

    return sub


def downsample_points(df, max_points=300, seed=42):
    """
    Downsample points for clean visualization.
    """
    if len(df) <= max_points:
        return df

    return df.sample(n=max_points, random_state=seed).copy()


def plot_one_pair(cid, gid, out_prefix):
    """
    Make one multi-panel plot for a selected cell-gene pair.
    """
    cid = safe_cell_id(cid)
    gid = safe_gene_id(gid)

    source_tables = {
        "Observed": mol_observed,
        "Learned 9E": mol_learned_9E,
        "Gene empirical": mol_gene_emp,
        "Cell-type gene empirical": mol_ct_gene_emp,
        "Spatial-kNN empirical": mol_spatial_knn_emp,
    }

    pair_data = {}

    for label, df in source_tables.items():
        sub = get_pair_molecules(df, cid, gid)
        pair_data[label] = sub

    combined_for_bounds = (
        pd.concat([d for d in pair_data.values() if len(d) > 0], ignore_index=True)
        if any(len(d) > 0 for d in pair_data.values())
        else pd.DataFrame()
    )

    bounds = get_geom_bounds(cid, fallback_df=combined_for_bounds, padding=5.0)

    n_panels = len(source_tables)

    fig, axes = plt.subplots(
        1,
        n_panels,
        figsize=(4.2 * n_panels, 4.4),
        sharex=True,
        sharey=True,
    )

    if n_panels == 1:
        axes = [axes]

    for ax, (label, sub) in zip(axes, pair_data.items()):

        # Plot exact polygons if they exist. Otherwise plot approximate CosMx geometry.
        polygon_plotted = False

        if cid in cell_polygons:
            _plot_polygon_boundary(
                ax,
                cell_polygons[cid],
                linestyle="-",
                linewidth=1.3,
                alpha=0.9,
            )
            polygon_plotted = True

        if cid in nuc_polygons:
            _plot_polygon_boundary(
                ax,
                nuc_polygons[cid],
                linestyle="--",
                linewidth=1.1,
                alpha=0.9,
            )

        if not polygon_plotted:
            plot_approx_cell_and_nucleus(ax, cid)

        sub_plot = downsample_points(
            sub,
            max_points=MAX_POINTS_PER_PANEL,
            seed=42,
        )

        if len(sub_plot) > 0:
            ax.scatter(
                pd.to_numeric(sub_plot["x"], errors="coerce"),
                pd.to_numeric(sub_plot["y"], errors="coerce"),
                s=POINT_SIZE,
                alpha=ALPHA_POINTS,
                marker="o",
                label=label,
            )

        ax.set_title(
            f"{label}\n n={len(sub):,}",
            fontsize=10,
        )

        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, linewidth=0.3, alpha=0.4)

        if bounds is not None:
            xmin, xmax, ymin, ymax = bounds
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)

        ax.set_xlabel("x global px")
        ax.set_ylabel("y global px")

    fig.suptitle(
        f"Cell {cid} — Gene {gid}\nObserved vs learned 9E vs empirical baseline imputed molecule locations",
        fontsize=13,
        y=1.04,
    )

    plt.tight_layout()

    png_path = f"{out_prefix}.png"
    pdf_path = f"{out_prefix}.pdf"

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    summary_rows = []

    for label, sub in pair_data.items():
        if len(sub) == 0:
            summary_rows.append({
                "cell_id": cid,
                "gene_id": gid,
                "source": label,
                "n_molecules": 0,
                "mean_r_norm": np.nan,
                "mean_z_rel": np.nan,
                "mean_p_nuclear": np.nan,
                "mean_x": np.nan,
                "mean_y": np.nan,
            })
        else:
            summary_rows.append({
                "cell_id": cid,
                "gene_id": gid,
                "source": label,
                "n_molecules": int(len(sub)),
                "mean_r_norm": float(pd.to_numeric(sub["r_norm"], errors="coerce").mean()) if "r_norm" in sub.columns else np.nan,
                "mean_z_rel": float(pd.to_numeric(sub["z_rel"], errors="coerce").mean()) if "z_rel" in sub.columns else np.nan,
                "mean_p_nuclear": float(pd.to_numeric(sub["p_nuclear"], errors="coerce").mean()) if "p_nuclear" in sub.columns else np.nan,
                "mean_x": float(pd.to_numeric(sub["x"], errors="coerce").mean()),
                "mean_y": float(pd.to_numeric(sub["y"], errors="coerce").mean()),
            })

    return pd.DataFrame(summary_rows), png_path, pdf_path

# ------------------------------------------------------------------------------
# 5. Select examples
# ------------------------------------------------------------------------------

selected_pairs_df = select_gene_cell_pairs()

print("\nSelected gene/cell pairs:")
display(selected_pairs_df)

selected_pairs_df.to_csv(SELECTION_CSV_PATH, index=False)

print("Saved selected pairs:")
print(f"  {SELECTION_CSV_PATH}")

# ------------------------------------------------------------------------------
# 6. Generate spatial maps
# ------------------------------------------------------------------------------

print("\nGenerating spatial molecule maps...")

all_summary_rows = []
figure_records = []

for i, row in selected_pairs_df.iterrows():
    cid = safe_cell_id(row["cell_id"])
    gid = safe_gene_id(row["gene_id"])

    safe_cid = safe_filename_token(cid)
    safe_gid = safe_filename_token(gid)

    out_prefix = SPATIAL_VIS_DIR / f"{RUN_NAME}_cell_{safe_cid}_gene_{safe_gid}"

    print(f"\n[{i+1}/{len(selected_pairs_df)}] Plotting cell_id={cid}, gene_id={gid}")

    summary_df, png_path, pdf_path = plot_one_pair(
        cid=cid,
        gid=gid,
        out_prefix=str(out_prefix),
    )

    all_summary_rows.append(summary_df)

    figure_records.append({
        "cell_id": cid,
        "gene_id": gid,
        "png_path": png_path,
        "pdf_path": pdf_path,
        "n_observed_selected": int(row.get("n_observed", -1)),
        "n_learned_imputed_selected": int(row.get("n_learned_imputed", -1)),
    })

# ------------------------------------------------------------------------------
# 7. Save summary tables
# ------------------------------------------------------------------------------

spatial_summary_df = pd.concat(all_summary_rows, ignore_index=True)
figure_manifest_df = pd.DataFrame(figure_records)

spatial_summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
figure_manifest_df.to_csv(FIGURE_MANIFEST_PATH, index=False)

print("\nSpatial map summary:")
display(spatial_summary_df.head(30))

print("\nFigure manifest:")
display(figure_manifest_df)

print("\nSaved spatial visualization outputs:")
print(f"  Selected pairs CSV : {SELECTION_CSV_PATH}")
print(f"  Summary CSV        : {SUMMARY_CSV_PATH}")
print(f"  Figure manifest    : {FIGURE_MANIFEST_PATH}")
print(f"  Figure directory   : {SPATIAL_VIS_DIR}")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 6 COMPLETE — Spatial map visualization")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 6B — Raw vs completed molecule dot maps
#
# Figure layout:
#   Panel 1: Raw observed
#   Panel 2: Raw + learned 9E
#   Panel 3: Raw + gene empirical
#   Panel 4: Raw + cell-type gene empirical
#   Panel 5: Raw + spatial-kNN empirical
#
# Biological/visual question:
#   How did the molecule map look before imputation, and how does it look after
#   learned 9E / empirical baseline imputation?
#
# Same algorithm/configs as Xenium Cell 6B:
#   MAX_GENE_CELL_PAIRS = 12
#   MIN_OBSERVED_FOR_PAIR = 1
#   MIN_LEARNED_IMPUTED_FOR_PAIR = 2
#   MAX_RAW_POINTS_PER_PANEL = 400
#   MAX_IMPUTED_POINTS_PER_PANEL = 400
#   RAW_POINT_SIZE = 18
#   IMPUTED_POINT_SIZE = 22
#   RAW_ALPHA = 0.75
#   IMPUTED_ALPHA = 0.85
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Keeps cell_id as string, e.g. "1_1"
#   - Uses mol_learned_9E / mol_gene_emp / mol_ct_gene_emp / mol_spatial_knn_emp
#   - Uses approximate CosMx cell/nucleus ellipses if exact polygons are unavailable
#
# GPU not needed.
# ==============================================================================

import os
import gc
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

print("=" * 100)
print("COSMX DOWNSTREAM CELL 6B — Raw vs completed molecule dot maps")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "IMPUTATION_DIR",
    "denoised_adata",
    "cell_idx_map",
    "mol_observed",
    "mol_learned_9E",
    "mol_gene_emp",
    "mol_ct_gene_emp",
    "mol_spatial_knn_emp",
    "shared_genes",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
IMPUTATION_DIR = Path(IMPUTATION_DIR)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

RAW_COMPLETED_VIS_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_raw_vs_completed_dot_maps"
RAW_COMPLETED_VIS_DIR.mkdir(parents=True, exist_ok=True)

SELECTION_CSV_PATH = RAW_COMPLETED_VIS_DIR / f"{RUN_NAME}_selected_gene_cell_pairs_raw_vs_completed.csv"

SUMMARY_CSV_PATH = RAW_COMPLETED_VIS_DIR / f"{RUN_NAME}_raw_vs_completed_dot_map_summary.csv"

FIGURE_MANIFEST_PATH = RAW_COMPLETED_VIS_DIR / f"{RUN_NAME}_raw_vs_completed_dot_map_figure_manifest.csv"

print(f"RAW_COMPLETED_VIS_DIR: {RAW_COMPLETED_VIS_DIR}")

# ------------------------------------------------------------------------------
# 2. Load exact polygons if available; otherwise use approximate CosMx geometry
# ------------------------------------------------------------------------------

CELL_POLYGONS_PATH = IMPUTATION_DIR / "cell_polygons.pkl"
NUC_POLYGONS_PATH = IMPUTATION_DIR / "nuc_polygons.pkl"

cell_polygons = {}
nuc_polygons = {}

if CELL_POLYGONS_PATH.exists() and CELL_POLYGONS_PATH.stat().st_size > 10:
    try:
        print(f"Loading cell polygons: {CELL_POLYGONS_PATH}")
        with open(CELL_POLYGONS_PATH, "rb") as f:
            cell_polygons = pickle.load(f)
    except Exception as e:
        print(f"Could not load cell polygons: {e}")
        cell_polygons = {}
else:
    print("Exact cell polygons are missing/empty. Will use approximate CosMx cell ellipses.")

if NUC_POLYGONS_PATH.exists() and NUC_POLYGONS_PATH.stat().st_size > 10:
    try:
        print(f"Loading nucleus polygons: {NUC_POLYGONS_PATH}")
        with open(NUC_POLYGONS_PATH, "rb") as f:
            nuc_polygons = pickle.load(f)
    except Exception as e:
        print(f"Could not load nucleus polygons: {e}")
        nuc_polygons = {}
else:
    print("Exact nucleus polygons are missing/empty. Will use approximate nucleus ellipses.")

print(f"cell_polygons entries: {len(cell_polygons):,}")
print(f"nuc_polygons entries : {len(nuc_polygons):,}")

print(
    "\nCosMx geometry note:\n"
    "  Exact Xenium-style polygons are not available in this Step5 setup.\n"
    "  Figures will show molecule dots with approximate cell/nucleus ellipses\n"
    "  based on CosMx centroid, width/height, and nuclear-area metadata."
)

# ------------------------------------------------------------------------------
# 3. Configuration
# ------------------------------------------------------------------------------

candidate_genes = [
    # Epithelial / tumor-like
    "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "KRT17", "KRT13",
    "CEACAM6", "TACSTD2", "MKI67", "TOP2A",

    # Stromal / ECM
    "LUM", "DCN", "COL1A1", "COL1A2", "COL6A1", "COL6A2",
    "IGFBP7", "MMP2", "TAGLN", "ACTA2",

    # Immune / myeloid / lymphoid
    "CD3D", "CD3E", "CD79A", "MS4A1", "CD74",
    "LYZ", "TYROBP", "FCER1G", "MARCO", "S100A8", "S100A9",
    "NKG7", "GZMB", "PRF1", "GNLY",

    # Endothelial / vascular
    "PECAM1", "VWF", "RAMP2", "CAV1", "CDH5", "ENG",

    # Mast
    "TPSAB1", "TPSB2", "CPA3", "HPGDS",
]

available_genes = set(str(g) for g in shared_genes)
selected_genes = [g for g in candidate_genes if g in available_genes]

print(f"\nSelected genes present in CosMx panel: {selected_genes}")

if len(selected_genes) == 0:
    raise RuntimeError("None of the candidate genes are present in shared_genes.")

MAX_GENE_CELL_PAIRS = 12

MIN_OBSERVED_FOR_PAIR = 1
MIN_LEARNED_IMPUTED_FOR_PAIR = 2

MAX_RAW_POINTS_PER_PANEL = 400
MAX_IMPUTED_POINTS_PER_PANEL = 400

RAW_POINT_SIZE = 18
IMPUTED_POINT_SIZE = 22

RAW_ALPHA = 0.75
IMPUTED_ALPHA = 0.85

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def safe_cell_id(x):
    return str(x)


def safe_gene_id(x):
    return str(x)


def safe_filename_token(x):
    s = str(x)
    for old, new in [
        ("/", "_"),
        ("\\", "_"),
        (" ", "_"),
        ("+", "plus"),
        ("|", "_"),
        (":", "_"),
    ]:
        s = s.replace(old, new)
    return s


def _plot_polygon_boundary(ax, geom, linestyle="-", linewidth=1.0, alpha=0.9):
    if geom is None:
        return

    try:
        if geom.geom_type == "Polygon":
            x, y = geom.exterior.xy
            ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)
        elif geom.geom_type == "MultiPolygon":
            for poly in geom.geoms:
                x, y = poly.exterior.xy
                ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)
    except Exception:
        pass


def get_cell_obs_row(cid):
    cid = safe_cell_id(cid)

    if cid not in cell_idx_map:
        return None

    idx = cell_idx_map[cid]
    return denoised_adata.obs.iloc[idx]


def get_cell_center_width_height(cid):
    row = get_cell_obs_row(cid)

    if row is None:
        return None

    x_candidates = [
        "center_x_global_px",
        "CenterX_global_px",
        "x_centroid",
        "center_x_um",
    ]

    y_candidates = [
        "center_y_global_px",
        "CenterY_global_px",
        "y_centroid",
        "center_y_um",
    ]

    cx = None
    cy = None

    for c in x_candidates:
        if c in row.index and pd.notna(row[c]):
            cx = float(row[c])
            break

    for c in y_candidates:
        if c in row.index and pd.notna(row[c]):
            cy = float(row[c])
            break

    if cx is None or cy is None:
        return None

    width = None
    height = None

    for c in ["width_px", "Width"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            width = float(row[c])
            break

    for c in ["height_px", "Height"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            height = float(row[c])
            break

    if width is None or height is None:
        area = None

        for c in ["cell_area_px", "label_cell_area_px", "Area"]:
            if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
                area = float(row[c])
                break

        if area is not None:
            radius = np.sqrt(area / np.pi)
            width = 2.0 * radius
            height = 2.0 * radius
        else:
            width = 20.0
            height = 20.0

    return cx, cy, width, height


def get_approx_nucleus_width_height(cid, cell_width, cell_height):
    row = get_cell_obs_row(cid)

    if row is None:
        return 0.55 * cell_width, 0.55 * cell_height

    nuc_area = None
    for c in ["label_nuclear_area_px", "nuc_area_px", "nuclear_area_px"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            nuc_area = float(row[c])
            break

    cell_area = None
    for c in ["label_cell_area_px", "cell_area_px", "Area"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            cell_area = float(row[c])
            break

    if nuc_area is not None and cell_area is not None and cell_area > 0:
        scale = np.sqrt(np.clip(nuc_area / cell_area, 0.05, 0.95))
        return cell_width * scale, cell_height * scale

    return 0.55 * cell_width, 0.55 * cell_height


def plot_approx_cell_and_nucleus(ax, cid):
    geom = get_cell_center_width_height(cid)

    if geom is None:
        return None

    cx, cy, width, height = geom

    cell_ellipse = Ellipse(
        xy=(cx, cy),
        width=width,
        height=height,
        angle=0,
        fill=False,
        linestyle="-",
        linewidth=1.3,
        alpha=0.9,
        edgecolor="black",
    )

    nw, nh = get_approx_nucleus_width_height(cid, width, height)

    nucleus_ellipse = Ellipse(
        xy=(cx, cy),
        width=nw,
        height=nh,
        angle=0,
        fill=False,
        linestyle="--",
        linewidth=1.1,
        alpha=0.9,
        edgecolor="gray",
    )

    ax.add_patch(cell_ellipse)
    ax.add_patch(nucleus_ellipse)

    return cx, cy, width, height


def get_geom_bounds(cid, fallback_df=None, padding=5.0):
    cid = safe_cell_id(cid)

    if cid in cell_polygons:
        try:
            minx, miny, maxx, maxy = cell_polygons[cid].bounds
            return minx - padding, maxx + padding, miny - padding, maxy + padding
        except Exception:
            pass

    geom = get_cell_center_width_height(cid)

    if geom is not None:
        cx, cy, width, height = geom
        half_w = max(width / 2.0, 5.0)
        half_h = max(height / 2.0, 5.0)

        return (
            cx - half_w - padding,
            cx + half_w + padding,
            cy - half_h - padding,
            cy + half_h + padding,
        )

    if fallback_df is not None and len(fallback_df) > 0:
        x = pd.to_numeric(fallback_df["x"], errors="coerce")
        y = pd.to_numeric(fallback_df["y"], errors="coerce")

        if x.notna().sum() > 0 and y.notna().sum() > 0:
            return (
                float(x.min()) - padding,
                float(x.max()) + padding,
                float(y.min()) - padding,
                float(y.max()) + padding,
            )

    return None


def get_pair_counts(df, label):
    d = df[["cell_id", "gene_id"]].copy()
    d["cell_id"] = d["cell_id"].astype(str)
    d["gene_id"] = d["gene_id"].astype(str)

    return (
        d.groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name=label)
    )


def get_pair_molecules(df, cid, gid):
    cid = safe_cell_id(cid)
    gid = safe_gene_id(gid)

    return df[
        (df["cell_id"].astype(str) == cid)
        & (df["gene_id"].astype(str) == gid)
    ].copy()


def downsample_points(df, max_points=400, seed=42):
    if len(df) <= max_points:
        return df.copy()

    return df.sample(n=max_points, random_state=seed).copy()


def select_gene_cell_pairs_for_raw_completed():
    print("\nSelecting gene/cell pairs for raw-vs-completed visualization...")

    observed_sub = mol_observed[
        mol_observed["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    learned_sub = mol_learned_9E[
        mol_learned_9E["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    observed_counts = get_pair_counts(observed_sub, "n_observed")
    learned_counts = get_pair_counts(learned_sub, "n_learned_imputed")

    merged = observed_counts.merge(
        learned_counts,
        on=["cell_id", "gene_id"],
        how="inner",
    )

    merged = merged[
        (merged["n_observed"] >= MIN_OBSERVED_FOR_PAIR)
        & (merged["n_learned_imputed"] >= MIN_LEARNED_IMPUTED_FOR_PAIR)
    ].copy()

    if len(merged) == 0:
        print("No pairs found with the current thresholds.")
        print("Relaxing to at least one observed and one learned-imputed molecule.")

        merged = observed_counts.merge(
            learned_counts,
            on=["cell_id", "gene_id"],
            how="inner",
        )

        merged = merged[
            (merged["n_observed"] >= 1)
            & (merged["n_learned_imputed"] >= 1)
        ].copy()

    if len(merged) == 0:
        raise RuntimeError(
            "Could not find any gene/cell pair with both raw and learned-imputed molecules."
        )

    merged["total_raw_plus_learned"] = merged["n_observed"] + merged["n_learned_imputed"]

    selected_rows = []

    for gene in selected_genes:
        sub = merged[merged["gene_id"] == gene].copy()

        if len(sub) == 0:
            continue

        sub = sub.sort_values(
            ["n_learned_imputed", "n_observed", "total_raw_plus_learned"],
            ascending=False,
        )

        selected_rows.append(sub.iloc[0])

        if len(selected_rows) >= MAX_GENE_CELL_PAIRS:
            break

    selected = pd.DataFrame(selected_rows)

    if len(selected) < min(MAX_GENE_CELL_PAIRS, len(merged)):
        remaining = merged.merge(
            selected[["cell_id", "gene_id"]],
            on=["cell_id", "gene_id"],
            how="left",
            indicator=True,
        )

        remaining = remaining[remaining["_merge"] == "left_only"].drop(columns=["_merge"])

        remaining = remaining.sort_values(
            ["n_learned_imputed", "n_observed", "total_raw_plus_learned"],
            ascending=False,
        )

        needed = MAX_GENE_CELL_PAIRS - len(selected)

        selected = pd.concat(
            [selected, remaining.head(needed)],
            ignore_index=True,
        )

    selected = selected.head(MAX_GENE_CELL_PAIRS).reset_index(drop=True)

    return selected


def plot_raw_plus_imputed_panel(
    ax,
    cid,
    gid,
    raw_df,
    imputed_df=None,
    title="",
    show_legend=True,
):
    cid = safe_cell_id(cid)
    gid = safe_gene_id(gid)

    raw_pair = get_pair_molecules(raw_df, cid, gid)

    if imputed_df is not None:
        imp_pair = get_pair_molecules(imputed_df, cid, gid)
    else:
        imp_pair = pd.DataFrame(columns=raw_pair.columns)

    combined = (
        pd.concat([raw_pair, imp_pair], ignore_index=True)
        if len(imp_pair) > 0
        else raw_pair
    )

    polygon_plotted = False

    if cid in cell_polygons:
        _plot_polygon_boundary(
            ax,
            cell_polygons[cid],
            linestyle="-",
            linewidth=1.3,
            alpha=0.9,
        )
        polygon_plotted = True

    if cid in nuc_polygons:
        _plot_polygon_boundary(
            ax,
            nuc_polygons[cid],
            linestyle="--",
            linewidth=1.1,
            alpha=0.9,
        )

    if not polygon_plotted:
        plot_approx_cell_and_nucleus(ax, cid)

    raw_plot = downsample_points(
        raw_pair,
        max_points=MAX_RAW_POINTS_PER_PANEL,
        seed=42,
    )

    imp_plot = downsample_points(
        imp_pair,
        max_points=MAX_IMPUTED_POINTS_PER_PANEL,
        seed=43,
    )

    if len(raw_plot) > 0:
        ax.scatter(
            pd.to_numeric(raw_plot["x"], errors="coerce"),
            pd.to_numeric(raw_plot["y"], errors="coerce"),
            s=RAW_POINT_SIZE,
            alpha=RAW_ALPHA,
            marker="o",
            label=f"raw observed n={len(raw_pair):,}",
        )

    if len(imp_plot) > 0:
        ax.scatter(
            pd.to_numeric(imp_plot["x"], errors="coerce"),
            pd.to_numeric(imp_plot["y"], errors="coerce"),
            s=IMPUTED_POINT_SIZE,
            alpha=IMPUTED_ALPHA,
            marker="x",
            label=f"imputed n={len(imp_pair):,}",
        )

    bounds = get_geom_bounds(cid, fallback_df=combined, padding=5.0)

    if bounds is not None:
        xmin, xmax, ymin, ymax = bounds
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linewidth=0.3, alpha=0.35)
    ax.set_xlabel("x global px")
    ax.set_ylabel("y global px")

    if show_legend:
        ax.legend(fontsize=7, loc="best", frameon=True)

    return {
        "n_raw_observed": int(len(raw_pair)),
        "n_imputed": int(len(imp_pair)),
        "n_completed": int(len(raw_pair) + len(imp_pair)),
    }


def plot_one_raw_completed_comparison(cid, gid, out_prefix):
    cid = safe_cell_id(cid)
    gid = safe_gene_id(gid)

    panels = [
        {
            "title": "Raw observed",
            "imputed_df": None,
            "method": "raw_observed",
        },
        {
            "title": "Raw + learned 9E",
            "imputed_df": mol_learned_9E,
            "method": "learned_9E",
        },
        {
            "title": "Raw + gene empirical",
            "imputed_df": mol_gene_emp,
            "method": "gene_emp",
        },
        {
            "title": "Raw + cell-type gene empirical",
            "imputed_df": mol_ct_gene_emp,
            "method": "ct_gene_emp",
        },
        {
            "title": "Raw + spatial-kNN empirical",
            "imputed_df": mol_spatial_knn_emp,
            "method": "spatial_knn_emp",
        },
    ]

    fig, axes = plt.subplots(
        1,
        len(panels),
        figsize=(4.3 * len(panels), 4.6),
        sharex=True,
        sharey=True,
    )

    if len(panels) == 1:
        axes = [axes]

    summary_rows = []

    for ax, panel in zip(axes, panels):
        counts = plot_raw_plus_imputed_panel(
            ax=ax,
            cid=cid,
            gid=gid,
            raw_df=mol_observed,
            imputed_df=panel["imputed_df"],
            title=panel["title"],
            show_legend=True,
        )

        summary_rows.append({
            "cell_id": cid,
            "gene_id": gid,
            "method": panel["method"],
            "panel_title": panel["title"],
            **counts,
        })

    fig.suptitle(
        f"Raw vs completed molecule maps\nCell {cid} — Gene {gid}",
        fontsize=14,
        y=1.05,
    )

    plt.tight_layout()

    png_path = f"{out_prefix}.png"
    pdf_path = f"{out_prefix}.pdf"

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    return pd.DataFrame(summary_rows), png_path, pdf_path


# ------------------------------------------------------------------------------
# 5. Select cell-gene examples
# ------------------------------------------------------------------------------

selected_pairs_df = select_gene_cell_pairs_for_raw_completed()

print("\nSelected gene/cell pairs for raw-vs-completed dot maps:")
display(selected_pairs_df)

selected_pairs_df.to_csv(SELECTION_CSV_PATH, index=False)

print("Saved selected pairs:")
print(f"  {SELECTION_CSV_PATH}")

# ------------------------------------------------------------------------------
# 6. Generate figures
# ------------------------------------------------------------------------------

print("\nGenerating raw-vs-completed dot map figures...")

all_summary_rows = []
figure_records = []

for i, row in selected_pairs_df.iterrows():
    cid = safe_cell_id(row["cell_id"])
    gid = safe_gene_id(row["gene_id"])

    safe_cid = safe_filename_token(cid)
    safe_gid = safe_filename_token(gid)

    out_prefix = RAW_COMPLETED_VIS_DIR / f"{RUN_NAME}_raw_vs_completed_cell_{safe_cid}_gene_{safe_gid}"

    print(f"\n[{i + 1}/{len(selected_pairs_df)}] Plotting cell_id={cid}, gene_id={gid}")

    summary_df, png_path, pdf_path = plot_one_raw_completed_comparison(
        cid=cid,
        gid=gid,
        out_prefix=str(out_prefix),
    )

    all_summary_rows.append(summary_df)

    figure_records.append({
        "cell_id": cid,
        "gene_id": gid,
        "png_path": png_path,
        "pdf_path": pdf_path,
        "n_observed_selected": int(row.get("n_observed", -1)),
        "n_learned_imputed_selected": int(row.get("n_learned_imputed", -1)),
    })

# ------------------------------------------------------------------------------
# 7. Save summaries
# ------------------------------------------------------------------------------

raw_completed_summary_df = pd.concat(all_summary_rows, ignore_index=True)
figure_manifest_df = pd.DataFrame(figure_records)

raw_completed_summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
figure_manifest_df.to_csv(FIGURE_MANIFEST_PATH, index=False)

print("\nRaw-vs-completed dot map summary:")
display(raw_completed_summary_df.head(40))

print("\nFigure manifest:")
display(figure_manifest_df)

print("\nSaved raw-vs-completed dot map outputs:")
print(f"  Selected pairs CSV : {SELECTION_CSV_PATH}")
print(f"  Summary CSV        : {SUMMARY_CSV_PATH}")
print(f"  Figure manifest    : {FIGURE_MANIFEST_PATH}")
print(f"  Figure directory   : {RAW_COMPLETED_VIS_DIR}")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 6B COMPLETE — Raw vs completed molecule dot maps")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 6C — Global raw vs completed molecule map
#
# Figure layout:
#   Panel 1: Raw observed molecules only
#   Panel 2: Completed molecules = raw observed + learned 9E imputed
#
# Purpose:
#   Tissue-level before/after visualization of molecule-level imputation.
#
# Same algorithm/configs as Xenium Cell 6C:
#   RANDOM_STATE = 42
#   MAX_RAW_POINTS = 600_000
#   MAX_IMPUTED_POINTS = 300_000
#   RAW_POINT_SIZE = 0.15
#   IMPUTED_POINT_SIZE = 0.25
#   RAW_ALPHA = 0.45
#   IMPUTED_ALPHA = 0.65
#   COLOR_BY_GENE = False
#   TOP_N_GENES_FOR_COLOR = 20
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E, not learned_imputed_df
#   - Keeps CosMx coordinate system in global pixels
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("COSMX DOWNSTREAM CELL 6C — Global raw vs completed molecule map")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_global_raw_vs_completed_maps"
GLOBAL_MAP_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_RAW_COMPLETED_PNG = GLOBAL_MAP_DIR / f"{RUN_NAME}_global_raw_vs_completed_molecule_map.png"

GLOBAL_RAW_COMPLETED_PDF = GLOBAL_MAP_DIR / f"{RUN_NAME}_global_raw_vs_completed_molecule_map.pdf"

GLOBAL_MAP_SUMMARY_CSV = GLOBAL_MAP_DIR / f"{RUN_NAME}_global_raw_vs_completed_molecule_map_summary.csv"

print(f"GLOBAL_MAP_DIR: {GLOBAL_MAP_DIR}")

# ------------------------------------------------------------------------------
# 2. Plotting configuration
# ------------------------------------------------------------------------------

RANDOM_STATE = 42

MAX_RAW_POINTS = 600_000
MAX_IMPUTED_POINTS = 300_000

RAW_POINT_SIZE = 0.15
IMPUTED_POINT_SIZE = 0.25

RAW_ALPHA = 0.45
IMPUTED_ALPHA = 0.65

COLOR_BY_GENE = False
TOP_N_GENES_FOR_COLOR = 20

print(f"MAX_RAW_POINTS    : {MAX_RAW_POINTS:,}")
print(f"MAX_IMPUTED_POINTS: {MAX_IMPUTED_POINTS:,}")
print(f"COLOR_BY_GENE     : {COLOR_BY_GENE}")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def sample_molecules_for_plot(df, max_points, seed=42):
    """
    Downsample molecule table for visualization.
    """
    if len(df) <= max_points:
        return df.copy()

    return df.sample(n=max_points, random_state=seed).copy()


def clean_xy(df):
    """
    Keep only rows with valid x/y.
    """
    out = df.copy()
    out["x"] = pd.to_numeric(out["x"], errors="coerce")
    out["y"] = pd.to_numeric(out["y"], errors="coerce")
    out = out.dropna(subset=["x", "y"])
    return out


def get_global_bounds(*dfs, padding_frac=0.03):
    """
    Use all provided dataframes to compute common x/y plot bounds.
    """
    xs = []
    ys = []

    for df in dfs:
        if df is None or len(df) == 0:
            continue

        xs.append(pd.to_numeric(df["x"], errors="coerce"))
        ys.append(pd.to_numeric(df["y"], errors="coerce"))

    if len(xs) == 0 or len(ys) == 0:
        raise ValueError("No valid x/y coordinates found for global bounds.")

    x_all = pd.concat(xs, ignore_index=True).dropna()
    y_all = pd.concat(ys, ignore_index=True).dropna()

    xmin, xmax = float(x_all.min()), float(x_all.max())
    ymin, ymax = float(y_all.min()), float(y_all.max())

    xpad = (xmax - xmin) * padding_frac
    ypad = (ymax - ymin) * padding_frac

    return xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad


def build_gene_color_map(df, top_n=20):
    """
    Make a simple gene-to-color map for top genes.
    Other genes become gray.
    """
    top_genes = (
        df["gene_id"]
        .astype(str)
        .value_counts()
        .head(top_n)
        .index
        .tolist()
    )

    cmap = plt.cm.get_cmap("tab20", len(top_genes))

    gene_to_color = {
        gene: cmap(i)
        for i, gene in enumerate(top_genes)
    }

    return gene_to_color, top_genes


def plot_gene_colored_points(
    ax,
    df,
    gene_to_color,
    default_color="lightgray",
    point_size=0.2,
    alpha=0.5,
    label_prefix="",
):
    """
    Plot points colored by top genes.
    """
    d = df.copy()
    d["gene_id"] = d["gene_id"].astype(str)

    top_genes = set(gene_to_color.keys())
    other = d[~d["gene_id"].isin(top_genes)]

    if len(other) > 0:
        ax.scatter(
            other["x"],
            other["y"],
            s=point_size,
            alpha=alpha * 0.4,
            c=default_color,
            linewidths=0,
            label=f"{label_prefix}other genes",
        )

    for gene, color in gene_to_color.items():
        sub = d[d["gene_id"] == gene]

        if len(sub) == 0:
            continue

        ax.scatter(
            sub["x"],
            sub["y"],
            s=point_size,
            alpha=alpha,
            c=[color],
            linewidths=0,
            label=f"{label_prefix}{gene}",
        )

# ------------------------------------------------------------------------------
# 4. Prepare sampled raw and imputed molecules
# ------------------------------------------------------------------------------

print("\nPreparing sampled molecule tables...")

if "status" in mol_observed.columns:
    raw_for_plot = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ].copy()
else:
    raw_for_plot = mol_observed.copy()

imputed_for_plot = mol_learned_9E.copy()

raw_for_plot = clean_xy(raw_for_plot)
imputed_for_plot = clean_xy(imputed_for_plot)

print(f"Raw observed molecules available : {len(raw_for_plot):,}")
print(f"Learned 9E imputed available     : {len(imputed_for_plot):,}")

raw_sample = sample_molecules_for_plot(
    raw_for_plot,
    max_points=MAX_RAW_POINTS,
    seed=RANDOM_STATE,
)

imputed_sample = sample_molecules_for_plot(
    imputed_for_plot,
    max_points=MAX_IMPUTED_POINTS,
    seed=RANDOM_STATE + 1,
)

completed_sample = pd.concat(
    [
        raw_sample.assign(plot_source="raw_observed"),
        imputed_sample.assign(plot_source="learned_9E_imputed"),
    ],
    ignore_index=True,
)

print(f"Raw sample       : {len(raw_sample):,}")
print(f"Imputed sample   : {len(imputed_sample):,}")
print(f"Completed sample : {len(completed_sample):,}")

xmin, xmax, ymin, ymax = get_global_bounds(raw_sample, imputed_sample)

print(f"Plot bounds: x=[{xmin:.1f}, {xmax:.1f}], y=[{ymin:.1f}, {ymax:.1f}]")

# ------------------------------------------------------------------------------
# 5. Plot raw vs completed
# ------------------------------------------------------------------------------

print("\nPlotting global raw vs completed molecule map...")

fig, axes = plt.subplots(
    1,
    2,
    figsize=(18, 8),
    sharex=True,
    sharey=True,
)

# Panel 1: raw observed only.
ax = axes[0]

if COLOR_BY_GENE:
    gene_to_color, top_genes = build_gene_color_map(raw_sample, TOP_N_GENES_FOR_COLOR)
    plot_gene_colored_points(
        ax,
        raw_sample,
        gene_to_color=gene_to_color,
        point_size=RAW_POINT_SIZE,
        alpha=RAW_ALPHA,
        label_prefix="raw: ",
    )
else:
    ax.scatter(
        raw_sample["x"],
        raw_sample["y"],
        s=RAW_POINT_SIZE,
        alpha=RAW_ALPHA,
        linewidths=0,
        label=f"raw observed sample n={len(raw_sample):,}",
    )

ax.set_title(
    f"Raw observed molecules\nsample n={len(raw_sample):,} / total {len(raw_for_plot):,}",
    fontsize=13,
)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x global px")
ax.set_ylabel("y global px")
ax.grid(False)

# Panel 2: completed = raw + learned imputed.
ax = axes[1]

if COLOR_BY_GENE:
    gene_to_color, top_genes = build_gene_color_map(completed_sample, TOP_N_GENES_FOR_COLOR)

    ax.scatter(
        raw_sample["x"],
        raw_sample["y"],
        s=RAW_POINT_SIZE,
        alpha=RAW_ALPHA * 0.5,
        linewidths=0,
        label=f"raw observed sample n={len(raw_sample):,}",
    )

    plot_gene_colored_points(
        ax,
        imputed_sample,
        gene_to_color=gene_to_color,
        point_size=IMPUTED_POINT_SIZE,
        alpha=IMPUTED_ALPHA,
        label_prefix="imputed: ",
    )
else:
    ax.scatter(
        raw_sample["x"],
        raw_sample["y"],
        s=RAW_POINT_SIZE,
        alpha=RAW_ALPHA * 0.45,
        linewidths=0,
        label=f"raw observed sample n={len(raw_sample):,}",
    )

    ax.scatter(
        imputed_sample["x"],
        imputed_sample["y"],
        s=IMPUTED_POINT_SIZE,
        alpha=IMPUTED_ALPHA,
        marker="x",
        linewidths=0.25,
        label=f"9E imputed sample n={len(imputed_sample):,}",
    )

ax.set_title(
    f"Completed molecule map: raw + learned 9E imputed\n"
    f"sample n={len(completed_sample):,} / total {len(raw_for_plot) + len(imputed_for_plot):,}",
    fontsize=13,
)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x global px")
ax.set_ylabel("y global px")
ax.grid(False)
ax.legend(loc="best", fontsize=8, markerscale=4)

fig.suptitle(
    "CosMx global tissue-level molecule map before and after learned 9E imputation",
    fontsize=16,
    y=1.02,
)

plt.tight_layout()

plt.savefig(GLOBAL_RAW_COMPLETED_PNG, dpi=300, bbox_inches="tight")
plt.savefig(GLOBAL_RAW_COMPLETED_PDF, bbox_inches="tight")
plt.show()

print(f"Saved PNG: {GLOBAL_RAW_COMPLETED_PNG}")
print(f"Saved PDF: {GLOBAL_RAW_COMPLETED_PDF}")

# ------------------------------------------------------------------------------
# 6. Save summary
# ------------------------------------------------------------------------------

summary = {
    "raw_total_molecules": int(len(raw_for_plot)),
    "learned_9E_imputed_total_molecules": int(len(imputed_for_plot)),
    "completed_total_molecules": int(len(raw_for_plot) + len(imputed_for_plot)),
    "raw_sampled_molecules": int(len(raw_sample)),
    "imputed_sampled_molecules": int(len(imputed_sample)),
    "completed_sampled_molecules": int(len(completed_sample)),
    "max_raw_points": int(MAX_RAW_POINTS),
    "max_imputed_points": int(MAX_IMPUTED_POINTS),
    "random_state": int(RANDOM_STATE),
    "color_by_gene": bool(COLOR_BY_GENE),
    "coordinate_system": "CosMx global pixel coordinates",
    "png_path": str(GLOBAL_RAW_COMPLETED_PNG),
    "pdf_path": str(GLOBAL_RAW_COMPLETED_PDF),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(GLOBAL_MAP_SUMMARY_CSV, index=False)

print("\nGlobal map summary:")
display(summary_df)

print(f"Saved summary CSV: {GLOBAL_MAP_SUMMARY_CSV}")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 6C COMPLETE — Global raw vs completed molecule map")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 6D_FIXED — All-molecule raw vs completed Datashader map
#
# This cell plots ALL molecules, not sampled molecules:
#   Panel 1: Raw observed molecules
#   Panel 2: Completed = raw observed + learned 9E imputed molecules
#
# Same algorithm/configs as original 6D_FIXED:
#   WIDTH  = 1600
#   HEIGHT = 1200
#   Raw image: black/white density
#   Completed image: raw = lightgray, imputed = red
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E, not learned_imputed_df
#   - Uses CosMx global pixel coordinates
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print("=" * 100)
print("COSMX DOWNSTREAM CELL 6D_FIXED — All-molecule raw vs completed Datashader map")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Install/import Datashader
# ------------------------------------------------------------------------------

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

# ------------------------------------------------------------------------------
# 1. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_global_raw_vs_completed_maps"
GLOBAL_MAP_DIR.mkdir(parents=True, exist_ok=True)

RAW_ALL_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_observed_datashader"

COMPLETED_ALL_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_completed_raw_plus_9E_datashader"

raw_png = str(RAW_ALL_PATH) + ".png"
completed_png = str(COMPLETED_ALL_PATH) + ".png"

combined_png = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_vs_completed_datashader_side_by_side.png"

SUMMARY_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_datashader_summary.csv"

print(f"GLOBAL_MAP_DIR : {GLOBAL_MAP_DIR}")
print(f"Raw image      : {raw_png}")
print(f"Completed image: {completed_png}")
print(f"Combined image : {combined_png}")

# ------------------------------------------------------------------------------
# 3. Prepare all molecule coordinates
# ------------------------------------------------------------------------------

print("\nPreparing all raw and imputed molecule coordinates...")

if "status" in mol_observed.columns:
    raw_all = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ][["x", "y"]].copy()
else:
    raw_all = mol_observed[["x", "y"]].copy()

imputed_all = mol_learned_9E[["x", "y"]].copy()

raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")

imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

raw_all["source"] = "raw"
imputed_all["source"] = "imputed"

raw_all["source"] = raw_all["source"].astype("category")
imputed_all["source"] = imputed_all["source"].astype("category")

completed_all = pd.concat([raw_all, imputed_all], ignore_index=True)
completed_all["source"] = completed_all["source"].astype("category")

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")
print(f"Completed points    : {len(completed_all):,}")

# ------------------------------------------------------------------------------
# 4. Shared plot bounds
# ------------------------------------------------------------------------------

xmin = float(completed_all["x"].min())
xmax = float(completed_all["x"].max())
ymin = float(completed_all["y"].min())
ymax = float(completed_all["y"].max())

xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03

x_range = (xmin - xpad, xmax + xpad)
y_range = (ymin - ypad, ymax + ypad)

print(f"x_range: {x_range}")
print(f"y_range: {y_range}")

# ------------------------------------------------------------------------------
# 5. Render all-molecule maps using Datashader
# ------------------------------------------------------------------------------

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

print("\nRendering raw observed all-molecule image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    str(RAW_ALL_PATH),
    fmt=".png",
)

print("Saved raw image:")
print(f"  {raw_png}")

print("\nRendering completed raw + learned 9E imputed all-molecule image...")

agg_completed = canvas.points(
    completed_all,
    x="x",
    y="y",
    agg=ds.count_cat("source"),
)

img_completed = tf.shade(
    agg_completed,
    color_key={
        "raw": "lightgray",
        "imputed": "red",
    },
    how="eq_hist",
)

export_image(
    img_completed,
    str(COMPLETED_ALL_PATH),
    fmt=".png",
)

print("Saved completed image:")
print(f"  {completed_png}")

# ------------------------------------------------------------------------------
# 6. Combine side by side
# ------------------------------------------------------------------------------

print("\nCombining raw and completed images side by side...")

if not os.path.exists(raw_png):
    raise FileNotFoundError(f"Raw image not found after export: {raw_png}")

if not os.path.exists(completed_png):
    raise FileNotFoundError(f"Completed image not found after export: {completed_png}")

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    img2 = img2.resize(img1.size)

w, h = img1.size

title_h = 100
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text((30, 58), f"n = {len(raw_all):,}", fill="gray")
draw.text((w + gap + 30, 58), f"raw n = {len(raw_all):,}, imputed n = {len(imputed_all):,}", fill="gray")

combined.save(combined_png)

print("Saved side-by-side image:")
print(f"  {combined_png}")

display(combined)

# ------------------------------------------------------------------------------
# 7. Save summary
# ------------------------------------------------------------------------------

summary = {
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(completed_all)),
    "plot_width": int(WIDTH),
    "plot_height": int(HEIGHT),
    "x_min": float(x_range[0]),
    "x_max": float(x_range[1]),
    "y_min": float(y_range[0]),
    "y_max": float(y_range[1]),
    "coordinate_system": "CosMx global pixel coordinates",
    "raw_image": raw_png,
    "completed_image": completed_png,
    "combined_image": str(combined_png),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nSummary:")
display(summary_df)

print("Saved summary:")
print(f"  {SUMMARY_PATH}")

# ------------------------------------------------------------------------------
# 8. Cleanup
# ------------------------------------------------------------------------------

del completed_all
gc.collect()

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 6D_FIXED COMPLETE — all-molecule raw vs completed map saved")
print("=" * 100)

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 6D_BW — All-molecule raw vs completed black/white map
#
# Panel 1: Raw observed molecules
# Panel 2: Completed = raw observed + learned 9E imputed
#
# Both panels use black background + white molecule density.
#
# Same algorithm/configs as original 6D_BW:
#   WIDTH  = 1600
#   HEIGHT = 1200
#   Both panels use count-density black/white Datashader rendering
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E, not learned_imputed_df
#   - Uses CosMx global pixel coordinates
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print("=" * 100)
print("COSMX DOWNSTREAM CELL 6D_BW — Black/white all-molecule raw vs completed Datashader map")
print("=" * 100)

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_global_raw_vs_completed_maps"
GLOBAL_MAP_DIR.mkdir(parents=True, exist_ok=True)

RAW_BW_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_observed_BW_datashader"

COMPLETED_BW_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_completed_raw_plus_9E_BW_datashader"

COMBINED_BW_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_vs_completed_BW_side_by_side.png"

SUMMARY_BW_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_vs_completed_BW_summary.csv"

print(f"GLOBAL_MAP_DIR : {GLOBAL_MAP_DIR}")
print(f"Raw BW image   : {str(RAW_BW_PATH)}.png")
print(f"Completed BW   : {str(COMPLETED_BW_PATH)}.png")
print(f"Combined BW    : {COMBINED_BW_PATH}")

# ------------------------------------------------------------------------------
# 2. Prepare coordinates
# ------------------------------------------------------------------------------

print("\nPreparing coordinates...")

if "status" in mol_observed.columns:
    raw_all = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ][["x", "y"]].copy()
else:
    raw_all = mol_observed[["x", "y"]].copy()

imputed_all = mol_learned_9E[["x", "y"]].copy()

raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")

imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

completed_all = pd.concat(
    [
        raw_all,
        imputed_all,
    ],
    ignore_index=True,
)

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")
print(f"Completed points    : {len(completed_all):,}")

# ------------------------------------------------------------------------------
# 3. Shared bounds and canvas
# ------------------------------------------------------------------------------

xmin = float(completed_all["x"].min())
xmax = float(completed_all["x"].max())
ymin = float(completed_all["y"].min())
ymax = float(completed_all["y"].max())

xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03

x_range = (xmin - xpad, xmax + xpad)
y_range = (ymin - ypad, ymax + ypad)

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

print(f"x_range: {x_range}")
print(f"y_range: {y_range}")
print(f"Canvas : {WIDTH} × {HEIGHT}")

# ------------------------------------------------------------------------------
# 4. Render raw black/white image
# ------------------------------------------------------------------------------

print("\nRendering raw observed black/white image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    str(RAW_BW_PATH),
    fmt=".png",
)

raw_png = str(RAW_BW_PATH) + ".png"

print(f"Saved raw BW image: {raw_png}")

# ------------------------------------------------------------------------------
# 5. Render completed black/white image
# ------------------------------------------------------------------------------

print("\nRendering completed black/white image...")

agg_completed = canvas.points(
    completed_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_completed = tf.shade(
    agg_completed,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_completed,
    str(COMPLETED_BW_PATH),
    fmt=".png",
)

completed_png = str(COMPLETED_BW_PATH) + ".png"

print(f"Saved completed BW image: {completed_png}")

# ------------------------------------------------------------------------------
# 6. Combine side-by-side
# ------------------------------------------------------------------------------

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    img2 = img2.resize(img1.size)

w, h = img1.size

title_h = 100
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text((30, 58), f"n = {len(raw_all):,}", fill="gray")
draw.text(
    (w + gap + 30, 58),
    f"raw n = {len(raw_all):,}, imputed n = {len(imputed_all):,}",
    fill="gray",
)

combined.save(COMBINED_BW_PATH)

display(combined)

print("\nSaved side-by-side BW image:")
print(f"  {COMBINED_BW_PATH}")

# ------------------------------------------------------------------------------
# 7. Save summary
# ------------------------------------------------------------------------------

summary_df = pd.DataFrame([{
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(completed_all)),
    "plot_width": int(WIDTH),
    "plot_height": int(HEIGHT),
    "x_min": float(x_range[0]),
    "x_max": float(x_range[1]),
    "y_min": float(y_range[0]),
    "y_max": float(y_range[1]),
    "coordinate_system": "CosMx global pixel coordinates",
    "raw_bw_image": raw_png,
    "completed_bw_image": completed_png,
    "combined_bw_image": str(COMBINED_BW_PATH),
}])

summary_df.to_csv(SUMMARY_BW_PATH, index=False)

print("Saved summary:")
print(f"  {SUMMARY_BW_PATH}")

# ------------------------------------------------------------------------------
# 8. Cleanup
# ------------------------------------------------------------------------------

del completed_all
gc.collect()

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 6D_BW COMPLETE")
print("=" * 100)

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 6D_OPTION2 — All-molecule raw vs completed grayscale Datashader map
#
# Option 2:
#   Panel 1: Raw observed molecules
#            black background + white raw density
#
#   Panel 2: Completed = raw observed + learned 9E imputed
#            black background
#            raw molecules     = dim gray
#            imputed molecules = white
#
# Purpose:
#   Tissue-level before/after visualization of molecule-level imputation,
#   while still distinguishing raw vs imputed molecules without using pink/red.
#
# Same algorithm/configs as original Cell 6D_OPTION2:
#   WIDTH  = 1600
#   HEIGHT = 1200
#   raw panel      : black background + white density
#   completed panel: raw = dimgray, imputed = white
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E, not learned_imputed_df
#   - Uses CosMx global pixel coordinates
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print("=" * 100)
print("COSMX DOWNSTREAM CELL 6D_OPTION2 — All-molecule raw vs completed grayscale Datashader map")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Install/import Datashader
# ------------------------------------------------------------------------------

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

# ------------------------------------------------------------------------------
# 1. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_global_raw_vs_completed_maps"
GLOBAL_MAP_DIR.mkdir(parents=True, exist_ok=True)

RAW_OPTION2_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_observed_option2_BW_datashader"

COMPLETED_OPTION2_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_completed_raw_dimgray_imputed_white_datashader"

COMBINED_OPTION2_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_vs_completed_dimgray_white_side_by_side.png"

SUMMARY_OPTION2_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_vs_completed_dimgray_white_summary.csv"

print(f"GLOBAL_MAP_DIR       : {GLOBAL_MAP_DIR}")
print(f"Raw image path       : {str(RAW_OPTION2_PATH)}.png")
print(f"Completed image path : {str(COMPLETED_OPTION2_PATH)}.png")
print(f"Combined image path  : {COMBINED_OPTION2_PATH}")

# ------------------------------------------------------------------------------
# 3. Prepare all raw and imputed molecule coordinates
# ------------------------------------------------------------------------------

print("\nPreparing all raw and imputed molecule coordinates...")

if "status" in mol_observed.columns:
    raw_all = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ][["x", "y"]].copy()
else:
    raw_all = mol_observed[["x", "y"]].copy()

imputed_all = mol_learned_9E[["x", "y"]].copy()

# Convert x/y to numeric float32 to reduce memory.
raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")

imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

# Remove invalid coordinate rows.
raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

# Add source labels for completed panel.
raw_all["source"] = "raw"
imputed_all["source"] = "imputed"

completed_all = pd.concat(
    [
        raw_all,
        imputed_all,
    ],
    ignore_index=True,
)

# Datashader count_cat needs categorical dtype.
completed_all["source"] = completed_all["source"].astype("category")

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")
print(f"Completed points    : {len(completed_all):,}")

# ------------------------------------------------------------------------------
# 4. Shared plot bounds
# ------------------------------------------------------------------------------

xmin = float(completed_all["x"].min())
xmax = float(completed_all["x"].max())
ymin = float(completed_all["y"].min())
ymax = float(completed_all["y"].max())

xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03

x_range = (xmin - xpad, xmax + xpad)
y_range = (ymin - ypad, ymax + ypad)

print(f"x_range: {x_range}")
print(f"y_range: {y_range}")

# ------------------------------------------------------------------------------
# 5. Canvas settings
# ------------------------------------------------------------------------------

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

print(f"Canvas: {WIDTH} × {HEIGHT}")

# ------------------------------------------------------------------------------
# 6. Render Panel 1: raw observed black/white density
# ------------------------------------------------------------------------------

print("\nRendering Panel 1: raw observed black/white density image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    str(RAW_OPTION2_PATH),
    fmt=".png",
)

raw_png = str(RAW_OPTION2_PATH) + ".png"

print("Saved raw image:")
print(f"  {raw_png}")

# ------------------------------------------------------------------------------
# 7. Render Panel 2: completed grayscale raw=dimgray, imputed=white
# ------------------------------------------------------------------------------

print("\nRendering Panel 2: completed grayscale image with raw=dimgray and imputed=white...")

agg_completed = canvas.points(
    completed_all,
    x="x",
    y="y",
    agg=ds.count_cat("source"),
)

img_completed = tf.shade(
    agg_completed,
    color_key={
        "raw": "dimgray",
        "imputed": "white",
    },
    how="eq_hist",
)

export_image(
    img_completed,
    str(COMPLETED_OPTION2_PATH),
    fmt=".png",
)

completed_png = str(COMPLETED_OPTION2_PATH) + ".png"

print("Saved completed image:")
print(f"  {completed_png}")

# ------------------------------------------------------------------------------
# 8. Combine side by side
# ------------------------------------------------------------------------------

print("\nCombining raw and completed images side by side...")

if not os.path.exists(raw_png):
    raise FileNotFoundError(f"Raw image not found after export: {raw_png}")

if not os.path.exists(completed_png):
    raise FileNotFoundError(f"Completed image not found after export: {completed_png}")

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    print(f"WARNING: image sizes differ: raw={img1.size}, completed={img2.size}")
    print("Resizing completed image to match raw image size.")
    img2 = img2.resize(img1.size)

w, h = img1.size

title_h = 110
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text(
    (30, 58),
    f"raw observed n = {len(raw_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 58),
    f"raw n = {len(raw_all):,}; imputed n = {len(imputed_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 82),
    "raw = dim gray; imputed = white",
    fill="gray",
)

combined.save(COMBINED_OPTION2_PATH)

print("Saved side-by-side image:")
print(f"  {COMBINED_OPTION2_PATH}")

display(combined)

# ------------------------------------------------------------------------------
# 9. Save summary
# ------------------------------------------------------------------------------

summary_df = pd.DataFrame([{
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(completed_all)),
    "plot_width": int(WIDTH),
    "plot_height": int(HEIGHT),
    "x_min": float(x_range[0]),
    "x_max": float(x_range[1]),
    "y_min": float(y_range[0]),
    "y_max": float(y_range[1]),
    "coordinate_system": "CosMx global pixel coordinates",
    "raw_image": raw_png,
    "completed_image": completed_png,
    "combined_image": str(COMBINED_OPTION2_PATH),
    "completed_panel_raw_color": "dimgray",
    "completed_panel_imputed_color": "white",
    "background_color": "black",
}])

summary_df.to_csv(SUMMARY_OPTION2_PATH, index=False)

print("\nSummary:")
display(summary_df)

print("Saved summary:")
print(f"  {SUMMARY_OPTION2_PATH}")

# ------------------------------------------------------------------------------
# 10. Cleanup
# ------------------------------------------------------------------------------

del completed_all
gc.collect()

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 6D_OPTION2 COMPLETE — grayscale raw/imputed all-molecule map saved")
print("=" * 100)

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 6D_OPTION4 — All-molecule raw vs completed map
#
# Panel 1:
#   Raw observed molecules
#   black background + white raw density
#
# Panel 2:
#   Completed = raw observed + learned 9E imputed
#   black background
#   white raw density, rendered exactly like Panel 1
#   imputed molecules overlaid in red
#
# Important:
#   Panel 2 is NOT using color_key raw=white/imputed=red directly.
#   Instead:
#     1. render raw density as black/white image
#     2. render imputed density as red transparent layer
#     3. alpha-composite red imputed layer on top of raw black/white layer
#
# Same algorithm/configs as original Cell 6D_OPTION4:
#   WIDTH  = 1600
#   HEIGHT = 1200
#   raw density = white
#   imputed overlay = red
#   alpha multiplier = 2.5
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E, not learned_imputed_df
#   - Uses CosMx global pixel coordinates
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print("=" * 100)
print("COSMX DOWNSTREAM CELL 6D_OPTION4 — Raw density + red imputed overlay")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Install/import Datashader
# ------------------------------------------------------------------------------

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

# ------------------------------------------------------------------------------
# 1. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_global_raw_vs_completed_maps"
GLOBAL_MAP_DIR.mkdir(parents=True, exist_ok=True)

RAW_OPTION4_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_observed_option4_BW_datashader"

RAW_FOR_COMPLETED_OPTION4_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_density_for_completed_option4_BW_datashader"

IMPUTED_RED_OPTION4_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_imputed_red_overlay_option4_datashader"

COMPLETED_OPTION4_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_completed_raw_BW_plus_imputed_red_overlay"

COMBINED_OPTION4_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_vs_completed_BW_red_overlay_side_by_side.png"

SUMMARY_OPTION4_PATH = GLOBAL_MAP_DIR / f"{RUN_NAME}_ALL_raw_vs_completed_BW_red_overlay_summary.csv"

print(f"GLOBAL_MAP_DIR       : {GLOBAL_MAP_DIR}")
print(f"Panel 1 raw image    : {str(RAW_OPTION4_PATH)}.png")
print(f"Panel 2 completed    : {str(COMPLETED_OPTION4_PATH)}.png")
print(f"Combined image       : {COMBINED_OPTION4_PATH}")

# ------------------------------------------------------------------------------
# 3. Prepare all raw and imputed coordinates
# ------------------------------------------------------------------------------

print("\nPreparing all raw and imputed molecule coordinates...")

if "status" in mol_observed.columns:
    raw_all = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ][["x", "y"]].copy()
else:
    raw_all = mol_observed[["x", "y"]].copy()

imputed_all = mol_learned_9E[["x", "y"]].copy()

raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")

imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")

# ------------------------------------------------------------------------------
# 4. Shared plot bounds
# ------------------------------------------------------------------------------

all_x_min = min(float(raw_all["x"].min()), float(imputed_all["x"].min()))
all_x_max = max(float(raw_all["x"].max()), float(imputed_all["x"].max()))
all_y_min = min(float(raw_all["y"].min()), float(imputed_all["y"].min()))
all_y_max = max(float(raw_all["y"].max()), float(imputed_all["y"].max()))

xpad = (all_x_max - all_x_min) * 0.03
ypad = (all_y_max - all_y_min) * 0.03

x_range = (all_x_min - xpad, all_x_max + xpad)
y_range = (all_y_min - ypad, all_y_max + ypad)

print(f"x_range: {x_range}")
print(f"y_range: {y_range}")

# ------------------------------------------------------------------------------
# 5. Canvas settings
# ------------------------------------------------------------------------------

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

print(f"Canvas: {WIDTH} × {HEIGHT}")

# ------------------------------------------------------------------------------
# 6. Render raw density image for Panel 1
# ------------------------------------------------------------------------------

print("\nRendering Panel 1: raw observed black/white density image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    str(RAW_OPTION4_PATH),
    fmt=".png",
)

raw_png = str(RAW_OPTION4_PATH) + ".png"

print("Saved Panel 1 raw image:")
print(f"  {raw_png}")

# ------------------------------------------------------------------------------
# 7. Render raw density image again for Panel 2 background
# ------------------------------------------------------------------------------

print("\nRendering Panel 2 background: raw black/white density image...")

# Use the same agg_raw and same shading so raw density in Panel 2 matches Panel 1.
img_raw_for_completed = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw_for_completed,
    str(RAW_FOR_COMPLETED_OPTION4_PATH),
    fmt=".png",
)

raw_for_completed_png = str(RAW_FOR_COMPLETED_OPTION4_PATH) + ".png"

print("Saved raw background for completed panel:")
print(f"  {raw_for_completed_png}")

# ------------------------------------------------------------------------------
# 8. Render imputed molecules as red overlay
# ------------------------------------------------------------------------------

print("\nRendering imputed molecules as red overlay...")

agg_imputed = canvas.points(
    imputed_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_imputed_red = tf.shade(
    agg_imputed,
    cmap=["black", "red"],
    how="eq_hist",
)

export_image(
    img_imputed_red,
    str(IMPUTED_RED_OPTION4_PATH),
    fmt=".png",
)

imputed_red_png = str(IMPUTED_RED_OPTION4_PATH) + ".png"

print("Saved red imputed overlay image:")
print(f"  {imputed_red_png}")

# ------------------------------------------------------------------------------
# 9. Alpha-composite red imputed layer over raw black/white density
# ------------------------------------------------------------------------------

print("\nCompositing completed panel: raw black/white density + red imputed overlay...")

raw_bg = Image.open(raw_for_completed_png).convert("RGBA")
red_layer = Image.open(imputed_red_png).convert("RGBA")

if raw_bg.size != red_layer.size:
    red_layer = red_layer.resize(raw_bg.size)

red_arr = np.array(red_layer).astype(np.uint8)

r = red_arr[:, :, 0].astype(np.int16)
g = red_arr[:, :, 1].astype(np.int16)
b = red_arr[:, :, 2].astype(np.int16)

# Identify red-like pixels. Datashader creates black background and red density.
red_strength = np.maximum(r - np.maximum(g, b), 0)

# Alpha controls red overlay strength.
# Same original multiplier.
alpha = np.clip(red_strength * 2.5, 0, 255).astype(np.uint8)

red_overlay = np.zeros_like(red_arr)
red_overlay[:, :, 0] = 255
red_overlay[:, :, 1] = 0
red_overlay[:, :, 2] = 0
red_overlay[:, :, 3] = alpha

red_overlay_img = Image.fromarray(red_overlay, mode="RGBA")

completed_img = Image.alpha_composite(raw_bg, red_overlay_img).convert("RGB")

completed_png = str(COMPLETED_OPTION4_PATH) + ".png"
completed_img.save(completed_png)

print("Saved completed panel:")
print(f"  {completed_png}")

# ------------------------------------------------------------------------------
# 10. Combine Panel 1 and Panel 2 side by side
# ------------------------------------------------------------------------------

print("\nCombining Panel 1 and Panel 2 side by side...")

if not os.path.exists(raw_png):
    raise FileNotFoundError(f"Raw image not found after export: {raw_png}")

if not os.path.exists(completed_png):
    raise FileNotFoundError(f"Completed image not found after export: {completed_png}")

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    print(f"WARNING: image sizes differ: raw={img1.size}, completed={img2.size}")
    print("Resizing completed image to match raw image size.")
    img2 = img2.resize(img1.size)

w, h = img1.size

title_h = 110
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text(
    (30, 58),
    f"raw observed n = {len(raw_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 58),
    f"raw n = {len(raw_all):,}; imputed n = {len(imputed_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 82),
    "raw density = white; imputed molecules = red",
    fill="gray",
)

combined.save(COMBINED_OPTION4_PATH)

print("Saved side-by-side image:")
print(f"  {COMBINED_OPTION4_PATH}")

display(combined)

# ------------------------------------------------------------------------------
# 11. Save summary
# ------------------------------------------------------------------------------

summary_df = pd.DataFrame([{
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(raw_all) + len(imputed_all)),
    "plot_width": int(WIDTH),
    "plot_height": int(HEIGHT),
    "x_min": float(x_range[0]),
    "x_max": float(x_range[1]),
    "y_min": float(y_range[0]),
    "y_max": float(y_range[1]),
    "coordinate_system": "CosMx global pixel coordinates",
    "panel1_raw_image": raw_png,
    "panel2_raw_background_image": raw_for_completed_png,
    "panel2_imputed_red_overlay_image": imputed_red_png,
    "panel2_completed_image": completed_png,
    "combined_image": str(COMBINED_OPTION4_PATH),
    "panel1_description": "raw observed, black background, white density",
    "panel2_description": "raw density same as panel 1 plus red imputed molecule overlay",
    "imputed_overlay_alpha_multiplier": 2.5,
}])

summary_df.to_csv(SUMMARY_OPTION4_PATH, index=False)

print("\nSummary:")
display(summary_df)

print("Saved summary:")
print(f"  {SUMMARY_OPTION4_PATH}")

# ------------------------------------------------------------------------------
# 12. Cleanup
# ------------------------------------------------------------------------------

gc.collect()

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 6D_OPTION4 COMPLETE — raw density + red imputed overlay saved")
print("=" * 100)

========RAN TILL THIS========

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 7 — Nuclear fraction / marker localization validation
#
# Biological question:
#   Are nuclear-associated / transcription-associated marker genes placed in
#   biologically plausible nuclear/perinuclear locations after imputation?
#
# Main comparison:
#   Raw observed
#   Learned 9E imputed / completed
#   Gene empirical imputed / completed
#   Cell-type gene empirical imputed / completed
#   Spatial-kNN empirical imputed / completed
#
# Main metric:
#   nuclear_fraction = number/probability of nuclear molecules / total molecules
#
# Same algorithm/task as Xenium Cell 7:
#   1. Aggregate nuclear indicator by gene for raw and each imputed table.
#   2. Build completed summaries = raw + imputed, without concatenating huge tables.
#   3. Compare nuclear-associated marker genes against reference genes.
#   4. Plot nuclear-fraction heatmap and delta-vs-raw heatmap.
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E / mol_gene_emp / mol_ct_gene_emp / mol_spatial_knn_emp
#   - Uses CosMx-relevant nuclear/proliferation/transcription marker set
#   - Uses p_nuclear or overlaps_nucleus from molecule tables
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("COSMX DOWNSTREAM CELL 7 — Nuclear fraction / marker localization validation")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
    "mol_gene_emp",
    "mol_ct_gene_emp",
    "mol_spatial_knn_emp",
    "shared_genes",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s) from reload/downstream cells:\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

NUCLEAR_VAL_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_nuclear_fraction_validation"
NUCLEAR_VAL_DIR.mkdir(parents=True, exist_ok=True)

NUCLEAR_GENE_SUMMARY_PATH = (
    NUCLEAR_VAL_DIR / f"{RUN_NAME}_nuclear_fraction_gene_summary.csv"
)

NUCLEAR_MARKER_SUMMARY_PATH = (
    NUCLEAR_VAL_DIR / f"{RUN_NAME}_nuclear_fraction_marker_summary.csv"
)

NUCLEAR_METHOD_SUMMARY_PATH = (
    NUCLEAR_VAL_DIR / f"{RUN_NAME}_nuclear_fraction_method_summary.csv"
)

NUCLEAR_HEATMAP_PATH = (
    NUCLEAR_VAL_DIR / f"{RUN_NAME}_nuclear_fraction_marker_heatmap.png"
)

NUCLEAR_DELTA_HEATMAP_PATH = (
    NUCLEAR_VAL_DIR / f"{RUN_NAME}_nuclear_fraction_delta_vs_raw_heatmap.png"
)

print(f"NUCLEAR_VAL_DIR: {NUCLEAR_VAL_DIR}")

# ------------------------------------------------------------------------------
# 2. Marker genes
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)

# CosMx/lung-safe nuclear-associated / transcription-associated / proliferation genes.
# These are not all strictly nuclear-localized transcripts. They are used here as
# nuclear/proliferation/transcription-associated marker genes for localization sanity checks.
candidate_nuclear_markers = [
    # proliferation / cell-cycle / nuclear-associated
    "MKI67", "TOP2A", "CCND1", "PCNA", "TYMS", "STMN1", "HMGB2",

    # transcription factors / lineage-state regulators often present in CosMx panels
    "SOX2", "SOX9", "TP63", "FOXA1", "FOXA2", "GATA3", "IRF7",
    "STAT1", "JUN", "FOS", "MYC", "E2F1",

    # immune/nuclear regulatory genes if present
    "NFKB1", "NFKBIA", "TBX21", "FOXP3", "BCL6",
]

# Broad cytoplasmic/membrane/ECM/reference genes for comparison.
candidate_non_nuclear_reference = [
    # epithelial / membrane / keratin
    "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "KRT17", "KRT13",
    "CEACAM6", "TACSTD2", "MUC1",

    # stromal / ECM
    "LUM", "DCN", "COL1A1", "COL1A2", "COL6A1", "COL6A2",
    "IGFBP7", "MMP2", "TAGLN", "ACTA2",

    # endothelial / vascular
    "PECAM1", "VWF", "RAMP2", "CAV1", "CDH5", "ENG",

    # immune / membrane/cytoplasmic
    "CD3D", "CD3E", "CD79A", "MS4A1", "CD74",
    "LYZ", "TYROBP", "FCER1G", "MARCO", "S100A8", "S100A9",
    "NKG7", "GZMB", "PRF1", "GNLY",

    # high-abundance transcript controls
    "MALAT1", "NEAT1", "B2M",
]

nuclear_marker_genes = [g for g in candidate_nuclear_markers if g in available_genes]
non_nuclear_reference_genes = [g for g in candidate_non_nuclear_reference if g in available_genes]

print(f"Nuclear-associated marker genes present: {nuclear_marker_genes}")
print(f"Reference/background genes present     : {non_nuclear_reference_genes}")

if len(nuclear_marker_genes) == 0:
    raise RuntimeError(
        "No nuclear-associated marker genes were found in shared_genes. "
        "Check the candidate_nuclear_markers list against your CosMx panel."
    )

if len(non_nuclear_reference_genes) == 0:
    raise RuntimeError(
        "No reference/background genes were found in shared_genes. "
        "Check the candidate_non_nuclear_reference list against your CosMx panel."
    )

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def get_nuclear_indicator(df):
    """
    Returns a numeric nuclear indicator for each molecule.

    Priority:
      1. p_nuclear column, if available
      2. overlaps_nucleus column, if available
      3. CellComp / compartment column, if available

    p_nuclear can be continuous or binary. For CosMx, it is often 0/1 or
    a model-estimated probability-like value.
    """

    if "p_nuclear" in df.columns:
        return (
            pd.to_numeric(df["p_nuclear"], errors="coerce")
            .fillna(0.0)
            .astype(np.float32)
        )

    if "overlaps_nucleus" in df.columns:
        return (
            pd.to_numeric(df["overlaps_nucleus"], errors="coerce")
            .fillna(0.0)
            .astype(np.float32)
        )

    # Optional CosMx compartment fallback.
    for comp_col in ["CellComp", "cell_compartment", "compartment"]:
        if comp_col in df.columns:
            comp = df[comp_col].astype(str).str.lower()
            nuclear = comp.str.contains("nuc", regex=False).astype(np.float32)
            return nuclear

    raise KeyError(
        "DataFrame has none of: 'p_nuclear', 'overlaps_nucleus', "
        "'CellComp', 'cell_compartment', or 'compartment'."
    )


def aggregate_nuclear_by_gene(df, method_name, source_type):
    """
    Aggregate nuclear statistics per gene for one molecule table.

    source_type:
      raw_observed
      learned_imputed
      baseline_imputed
      completed
    """

    required_cols = ["gene_id"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"{method_name} missing required columns: {missing}")

    temp = pd.DataFrame({
        "gene_id": df["gene_id"].astype(str).values,
        "nuclear_value": get_nuclear_indicator(df).values,
    })

    out = (
        temp
        .groupby("gene_id", observed=True)
        .agg(
            n_molecules=("nuclear_value", "size"),
            nuclear_sum=("nuclear_value", "sum"),
            nuclear_fraction=("nuclear_value", "mean"),
        )
        .reset_index()
    )

    out["method"] = method_name
    out["source_type"] = source_type

    return out


def combine_raw_and_imputed_gene_aggs(raw_agg, imp_agg, completed_method_name):
    """
    Build completed per-gene nuclear fraction without concatenating huge tables.

    completed = raw observed + imputed molecules
    """

    raw_sub = raw_agg[["gene_id", "n_molecules", "nuclear_sum"]].rename(
        columns={
            "n_molecules": "raw_n_molecules",
            "nuclear_sum": "raw_nuclear_sum",
        }
    )

    imp_sub = imp_agg[["gene_id", "n_molecules", "nuclear_sum"]].rename(
        columns={
            "n_molecules": "imputed_n_molecules",
            "nuclear_sum": "imputed_nuclear_sum",
        }
    )

    merged = raw_sub.merge(imp_sub, on="gene_id", how="outer").fillna(0)

    merged["n_molecules"] = (
        merged["raw_n_molecules"] + merged["imputed_n_molecules"]
    )

    merged["nuclear_sum"] = (
        merged["raw_nuclear_sum"] + merged["imputed_nuclear_sum"]
    )

    merged["nuclear_fraction"] = (
        merged["nuclear_sum"] / merged["n_molecules"].replace(0, np.nan)
    )

    out = merged[["gene_id", "n_molecules", "nuclear_sum", "nuclear_fraction"]].copy()
    out["method"] = completed_method_name
    out["source_type"] = "completed"

    return out


def add_marker_labels(df):
    df = df.copy()
    df["is_nuclear_marker"] = df["gene_id"].isin(nuclear_marker_genes)
    df["is_reference_gene"] = df["gene_id"].isin(non_nuclear_reference_genes)
    return df


# ------------------------------------------------------------------------------
# 4. Aggregate raw, imputed, and completed nuclear fractions
# ------------------------------------------------------------------------------

print("\nAggregating nuclear fractions by gene...")

if "status" in mol_observed.columns:
    mol_observed_raw = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ].copy()
else:
    mol_observed_raw = mol_observed.copy()

raw_gene_agg = aggregate_nuclear_by_gene(
    mol_observed_raw,
    method_name="Raw observed",
    source_type="raw_observed",
)

learned_imp_agg = aggregate_nuclear_by_gene(
    mol_learned_9E,
    method_name="Learned 9E imputed only",
    source_type="learned_imputed",
)

gene_emp_imp_agg = aggregate_nuclear_by_gene(
    mol_gene_emp,
    method_name="Gene empirical imputed only",
    source_type="baseline_imputed",
)

ct_gene_emp_imp_agg = aggregate_nuclear_by_gene(
    mol_ct_gene_emp,
    method_name="Cell-type gene empirical imputed only",
    source_type="baseline_imputed",
)

spatial_knn_imp_agg = aggregate_nuclear_by_gene(
    mol_spatial_knn_emp,
    method_name="Spatial-kNN empirical imputed only",
    source_type="baseline_imputed",
)

learned_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    learned_imp_agg,
    completed_method_name="Learned 9E completed",
)

gene_emp_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    gene_emp_imp_agg,
    completed_method_name="Gene empirical completed",
)

ct_gene_emp_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    ct_gene_emp_imp_agg,
    completed_method_name="Cell-type gene empirical completed",
)

spatial_knn_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    spatial_knn_imp_agg,
    completed_method_name="Spatial-kNN empirical completed",
)

nuclear_gene_summary_df = pd.concat(
    [
        raw_gene_agg,
        learned_imp_agg,
        gene_emp_imp_agg,
        ct_gene_emp_imp_agg,
        spatial_knn_imp_agg,
        learned_completed_agg,
        gene_emp_completed_agg,
        ct_gene_emp_completed_agg,
        spatial_knn_completed_agg,
    ],
    ignore_index=True,
)

nuclear_gene_summary_df = add_marker_labels(nuclear_gene_summary_df)

print(f"nuclear_gene_summary_df shape: {nuclear_gene_summary_df.shape}")
display(nuclear_gene_summary_df.head(20))

# ------------------------------------------------------------------------------
# 5. Marker-only table
# ------------------------------------------------------------------------------

marker_methods_order = [
    "Raw observed",
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
    "Learned 9E completed",
    "Gene empirical completed",
    "Cell-type gene empirical completed",
    "Spatial-kNN empirical completed",
]

nuclear_marker_summary_df = nuclear_gene_summary_df[
    nuclear_gene_summary_df["gene_id"].isin(nuclear_marker_genes)
].copy()

nuclear_marker_summary_df["method"] = pd.Categorical(
    nuclear_marker_summary_df["method"],
    categories=marker_methods_order,
    ordered=True,
)

nuclear_marker_summary_df = nuclear_marker_summary_df.sort_values(
    ["gene_id", "method"]
).reset_index(drop=True)

print("\nNuclear marker summary:")
display(nuclear_marker_summary_df)

# ------------------------------------------------------------------------------
# 6. Method-level summary: nuclear markers vs reference/background genes
# ------------------------------------------------------------------------------

method_summary_rows = []

for method in marker_methods_order:
    sub = nuclear_gene_summary_df[
        nuclear_gene_summary_df["method"].astype(str) == method
    ].copy()

    nuc = sub[sub["gene_id"].isin(nuclear_marker_genes)]
    ref = sub[sub["gene_id"].isin(non_nuclear_reference_genes)]

    method_summary_rows.append({
        "method": method,
        "n_nuclear_marker_genes": int(len(nuc)),
        "n_reference_genes": int(len(ref)),

        "mean_nuclear_fraction_nuclear_markers": (
            float(nuc["nuclear_fraction"].mean()) if len(nuc) else np.nan
        ),
        "median_nuclear_fraction_nuclear_markers": (
            float(nuc["nuclear_fraction"].median()) if len(nuc) else np.nan
        ),

        "mean_nuclear_fraction_reference_genes": (
            float(ref["nuclear_fraction"].mean()) if len(ref) else np.nan
        ),
        "median_nuclear_fraction_reference_genes": (
            float(ref["nuclear_fraction"].median()) if len(ref) else np.nan
        ),
    })

nuclear_method_summary_df = pd.DataFrame(method_summary_rows)

nuclear_method_summary_df["mean_marker_minus_reference"] = (
    nuclear_method_summary_df["mean_nuclear_fraction_nuclear_markers"]
    - nuclear_method_summary_df["mean_nuclear_fraction_reference_genes"]
)

nuclear_method_summary_df["median_marker_minus_reference"] = (
    nuclear_method_summary_df["median_nuclear_fraction_nuclear_markers"]
    - nuclear_method_summary_df["median_nuclear_fraction_reference_genes"]
)

# Add deltas vs raw observed.
raw_method_row = nuclear_method_summary_df[
    nuclear_method_summary_df["method"] == "Raw observed"
].iloc[0]

for col in [
    "mean_nuclear_fraction_nuclear_markers",
    "median_nuclear_fraction_nuclear_markers",
    "mean_marker_minus_reference",
    "median_marker_minus_reference",
]:
    nuclear_method_summary_df[f"delta_vs_raw_{col}"] = (
        nuclear_method_summary_df[col] - raw_method_row[col]
    )

print("\nMethod-level nuclear localization summary:")
display(nuclear_method_summary_df)

# ------------------------------------------------------------------------------
# 7. Heatmap: nuclear fraction for marker genes
# ------------------------------------------------------------------------------

heatmap_df = nuclear_marker_summary_df.pivot_table(
    index="gene_id",
    columns="method",
    values="nuclear_fraction",
    aggfunc="mean",
)

main_heatmap_methods = [
    "Raw observed",
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
    "Learned 9E completed",
]

main_heatmap_methods = [m for m in main_heatmap_methods if m in heatmap_df.columns]

heatmap_df = heatmap_df[main_heatmap_methods]

fig_w = max(10, 1.4 * len(main_heatmap_methods))
fig_h = max(4, 0.5 * len(heatmap_df.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = heatmap_df.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="viridis",
)

ax.set_title("CosMx nuclear fraction of nuclear-associated marker genes", fontsize=14, pad=14)
ax.set_xlabel("Dataset / method")
ax.set_ylabel("Marker gene")

ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index, fontsize=10)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if val > 0.55 else "black",
            )

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Nuclear fraction", fontsize=11)

plt.tight_layout()
plt.savefig(NUCLEAR_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved nuclear fraction heatmap:")
print(f"  {NUCLEAR_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 8. Delta heatmap: method - raw observed
# ------------------------------------------------------------------------------

raw_values = heatmap_df["Raw observed"] if "Raw observed" in heatmap_df.columns else None

if raw_values is not None:
    delta_heatmap_df = heatmap_df.copy()

    for col in delta_heatmap_df.columns:
        delta_heatmap_df[col] = delta_heatmap_df[col] - raw_values

    if "Raw observed" in delta_heatmap_df.columns:
        delta_heatmap_df = delta_heatmap_df.drop(columns=["Raw observed"])

    fig_w = max(10, 1.4 * len(delta_heatmap_df.columns))
    fig_h = max(4, 0.5 * len(delta_heatmap_df.index) + 2)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    arr = delta_heatmap_df.values.astype(float)
    finite_vals = arr[np.isfinite(arr)]

    vmax = np.nanpercentile(np.abs(finite_vals), 95) if len(finite_vals) else 0.2
    vmax = max(float(vmax), 0.05)

    im = ax.imshow(
        arr,
        aspect="auto",
        vmin=-vmax,
        vmax=vmax,
        cmap="coolwarm",
    )

    ax.set_title("CosMx change in nuclear fraction vs raw observed", fontsize=14, pad=14)
    ax.set_xlabel("Dataset / method")
    ax.set_ylabel("Marker gene")

    ax.set_xticks(np.arange(len(delta_heatmap_df.columns)))
    ax.set_xticklabels(delta_heatmap_df.columns, rotation=45, ha="right", fontsize=9)

    ax.set_yticks(np.arange(len(delta_heatmap_df.index)))
    ax.set_yticklabels(delta_heatmap_df.index, fontsize=10)

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = arr[i, j]
            if np.isfinite(val):
                ax.text(
                    j,
                    i,
                    f"{val:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black",
                )

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Δ nuclear fraction vs raw", fontsize=11)

    plt.tight_layout()
    plt.savefig(NUCLEAR_DELTA_HEATMAP_PATH, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved nuclear delta heatmap:")
    print(f"  {NUCLEAR_DELTA_HEATMAP_PATH}")

else:
    print("Raw observed column not found in heatmap_df; skipping delta heatmap.")

# ------------------------------------------------------------------------------
# 9. Save tables
# ------------------------------------------------------------------------------

nuclear_gene_summary_df.to_csv(NUCLEAR_GENE_SUMMARY_PATH, index=False)
nuclear_marker_summary_df.to_csv(NUCLEAR_MARKER_SUMMARY_PATH, index=False)
nuclear_method_summary_df.to_csv(NUCLEAR_METHOD_SUMMARY_PATH, index=False)

print("\nSaved nuclear localization validation outputs:")
print(f"  Gene summary   : {NUCLEAR_GENE_SUMMARY_PATH}")
print(f"  Marker summary : {NUCLEAR_MARKER_SUMMARY_PATH}")
print(f"  Method summary : {NUCLEAR_METHOD_SUMMARY_PATH}")
print(f"  Heatmap        : {NUCLEAR_HEATMAP_PATH}")
print(f"  Delta heatmap  : {NUCLEAR_DELTA_HEATMAP_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: nuclear-associated markers have higher nuclear_fraction than reference genes.")
print("  Good sign 2: Learned 9E imputed/completed nuclear fractions are biologically plausible.")
print("  Good sign 3: Learned 9E is not blindly more nuclear for every gene; check marker-specific behavior.")
print("  Caution: very high nuclear fraction for all genes may indicate over-central placement.")
print("  CosMx note: this is compartment/probability-based nuclear validation, not exact polygon validation.")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 7 COMPLETE — Nuclear fraction / marker localization validation")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 8 — Radial distribution comparison for marker genes
#
# Biological question:
#   Does learned 9E capture subcellular radial localization better than
#   empirical baselines?
#
# Main comparison:
#   Learned 9E imputed-only radial distribution
#   Gene empirical imputed-only radial distribution
#   Cell-type gene empirical imputed-only radial distribution
#   Spatial-kNN empirical imputed-only radial distribution
#
# Reference:
#   Raw observed radial distribution for the same gene.
#
# Main metrics:
#   mean r_norm
#   median r_norm
#   central fraction: r_norm <= 0.35
#   peripheral fraction: r_norm >= 0.75
#   Wasserstein distance to raw observed distribution
#   KS statistic to raw observed distribution
#
# Same algorithm/configs as original Cell 8.
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E / mol_gene_emp / mol_ct_gene_emp / mol_spatial_knn_emp
#   - Keeps CosMx molecule IDs as strings where relevant
#   - Uses CosMx-relevant marker genes
#   - Uses r_norm from Step5 molecule tables
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wasserstein_distance, ks_2samp

print("=" * 100)
print("COSMX DOWNSTREAM CELL 8 — Radial distribution comparison for marker genes")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
    "mol_gene_emp",
    "mol_ct_gene_emp",
    "mol_spatial_knn_emp",
    "shared_genes",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s) from reload/downstream cells:\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

RADIAL_VAL_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_radial_distribution_validation"
RADIAL_VAL_DIR.mkdir(parents=True, exist_ok=True)

RADIAL_GENE_SUMMARY_PATH = (
    RADIAL_VAL_DIR / f"{RUN_NAME}_radial_distribution_gene_summary.csv"
)

RADIAL_DISTANCE_SUMMARY_PATH = (
    RADIAL_VAL_DIR / f"{RUN_NAME}_radial_distribution_distance_to_raw.csv"
)

RADIAL_METHOD_SUMMARY_PATH = (
    RADIAL_VAL_DIR / f"{RUN_NAME}_radial_distribution_method_summary.csv"
)

RADIAL_BEST_COUNTS_PATH = (
    RADIAL_VAL_DIR / f"{RUN_NAME}_radial_distribution_best_method_counts.csv"
)

RADIAL_WASSERSTEIN_HEATMAP_PATH = (
    RADIAL_VAL_DIR / f"{RUN_NAME}_radial_wasserstein_to_raw_heatmap.png"
)

RADIAL_MEAN_HEATMAP_PATH = (
    RADIAL_VAL_DIR / f"{RUN_NAME}_mean_r_norm_marker_heatmap.png"
)

RADIAL_HIST_DIR = RADIAL_VAL_DIR / "marker_radial_histograms"
RADIAL_HIST_DIR.mkdir(parents=True, exist_ok=True)

print(f"RADIAL_VAL_DIR: {RADIAL_VAL_DIR}")

# ------------------------------------------------------------------------------
# 2. Marker genes
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)

# CosMx/lung/NSCLC-relevant marker genes.
# Includes nuclear/proliferation, epithelial, stromal, immune, endothelial, and mast genes.
candidate_marker_genes = [
    # Nuclear-associated / proliferation / transcription-state markers
    "MKI67", "TOP2A", "CCND1", "PCNA", "TYMS", "STMN1", "HMGB2",
    "SOX2", "SOX9", "GATA3", "STAT1", "JUN", "FOS", "MYC",
    "NFKB1", "NFKBIA", "TBX21", "FOXP3",

    # Epithelial / tumor-like
    "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "KRT17", "KRT13",
    "CEACAM6", "TACSTD2", "MUC1",

    # Stromal / ECM
    "LUM", "DCN", "COL1A1", "COL1A2", "COL6A1", "COL6A2",
    "IGFBP7", "MMP2", "TAGLN", "ACTA2",

    # Immune / macrophage / lymphoid
    "CD3D", "CD3E", "CD79A", "MS4A1", "CD74",
    "LYZ", "TYROBP", "FCER1G", "MARCO", "S100A8", "S100A9",
    "NKG7", "GZMB", "PRF1", "GNLY",

    # Endothelial / vascular
    "PECAM1", "VWF", "RAMP2", "CAV1", "CDH5", "ENG",

    # Mast
    "TPSAB1", "TPSB2", "CPA3", "HPGDS",

    # High-abundance transcript controls
    "MALAT1", "NEAT1", "B2M",
]

marker_genes = [g for g in candidate_marker_genes if g in available_genes]

print(f"Marker genes present for radial validation ({len(marker_genes)}):")
print(marker_genes)

if len(marker_genes) == 0:
    raise RuntimeError("No marker genes found in shared_genes.")

# To avoid overly crowded figures, make histograms for these priority genes.
priority_hist_genes = [
    g for g in [
        "MKI67", "TOP2A", "CCND1", "STMN1",
        "EPCAM", "KRT7", "KRT8", "KRT18",
        "LUM", "DCN", "COL1A1",
        "CD3D", "CD79A", "LYZ", "MARCO",
        "NKG7", "GZMB",
        "PECAM1", "VWF", "CAV1",
        "TPSAB1", "HPGDS",
        "MALAT1", "NEAT1",
    ]
    if g in marker_genes
]

MAX_HIST_GENES = 12
priority_hist_genes = priority_hist_genes[:MAX_HIST_GENES]

print(f"Priority genes for radial histogram plots ({len(priority_hist_genes)}):")
print(priority_hist_genes)

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def clean_rnorm_table(df, method_name, source_type, genes):
    """
    Keep gene_id and r_norm for selected genes only.
    """
    if "gene_id" not in df.columns:
        raise KeyError(f"{method_name} is missing gene_id column.")
    if "r_norm" not in df.columns:
        raise KeyError(f"{method_name} is missing r_norm column.")

    out = df[["gene_id", "r_norm"]].copy()
    out["gene_id"] = out["gene_id"].astype(str)
    out = out[out["gene_id"].isin(genes)].copy()

    out["r_norm"] = pd.to_numeric(out["r_norm"], errors="coerce")
    out = out.dropna(subset=["r_norm"])

    # r_norm should be in [0, 1]. Clip small numeric errors.
    out["r_norm"] = out["r_norm"].clip(0, 1).astype(np.float32)

    out["method"] = method_name
    out["source_type"] = source_type

    return out


def summarize_rnorm_by_gene(df):
    """
    Summarize radial distribution per method/gene.
    """
    rows = []

    for (method, source_type, gene), sub in df.groupby(
        ["method", "source_type", "gene_id"],
        observed=True,
    ):
        vals = sub["r_norm"].to_numpy(dtype=np.float32)

        if len(vals) == 0:
            continue

        rows.append({
            "method": method,
            "source_type": source_type,
            "gene_id": gene,
            "n_molecules": int(len(vals)),
            "mean_r_norm": float(np.mean(vals)),
            "median_r_norm": float(np.median(vals)),
            "std_r_norm": float(np.std(vals)),
            "q25_r_norm": float(np.quantile(vals, 0.25)),
            "q75_r_norm": float(np.quantile(vals, 0.75)),
            "central_fraction_r_le_0_35": float(np.mean(vals <= 0.35)),
            "mid_fraction_0_35_lt_r_lt_0_75": float(np.mean((vals > 0.35) & (vals < 0.75))),
            "peripheral_fraction_r_ge_0_75": float(np.mean(vals >= 0.75)),
        })

    return pd.DataFrame(rows)


def compute_distance_to_raw(all_rnorm_df):
    """
    Compare each method/gene radial distribution against raw observed distribution
    for the same gene.
    """
    rows = []

    raw_df = all_rnorm_df[all_rnorm_df["method"] == "Raw observed"].copy()

    method_names = [
        "Learned 9E imputed only",
        "Gene empirical imputed only",
        "Cell-type gene empirical imputed only",
        "Spatial-kNN empirical imputed only",
    ]

    for gene in marker_genes:
        raw_vals = raw_df[raw_df["gene_id"] == gene]["r_norm"].to_numpy(dtype=np.float32)

        if len(raw_vals) < 5:
            continue

        for method in method_names:
            vals = all_rnorm_df[
                (all_rnorm_df["method"] == method)
                & (all_rnorm_df["gene_id"] == gene)
            ]["r_norm"].to_numpy(dtype=np.float32)

            if len(vals) < 5:
                rows.append({
                    "gene_id": gene,
                    "method": method,
                    "n_raw": int(len(raw_vals)),
                    "n_method": int(len(vals)),
                    "wasserstein_to_raw": np.nan,
                    "ks_stat_to_raw": np.nan,
                    "ks_pvalue_to_raw": np.nan,
                    "mean_r_norm_raw": float(np.mean(raw_vals)),
                    "mean_r_norm_method": np.nan,
                    "delta_mean_r_norm_method_minus_raw": np.nan,
                })
                continue

            ks = ks_2samp(raw_vals, vals)

            rows.append({
                "gene_id": gene,
                "method": method,
                "n_raw": int(len(raw_vals)),
                "n_method": int(len(vals)),
                "wasserstein_to_raw": float(wasserstein_distance(raw_vals, vals)),
                "ks_stat_to_raw": float(ks.statistic),
                "ks_pvalue_to_raw": float(ks.pvalue),
                "mean_r_norm_raw": float(np.mean(raw_vals)),
                "mean_r_norm_method": float(np.mean(vals)),
                "delta_mean_r_norm_method_minus_raw": float(np.mean(vals) - np.mean(raw_vals)),
            })

    return pd.DataFrame(rows)


def plot_radial_histograms_for_gene(all_rnorm_df, gene, out_path):
    """
    Plot radial histograms for one gene:
      raw observed vs learned 9E vs baselines.
    """
    methods = [
        "Raw observed",
        "Learned 9E imputed only",
        "Gene empirical imputed only",
        "Cell-type gene empirical imputed only",
        "Spatial-kNN empirical imputed only",
    ]

    bins = np.linspace(0, 1, 31)

    fig, ax = plt.subplots(figsize=(9, 5))

    for method in methods:
        vals = all_rnorm_df[
            (all_rnorm_df["method"] == method)
            & (all_rnorm_df["gene_id"] == gene)
        ]["r_norm"].to_numpy(dtype=np.float32)

        if len(vals) == 0:
            continue

        ax.hist(
            vals,
            bins=bins,
            density=True,
            histtype="step",
            linewidth=1.8,
            label=f"{method} (n={len(vals):,})",
        )

    ax.set_title(f"CosMx radial distribution for marker gene {gene}", fontsize=14)
    ax.set_xlabel("r_norm: 0 = nuclear/center, 1 = cell edge")
    ax.set_ylabel("Density")
    ax.set_xlim(0, 1)
    ax.grid(True, linewidth=0.3, alpha=0.4)
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved radial histogram for {gene}: {out_path}")


# ------------------------------------------------------------------------------
# 4. Build radial tables
# ------------------------------------------------------------------------------

print("\nBuilding radial distribution tables...")

if "status" in mol_observed.columns:
    mol_observed_raw = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ].copy()
else:
    mol_observed_raw = mol_observed.copy()

raw_rnorm = clean_rnorm_table(
    mol_observed_raw,
    method_name="Raw observed",
    source_type="raw_observed",
    genes=marker_genes,
)

learned_rnorm = clean_rnorm_table(
    mol_learned_9E,
    method_name="Learned 9E imputed only",
    source_type="learned_imputed",
    genes=marker_genes,
)

gene_emp_rnorm = clean_rnorm_table(
    mol_gene_emp,
    method_name="Gene empirical imputed only",
    source_type="baseline_imputed",
    genes=marker_genes,
)

ct_gene_emp_rnorm = clean_rnorm_table(
    mol_ct_gene_emp,
    method_name="Cell-type gene empirical imputed only",
    source_type="baseline_imputed",
    genes=marker_genes,
)

spatial_knn_rnorm = clean_rnorm_table(
    mol_spatial_knn_emp,
    method_name="Spatial-kNN empirical imputed only",
    source_type="baseline_imputed",
    genes=marker_genes,
)

all_rnorm_df = pd.concat(
    [
        raw_rnorm,
        learned_rnorm,
        gene_emp_rnorm,
        ct_gene_emp_rnorm,
        spatial_knn_rnorm,
    ],
    ignore_index=True,
)

print(f"all_rnorm_df shape: {all_rnorm_df.shape}")
print("Molecules per method:")
display(
    all_rnorm_df
    .groupby("method", observed=True)
    .size()
    .reset_index(name="n_marker_molecules")
)

# ------------------------------------------------------------------------------
# 5. Per-gene radial summaries
# ------------------------------------------------------------------------------

radial_gene_summary_df = summarize_rnorm_by_gene(all_rnorm_df)

print("\nRadial gene summary:")
display(radial_gene_summary_df.head(30))

# ------------------------------------------------------------------------------
# 6. Distance to raw observed distribution
# ------------------------------------------------------------------------------

radial_distance_df = compute_distance_to_raw(all_rnorm_df)

print("\nDistance of imputed radial distributions to raw observed distribution:")
display(radial_distance_df.head(30))

# Method-level summary: lower distance is better.
radial_method_summary_df = (
    radial_distance_df
    .groupby("method", observed=True)
    .agg(
        n_genes_evaluated=("gene_id", "nunique"),
        mean_wasserstein_to_raw=("wasserstein_to_raw", "mean"),
        median_wasserstein_to_raw=("wasserstein_to_raw", "median"),
        mean_ks_stat_to_raw=("ks_stat_to_raw", "mean"),
        median_ks_stat_to_raw=("ks_stat_to_raw", "median"),
        mean_abs_delta_mean_r_norm=(
            "delta_mean_r_norm_method_minus_raw",
            lambda x: float(np.nanmean(np.abs(x))),
        ),
        median_abs_delta_mean_r_norm=(
            "delta_mean_r_norm_method_minus_raw",
            lambda x: float(np.nanmedian(np.abs(x))),
        ),
    )
    .reset_index()
)

radial_method_summary_df = radial_method_summary_df.sort_values(
    "mean_wasserstein_to_raw",
    ascending=True,
).reset_index(drop=True)

print("\nMethod-level radial distribution summary:")
display(radial_method_summary_df)

# ------------------------------------------------------------------------------
# 7. Ranking: which method is closest to raw for each gene?
# ------------------------------------------------------------------------------

rank_df = radial_distance_df.copy()

rank_df["wasserstein_rank_for_gene"] = (
    rank_df
    .groupby("gene_id")["wasserstein_to_raw"]
    .rank(method="min", ascending=True)
)

rank_df["is_best_for_gene"] = rank_df["wasserstein_rank_for_gene"] == 1

best_counts_df = (
    rank_df
    .groupby("method", observed=True)
    .agg(
        n_best_genes=("is_best_for_gene", "sum"),
        mean_rank=("wasserstein_rank_for_gene", "mean"),
    )
    .reset_index()
    .sort_values(["n_best_genes", "mean_rank"], ascending=[False, True])
)

print("\nBest method counts by gene based on Wasserstein distance to raw:")
display(best_counts_df)

# ------------------------------------------------------------------------------
# 8. Heatmap: Wasserstein distance to raw observed
# ------------------------------------------------------------------------------

wasserstein_matrix = radial_distance_df.pivot_table(
    index="gene_id",
    columns="method",
    values="wasserstein_to_raw",
    aggfunc="mean",
)

method_order = [
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
]

method_order = [m for m in method_order if m in wasserstein_matrix.columns]
wasserstein_matrix = wasserstein_matrix[method_order]

# Order genes by learned 9E distance if available.
if "Learned 9E imputed only" in wasserstein_matrix.columns:
    gene_order = wasserstein_matrix["Learned 9E imputed only"].sort_values().index
    wasserstein_matrix = wasserstein_matrix.loc[gene_order]

fig_w = max(9, 1.8 * len(method_order))
fig_h = max(6, 0.35 * len(wasserstein_matrix.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = wasserstein_matrix.values.astype(float)
vmax = np.nanpercentile(arr, 95) if np.isfinite(arr).any() else 0.2
vmax = max(float(vmax), 0.05)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=0,
    vmax=vmax,
    cmap="magma_r",
)

ax.set_title(
    "CosMx radial Wasserstein distance to raw observed distribution\nLower is better",
    fontsize=14,
    pad=14,
)
ax.set_xlabel("Imputation method")
ax.set_ylabel("Marker gene")

ax.set_xticks(np.arange(len(wasserstein_matrix.columns)))
ax.set_xticklabels(wasserstein_matrix.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(wasserstein_matrix.index)))
ax.set_yticklabels(wasserstein_matrix.index, fontsize=9)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:.3f}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if val > 0.5 * vmax else "black",
            )

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Wasserstein distance to raw", fontsize=11)

plt.tight_layout()
plt.savefig(RADIAL_WASSERSTEIN_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved radial Wasserstein heatmap:")
print(f"  {RADIAL_WASSERSTEIN_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 9. Heatmap: mean r_norm by method
# ------------------------------------------------------------------------------

mean_r_matrix = radial_gene_summary_df.pivot_table(
    index="gene_id",
    columns="method",
    values="mean_r_norm",
    aggfunc="mean",
)

mean_r_method_order = [
    "Raw observed",
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
]

mean_r_method_order = [m for m in mean_r_method_order if m in mean_r_matrix.columns]
mean_r_matrix = mean_r_matrix[mean_r_method_order]

if "Raw observed" in mean_r_matrix.columns:
    gene_order = mean_r_matrix["Raw observed"].sort_values().index
    mean_r_matrix = mean_r_matrix.loc[gene_order]

fig_w = max(10, 1.6 * len(mean_r_method_order))
fig_h = max(6, 0.35 * len(mean_r_matrix.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = mean_r_matrix.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="viridis",
)

ax.set_title(
    "CosMx mean radial position of marker genes\n0 = nuclear/central, 1 = cell edge",
    fontsize=14,
    pad=14,
)
ax.set_xlabel("Dataset / method")
ax.set_ylabel("Marker gene")

ax.set_xticks(np.arange(len(mean_r_matrix.columns)))
ax.set_xticklabels(mean_r_matrix.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(mean_r_matrix.index)))
ax.set_yticklabels(mean_r_matrix.index, fontsize=9)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if val > 0.55 else "black",
            )

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Mean r_norm", fontsize=11)

plt.tight_layout()
plt.savefig(RADIAL_MEAN_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved mean r_norm heatmap:")
print(f"  {RADIAL_MEAN_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 10. Histogram plots for selected marker genes
# ------------------------------------------------------------------------------

print("\nGenerating radial histogram plots for priority genes...")

hist_records = []

for gene in priority_hist_genes:
    out_path = RADIAL_HIST_DIR / f"{RUN_NAME}_radial_histogram_{gene}.png"

    plot_radial_histograms_for_gene(
        all_rnorm_df=all_rnorm_df,
        gene=gene,
        out_path=out_path,
    )

    hist_records.append({
        "gene_id": gene,
        "histogram_path": str(out_path),
    })

radial_hist_manifest_df = pd.DataFrame(hist_records)

RADIAL_HIST_MANIFEST_PATH = (
    RADIAL_VAL_DIR / f"{RUN_NAME}_radial_histogram_manifest.csv"
)

radial_hist_manifest_df.to_csv(RADIAL_HIST_MANIFEST_PATH, index=False)

# ------------------------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------------------------

radial_gene_summary_df.to_csv(RADIAL_GENE_SUMMARY_PATH, index=False)
radial_distance_df.to_csv(RADIAL_DISTANCE_SUMMARY_PATH, index=False)
radial_method_summary_df.to_csv(RADIAL_METHOD_SUMMARY_PATH, index=False)
best_counts_df.to_csv(RADIAL_BEST_COUNTS_PATH, index=False)

print("\nSaved radial distribution validation outputs:")
print(f"  Gene summary           : {RADIAL_GENE_SUMMARY_PATH}")
print(f"  Distance to raw        : {RADIAL_DISTANCE_SUMMARY_PATH}")
print(f"  Method summary         : {RADIAL_METHOD_SUMMARY_PATH}")
print(f"  Best method counts     : {RADIAL_BEST_COUNTS_PATH}")
print(f"  Wasserstein heatmap    : {RADIAL_WASSERSTEIN_HEATMAP_PATH}")
print(f"  Mean r_norm heatmap    : {RADIAL_MEAN_HEATMAP_PATH}")
print(f"  Histogram manifest     : {RADIAL_HIST_MANIFEST_PATH}")
print(f"  Histogram directory    : {RADIAL_HIST_DIR}")

print("\nInterpretation guide:")
print("  Good sign 1: Learned 9E has lower mean/median Wasserstein distance to raw than baselines.")
print("  Good sign 2: Learned 9E mean r_norm is close to raw marker-specific mean r_norm.")
print("  Good sign 3: For nuclear markers, learned 9E should not be pushed too peripheral.")
print("  Good sign 4: For membrane/ECM markers, learned 9E should not collapse everything to the nucleus.")
print("  Caution: Raw observed molecules are sparse/noisy, so use this with held-out recovery and nuclear validation.")
print("  CosMx note: r_norm is based on Step5 centroid/radius-normalized CosMx geometry, not exact Xenium polygons.")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 8 COMPLETE — Radial distribution comparison for marker genes")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 9 — Per-gene / per-cell-type localization summaries
#
# Biological question:
#   Which genes and cell types show meaningful localization patterns after
#   learned 9E imputation, and how do they compare with empirical baselines?
#
# Main comparison:
#   Raw observed
#   Learned 9E imputed / completed
#   Gene empirical imputed / completed
#   Cell-type gene empirical imputed / completed
#   Spatial-kNN empirical imputed / completed
#
# Main outputs:
#   Per (gene, cell type, method) summaries of:
#     - number of molecules
#     - mean r_norm
#     - nuclear fraction
#     - mean z_rel
#     - central/peripheral fractions
#     - imputed fraction for completed tables
#
# Same algorithm/task as Xenium Cell 9.
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses mol_learned_9E / mol_gene_emp / mol_ct_gene_emp / mol_spatial_knn_emp
#   - Uses CosMx-relevant marker genes
#   - Accepts CosMx cell-type columns
#   - Keeps cell IDs as strings where needed
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("COSMX DOWNSTREAM CELL 9 — Per-gene / per-cell-type localization summaries")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
    "mol_gene_emp",
    "mol_ct_gene_emp",
    "mol_spatial_knn_emp",
    "shared_genes",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s) from reload/downstream cells:\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

LOC_SUMMARY_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_per_gene_celltype_localization_summary"
LOC_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

LOC_IMPUTED_ONLY_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_localization_summary_imputed_only_by_gene_celltype.csv"
)

LOC_COMPLETED_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_localization_summary_completed_by_gene_celltype.csv"
)

LOC_DELTA_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_localization_delta_learned_vs_baselines.csv"
)

LOC_METHOD_SUMMARY_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_localization_method_summary.csv"
)

LOC_NUC_HEATMAP_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_learned_completed_nuclear_fraction_gene_celltype_heatmap.png"
)

LOC_R_HEATMAP_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_learned_completed_mean_r_norm_gene_celltype_heatmap.png"
)

LOC_HEATMAP_NUC_MATRIX_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_learned_completed_nuclear_fraction_gene_celltype_matrix.csv"
)

LOC_HEATMAP_R_MATRIX_PATH = (
    LOC_SUMMARY_DIR / f"{RUN_NAME}_learned_completed_mean_r_norm_gene_celltype_matrix.csv"
)

print(f"LOC_SUMMARY_DIR: {LOC_SUMMARY_DIR}")

# ------------------------------------------------------------------------------
# 2. CosMx marker genes and helper functions
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)

candidate_summary_genes = [
    # Nuclear / proliferation / transcription-state markers
    "MKI67", "TOP2A", "CCND1", "PCNA", "TYMS", "STMN1", "HMGB2",
    "SOX2", "SOX9", "GATA3", "STAT1", "JUN", "FOS", "MYC",
    "NFKB1", "NFKBIA", "TBX21", "FOXP3",

    # Epithelial / tumor-like
    "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "KRT17", "KRT13",
    "CEACAM6", "TACSTD2", "MUC1",

    # Stromal / ECM
    "LUM", "DCN", "COL1A1", "COL1A2", "COL6A1", "COL6A2",
    "IGFBP7", "MMP2", "TAGLN", "ACTA2",

    # Immune / lymphoid / myeloid
    "CD3D", "CD3E", "CD8A", "CD8B",
    "MS4A1", "CD79A", "CD79B", "CD74",
    "LYZ", "LST1", "TYROBP", "FCER1G", "MARCO",
    "S100A8", "S100A9", "CD68", "CD163",
    "NKG7", "GZMB", "PRF1", "GNLY",

    # Endothelial / vascular
    "PECAM1", "VWF", "CAV1", "RAMP2", "CDH5", "ENG",

    # Mast
    "TPSAB1", "TPSB2", "CPA3", "HPGDS",

    # High-abundance localization controls
    "MALAT1", "NEAT1", "B2M",
]

summary_genes = [g for g in candidate_summary_genes if g in available_genes]

if len(summary_genes) == 0:
    print("WARNING: none of the candidate genes were found. Using all shared genes.")
    summary_genes = [str(g) for g in shared_genes]

print(f"Genes used for focused heatmaps/summaries: {len(summary_genes)}")
print(summary_genes[:80])


def get_celltype_col(df):
    """
    Find cell-type column in a CosMx molecule table.
    """
    candidates = [
        "cell_type",
        "Final_CosMx_Cell_Type",
        "Assigned_CosMx_Cell_Type",
        "Assigned_Xenium_Cell_Type",
        "Reference_Cell_Type",
        "ct_label",
        "celltype",
        "Cell_Type",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError(
        "Could not find a cell-type column. Expected one of: "
        f"{candidates}. Existing columns: {list(df.columns)}"
    )


def numeric_col_or_nan(df, col):
    """
    Return numeric column if present, otherwise all NaN.
    """
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").astype(np.float32)

    return pd.Series(np.nan, index=df.index, dtype=np.float32)


def nuclear_indicator(df):
    """
    Use p_nuclear if present; otherwise overlaps_nucleus; otherwise CosMx compartment.
    """
    if "p_nuclear" in df.columns:
        return (
            pd.to_numeric(df["p_nuclear"], errors="coerce")
            .fillna(0)
            .astype(np.float32)
        )

    if "overlaps_nucleus" in df.columns:
        return (
            pd.to_numeric(df["overlaps_nucleus"], errors="coerce")
            .fillna(0)
            .astype(np.float32)
        )

    for comp_col in ["CellComp", "cell_compartment", "compartment"]:
        if comp_col in df.columns:
            comp = df[comp_col].astype(str).str.lower()
            return comp.str.contains("nuc", regex=False).astype(np.float32)

    return pd.Series(np.nan, index=df.index, dtype=np.float32)


# ------------------------------------------------------------------------------
# 3. Aggregation functions
# ------------------------------------------------------------------------------

def aggregate_localization(df, method, source_type, genes=None):
    """
    Aggregate localization statistics per gene and cell type.

    source_type examples:
      raw_observed
      learned_imputed
      baseline_imputed
    """
    if "gene_id" not in df.columns:
        raise KeyError(f"{method} missing gene_id column.")

    if genes is not None:
        d = df[df["gene_id"].astype(str).isin(genes)].copy()
    else:
        d = df.copy()

    if len(d) == 0:
        print(f"WARNING: no molecules found for {method} after gene filtering.")
        return pd.DataFrame(columns=[
            "gene_id",
            "cell_type",
            "n_molecules",
            "mean_r_norm",
            "median_r_norm",
            "std_r_norm",
            "mean_z_rel",
            "median_z_rel",
            "std_z_rel",
            "nuclear_fraction",
            "central_fraction_r_le_0_35",
            "peripheral_fraction_r_ge_0_75",
            "method",
            "source_type",
        ])

    ct_col = get_celltype_col(d)

    temp = pd.DataFrame({
        "gene_id": d["gene_id"].astype(str).values,
        "cell_type": d[ct_col].astype(str).values,
        "r_norm": numeric_col_or_nan(d, "r_norm").values,
        "z_rel": numeric_col_or_nan(d, "z_rel").values,
        "p_nuclear": nuclear_indicator(d).values,
    })

    temp["central"] = temp["r_norm"] <= 0.35
    temp["peripheral"] = temp["r_norm"] >= 0.75

    out = (
        temp
        .groupby(["gene_id", "cell_type"], observed=True)
        .agg(
            n_molecules=("gene_id", "size"),

            mean_r_norm=("r_norm", "mean"),
            median_r_norm=("r_norm", "median"),
            std_r_norm=("r_norm", "std"),

            mean_z_rel=("z_rel", "mean"),
            median_z_rel=("z_rel", "median"),
            std_z_rel=("z_rel", "std"),

            nuclear_fraction=("p_nuclear", "mean"),
            central_fraction_r_le_0_35=("central", "mean"),
            peripheral_fraction_r_ge_0_75=("peripheral", "mean"),
        )
        .reset_index()
    )

    out["method"] = method
    out["source_type"] = source_type

    return out


def combine_raw_and_imputed_localization(raw_agg, imp_agg, completed_method):
    """
    Combine raw and imputed aggregate tables into completed aggregate table
    using weighted means.

    This avoids concatenating huge raw + imputed molecule tables.
    """
    keys = ["gene_id", "cell_type"]

    raw = raw_agg.copy()
    imp = imp_agg.copy()

    raw = raw.rename(columns={c: f"raw_{c}" for c in raw.columns if c not in keys})
    imp = imp.rename(columns={c: f"imp_{c}" for c in imp.columns if c not in keys})

    merged = raw.merge(imp, on=keys, how="outer")

    count_cols = ["raw_n_molecules", "imp_n_molecules"]

    for c in count_cols:
        if c not in merged.columns:
            merged[c] = 0
        merged[c] = merged[c].fillna(0).astype(float)

    raw_n = merged["raw_n_molecules"].to_numpy(dtype=float)
    imp_n = merged["imp_n_molecules"].to_numpy(dtype=float)
    total_n = raw_n + imp_n

    out = merged[keys].copy()

    out["n_molecules"] = total_n.astype(int)
    out["n_raw_molecules"] = raw_n.astype(int)
    out["n_imputed_molecules"] = imp_n.astype(int)
    out["imputed_fraction"] = np.divide(
        imp_n,
        np.maximum(total_n, 1),
    )

    weighted_metrics = [
        "mean_r_norm",
        "mean_z_rel",
        "nuclear_fraction",
        "central_fraction_r_le_0_35",
        "peripheral_fraction_r_ge_0_75",
    ]

    for m in weighted_metrics:
        raw_col = f"raw_{m}"
        imp_col = f"imp_{m}"

        raw_vals = (
            merged[raw_col].to_numpy(dtype=float)
            if raw_col in merged.columns
            else np.full(len(merged), np.nan)
        )

        imp_vals = (
            merged[imp_col].to_numpy(dtype=float)
            if imp_col in merged.columns
            else np.full(len(merged), np.nan)
        )

        raw_vals = np.nan_to_num(raw_vals, nan=0.0)
        imp_vals = np.nan_to_num(imp_vals, nan=0.0)

        out[m] = np.divide(
            raw_vals * raw_n + imp_vals * imp_n,
            np.maximum(total_n, 1),
        )

    # Median/std cannot be combined exactly from aggregate summaries.
    out["median_r_norm"] = np.nan
    out["std_r_norm"] = np.nan
    out["median_z_rel"] = np.nan
    out["std_z_rel"] = np.nan

    out["method"] = completed_method
    out["source_type"] = "completed"

    return out


# ------------------------------------------------------------------------------
# 4. Build imputed-only and completed summaries
# ------------------------------------------------------------------------------

print("\nAggregating raw and imputed localization summaries...")

if "status" in mol_observed.columns:
    raw_obs = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ].copy()
else:
    raw_obs = mol_observed.copy()

raw_agg = aggregate_localization(
    raw_obs,
    method="Raw observed",
    source_type="raw_observed",
    genes=summary_genes,
)

learned_imp_agg = aggregate_localization(
    mol_learned_9E,
    method="Learned 9E imputed only",
    source_type="learned_imputed",
    genes=summary_genes,
)

gene_emp_imp_agg = aggregate_localization(
    mol_gene_emp,
    method="Gene empirical imputed only",
    source_type="baseline_imputed",
    genes=summary_genes,
)

ct_gene_emp_imp_agg = aggregate_localization(
    mol_ct_gene_emp,
    method="Cell-type gene empirical imputed only",
    source_type="baseline_imputed",
    genes=summary_genes,
)

spatial_knn_imp_agg = aggregate_localization(
    mol_spatial_knn_emp,
    method="Spatial-kNN empirical imputed only",
    source_type="baseline_imputed",
    genes=summary_genes,
)

imputed_only_summary_df = pd.concat(
    [
        learned_imp_agg,
        gene_emp_imp_agg,
        ct_gene_emp_imp_agg,
        spatial_knn_imp_agg,
    ],
    ignore_index=True,
)

learned_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    learned_imp_agg,
    completed_method="Learned 9E completed",
)

gene_emp_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    gene_emp_imp_agg,
    completed_method="Gene empirical completed",
)

ct_gene_emp_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    ct_gene_emp_imp_agg,
    completed_method="Cell-type gene empirical completed",
)

spatial_knn_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    spatial_knn_imp_agg,
    completed_method="Spatial-kNN empirical completed",
)

completed_summary_df = pd.concat(
    [
        raw_agg,
        learned_completed_agg,
        gene_emp_completed_agg,
        ct_gene_emp_completed_agg,
        spatial_knn_completed_agg,
    ],
    ignore_index=True,
)

print(f"Raw summary rows          : {len(raw_agg):,}")
print(f"Imputed-only summary rows : {len(imputed_only_summary_df):,}")
print(f"Completed summary rows    : {len(completed_summary_df):,}")

print("\nPreview completed summary:")
display(completed_summary_df.head(20))

# ------------------------------------------------------------------------------
# 5. Learned 9E vs empirical baselines deltas
# ------------------------------------------------------------------------------

print("\nComputing learned 9E vs baseline localization deltas...")

keys = ["gene_id", "cell_type"]

learned_comp = learned_completed_agg.copy()
learned_comp = learned_comp.rename(
    columns={c: f"learned_{c}" for c in learned_comp.columns if c not in keys}
)

delta_rows = []

baseline_completed_tables = {
    "Gene empirical completed": gene_emp_completed_agg,
    "Cell-type gene empirical completed": ct_gene_emp_completed_agg,
    "Spatial-kNN empirical completed": spatial_knn_completed_agg,
}

metrics_for_delta = [
    "mean_r_norm",
    "mean_z_rel",
    "nuclear_fraction",
    "central_fraction_r_le_0_35",
    "peripheral_fraction_r_ge_0_75",
    "imputed_fraction",
]

for baseline_name, base_df in baseline_completed_tables.items():
    base = base_df.copy()
    base = base.rename(columns={c: f"baseline_{c}" for c in base.columns if c not in keys})

    merged = learned_comp.merge(base, on=keys, how="inner")
    merged["baseline_method"] = baseline_name

    for m in metrics_for_delta:
        lc = f"learned_{m}"
        bc = f"baseline_{m}"

        if lc in merged.columns and bc in merged.columns:
            merged[f"delta_{m}_learned_minus_baseline"] = merged[lc] - merged[bc]

    delta_rows.append(merged)

loc_delta_df = pd.concat(delta_rows, ignore_index=True)

print(f"loc_delta_df shape: {loc_delta_df.shape}")
display(loc_delta_df.head(20))

# ------------------------------------------------------------------------------
# 6. Method-level summary
# ------------------------------------------------------------------------------

method_summary_df = (
    completed_summary_df
    .groupby("method", observed=True)
    .agg(
        n_gene_celltype_pairs=("gene_id", "size"),
        total_molecules=("n_molecules", "sum"),
        mean_r_norm=("mean_r_norm", "mean"),
        mean_z_rel=("mean_z_rel", "mean"),
        mean_nuclear_fraction=("nuclear_fraction", "mean"),
        mean_central_fraction=("central_fraction_r_le_0_35", "mean"),
        mean_peripheral_fraction=("peripheral_fraction_r_ge_0_75", "mean"),
    )
    .reset_index()
)

print("\nMethod-level localization summary:")
display(method_summary_df)

# ------------------------------------------------------------------------------
# 7. Heatmaps for learned 9E completed
# ------------------------------------------------------------------------------

MIN_MOLECULES_HEATMAP = 20

learned_heat = learned_completed_agg[
    learned_completed_agg["n_molecules"] >= MIN_MOLECULES_HEATMAP
].copy()

learned_heat = learned_heat[learned_heat["gene_id"].isin(summary_genes)].copy()

# Keep biologically readable cell-type order.
preferred_celltype_order = [
    "Epithelial cells",
    "Fibroblasts",
    "Endothelial cells",
    "Myeloid cells",
    "T lymphocytes",
    "B lymphocytes",
    "NK cells",
    "MAST cells",
    "Oligodendrocytes",
    "Unlabeled",
]

gene_order = [g for g in summary_genes if g in learned_heat["gene_id"].unique()]
available_celltypes = learned_heat["cell_type"].astype(str).unique().tolist()

celltype_order = [ct for ct in preferred_celltype_order if ct in available_celltypes]
celltype_order += sorted([ct for ct in available_celltypes if ct not in celltype_order])


def plot_gene_celltype_heatmap(
    df,
    value_col,
    title,
    cbar_label,
    out_path,
    matrix_out_path=None,
    vmin=None,
    vmax=None,
    cmap="viridis",
):
    matrix = df.pivot_table(
        index="gene_id",
        columns="cell_type",
        values=value_col,
        aggfunc="mean",
    )

    matrix = matrix.reindex(index=gene_order, columns=celltype_order)

    # Drop all-empty rows/columns.
    matrix = matrix.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if matrix.empty:
        print(f"WARNING: empty heatmap matrix for {value_col}. Skipping.")
        return matrix

    if matrix_out_path is not None:
        matrix.to_csv(matrix_out_path)

    fig_w = max(12, 0.75 * len(matrix.columns) + 4)
    fig_h = max(7, 0.35 * len(matrix.index) + 3)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    arr = matrix.values.astype(float)

    im = ax.imshow(
        arr,
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )

    ax.set_title(title, fontsize=14, pad=14)
    ax.set_xlabel("Cell type")
    ax.set_ylabel("Gene")

    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right", fontsize=8)

    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index, fontsize=8)

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label(cbar_label, fontsize=11)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved heatmap: {out_path}")

    if matrix_out_path is not None:
        print(f"Saved heatmap matrix: {matrix_out_path}")

    return matrix


if len(learned_heat) > 0:
    nuc_matrix = plot_gene_celltype_heatmap(
        learned_heat,
        value_col="nuclear_fraction",
        title="CosMx learned 9E completed: nuclear fraction by gene and cell type",
        cbar_label="Nuclear fraction",
        out_path=LOC_NUC_HEATMAP_PATH,
        matrix_out_path=LOC_HEATMAP_NUC_MATRIX_PATH,
        vmin=0,
        vmax=1,
        cmap="viridis",
    )

    r_matrix = plot_gene_celltype_heatmap(
        learned_heat,
        value_col="mean_r_norm",
        title="CosMx learned 9E completed: mean radial position by gene and cell type",
        cbar_label="Mean r_norm",
        out_path=LOC_R_HEATMAP_PATH,
        matrix_out_path=LOC_HEATMAP_R_MATRIX_PATH,
        vmin=0,
        vmax=1,
        cmap="viridis",
    )
else:
    print("WARNING: no gene/cell-type pairs passed MIN_MOLECULES_HEATMAP.")
    nuc_matrix = pd.DataFrame()
    r_matrix = pd.DataFrame()

# ------------------------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------------------------

imputed_only_summary_df.to_csv(LOC_IMPUTED_ONLY_PATH, index=False)
completed_summary_df.to_csv(LOC_COMPLETED_PATH, index=False)
loc_delta_df.to_csv(LOC_DELTA_PATH, index=False)
method_summary_df.to_csv(LOC_METHOD_SUMMARY_PATH, index=False)

print("\nSaved per-gene/per-cell-type localization summary outputs:")
print(f"  Imputed-only summary : {LOC_IMPUTED_ONLY_PATH}")
print(f"  Completed summary    : {LOC_COMPLETED_PATH}")
print(f"  Learned-vs-baselines : {LOC_DELTA_PATH}")
print(f"  Method summary       : {LOC_METHOD_SUMMARY_PATH}")
print(f"  Nuclear heatmap      : {LOC_NUC_HEATMAP_PATH}")
print(f"  r_norm heatmap       : {LOC_R_HEATMAP_PATH}")
print(f"  Nuclear matrix       : {LOC_HEATMAP_NUC_MATRIX_PATH}")
print(f"  r_norm matrix        : {LOC_HEATMAP_R_MATRIX_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: Known nuclear/TF genes have higher nuclear_fraction in expected cell types.")
print("  Good sign 2: Stromal/ECM or membrane-associated genes do not collapse fully to the nucleus.")
print("  Good sign 3: Learned 9E differs from baselines in biologically plausible ways, not randomly.")
print("  Caution: this is descriptive; use it with nuclear/radial statistical summaries.")
print("  CosMx note: r_norm/nuclear_fraction are based on Step5 CosMx geometry/compartment fields.")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 9 COMPLETE — Per-gene / per-cell-type localization summaries")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 10 — Spatial domain / tissue-region signal strengthening
#
# Biological question:
#   Does completed 9E count data make tissue-region / spatial biological
#   patterns clearer than raw observed counts?
#
# Main comparison:
#   Raw observed counts
#   Learned 9E completed counts
#
# Optional comparison:
#   Step4 denoised counts, if X_denoised is available
#
# Main metrics:
#   - Moran's I on cell-coordinate kNN graph
#   - neighbor smoothness / neighbor correlation
#   - expected-domain contrast for marker modules
#   - AUROC of module score for expected tissue/cell-type domain
#
# Same algorithm/configs as original Cell 10:
#   RANDOM_STATE = 42
#   SAMPLE_SIZE  = 50_000
#   SPATIAL_K    = 12
#   MAX_MAP_POINTS = 50_000
#
# CosMx-safe:
#   - Uses DOWNSTREAM_DIR, not CHECKPOINT_DIR
#   - Uses denoised_adata.obs[ct_col] for CosMx cell-type labels
#   - Uses CosMx-compatible broad biological modules
#   - Uses CosMx centroid/global-pixel coordinate columns when available
#   - Falls back to molecule-derived cell centroids if needed
#   - Keeps CosMx cell IDs as strings
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

print("=" * 100)
print("COSMX DOWNSTREAM CELL 10 — Spatial domain / tissue-region signal strengthening")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "denoised_adata",
    "X_raw_counts",
    "X_completed_9E",
    "shared_genes",
    "ct_col",
    "cell_ids_step4",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 and Cell 2 first. "
        "Cell 2 is needed because it creates X_completed_9E."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

SPATIAL_DOMAIN_DIR = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_spatial_domain_signal_strengthening"
)
SPATIAL_DOMAIN_DIR.mkdir(parents=True, exist_ok=True)

SPATIAL_MODULE_METRICS_PATH = (
    SPATIAL_DOMAIN_DIR / f"{RUN_NAME}_spatial_domain_module_metrics.csv"
)

SPATIAL_MODULE_GAIN_PATH = (
    SPATIAL_DOMAIN_DIR / f"{RUN_NAME}_spatial_domain_module_gain_vs_raw.csv"
)

SPATIAL_MODULE_HEATMAP_PATH = (
    SPATIAL_DOMAIN_DIR / f"{RUN_NAME}_spatial_domain_metric_gain_heatmap.png"
)

SPATIAL_MODULE_MAP_DIR = SPATIAL_DOMAIN_DIR / "module_spatial_maps"
SPATIAL_MODULE_MAP_DIR.mkdir(parents=True, exist_ok=True)

SPATIAL_MODULE_MAP_MANIFEST_PATH = (
    SPATIAL_DOMAIN_DIR / f"{RUN_NAME}_spatial_module_map_manifest.csv"
)

SPATIAL_SAMPLE_IDX_PATH = (
    SPATIAL_DOMAIN_DIR / f"{RUN_NAME}_spatial_domain_sample_idx.npy"
)

SPATIAL_COORDINATE_TABLE_PATH = (
    SPATIAL_DOMAIN_DIR / f"{RUN_NAME}_spatial_domain_sample_coordinates.csv"
)

print(f"SPATIAL_DOMAIN_DIR: {SPATIAL_DOMAIN_DIR}")

# ------------------------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------------------------

RANDOM_STATE = 42
SAMPLE_SIZE = 50_000

# kNN graph for spatial autocorrelation.
SPATIAL_K = 12

# Plotting sample. Use same cells as metrics if possible.
MAX_MAP_POINTS = 50_000

print(f"SAMPLE_SIZE    : {SAMPLE_SIZE:,}")
print(f"SPATIAL_K      : {SPATIAL_K}")
print(f"MAX_MAP_POINTS : {MAX_MAP_POINTS:,}")
print(f"RANDOM_STATE   : {RANDOM_STATE}")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_xy_from_adata_or_molecules(adata):
    """
    Get cell x/y coordinates.

    Priority:
      1. Coordinate columns in denoised_adata.obs.
      2. Fallback: molecule-derived mean x/y per cell from mol_observed if available.
    """

    x_candidates = [
        "center_x_global_px",
        "CenterX_global_px",
        "x_centroid",
        "x_centroid_px",
        "center_x",
        "cell_x",
        "x",
    ]

    y_candidates = [
        "center_y_global_px",
        "CenterY_global_px",
        "y_centroid",
        "y_centroid_px",
        "center_y",
        "cell_y",
        "y",
    ]

    x_col = None
    y_col = None

    for c in x_candidates:
        if c in adata.obs.columns:
            x_col = c
            break

    for c in y_candidates:
        if c in adata.obs.columns:
            y_col = c
            break

    if x_col is not None and y_col is not None:
        xy = adata.obs[[x_col, y_col]].copy()
        xy.columns = ["x", "y"]

        xy["x"] = pd.to_numeric(xy["x"], errors="coerce")
        xy["y"] = pd.to_numeric(xy["y"], errors="coerce")

        return xy.to_numpy(dtype=np.float32), x_col, y_col, "denoised_adata.obs"

    # Fallback from molecules.
    if "mol_observed" in globals():
        print(
            "Could not find coordinate columns in denoised_adata.obs. "
            "Falling back to mean molecule coordinates per cell."
        )

        mol_tmp = mol_observed[["cell_id", "x", "y"]].copy()
        mol_tmp["cell_id"] = mol_tmp["cell_id"].astype(str)
        mol_tmp["x"] = pd.to_numeric(mol_tmp["x"], errors="coerce")
        mol_tmp["y"] = pd.to_numeric(mol_tmp["y"], errors="coerce")
        mol_tmp = mol_tmp.dropna(subset=["x", "y"])

        centroid_df = (
            mol_tmp
            .groupby("cell_id", observed=True)
            .agg(x=("x", "mean"), y=("y", "mean"))
        )

        xy = np.full((adata.n_obs, 2), np.nan, dtype=np.float32)

        for i, cid in enumerate(np.asarray(cell_ids_step4).astype(str)):
            if cid in centroid_df.index:
                xy[i, 0] = float(centroid_df.loc[cid, "x"])
                xy[i, 1] = float(centroid_df.loc[cid, "y"])

        return xy, "molecule_mean_x", "molecule_mean_y", "mol_observed fallback"

    raise KeyError(
        "Could not find x/y centroid columns in denoised_adata.obs, and "
        "mol_observed is not available for fallback.\n"
        f"obs columns: {list(adata.obs.columns)}"
    )


def log_norm_counts(X):
    """
    Library-size normalize counts and log1p transform.
    """
    X = ensure_dense(X).astype(np.float32)

    lib = X.sum(axis=1, keepdims=True)
    lib = np.maximum(lib, 1.0)

    X_norm = X / lib * 1e4
    X_log = np.log1p(X_norm).astype(np.float32)

    return X_log


def safe_auroc(y_true, scores):
    try:
        y_true = np.asarray(y_true).astype(int)
        scores = np.asarray(scores).astype(float)

        if len(np.unique(y_true)) < 2:
            return np.nan

        return float(roc_auc_score(y_true, scores))
    except Exception:
        return np.nan


def build_spatial_knn_edges(xy, k=12):
    """
    Build directed kNN edges from spatial coordinates.
    Returns source indices and neighbor indices.
    """
    n = xy.shape[0]

    if n <= k:
        raise ValueError(f"Not enough cells to build kNN graph: n={n}, k={k}")

    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm="ball_tree")
    nbrs.fit(xy)

    distances, indices = nbrs.kneighbors(xy)

    # Drop self neighbor at column 0.
    neigh = indices[:, 1:]
    src = np.repeat(np.arange(xy.shape[0]), k)
    dst = neigh.reshape(-1)

    return src.astype(np.int64), dst.astype(np.int64)


def morans_i_knn(values, src, dst):
    """
    Approximate Moran's I using unweighted directed kNN edges.

    Higher positive value means nearby cells have more similar module scores.
    """
    x = np.asarray(values, dtype=np.float64)
    x = x - np.nanmean(x)

    denom = np.nansum(x ** 2)

    if denom <= 1e-12:
        return np.nan

    w = len(src)
    n = len(x)

    num = np.nansum(x[src] * x[dst])

    return float((n / w) * (num / denom))


def neighbor_correlation(values, src, dst):
    """
    Pearson correlation between each cell's score and its neighbor's score.
    """
    a = np.asarray(values[src], dtype=np.float64)
    b = np.asarray(values[dst], dtype=np.float64)

    if np.nanstd(a) <= 1e-12 or np.nanstd(b) <= 1e-12:
        return np.nan

    return float(np.corrcoef(a, b)[0, 1])


def neighbor_smoothness(values, src, dst):
    """
    Mean absolute score difference between neighboring cells.
    Lower is smoother.
    """
    v = np.asarray(values, dtype=np.float64)
    return float(np.nanmean(np.abs(v[src] - v[dst])))


def module_score(X_log, genes, gene_to_col):
    """
    Average log-normalized expression over available module genes.
    """
    genes_used = [g for g in genes if g in gene_to_col]
    cols = [gene_to_col[g] for g in genes_used]

    if len(cols) == 0:
        return None, []

    score = X_log[:, cols].mean(axis=1)

    return np.asarray(score, dtype=np.float32), genes_used


def expected_domain_mask(cell_types, expected_celltypes):
    """
    Boolean mask for expected cell types.
    Allows partial matching if exact cell type is not found.
    """
    ct = pd.Series(np.asarray(cell_types).astype(str))

    mask = np.zeros(len(ct), dtype=bool)

    for expected in expected_celltypes:
        exact = (ct == expected).to_numpy()

        if exact.sum() > 0:
            mask |= exact
        else:
            # Fallback partial matching.
            mask |= ct.str.contains(expected, case=False, regex=False).to_numpy()

    return mask


# ------------------------------------------------------------------------------
# 4. Define CosMx biological modules and expected domains
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)
gene_to_col = {str(g): j for j, g in enumerate(shared_genes)}

cell_type_labels = denoised_adata.obs[ct_col].astype(str).values
unique_cell_types = sorted(pd.Series(cell_type_labels).unique())

print("\nCosMx cell types:")
print(unique_cell_types)

# CosMx broad module definitions.
# These are matched to your current broad CosMx cell-type labels:
# Epithelial cells, Fibroblasts, Endothelial cells, Myeloid cells,
# T lymphocytes, B lymphocytes, NK cells, MAST cells, Oligodendrocytes, Unlabeled.
module_defs = {
    "Epithelial_Tumor": {
        "genes": [
            "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "KRT17", "KRT13",
            "CEACAM6", "TACSTD2", "MUC1", "SOX2", "SOX9", "TP63"
        ],
        "expected_celltypes": ["Epithelial cells"],
    },

    "Fibroblast_ECM": {
        "genes": [
            "LUM", "DCN", "COL1A1", "COL1A2", "COL6A1", "COL6A2",
            "IGFBP7", "MMP2", "TAGLN", "ACTA2", "PDGFRA", "PDGFRB"
        ],
        "expected_celltypes": ["Fibroblasts"],
    },

    "Endothelial_Vascular": {
        "genes": [
            "PECAM1", "VWF", "CAV1", "RAMP2", "CDH5", "ENG",
            "CLDN5", "KDR", "FLT1", "ACKR1", "ICAM1"
        ],
        "expected_celltypes": ["Endothelial cells"],
    },

    "T_Cell": {
        "genes": [
            "CD3D", "CD3E", "CD3G", "CD4", "CD8A", "CD8B",
            "IL7R", "CCR7", "LTB", "TRAC", "TRBC1", "TRBC2"
        ],
        "expected_celltypes": ["T lymphocytes"],
    },

    "B_Cell": {
        "genes": [
            "MS4A1", "CD79A", "CD79B", "BANK1", "CD19", "CD22",
            "CD74", "HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DPB1",
            "IGKC", "JCHAIN", "MZB1", "XBP1"
        ],
        "expected_celltypes": ["B lymphocytes"],
    },

    "Myeloid_Macrophage": {
        "genes": [
            "LYZ", "LST1", "TYROBP", "FCER1G", "MARCO", "S100A8", "S100A9",
            "CD68", "CD163", "C1QA", "C1QB", "C1QC", "HLA-DRA",
            "HLA-DPA1", "HLA-DPB1", "CSF1R", "ITGAM", "ITGAX"
        ],
        "expected_celltypes": ["Myeloid cells"],
    },

    "NK_Cytotoxic": {
        "genes": [
            "NKG7", "GZMB", "GZMA", "PRF1", "GNLY", "KLRD1", "KLRB1",
            "KLRF1", "FCGR3A", "TYROBP"
        ],
        "expected_celltypes": ["NK cells"],
    },

    "Mast_Cell": {
        "genes": [
            "TPSAB1", "TPSB2", "CPA3", "KIT", "MS4A2", "HPGDS", "HDC", "GATA2"
        ],
        "expected_celltypes": ["MAST cells"],
    },

    "Proliferation": {
        "genes": [
            "MKI67", "TOP2A", "PCNA", "TYMS", "STMN1", "HMGB2", "CCND1"
        ],
        # Broad CosMx labels do not have a separate proliferative subtype.
        # Epithelial domain is used as the expected tumor/proliferative compartment.
        "expected_celltypes": ["Epithelial cells"],
    },

    "Interferon_Inflammatory": {
        "genes": [
            "STAT1", "IRF7", "NFKBIA", "NFKB1", "CXCL10", "ISG15", "IFIT1", "IFIT3"
        ],
        # In broad CosMx annotations, this may appear across immune compartments.
        "expected_celltypes": ["Myeloid cells", "T lymphocytes", "B lymphocytes", "NK cells"],
    },
}

# Keep only modules with at least 2 genes present and at least some expected-domain cells.
filtered_modules = {}

for module_name, spec in module_defs.items():
    present = [g for g in spec["genes"] if g in available_genes]
    domain_mask_test = expected_domain_mask(cell_type_labels, spec["expected_celltypes"])

    if len(present) >= 2 and domain_mask_test.sum() > 0:
        filtered_modules[module_name] = {
            "genes": present,
            "expected_celltypes": spec["expected_celltypes"],
        }

print("\nModules with available genes and expected CosMx domains:")
for m, spec in filtered_modules.items():
    print(
        f"  {m:25s}: "
        f"n_genes={len(spec['genes']):2d}, "
        f"genes={spec['genes']}, "
        f"expected={spec['expected_celltypes']}"
    )

if len(filtered_modules) == 0:
    raise RuntimeError("No modules have at least 2 available genes and valid expected-domain cells.")

# ------------------------------------------------------------------------------
# 5. Sample cells and prepare spatial graph
# ------------------------------------------------------------------------------

print("\nPreparing cell coordinates and sample...")

xy_all, x_col, y_col, coord_source = get_xy_from_adata_or_molecules(denoised_adata)

valid_xy = np.isfinite(xy_all).all(axis=1)
all_indices = np.where(valid_xy)[0]

if len(all_indices) == 0:
    raise RuntimeError("No cells have valid spatial coordinates for Cell 10.")

rng = np.random.default_rng(RANDOM_STATE)

if len(all_indices) > SAMPLE_SIZE:
    sample_idx = rng.choice(all_indices, size=SAMPLE_SIZE, replace=False)
else:
    sample_idx = all_indices.copy()

# Keep stable order.
sample_idx = np.sort(sample_idx)

xy_sample = xy_all[sample_idx, :]
cell_types_sample = np.asarray(cell_type_labels).astype(str)[sample_idx]
sample_cell_ids = np.asarray(cell_ids_step4).astype(str)[sample_idx]

np.save(SPATIAL_SAMPLE_IDX_PATH, sample_idx)

coordinate_df = pd.DataFrame({
    "sample_local_index": np.arange(len(sample_idx)),
    "adata_index": sample_idx,
    "cell_id": sample_cell_ids,
    "x": xy_sample[:, 0],
    "y": xy_sample[:, 1],
    "cell_type": cell_types_sample,
})
coordinate_df.to_csv(SPATIAL_COORDINATE_TABLE_PATH, index=False)

print(f"Coordinate source      : {coord_source}")
print(f"Coordinate columns     : {x_col}, {y_col}")
print(f"Valid coordinate cells : {len(all_indices):,}")
print(f"Sample cells used      : {len(sample_idx):,}")
print(f"Saved sample index     : {SPATIAL_SAMPLE_IDX_PATH}")
print(f"Saved coordinate table : {SPATIAL_COORDINATE_TABLE_PATH}")

print("\nBuilding spatial kNN graph...")
src, dst = build_spatial_knn_edges(xy_sample, k=SPATIAL_K)
print(f"kNN edges: {len(src):,}")

# ------------------------------------------------------------------------------
# 6. Build datasets
# ------------------------------------------------------------------------------

datasets = {
    "Raw observed counts": X_raw_counts,
    "Learned 9E completed counts": X_completed_9E,
}

if "X_denoised" in globals():
    datasets["Step4 denoised counts"] = X_denoised

print("\nDatasets to evaluate:")
for name, X in datasets.items():
    print(f"  {name:30s}: {ensure_dense(X).shape}")

# ------------------------------------------------------------------------------
# 7. Compute module spatial-domain metrics
# ------------------------------------------------------------------------------

print("\nComputing spatial-domain metrics...")

metric_rows = []
module_score_cache = {}

for dataset_name, X_counts in datasets.items():
    print(f"\nProcessing dataset: {dataset_name}")

    X_counts_dense = ensure_dense(X_counts)

    if X_counts_dense.shape != denoised_adata.shape:
        raise ValueError(
            f"{dataset_name} shape {X_counts_dense.shape} does not match "
            f"denoised_adata shape {denoised_adata.shape}"
        )

    X_sub = X_counts_dense[sample_idx, :]
    X_log = log_norm_counts(X_sub)

    for module_name, spec in filtered_modules.items():
        score, genes_used = module_score(X_log, spec["genes"], gene_to_col)

        if score is None:
            continue

        domain_mask = expected_domain_mask(
            cell_types_sample,
            spec["expected_celltypes"],
        )

        in_domain = score[domain_mask]
        out_domain = score[~domain_mask]

        domain_mean = float(np.mean(in_domain)) if len(in_domain) else np.nan
        background_mean = float(np.mean(out_domain)) if len(out_domain) else np.nan
        domain_contrast = domain_mean - background_mean

        domain_ratio = (domain_mean + 1e-6) / (background_mean + 1e-6)

        y_true = domain_mask.astype(int)
        auc = safe_auroc(y_true, score)

        moran = morans_i_knn(score, src, dst)
        neigh_corr = neighbor_correlation(score, src, dst)
        neigh_smooth = neighbor_smoothness(score, src, dst)

        metric_rows.append({
            "dataset": dataset_name,
            "module": module_name,
            "genes_used": ",".join(genes_used),
            "n_genes_used": int(len(genes_used)),
            "expected_celltypes": ",".join(spec["expected_celltypes"]),

            "n_domain_cells": int(domain_mask.sum()),
            "n_background_cells": int((~domain_mask).sum()),

            "mean_score_domain": domain_mean,
            "mean_score_background": background_mean,
            "domain_contrast_domain_minus_background": float(domain_contrast),
            "domain_ratio_domain_over_background": float(domain_ratio),
            "AUROC_domain_vs_background": auc,

            "morans_I_spatial_autocorrelation": moran,
            "neighbor_correlation": neigh_corr,
            "neighbor_smoothness_absdiff": neigh_smooth,
        })

        module_score_cache[(dataset_name, module_name)] = score.astype(np.float32)

    del X_sub, X_log
    gc.collect()

spatial_module_metrics_df = pd.DataFrame(metric_rows)

print("\nSpatial module metrics:")
display(spatial_module_metrics_df)

if len(spatial_module_metrics_df) == 0:
    raise RuntimeError("No spatial module metrics were computed.")

# ------------------------------------------------------------------------------
# 8. Raw vs completed gain table
# ------------------------------------------------------------------------------

print("\nComputing raw-vs-completed spatial-domain gains...")

raw_df = spatial_module_metrics_df[
    spatial_module_metrics_df["dataset"] == "Raw observed counts"
].copy()

completed_df = spatial_module_metrics_df[
    spatial_module_metrics_df["dataset"] == "Learned 9E completed counts"
].copy()

merge_keys = ["module"]

raw_renamed = raw_df.rename(columns={
    c: f"raw_{c}" for c in raw_df.columns if c not in merge_keys
})

comp_renamed = completed_df.rename(columns={
    c: f"completed_{c}" for c in completed_df.columns if c not in merge_keys
})

gain_df = raw_renamed.merge(
    comp_renamed,
    on=merge_keys,
    how="inner",
)

gain_metrics = [
    "domain_contrast_domain_minus_background",
    "domain_ratio_domain_over_background",
    "AUROC_domain_vs_background",
    "morans_I_spatial_autocorrelation",
    "neighbor_correlation",
    "neighbor_smoothness_absdiff",
]

for m in gain_metrics:
    raw_col = f"raw_{m}"
    comp_col = f"completed_{m}"

    if raw_col in gain_df.columns and comp_col in gain_df.columns:
        gain_df[f"delta_{m}_completed_minus_raw"] = (
            gain_df[comp_col] - gain_df[raw_col]
        )

# For neighbor smoothness, lower is better, so define improvement as raw - completed.
if (
    "raw_neighbor_smoothness_absdiff" in gain_df.columns
    and "completed_neighbor_smoothness_absdiff" in gain_df.columns
):
    gain_df["improvement_neighbor_smoothness_raw_minus_completed"] = (
        gain_df["raw_neighbor_smoothness_absdiff"]
        - gain_df["completed_neighbor_smoothness_absdiff"]
    )

print("\nRaw vs learned 9E completed gain table:")
display(gain_df)

# ------------------------------------------------------------------------------
# 9. Heatmap of gains
# ------------------------------------------------------------------------------

heatmap_metrics = [
    "delta_domain_contrast_domain_minus_background_completed_minus_raw",
    "delta_AUROC_domain_vs_background_completed_minus_raw",
    "delta_morans_I_spatial_autocorrelation_completed_minus_raw",
    "delta_neighbor_correlation_completed_minus_raw",
    "improvement_neighbor_smoothness_raw_minus_completed",
]

heatmap_metrics = [m for m in heatmap_metrics if m in gain_df.columns]

heat_df = gain_df.set_index("module")[heatmap_metrics].copy()

rename_map = {
    "delta_domain_contrast_domain_minus_background_completed_minus_raw": "Δ domain contrast",
    "delta_AUROC_domain_vs_background_completed_minus_raw": "Δ AUROC",
    "delta_morans_I_spatial_autocorrelation_completed_minus_raw": "Δ Moran's I",
    "delta_neighbor_correlation_completed_minus_raw": "Δ neighbor corr",
    "improvement_neighbor_smoothness_raw_minus_completed": "Smoothness improvement",
}

heat_df = heat_df.rename(columns=rename_map)

fig_w = max(10, 1.7 * len(heat_df.columns) + 3)
fig_h = max(5, 0.6 * len(heat_df.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = heat_df.values.astype(float)
finite = arr[np.isfinite(arr)]

vmax = np.nanpercentile(np.abs(finite), 95) if len(finite) else 1.0
vmax = max(float(vmax), 0.01)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=-vmax,
    vmax=vmax,
    cmap="coolwarm",
)

ax.set_title(
    "CosMx spatial-domain signal gain: Learned 9E completed − Raw observed",
    fontsize=14,
    pad=14,
)

ax.set_xlabel("Metric gain")
ax.set_ylabel("Biological module")

ax.set_xticks(np.arange(len(heat_df.columns)))
ax.set_xticklabels(heat_df.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(heat_df.index)))
ax.set_yticklabels(heat_df.index, fontsize=10)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{val:+.3f}",
                ha="center",
                va="center",
                fontsize=8,
                color="black",
            )

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Gain vs raw", fontsize=11)

plt.tight_layout()
plt.savefig(SPATIAL_MODULE_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved spatial-domain gain heatmap:")
print(f"  {SPATIAL_MODULE_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 10. Spatial maps for selected modules
# ------------------------------------------------------------------------------

print("\nGenerating spatial maps for module scores...")

if len(sample_idx) > MAX_MAP_POINTS:
    map_local_idx = rng.choice(
        np.arange(len(sample_idx)),
        size=MAX_MAP_POINTS,
        replace=False,
    )
else:
    map_local_idx = np.arange(len(sample_idx))

map_x = xy_sample[map_local_idx, 0]
map_y = xy_sample[map_local_idx, 1]

map_records = []

modules_to_plot = list(filtered_modules.keys())

for module_name in modules_to_plot:
    raw_key = ("Raw observed counts", module_name)
    comp_key = ("Learned 9E completed counts", module_name)

    if raw_key not in module_score_cache or comp_key not in module_score_cache:
        continue

    raw_score = module_score_cache[raw_key][map_local_idx]
    comp_score = module_score_cache[comp_key][map_local_idx]
    delta_score = comp_score - raw_score

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 5),
        sharex=True,
        sharey=True,
    )

    panels = [
        ("Raw observed", raw_score, "viridis"),
        ("Learned 9E completed", comp_score, "viridis"),
        ("Completed − Raw", delta_score, "coolwarm"),
    ]

    for ax, (title, vals, cmap) in zip(axes, panels):
        if title == "Completed − Raw":
            finite_vals = vals[np.isfinite(vals)]
            vmax_delta = (
                np.nanpercentile(np.abs(finite_vals), 95)
                if len(finite_vals)
                else 1.0
            )
            vmax_delta = max(float(vmax_delta), 0.01)

            sca = ax.scatter(
                map_x,
                map_y,
                c=vals,
                s=2,
                cmap=cmap,
                vmin=-vmax_delta,
                vmax=vmax_delta,
                linewidths=0,
            )
        else:
            sca = ax.scatter(
                map_x,
                map_y,
                c=vals,
                s=2,
                cmap=cmap,
                linewidths=0,
            )

        ax.set_title(title, fontsize=12)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("x global px")
        ax.set_ylabel("y global px")
        ax.grid(False)

        cbar = plt.colorbar(sca, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label("Module score", fontsize=9)

    fig.suptitle(f"CosMx spatial module map: {module_name}", fontsize=15, y=1.02)
    plt.tight_layout()

    out_path = SPATIAL_MODULE_MAP_DIR / f"{RUN_NAME}_spatial_module_map_{module_name}.png"

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    map_records.append({
        "module": module_name,
        "map_path": str(out_path),
    })

    print(f"Saved module map for {module_name}: {out_path}")

module_map_manifest_df = pd.DataFrame(map_records)
module_map_manifest_df.to_csv(SPATIAL_MODULE_MAP_MANIFEST_PATH, index=False)

# ------------------------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------------------------

spatial_module_metrics_df.to_csv(SPATIAL_MODULE_METRICS_PATH, index=False)
gain_df.to_csv(SPATIAL_MODULE_GAIN_PATH, index=False)

print("\nSaved spatial-domain validation outputs:")
print(f"  Module metrics      : {SPATIAL_MODULE_METRICS_PATH}")
print(f"  Gain vs raw         : {SPATIAL_MODULE_GAIN_PATH}")
print(f"  Gain heatmap        : {SPATIAL_MODULE_HEATMAP_PATH}")
print(f"  Module map manifest : {SPATIAL_MODULE_MAP_MANIFEST_PATH}")
print(f"  Module map dir      : {SPATIAL_MODULE_MAP_DIR}")
print(f"  Sample index        : {SPATIAL_SAMPLE_IDX_PATH}")
print(f"  Coordinate table    : {SPATIAL_COORDINATE_TABLE_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: completed data has higher domain contrast than raw.")
print("  Good sign 2: completed data has higher AUROC for expected CosMx cell-type domains.")
print("  Good sign 3: completed data has higher Moran's I or neighbor correlation.")
print("  Good sign 4: completed data has lower neighbor smoothness absdiff.")
print("  Caution: too much smoothing can inflate spatial autocorrelation; check AUROC and module maps together.")
print("  CosMx note: module domains use broad CosMx cell-type labels, not Xenium breast cancer subtypes.")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 10 COMPLETE — Spatial domain / tissue-region signal strengthening")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 11_OFFICIAL_SPRAWL_APPROX — Run official SPRAWL on CosMx
#
# Goal:
#   Compute official SPRAWL scores:
#       1. peripheral
#       2. central
#       3. punctate
#       4. radial
#
# Main comparison:
#   Raw observed vs Learned 9E completed
#
# Optional:
#   Also run empirical baseline completed molecule tables.
#
# Important CosMx limitation:
#   Exact Xenium-style cell polygons are not available in the current CosMx Step5
#   setup. If real cell_polygons.pkl is missing/empty, this cell builds
#   approximate ellipse boundaries from CosMx cell centroid/area/width/height.
#
# Interpretation:
#   - If real polygons are available: this is official SPRAWL on real boundaries.
#   - If fallback ellipses are used: this is official SPRAWL scoring on approximate
#     CosMx 2D cell-boundary proxies.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import sys
import pickle
import subprocess
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm

print("=" * 100)
print("COSMX DOWNSTREAM CELL 11_OFFICIAL_SPRAWL_APPROX — Run official SPRAWL on CosMx")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "IMPUTATION_DIR",
    "denoised_adata",
    "cell_ids_step4",
    "cell_idx_map",
    "mol_observed",
    "mol_learned_9E",
    "shared_genes",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
IMPUTATION_DIR = Path(IMPUTATION_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# Baseline tables are optional depending on RUN_BASELINES_TOO.
has_baselines = all(
    v in globals()
    for v in ["mol_gene_emp", "mol_ct_gene_emp", "mol_spatial_knn_emp"]
)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

OFFICIAL_SPRAWL_DIR = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_official_sprawl_validation"
)
OFFICIAL_SPRAWL_DIR.mkdir(parents=True, exist_ok=True)

SPRAWL_H5_DIR = OFFICIAL_SPRAWL_DIR / "sprawl_h5_inputs"
SPRAWL_H5_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_SPRAWL_SCORES_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_cell_gene_scores.csv"
)

OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_method_summary.csv"
)

OFFICIAL_SPRAWL_GENE_GAIN_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_raw_vs_learned_gene_gain.csv"
)

OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_pattern_summary.csv"
)

OFFICIAL_SPRAWL_H5_MANIFEST_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_h5_manifest.csv"
)

OFFICIAL_SPRAWL_FIG_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_summary.png"
)

OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_scorable_pairs_heatmap.png"
)

OFFICIAL_SPRAWL_CELL_SELECTION_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_sampled_cells.csv"
)

print(f"OFFICIAL_SPRAWL_DIR: {OFFICIAL_SPRAWL_DIR}")

# ------------------------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------------------------

RANDOM_STATE = 42

# Start small. Official SPRAWL radial/punctate are expensive.
SAMPLE_CELLS = 750

MIN_TOTAL_MOLS_PER_CELL = 10

# Minimum molecules per gene per cell.
MIN_GENE_MOLS_FOR_INPUT = 3

SPRAWL_METRICS = ["peripheral", "central", "punctate", "radial"]

# For expensive permutation metrics.
SPRAWL_NUM_ITERATIONS = 200
SPRAWL_NUM_PAIRS = 4

SPRAWL_PROCESSES = 2

# Start False. Set True only after raw/learned works.
RUN_BASELINES_TOO = False

# Restrict to CosMx-relevant marker genes for faster biological validation.
RESTRICT_TO_MARKER_GENES = True

SIG_THRESHOLD = 0.05

candidate_marker_genes = [
    # Nuclear / proliferation / state
    "MKI67", "TOP2A", "CCND1", "PCNA", "TYMS", "STMN1", "HMGB2",
    "SOX2", "SOX9", "GATA3", "STAT1", "JUN", "FOS", "MYC",
    "NFKB1", "NFKBIA", "TBX21", "FOXP3",

    # Epithelial / tumor-like
    "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "KRT17", "KRT13",
    "CEACAM6", "TACSTD2", "MUC1",

    # Stromal / ECM
    "LUM", "DCN", "COL1A1", "COL1A2", "COL6A1", "COL6A2",
    "IGFBP7", "MMP2", "TAGLN", "ACTA2",

    # Immune / myeloid / lymphoid
    "CD3D", "CD3E", "CD8A", "CD8B",
    "MS4A1", "CD79A", "CD79B", "CD74",
    "LYZ", "LST1", "TYROBP", "FCER1G", "MARCO",
    "S100A8", "S100A9", "CD68", "CD163",
    "NKG7", "GZMB", "PRF1", "GNLY",

    # Endothelial / vascular
    "PECAM1", "VWF", "CAV1", "RAMP2", "CDH5", "ENG",

    # Mast
    "TPSAB1", "TPSB2", "CPA3", "HPGDS",

    # High-abundance controls
    "MALAT1", "NEAT1", "B2M",
]

available_genes = set(str(g) for g in shared_genes)
marker_genes = [g for g in candidate_marker_genes if g in available_genes]

print(f"SAMPLE_CELLS              : {SAMPLE_CELLS:,}")
print(f"MIN_TOTAL_MOLS_PER_CELL   : {MIN_TOTAL_MOLS_PER_CELL}")
print(f"MIN_GENE_MOLS_FOR_INPUT   : {MIN_GENE_MOLS_FOR_INPUT}")
print(f"SPRAWL_METRICS            : {SPRAWL_METRICS}")
print(f"SPRAWL_NUM_ITERATIONS     : {SPRAWL_NUM_ITERATIONS}")
print(f"SPRAWL_NUM_PAIRS          : {SPRAWL_NUM_PAIRS}")
print(f"SPRAWL_PROCESSES          : {SPRAWL_PROCESSES}")
print(f"RUN_BASELINES_TOO         : {RUN_BASELINES_TOO}")
print(f"RESTRICT_TO_MARKER_GENES  : {RESTRICT_TO_MARKER_GENES}")
print(f"Marker genes available    : {marker_genes}")

if RUN_BASELINES_TOO and not has_baselines:
    raise NameError(
        "RUN_BASELINES_TOO=True but baseline molecule tables are not available. "
        "Run Cell 1 that loads mol_gene_emp, mol_ct_gene_emp, and mol_spatial_knn_emp."
    )

# ------------------------------------------------------------------------------
# 3. Install/import official SPRAWL
# ------------------------------------------------------------------------------

def pip_install(cmd):
    print(f"Running: pip {cmd}")
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd.split())


def install_and_import_sprawl():
    try:
        import sprawl
        from sprawl import hdf5, scoring
        print("Official SPRAWL imported successfully.")
        return sprawl, hdf5, scoring

    except Exception as e1:
        print(f"Initial SPRAWL import failed: {e1}")
        print("Trying PyPI package: subcellular-sprawl")

        try:
            pip_install("install -q subcellular-sprawl")
            import sprawl
            from sprawl import hdf5, scoring
            print("Official SPRAWL imported successfully from PyPI.")
            return sprawl, hdf5, scoring

        except Exception as e2:
            print(f"PyPI install/import failed: {e2}")
            print("Trying GitHub install from salzman-lab/SPRAWL package subdirectory.")

            try:
                pip_install("install -q git+https://github.com/salzman-lab/SPRAWL.git#subdirectory=package")
                import sprawl
                from sprawl import hdf5, scoring
                print("Official SPRAWL imported successfully from GitHub.")
                return sprawl, hdf5, scoring

            except Exception as e3:
                raise ImportError(
                    "Could not install/import official SPRAWL.\n\n"
                    f"Initial error: {e1}\n"
                    f"PyPI error   : {e2}\n"
                    f"GitHub error : {e3}"
                )

sprawl, sprawl_hdf5, sprawl_scoring = install_and_import_sprawl()

print("\nSPRAWL scoring metrics available:")
try:
    print(list(sprawl_scoring.available_metrics.keys()))
except Exception:
    print("Could not inspect scoring.available_metrics")

# ------------------------------------------------------------------------------
# 4. Load real polygons if available; otherwise prepare approximate boundaries
# ------------------------------------------------------------------------------

CELL_POLYGONS_PATH = IMPUTATION_DIR / "cell_polygons.pkl"
NUC_POLYGONS_PATH = IMPUTATION_DIR / "nuc_polygons.pkl"

cell_polygons = {}
nuc_polygons = {}

if CELL_POLYGONS_PATH.exists() and CELL_POLYGONS_PATH.stat().st_size > 10:
    try:
        with open(CELL_POLYGONS_PATH, "rb") as f:
            cell_polygons = pickle.load(f)
        cell_polygons = {str(k): v for k, v in cell_polygons.items()}
        print(f"Loaded real cell polygons: {len(cell_polygons):,}")
    except Exception as e:
        print(f"Could not load cell polygons: {e}")
        cell_polygons = {}
else:
    print("Real cell_polygons.pkl missing/empty. Will use approximate ellipse boundaries.")

if NUC_POLYGONS_PATH.exists() and NUC_POLYGONS_PATH.stat().st_size > 10:
    try:
        with open(NUC_POLYGONS_PATH, "rb") as f:
            nuc_polygons = pickle.load(f)
        nuc_polygons = {str(k): v for k, v in nuc_polygons.items()}
        print(f"Loaded nucleus polygons: {len(nuc_polygons):,}")
    except Exception as e:
        print(f"Could not load nucleus polygons: {e}")
        nuc_polygons = {}
else:
    print("Nucleus polygons missing/empty. SPRAWL will use cell boundaries only.")

USE_APPROX_BOUNDARIES = len(cell_polygons) == 0

print(f"USE_APPROX_BOUNDARIES: {USE_APPROX_BOUNDARIES}")

# ------------------------------------------------------------------------------
# 5. SPRAWL-compatible cell class
# ------------------------------------------------------------------------------

class SimpleSprawlCell:
    """
    Minimal object compatible with sprawl.hdf5.HDF5.write_cells().

    SPRAWL expects each cell to have:
      - cell_id
      - annotation
      - zslices
      - boundaries[zslice]
      - spot_coords[zslice]
      - spot_genes[zslice]
      - gene_counts
      - genes
      - gene_vars
    """

    def __init__(self, cell_id, annotation, boundary_xy, spot_xy, spot_genes):
        self.cell_id = str(cell_id)
        self.annotation = str(annotation)

        # Collapse CosMx molecules into one 2D z-slice.
        self.zslices = ["z0"]

        self.boundaries = {
            "z0": np.asarray(boundary_xy, dtype=np.float32)
        }

        self.spot_coords = {
            "z0": np.asarray(spot_xy, dtype=np.float32)
        }

        self.spot_genes = {
            "z0": np.asarray([str(g) for g in spot_genes])
        }

        self.gene_counts = Counter([str(g) for g in spot_genes])
        self.genes = sorted(list(self.gene_counts.keys()))

        self.gene_vars = {}
        self.n_per_z = {"z0": len(spot_genes)}
        self.n = len(spot_genes)

        self.ranked = False
        self.spot_ranks = {"z0": []}
        self.spot_values = {"z0": []}
        self.gene_med_ranks = {}

    def filter_genes_by_count(self, min_gene_spots=1, max_gene_spots=None):
        """
        SPRAWL radial/punctate metrics may call this method.
        """
        if max_gene_spots is None:
            max_gene_spots = max(self.gene_counts.values()) if self.gene_counts else 0

        keep_genes = {
            g for g, c in self.gene_counts.items()
            if min_gene_spots <= c <= max_gene_spots
        }

        new_spot_genes = []
        new_spot_coords = []

        for gene, xy in zip(self.spot_genes["z0"], self.spot_coords["z0"]):
            if gene in keep_genes:
                new_spot_genes.append(gene)
                new_spot_coords.append(xy)

        self.spot_genes["z0"] = np.asarray(new_spot_genes)
        self.spot_coords["z0"] = np.asarray(new_spot_coords, dtype=np.float32)

        self.gene_counts = Counter([str(g) for g in self.spot_genes["z0"]])
        self.genes = sorted(list(self.gene_counts.keys()))
        self.gene_vars = {}

        self.n = len(self.spot_genes["z0"])
        self.n_per_z = {"z0": self.n}

        if self.n == 0:
            self.zslices = []
            self.boundaries = {}
            self.spot_coords = {}
            self.spot_genes = {}
            self.n_per_z = {}

        return self

# ------------------------------------------------------------------------------
# 6. Helper functions
# ------------------------------------------------------------------------------

def get_celltype_col(df):
    candidates = [
        "cell_type",
        "Final_CosMx_Cell_Type",
        "Assigned_CosMx_Cell_Type",
        "Assigned_Xenium_Cell_Type",
        "Reference_Cell_Type",
        "ct_label",
        "celltype",
        "Cell_Type",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    return None


def get_obs_row_for_cell(cid):
    cid = str(cid)

    if cid not in cell_idx_map:
        return None

    return denoised_adata.obs.iloc[cell_idx_map[cid]]


def get_cell_center_width_height(cid):
    """
    Return approximate center/width/height for CosMx cell.
    """
    row = get_obs_row_for_cell(cid)

    if row is None:
        return None

    x_candidates = [
        "center_x_global_px",
        "CenterX_global_px",
        "x_centroid",
        "x_centroid_px",
        "center_x",
        "cell_x",
        "x",
    ]

    y_candidates = [
        "center_y_global_px",
        "CenterY_global_px",
        "y_centroid",
        "y_centroid_px",
        "center_y",
        "cell_y",
        "y",
    ]

    cx = None
    cy = None

    for c in x_candidates:
        if c in row.index and pd.notna(row[c]):
            cx = float(row[c])
            break

    for c in y_candidates:
        if c in row.index and pd.notna(row[c]):
            cy = float(row[c])
            break

    if cx is None or cy is None:
        return None

    width = None
    height = None

    for c in ["width_px", "Width", "cell_width_px"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            width = float(row[c])
            break

    for c in ["height_px", "Height", "cell_height_px"]:
        if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
            height = float(row[c])
            break

    if width is None or height is None:
        area = None

        for c in ["cell_area_px", "label_cell_area_px", "Area"]:
            if c in row.index and pd.notna(row[c]) and float(row[c]) > 0:
                area = float(row[c])
                break

        if area is not None:
            radius = np.sqrt(area / np.pi)
            width = 2.0 * radius
            height = 2.0 * radius
        else:
            width = 20.0
            height = 20.0

    return cx, cy, width, height


def ellipse_boundary_xy(cx, cy, width, height, n_points=96):
    """
    Build closed ellipse boundary.
    """
    theta = np.linspace(0, 2 * np.pi, n_points, endpoint=True)
    x = cx + 0.5 * width * np.cos(theta)
    y = cy + 0.5 * height * np.sin(theta)

    return np.vstack([x, y]).T.astype(np.float32)


def polygon_to_boundary_xy(poly):
    """
    Convert shapely Polygon/MultiPolygon to one boundary coordinate array.
    If MultiPolygon, use the largest component.
    """
    if poly is None:
        return None

    try:
        if poly.geom_type == "Polygon":
            return np.asarray(poly.exterior.coords, dtype=np.float32)

        if poly.geom_type == "MultiPolygon":
            largest = max(poly.geoms, key=lambda p: p.area)
            return np.asarray(largest.exterior.coords, dtype=np.float32)

    except Exception:
        return None

    return None


def get_boundary_for_cell(cid):
    """
    Return real polygon boundary if available.
    Otherwise return approximate ellipse boundary.
    """
    cid = str(cid)

    if cid in cell_polygons:
        boundary = polygon_to_boundary_xy(cell_polygons[cid])

        if boundary is not None and len(boundary) >= 4:
            return boundary, "real_polygon"

    geom = get_cell_center_width_height(cid)

    if geom is not None:
        cx, cy, width, height = geom
        return ellipse_boundary_xy(cx, cy, width, height), "approx_ellipse"

    return None, "missing"


def prepare_sprawl_molecule_table(df, sample_cids, marker_genes=None):
    """
    Keep columns required for SPRAWL:
      cell_id, gene_id, x, y, cell_type

    CosMx-safe:
      cell_id remains string.
    """
    required_cols = ["cell_id", "gene_id", "x", "y"]
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise KeyError(f"Molecule table missing required columns for SPRAWL: {missing}")

    sample_cids = set(str(c) for c in sample_cids)

    ct_col = get_celltype_col(df)

    keep_cols = ["cell_id", "gene_id", "x", "y"]
    if ct_col is not None:
        keep_cols.append(ct_col)

    d = df.loc[df["cell_id"].astype(str).isin(sample_cids), keep_cols].copy()

    d["cell_id"] = d["cell_id"].astype(str)
    d["gene_id"] = d["gene_id"].astype(str)
    d["x"] = pd.to_numeric(d["x"], errors="coerce").astype(np.float32)
    d["y"] = pd.to_numeric(d["y"], errors="coerce").astype(np.float32)

    d = d.dropna(subset=["x", "y", "gene_id"])

    if marker_genes is not None:
        d = d[d["gene_id"].isin(marker_genes)].copy()

    if ct_col is not None:
        d = d.rename(columns={ct_col: "cell_type"})
        d["cell_type"] = d["cell_type"].astype(str)
    else:
        d["cell_type"] = "Unknown"

    return d


def build_sprawl_cells_from_molecules(mol_df, sample_cids, label):
    """
    Build SimpleSprawlCell objects from a molecule table.
    """
    cells = []
    boundary_source_counts = Counter()

    sample_cids = set(str(c) for c in sample_cids)

    grouped = mol_df.groupby("cell_id", observed=True)

    for cid, g in grouped:
        cid = str(cid)

        if cid not in sample_cids:
            continue

        boundary_xy, boundary_source = get_boundary_for_cell(cid)

        if boundary_xy is None or len(boundary_xy) < 4:
            boundary_source_counts["missing_boundary"] += 1
            continue

        spot_xy = g[["x", "y"]].to_numpy(dtype=np.float32)
        spot_genes = g["gene_id"].astype(str).to_numpy()

        if len(spot_genes) < MIN_TOTAL_MOLS_PER_CELL:
            boundary_source_counts["too_few_molecules"] += 1
            continue

        # Optional: remove genes with too few molecules per cell.
        gene_counts = pd.Series(spot_genes).value_counts()
        keep_genes = set(gene_counts[gene_counts >= MIN_GENE_MOLS_FOR_INPUT].index)

        keep_mask = np.array([g in keep_genes for g in spot_genes])

        spot_xy = spot_xy[keep_mask]
        spot_genes = spot_genes[keep_mask]

        if len(spot_genes) < MIN_TOTAL_MOLS_PER_CELL:
            boundary_source_counts["too_few_after_gene_filter"] += 1
            continue

        annotation = str(g["cell_type"].iloc[0]) if "cell_type" in g.columns else "Unknown"

        cells.append(
            SimpleSprawlCell(
                cell_id=cid,
                annotation=annotation,
                boundary_xy=boundary_xy,
                spot_xy=spot_xy,
                spot_genes=spot_genes,
            )
        )

        boundary_source_counts[boundary_source] += 1

    print(f"  {label}: built {len(cells):,} SPRAWL cells")
    print(f"  Boundary/filter counts: {dict(boundary_source_counts)}")

    return cells, boundary_source_counts


def make_completed_sprawl_table(raw_df, imputed_df, sample_cids, marker_genes=None):
    """
    completed = raw observed + imputed
    only for sampled cells.
    """
    raw_min = prepare_sprawl_molecule_table(
        raw_df,
        sample_cids=sample_cids,
        marker_genes=marker_genes,
    )

    imp_min = prepare_sprawl_molecule_table(
        imputed_df,
        sample_cids=sample_cids,
        marker_genes=marker_genes,
    )

    completed = pd.concat([raw_min, imp_min], ignore_index=True)

    return completed


def write_and_reload_sprawl_cells(cells, out_h5_path, hdf5_module):
    """
    Write SimpleSprawlCell objects to official SPRAWL HDF5,
    then read them back as official sprawl.cell.Cell objects.
    """
    out_h5_path = str(out_h5_path)

    if os.path.exists(out_h5_path):
        os.remove(out_h5_path)

    hdf5_module.HDF5.write_cells(cells, out_h5_path)

    h = hdf5_module.HDF5(out_h5_path)
    official_cells = h.cells()

    return official_cells


def run_sprawl_metric(cells, metric_name, scoring_module):
    """
    Run one official SPRAWL metric.
    """
    print(f"    Running official SPRAWL metric: {metric_name}")

    if metric_name in ["radial", "punctate"]:
        df = scoring_module.iter_scores(
            cells,
            metric=metric_name,
            processes=SPRAWL_PROCESSES,
            num_iterations=SPRAWL_NUM_ITERATIONS,
            num_pairs=SPRAWL_NUM_PAIRS,
        )
    else:
        df = scoring_module.iter_scores(
            cells,
            metric=metric_name,
            processes=SPRAWL_PROCESSES,
        )

    df = pd.DataFrame(df)
    df["metric"] = metric_name

    return df


def run_all_sprawl_metrics_for_method(cells, method_name, scoring_module):
    """
    Run all selected official SPRAWL metrics for one method.
    """
    metric_tables = []

    for metric_name in SPRAWL_METRICS:
        try:
            metric_df = run_sprawl_metric(
                cells=cells,
                metric_name=metric_name,
                scoring_module=scoring_module,
            )

            if len(metric_df) == 0:
                print(f"      {metric_name}: 0 rows")
                continue

            metric_df["method"] = method_name
            metric_tables.append(metric_df)

            print(f"      {metric_name}: {len(metric_df):,} rows")

        except Exception as e:
            print(f"      WARNING: SPRAWL metric {metric_name} failed for {method_name}: {e}")

    if len(metric_tables) == 0:
        return pd.DataFrame()

    return pd.concat(metric_tables, ignore_index=True)


def standardize_sprawl_columns(df):
    """
    Normalize possible SPRAWL output column names.
    """
    out = df.copy()

    rename_candidates = {
        "gene_name": "gene",
        "cell": "cell_id",
        "cellID": "cell_id",
        "cell_id": "cell_id",
        "score": "score",
        "variance": "variance",
        "num_gene_spots": "num_gene_spots",
        "n": "num_gene_spots",
    }

    for old, new in rename_candidates.items():
        if old in out.columns and new not in out.columns:
            out = out.rename(columns={old: new})

    if "gene" not in out.columns:
        if "genes" in out.columns:
            out = out.rename(columns={"genes": "gene"})
        else:
            out["gene"] = "unknown"

    if "cell_id" not in out.columns:
        out["cell_id"] = "unknown"

    if "score" not in out.columns:
        raise KeyError(f"SPRAWL output does not contain a score column. Columns: {list(out.columns)}")

    if "variance" not in out.columns:
        out["variance"] = np.nan

    if "num_gene_spots" not in out.columns:
        out["num_gene_spots"] = np.nan

    out["gene"] = out["gene"].astype(str)
    out["cell_id"] = out["cell_id"].astype(str)

    return out


def add_sprawl_significance(score_df):
    """
    Add approximate z and p values using SPRAWL score and variance.
    If variance is unavailable, p-value is NaN.
    """
    df = score_df.copy()

    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df["variance"] = pd.to_numeric(df["variance"], errors="coerce")

    df["z"] = df["score"] / np.sqrt(df["variance"].replace(0, np.nan))
    df["approx_p_value"] = 2.0 * norm.sf(np.abs(df["z"]))

    df["is_significant"] = df["approx_p_value"] < SIG_THRESHOLD

    return df


def summarize_sprawl_by_method(score_df):
    """
    Method-level SPRAWL summary.
    """
    summary = (
        score_df
        .groupby(["method", "metric"], observed=True)
        .agg(
            n_scored_cell_gene_pairs=("gene", "size"),
            n_significant_pairs=("is_significant", "sum"),
            mean_score=("score", "mean"),
            median_score=("score", "median"),
            mean_num_gene_spots=("num_gene_spots", "mean"),
            median_num_gene_spots=("num_gene_spots", "median"),
        )
        .reset_index()
    )

    summary["fraction_significant"] = (
        summary["n_significant_pairs"]
        / summary["n_scored_cell_gene_pairs"].replace(0, np.nan)
    )

    return summary


def classify_sprawl_pattern(row):
    """
    Simple pattern label from metric score and significance.
    """
    if not bool(row["is_significant"]):
        return "non-significant"

    metric = str(row["metric"])
    score = float(row["score"])

    if metric == "peripheral":
        return "peripheral" if score > 0 else "anti-peripheral"

    if metric == "central":
        return "central" if score > 0 else "anti-central"

    if metric == "punctate":
        return "punctate" if score > 0 else "dispersed"

    if metric == "radial":
        return "radial" if score > 0 else "anti-radial"

    return "unknown"

# ------------------------------------------------------------------------------
# 7. Select cells
# ------------------------------------------------------------------------------

print("\nSelecting sampled cells for official SPRAWL...")

if "status" in mol_observed.columns:
    raw_obs_full = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ].copy()
else:
    raw_obs_full = mol_observed.copy()

raw_obs_full["cell_id"] = raw_obs_full["cell_id"].astype(str)

# Candidate cells must have enough raw molecules and boundary information.
cell_mol_counts = raw_obs_full.groupby("cell_id", observed=True).size()

candidate_cids = [
    str(cid)
    for cid, n in cell_mol_counts.items()
    if n >= MIN_TOTAL_MOLS_PER_CELL
]

# Keep only cells with real or approximate boundary.
valid_cids = []

boundary_source_for_selection = []

for cid in candidate_cids:
    boundary, source = get_boundary_for_cell(cid)

    if boundary is not None and len(boundary) >= 4:
        valid_cids.append(cid)
        boundary_source_for_selection.append(source)

valid_cids = sorted(valid_cids)

rng = np.random.default_rng(RANDOM_STATE)

if len(valid_cids) > SAMPLE_CELLS:
    sample_cids = set(rng.choice(valid_cids, size=SAMPLE_CELLS, replace=False).tolist())
else:
    sample_cids = set(valid_cids)

print(f"Candidate raw cells with enough molecules : {len(candidate_cids):,}")
print(f"Valid cells with usable boundary          : {len(valid_cids):,}")
print(f"Sampled cells                             : {len(sample_cids):,}")
print(f"Boundary sources among valid candidates   : {dict(Counter(boundary_source_for_selection))}")

selected_cell_df = pd.DataFrame({
    "cell_id": sorted(list(sample_cids)),
})
selected_cell_df.to_csv(OFFICIAL_SPRAWL_CELL_SELECTION_PATH, index=False)

if len(sample_cids) == 0:
    raise RuntimeError("No valid cells available for SPRAWL.")

marker_genes_for_run = None

if RESTRICT_TO_MARKER_GENES:
    marker_genes_for_run = marker_genes

    if len(marker_genes_for_run) == 0:
        raise RuntimeError("RESTRICT_TO_MARKER_GENES=True, but no marker genes are available.")

    print(f"Restricting SPRAWL input to marker genes: {marker_genes_for_run}")

# ------------------------------------------------------------------------------
# 8. Build method-specific SPRAWL molecule tables
# ------------------------------------------------------------------------------

print("\nPreparing molecule tables for official SPRAWL...")

raw_sprawl_df = prepare_sprawl_molecule_table(
    raw_obs_full,
    sample_cids=sample_cids,
    marker_genes=marker_genes_for_run,
)

learned_completed_df = make_completed_sprawl_table(
    raw_obs_full,
    mol_learned_9E,
    sample_cids=sample_cids,
    marker_genes=marker_genes_for_run,
)

method_molecule_tables = {
    "Raw observed": raw_sprawl_df,
    "Learned 9E completed": learned_completed_df,
}

if RUN_BASELINES_TOO:
    gene_emp_completed_df = make_completed_sprawl_table(
        raw_obs_full,
        mol_gene_emp,
        sample_cids=sample_cids,
        marker_genes=marker_genes_for_run,
    )

    ct_gene_emp_completed_df = make_completed_sprawl_table(
        raw_obs_full,
        mol_ct_gene_emp,
        sample_cids=sample_cids,
        marker_genes=marker_genes_for_run,
    )

    spatial_knn_completed_df = make_completed_sprawl_table(
        raw_obs_full,
        mol_spatial_knn_emp,
        sample_cids=sample_cids,
        marker_genes=marker_genes_for_run,
    )

    method_molecule_tables.update({
        "Gene empirical completed": gene_emp_completed_df,
        "Cell-type gene empirical completed": ct_gene_emp_completed_df,
        "Spatial-kNN empirical completed": spatial_knn_completed_df,
    })

for method_name, df in method_molecule_tables.items():
    print(f"  {method_name:35s}: {len(df):,} molecules")

# ------------------------------------------------------------------------------
# 9. Build/write/read official SPRAWL cells and run metrics
# ------------------------------------------------------------------------------

print("\nBuilding official SPRAWL HDF5 files and running metrics...")

all_score_tables = []
h5_manifest_rows = []

for method_name, mol_df in method_molecule_tables.items():
    print("\n" + "-" * 90)
    print(f"Method: {method_name}")
    print("-" * 90)

    simple_cells, boundary_counts = build_sprawl_cells_from_molecules(
        mol_df=mol_df,
        sample_cids=sample_cids,
        label=method_name,
    )

    if len(simple_cells) == 0:
        print(f"  WARNING: no SPRAWL cells for {method_name}; skipping.")
        continue

    safe_method = (
        method_name
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
    )

    h5_path = SPRAWL_H5_DIR / f"{RUN_NAME}_official_sprawl_{safe_method}.h5"

    official_cells = write_and_reload_sprawl_cells(
        cells=simple_cells,
        out_h5_path=h5_path,
        hdf5_module=sprawl_hdf5,
    )

    print(f"  Wrote/read official SPRAWL HDF5: {h5_path}")
    print(f"  Official SPRAWL cells loaded: {len(official_cells):,}")

    h5_manifest_rows.append({
        "method": method_name,
        "h5_path": str(h5_path),
        "n_cells": int(len(official_cells)),
        "n_molecules_in_input": int(len(mol_df)),
        "boundary_counts": str(dict(boundary_counts)),
        "used_approx_boundaries": bool(USE_APPROX_BOUNDARIES),
    })

    score_df = run_all_sprawl_metrics_for_method(
        cells=official_cells,
        method_name=method_name,
        scoring_module=sprawl_scoring,
    )

    if len(score_df) == 0:
        print(f"  WARNING: no SPRAWL scores generated for {method_name}.")
        continue

    score_df = standardize_sprawl_columns(score_df)
    all_score_tables.append(score_df)

    del simple_cells, official_cells, score_df
    gc.collect()

if len(all_score_tables) == 0:
    raise RuntimeError("Official SPRAWL produced no score tables. Check package/API/input format.")

official_sprawl_scores_df = pd.concat(all_score_tables, ignore_index=True)

print("\nOfficial SPRAWL raw score table:")
display(official_sprawl_scores_df.head(20))
print(f"official_sprawl_scores_df shape: {official_sprawl_scores_df.shape}")

# ------------------------------------------------------------------------------
# 10. Add approximate significance and pattern labels
# ------------------------------------------------------------------------------

official_sprawl_scores_df = add_sprawl_significance(official_sprawl_scores_df)

official_sprawl_scores_df["pattern_call"] = official_sprawl_scores_df.apply(
    classify_sprawl_pattern,
    axis=1,
)

print("\nOfficial SPRAWL score table with approximate significance:")
display(official_sprawl_scores_df.head(20))

# ------------------------------------------------------------------------------
# 11. Summaries
# ------------------------------------------------------------------------------

official_method_summary_df = summarize_sprawl_by_method(official_sprawl_scores_df)

print("\nOfficial SPRAWL method summary:")
display(official_method_summary_df)

pattern_summary_df = (
    official_sprawl_scores_df
    .groupby(["method", "metric", "pattern_call"], observed=True)
    .size()
    .reset_index(name="n_pairs")
)

pattern_summary_df["fraction_within_method_metric"] = (
    pattern_summary_df["n_pairs"]
    / pattern_summary_df.groupby(["method", "metric"])["n_pairs"].transform("sum")
)

print("\nOfficial SPRAWL pattern summary:")
display(pattern_summary_df)

# Gene-level gain: Learned 9E completed vs Raw observed.
raw_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Raw observed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

learned_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Learned 9E completed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

raw_gene_metric = (
    raw_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("raw_significant_cells")
)

learned_gene_metric = (
    learned_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("learned_significant_cells")
)

official_gene_gain_df = (
    pd.concat([raw_gene_metric, learned_gene_metric], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

official_gene_gain_df["gain_significant_cells"] = (
    official_gene_gain_df["learned_significant_cells"]
    - official_gene_gain_df["raw_significant_cells"]
)

official_gene_gain_df = official_gene_gain_df.sort_values(
    "gain_significant_cells",
    ascending=False,
).reset_index(drop=True)

print("\nTop genes by gain in significant official SPRAWL-localized cells:")
display(official_gene_gain_df.head(30))

h5_manifest_df = pd.DataFrame(h5_manifest_rows)

# ------------------------------------------------------------------------------
# 12. Visualization
# ------------------------------------------------------------------------------

print("\nGenerating official SPRAWL summary figure...")

plot_methods = [
    "Raw observed",
    "Learned 9E completed",
    "Gene empirical completed",
    "Cell-type gene empirical completed",
    "Spatial-kNN empirical completed",
]

plot_methods = [
    m for m in plot_methods
    if m in official_sprawl_scores_df["method"].astype(str).unique()
]

plot_metrics = SPRAWL_METRICS

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.ravel()

for ax, metric in zip(axes, plot_metrics):
    sub = official_sprawl_scores_df[
        official_sprawl_scores_df["metric"] == metric
    ].copy()

    for method in plot_methods:
        vals = sub[sub["method"] == method]["score"].dropna()

        if len(vals) == 0:
            continue

        ax.hist(
            vals,
            bins=50,
            density=True,
            histtype="step",
            linewidth=1.6,
            label=method,
        )

    ax.axvline(0, linestyle="--", linewidth=1.0)
    ax.set_title(f"Official SPRAWL {metric} score distribution")
    ax.set_xlabel("SPRAWL score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OFFICIAL_SPRAWL_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved official SPRAWL figure:")
print(f"  {OFFICIAL_SPRAWL_FIG_PATH}")

# Heatmap of scorable pairs per method/metric.
scorable_matrix = official_method_summary_df.pivot_table(
    index="metric",
    columns="method",
    values="n_scored_cell_gene_pairs",
    aggfunc="sum",
).reindex(index=SPRAWL_METRICS, columns=plot_methods)

fig, ax = plt.subplots(figsize=(12, 5))

arr = scorable_matrix.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    cmap="viridis",
)

ax.set_title("Official SPRAWL: scorable cell-gene pairs")
ax.set_xlabel("Method")
ax.set_ylabel("Metric")

ax.set_xticks(np.arange(len(scorable_matrix.columns)))
ax.set_xticklabels(scorable_matrix.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(len(scorable_matrix.index)))
ax.set_yticklabels(scorable_matrix.index)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(
                j,
                i,
                f"{int(val):,}",
                ha="center",
                va="center",
                fontsize=8,
            )

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Scored pairs")

plt.tight_layout()
plt.savefig(OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved scorable-pairs heatmap:")
print(f"  {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------------------------

official_sprawl_scores_df.to_csv(OFFICIAL_SPRAWL_SCORES_PATH, index=False)
official_method_summary_df.to_csv(OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH, index=False)
official_gene_gain_df.to_csv(OFFICIAL_SPRAWL_GENE_GAIN_PATH, index=False)
pattern_summary_df.to_csv(OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH, index=False)
h5_manifest_df.to_csv(OFFICIAL_SPRAWL_H5_MANIFEST_PATH, index=False)

print("\nSaved official SPRAWL outputs:")
print(f"  Cell-gene scores     : {OFFICIAL_SPRAWL_SCORES_PATH}")
print(f"  Method summary       : {OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH}")
print(f"  Gene gain table      : {OFFICIAL_SPRAWL_GENE_GAIN_PATH}")
print(f"  Pattern summary      : {OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH}")
print(f"  HDF5 manifest        : {OFFICIAL_SPRAWL_H5_MANIFEST_PATH}")
print(f"  Sampled cells        : {OFFICIAL_SPRAWL_CELL_SELECTION_PATH}")
print(f"  Score figure         : {OFFICIAL_SPRAWL_FIG_PATH}")
print(f"  Scorable heatmap     : {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: Learned 9E completed has more scorable cell-gene pairs than raw.")
print("  Good sign 2: Learned 9E completed has more significant localized pairs than raw.")
print("  Good sign 3: Learned 9E score distributions are not collapsed to only one pattern.")
print("  Good sign 4: Learned 9E should not make all genes uniformly central/peripheral.")
print("  Caution 1: radial and punctate are permutation-heavy; increase SPRAWL_NUM_ITERATIONS for final reporting.")
print("  Caution 2: This CosMx run likely uses approximate ellipse boundaries unless real cell_polygons.pkl exists.")
print("  Caution 3: If approximate boundaries are used, report this as SPRAWL-compatible approximate 2D localization analysis, not exact polygon SPRAWL.")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 11 COMPLETE — Official SPRAWL / approximate-boundary localization validation")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX DOWNSTREAM CELL 11B — Continue official SPRAWL post-processing
#
# Purpose:
#   Continue official SPRAWL post-processing without rerunning expensive SPRAWL
#   scoring.
#
# Use this if:
#   1. official_sprawl_scores_df already exists in memory, OR
#   2. the official SPRAWL score CSV was already saved by Cell 11.
#
# Main fixes compared with the original Cell 11B:
#   - Uses DOWNSTREAM_DIR instead of CHECKPOINT_DIR
#   - Works with CosMx corrected Cell 11 output paths
#   - Keeps CosMx cell IDs as strings
#   - Can reload official_sprawl_scores_df from saved CSV if not in memory
#   - Robustly handles missing variance / num_spots / annotation columns
#
# GPU not needed.
# ==============================================================================

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

print("=" * 100)
print("COSMX DOWNSTREAM CELL 11B — Continue official SPRAWL post-processing")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

OFFICIAL_SPRAWL_DIR = (
    DOWNSTREAM_DIR / f"{RUN_NAME}_official_sprawl_validation"
)
OFFICIAL_SPRAWL_DIR.mkdir(parents=True, exist_ok=True)

SPRAWL_OFFICIAL_DIR = OFFICIAL_SPRAWL_DIR

OFFICIAL_SPRAWL_SCORES_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_cell_gene_scores.csv"
)

OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_method_summary.csv"
)

OFFICIAL_SPRAWL_GENE_GAIN_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_raw_vs_learned_gene_gain.csv"
)

OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_pattern_summary.csv"
)

OFFICIAL_SPRAWL_H5_MANIFEST_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_h5_manifest.csv"
)

OFFICIAL_SPRAWL_FIG_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_score_distributions.png"
)

OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_scorable_pairs_heatmap.png"
)

OFFICIAL_SPRAWL_TOP_GAIN_PATH = (
    OFFICIAL_SPRAWL_DIR / f"{RUN_NAME}_official_sprawl_top_gene_metric_gains.csv"
)

print(f"OFFICIAL_SPRAWL_DIR: {OFFICIAL_SPRAWL_DIR}")
print(f"Score CSV path      : {OFFICIAL_SPRAWL_SCORES_PATH}")

# ------------------------------------------------------------------------------
# 2. Configuration defaults
# ------------------------------------------------------------------------------

if "SIG_THRESHOLD" not in globals():
    SIG_THRESHOLD = 0.05

if "SPRAWL_METRICS" not in globals():
    SPRAWL_METRICS = ["peripheral", "central", "punctate", "radial"]

plot_methods = [
    "Raw observed",
    "Learned 9E completed",
    "Gene empirical completed",
    "Cell-type gene empirical completed",
    "Spatial-kNN empirical completed",
]

print(f"SIG_THRESHOLD: {SIG_THRESHOLD}")
print(f"SPRAWL_METRICS: {SPRAWL_METRICS}")

# ------------------------------------------------------------------------------
# 3. Load official_sprawl_scores_df if needed
# ------------------------------------------------------------------------------

if "official_sprawl_scores_df" in globals():
    print("Using official_sprawl_scores_df from memory.")
else:
    if OFFICIAL_SPRAWL_SCORES_PATH.exists():
        print("official_sprawl_scores_df not in memory.")
        print("Loading saved official SPRAWL score CSV:")
        print(f"  {OFFICIAL_SPRAWL_SCORES_PATH}")

        official_sprawl_scores_df = pd.read_csv(OFFICIAL_SPRAWL_SCORES_PATH)
    else:
        raise FileNotFoundError(
            "official_sprawl_scores_df is not in memory and saved score CSV was not found:\n"
            f"{OFFICIAL_SPRAWL_SCORES_PATH}\n\n"
            "You need to run corrected CosMx Cell 11 first."
        )

print(f"official_sprawl_scores_df shape: {official_sprawl_scores_df.shape}")
print("Columns:")
print(list(official_sprawl_scores_df.columns))

if len(official_sprawl_scores_df) == 0:
    raise RuntimeError("official_sprawl_scores_df is empty. Rerun Cell 11.")

# ------------------------------------------------------------------------------
# 4. Robustly standardize/check official SPRAWL score table
# ------------------------------------------------------------------------------

def standardize_sprawl_columns_for_postprocess(df):
    """
    Make SPRAWL score table robust across possible official SPRAWL output schemas.
    """
    out = df.copy()

    # Common column-name variants.
    rename_map = {}

    if "gene_name" in out.columns and "gene" not in out.columns:
        rename_map["gene_name"] = "gene"

    if "genes" in out.columns and "gene" not in out.columns:
        rename_map["genes"] = "gene"

    if "cell" in out.columns and "cell_id" not in out.columns:
        rename_map["cell"] = "cell_id"

    if "cellID" in out.columns and "cell_id" not in out.columns:
        rename_map["cellID"] = "cell_id"

    if "n" in out.columns and "num_gene_spots" not in out.columns:
        rename_map["n"] = "num_gene_spots"

    if rename_map:
        out = out.rename(columns=rename_map)

    # Required logical columns.
    if "metric" not in out.columns:
        raise KeyError(
            "official_sprawl_scores_df is missing 'metric'. "
            f"Columns: {list(out.columns)}"
        )

    if "method" not in out.columns:
        raise KeyError(
            "official_sprawl_scores_df is missing 'method'. "
            f"Columns: {list(out.columns)}"
        )

    if "score" not in out.columns:
        raise KeyError(
            "official_sprawl_scores_df is missing 'score'. "
            f"Columns: {list(out.columns)}"
        )

    if "gene" not in out.columns:
        out["gene"] = "unknown"

    if "cell_id" not in out.columns:
        out["cell_id"] = "unknown"

    if "annotation" not in out.columns:
        out["annotation"] = "Unknown"

    if "num_spots" not in out.columns:
        out["num_spots"] = np.nan

    if "num_gene_spots" not in out.columns:
        out["num_gene_spots"] = np.nan

    if "variance" not in out.columns:
        out["variance"] = np.nan

    # Type cleanup.
    out["metric"] = out["metric"].astype(str)
    out["method"] = out["method"].astype(str)
    out["gene"] = out["gene"].astype(str)
    out["cell_id"] = out["cell_id"].astype(str)
    out["annotation"] = out["annotation"].astype(str)

    out["score"] = pd.to_numeric(out["score"], errors="coerce")
    out["variance"] = pd.to_numeric(out["variance"], errors="coerce")
    out["num_gene_spots"] = pd.to_numeric(out["num_gene_spots"], errors="coerce")
    out["num_spots"] = pd.to_numeric(out["num_spots"], errors="coerce")

    return out


official_sprawl_scores_df = standardize_sprawl_columns_for_postprocess(
    official_sprawl_scores_df
)

print("\nStandardized score table columns:")
print(list(official_sprawl_scores_df.columns))

# ------------------------------------------------------------------------------
# 5. Add approximate z-score, p-value, and significance
# ------------------------------------------------------------------------------

print("\nAdding approximate significance values...")

safe_variance = official_sprawl_scores_df["variance"].where(
    official_sprawl_scores_df["variance"] > 0,
    np.nan,
)

official_sprawl_scores_df["z"] = (
    official_sprawl_scores_df["score"]
    / np.sqrt(safe_variance)
)

official_sprawl_scores_df["approx_p_value"] = 2.0 * norm.sf(
    np.abs(official_sprawl_scores_df["z"].astype(float))
)

official_sprawl_scores_df["is_significant"] = (
    official_sprawl_scores_df["approx_p_value"] < SIG_THRESHOLD
)

print("Added/updated columns:")
print("  z")
print("  approx_p_value")
print("  is_significant")

display(official_sprawl_scores_df.head(20))

n_missing_variance = int(official_sprawl_scores_df["variance"].isna().sum())
n_valid_p = int(official_sprawl_scores_df["approx_p_value"].notna().sum())

print(f"\nRows with missing variance : {n_missing_variance:,}")
print(f"Rows with valid p-values   : {n_valid_p:,}")

if n_valid_p == 0:
    print(
        "WARNING: No valid approximate p-values were computed, probably because "
        "variance is missing/NaN in the SPRAWL output. Score summaries and plots "
        "are still valid, but significance-based summaries will be empty/zero."
    )

# ------------------------------------------------------------------------------
# 6. Pattern classification
# ------------------------------------------------------------------------------

def classify_sprawl_pattern(row):
    """
    Simple pattern label from official SPRAWL metric score and approximate significance.
    """
    if not bool(row["is_significant"]):
        return "non-significant"

    metric = str(row["metric"])
    score = float(row["score"])

    if metric == "peripheral":
        return "peripheral" if score > 0 else "anti-peripheral"

    if metric == "central":
        return "central" if score > 0 else "anti-central"

    if metric == "punctate":
        return "punctate" if score > 0 else "dispersed"

    if metric == "radial":
        return "radial" if score > 0 else "anti-radial"

    return "unknown"


official_sprawl_scores_df["pattern_call"] = official_sprawl_scores_df.apply(
    classify_sprawl_pattern,
    axis=1,
)

print("\nPattern-call preview:")
display(
    official_sprawl_scores_df[
        [
            "method",
            "metric",
            "cell_id",
            "gene",
            "score",
            "variance",
            "approx_p_value",
            "is_significant",
            "pattern_call",
        ]
    ].head(20)
)

# ------------------------------------------------------------------------------
# 7. Method-level summary
# ------------------------------------------------------------------------------

official_method_summary_df = (
    official_sprawl_scores_df
    .groupby(["method", "metric"], observed=True)
    .agg(
        n_scored_cell_gene_pairs=("gene", "size"),
        n_significant_pairs=("is_significant", "sum"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        mean_abs_score=("score", lambda x: float(np.nanmean(np.abs(x)))),
        median_abs_score=("score", lambda x: float(np.nanmedian(np.abs(x)))),
        mean_num_gene_spots=("num_gene_spots", "mean"),
        median_num_gene_spots=("num_gene_spots", "median"),
        mean_num_spots=("num_spots", "mean"),
        median_num_spots=("num_spots", "median"),
    )
    .reset_index()
)

official_method_summary_df["fraction_significant"] = (
    official_method_summary_df["n_significant_pairs"]
    / official_method_summary_df["n_scored_cell_gene_pairs"].replace(0, np.nan)
)

print("\nOfficial SPRAWL method summary:")
display(official_method_summary_df)

# ------------------------------------------------------------------------------
# 8. Pattern summary
# ------------------------------------------------------------------------------

pattern_summary_df = (
    official_sprawl_scores_df
    .groupby(["method", "metric", "pattern_call"], observed=True)
    .size()
    .reset_index(name="n_pairs")
)

pattern_summary_df["fraction_within_method_metric"] = (
    pattern_summary_df["n_pairs"]
    / pattern_summary_df.groupby(["method", "metric"])["n_pairs"].transform("sum")
)

print("\nOfficial SPRAWL pattern summary:")
display(pattern_summary_df)

# ------------------------------------------------------------------------------
# 9. Gene-level gain: Learned 9E completed vs Raw observed
# ------------------------------------------------------------------------------

raw_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Raw observed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

learned_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Learned 9E completed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

raw_gene_metric = (
    raw_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("raw_significant_cells")
)

learned_gene_metric = (
    learned_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("learned_significant_cells")
)

official_gene_gain_df = (
    pd.concat([raw_gene_metric, learned_gene_metric], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

if len(official_gene_gain_df) > 0:
    official_gene_gain_df["gain_significant_cells"] = (
        official_gene_gain_df["learned_significant_cells"]
        - official_gene_gain_df["raw_significant_cells"]
    )

    official_gene_gain_df = official_gene_gain_df.sort_values(
        "gain_significant_cells",
        ascending=False,
    ).reset_index(drop=True)
else:
    official_gene_gain_df = pd.DataFrame(
        columns=[
            "metric",
            "gene",
            "raw_significant_cells",
            "learned_significant_cells",
            "gain_significant_cells",
        ]
    )

print("\nTop genes by gain in significant official SPRAWL-localized cells:")
display(official_gene_gain_df.head(30))

# ------------------------------------------------------------------------------
# 10. Score-based gain summary, independent of p-values
# ------------------------------------------------------------------------------

raw_scores = official_sprawl_scores_df[
    official_sprawl_scores_df["method"] == "Raw observed"
].copy()

learned_scores = official_sprawl_scores_df[
    official_sprawl_scores_df["method"] == "Learned 9E completed"
].copy()

raw_gene_score = (
    raw_scores
    .groupby(["metric", "gene"], observed=True)
    .agg(
        raw_n_pairs=("score", "size"),
        raw_mean_score=("score", "mean"),
        raw_median_score=("score", "median"),
        raw_mean_abs_score=("score", lambda x: float(np.nanmean(np.abs(x)))),
    )
)

learned_gene_score = (
    learned_scores
    .groupby(["metric", "gene"], observed=True)
    .agg(
        learned_n_pairs=("score", "size"),
        learned_mean_score=("score", "mean"),
        learned_median_score=("score", "median"),
        learned_mean_abs_score=("score", lambda x: float(np.nanmean(np.abs(x)))),
    )
)

score_gain_df = (
    raw_gene_score
    .join(learned_gene_score, how="outer")
    .reset_index()
)

for c in [
    "raw_n_pairs",
    "learned_n_pairs",
    "raw_mean_score",
    "learned_mean_score",
    "raw_median_score",
    "learned_median_score",
    "raw_mean_abs_score",
    "learned_mean_abs_score",
]:
    if c in score_gain_df.columns:
        score_gain_df[c] = pd.to_numeric(score_gain_df[c], errors="coerce")

score_gain_df["gain_n_scored_pairs"] = (
    score_gain_df["learned_n_pairs"].fillna(0)
    - score_gain_df["raw_n_pairs"].fillna(0)
)

score_gain_df["delta_mean_score_learned_minus_raw"] = (
    score_gain_df["learned_mean_score"] - score_gain_df["raw_mean_score"]
)

score_gain_df["delta_mean_abs_score_learned_minus_raw"] = (
    score_gain_df["learned_mean_abs_score"] - score_gain_df["raw_mean_abs_score"]
)

score_gain_df = score_gain_df.sort_values(
    ["gain_n_scored_pairs", "delta_mean_abs_score_learned_minus_raw"],
    ascending=[False, False],
).reset_index(drop=True)

print("\nTop gene/metric gains by scorable pairs and mean absolute score:")
display(score_gain_df.head(30))

# ------------------------------------------------------------------------------
# 11. Optional HDF5 manifest
# ------------------------------------------------------------------------------

if "h5_manifest_df" in globals():
    h5_manifest_df = h5_manifest_df.copy()
elif "h5_manifest_rows" in globals():
    h5_manifest_df = pd.DataFrame(h5_manifest_rows)
elif OFFICIAL_SPRAWL_H5_MANIFEST_PATH.exists():
    try:
        h5_manifest_df = pd.read_csv(OFFICIAL_SPRAWL_H5_MANIFEST_PATH)
    except Exception:
        h5_manifest_df = pd.DataFrame()
else:
    h5_manifest_df = pd.DataFrame()

# ------------------------------------------------------------------------------
# 12. Visualization: score distributions
# ------------------------------------------------------------------------------

print("\nGenerating official SPRAWL score-distribution figure...")

available_methods = official_sprawl_scores_df["method"].astype(str).unique().tolist()
plot_methods_present = [m for m in plot_methods if m in available_methods]
plot_methods_present += sorted([m for m in available_methods if m not in plot_methods_present])

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.ravel()

for ax, metric in zip(axes, SPRAWL_METRICS):
    sub = official_sprawl_scores_df[
        official_sprawl_scores_df["metric"] == metric
    ].copy()

    for method in plot_methods_present:
        vals = sub[sub["method"] == method]["score"].dropna()

        if len(vals) == 0:
            continue

        ax.hist(
            vals,
            bins=50,
            density=True,
            histtype="step",
            linewidth=1.6,
            label=method,
        )

    ax.axvline(0, linestyle="--", linewidth=1.0)
    ax.set_title(f"Official SPRAWL {metric} score distribution")
    ax.set_xlabel("SPRAWL score")
    ax.set_ylabel("Density")

    if len(sub) > 0:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OFFICIAL_SPRAWL_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved official SPRAWL score figure:")
print(f"  {OFFICIAL_SPRAWL_FIG_PATH}")

# ------------------------------------------------------------------------------
# 13. Heatmap: scorable pairs
# ------------------------------------------------------------------------------

scorable_matrix = official_method_summary_df.pivot_table(
    index="metric",
    columns="method",
    values="n_scored_cell_gene_pairs",
    aggfunc="sum",
)

scorable_matrix = scorable_matrix.reindex(
    index=SPRAWL_METRICS,
    columns=[m for m in plot_methods_present if m in scorable_matrix.columns],
)

fig, ax = plt.subplots(figsize=(12, 5))

arr = scorable_matrix.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    cmap="viridis",
)

ax.set_title("Official SPRAWL: scorable cell-gene pairs")
ax.set_xlabel("Method")
ax.set_ylabel("Metric")

ax.set_xticks(np.arange(len(scorable_matrix.columns)))
ax.set_xticklabels(scorable_matrix.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(len(scorable_matrix.index)))
ax.set_yticklabels(scorable_matrix.index)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{int(val):,}", ha="center", va="center", fontsize=8)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Scored pairs")

plt.tight_layout()
plt.savefig(OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved scorable-pairs heatmap:")
print(f"  {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 14. Save outputs
# ------------------------------------------------------------------------------

official_sprawl_scores_df.to_csv(OFFICIAL_SPRAWL_SCORES_PATH, index=False)
official_method_summary_df.to_csv(OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH, index=False)
official_gene_gain_df.to_csv(OFFICIAL_SPRAWL_GENE_GAIN_PATH, index=False)
pattern_summary_df.to_csv(OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH, index=False)
score_gain_df.to_csv(OFFICIAL_SPRAWL_TOP_GAIN_PATH, index=False)

if len(h5_manifest_df) > 0:
    h5_manifest_df.to_csv(OFFICIAL_SPRAWL_H5_MANIFEST_PATH, index=False)

print("\nSaved official SPRAWL post-processing outputs:")
print(f"  Cell-gene scores     : {OFFICIAL_SPRAWL_SCORES_PATH}")
print(f"  Method summary       : {OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH}")
print(f"  Gene gain table      : {OFFICIAL_SPRAWL_GENE_GAIN_PATH}")
print(f"  Pattern summary      : {OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH}")
print(f"  Score gain table     : {OFFICIAL_SPRAWL_TOP_GAIN_PATH}")
if len(h5_manifest_df) > 0:
    print(f"  HDF5 manifest        : {OFFICIAL_SPRAWL_H5_MANIFEST_PATH}")
print(f"  Score figure         : {OFFICIAL_SPRAWL_FIG_PATH}")
print(f"  Scorable heatmap     : {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: Learned 9E completed has more scorable SPRAWL cell-gene pairs than raw.")
print("  Good sign 2: Learned 9E completed has more significant localized pairs than raw.")
print("  Good sign 3: Learned 9E should not collapse all scores to only one pattern.")
print("  Good sign 4: If variance is missing, use score distributions and scorable-pair gains rather than significance.")
print("  Caution: approx_p_value is computed from score / sqrt(variance); use it mainly for comparative screening.")
print("  CosMx note: if Cell 11 used approximate ellipse boundaries, report this as approximate-boundary SPRAWL-compatible analysis.")

print("\n" + "=" * 100)
print("COSMX DOWNSTREAM CELL 11B COMPLETE — Official SPRAWL post-processing")
print("=" * 100)

gc.collect()

==========================THE END=============================

In [ ]:
###############################################################################
# COSMX DOWNSTREAM CELL — ACTUAL InSTAnT gene-gene colocalization
#
# Purpose:
#   Run the official InSTAnT package on CosMx molecule coordinates.
#
# Main comparison:
#   1. Raw observed molecules
#   2. Completed molecules = raw observed + learned 9E imputed molecules
#
# Official InSTAnT workflow:
#   1. Prepare transcript CSV with columns:
#        gene, uID, absX, absY
#   2. Run:
#        obj.load_preprocessed_data(...)
#        obj.run_ProximalPairs(...)
#        obj.run_GlobalColocalization(...)
#
# Important:
#   This cell calls the official InSTAnT package.
#   This is different from the previous manually implemented CPB-like permutation cell.
#
# CosMx-safe:
#   - Uses mol_observed and mol_learned_9E from Cell 1
#   - Keeps original CosMx cell IDs in a mapping file
#   - Maps CosMx string cell IDs like "27_1455" to integer uID values for InSTAnT
#   - Saves all outputs to DOWNSTREAM_DIR
#
# GPU not needed.
###############################################################################

import os
import sys
import gc
import json
import pickle
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("COSMX DOWNSTREAM — ACTUAL InSTAnT gene-gene colocalization")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables from Cell 1
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
    "mol_observed",
    "mol_learned_9E",
    "shared_genes",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

INSTANT_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_official_instant_colocalization"
INSTANT_DIR.mkdir(parents=True, exist_ok=True)

INSTANT_INPUT_DIR = INSTANT_DIR / "instant_inputs"
INSTANT_INPUT_DIR.mkdir(parents=True, exist_ok=True)

INSTANT_OUTPUT_DIR = INSTANT_DIR / "instant_outputs"
INSTANT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_INSTANT_CSV = INSTANT_INPUT_DIR / f"{RUN_NAME}_raw_observed_instant_input.csv"
COMPLETED_INSTANT_CSV = INSTANT_INPUT_DIR / f"{RUN_NAME}_completed_9E_instant_input.csv"

CELL_ID_MAPPING_PATH = INSTANT_INPUT_DIR / f"{RUN_NAME}_instant_cell_id_mapping.csv"
SAMPLED_CELLS_PATH = INSTANT_INPUT_DIR / f"{RUN_NAME}_instant_sampled_cells.csv"
TOP_GENES_PATH = INSTANT_INPUT_DIR / f"{RUN_NAME}_instant_top_genes.csv"

RAW_PREFIX = INSTANT_OUTPUT_DIR / f"{RUN_NAME}_raw_observed"
COMPLETED_PREFIX = INSTANT_OUTPUT_DIR / f"{RUN_NAME}_completed_9E"

SUMMARY_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_summary.csv"
GENE_PAIR_GAIN_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_gene_pair_gain.csv"
FIG_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_summary.png"
RUN_CONFIG_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_run_config.json"

print(f"INSTANT_DIR       : {INSTANT_DIR}")
print(f"INSTANT_INPUT_DIR : {INSTANT_INPUT_DIR}")
print(f"INSTANT_OUTPUT_DIR: {INSTANT_OUTPUT_DIR}")

# ------------------------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------------------------

# Kept aligned with your previous InSTAnT-like/CLQ cell where possible.
DISTANCE_THRESHOLD = 2.0
MIN_MOLS_PER_GENE = 3
SAMPLE_CELLS = 3000
TOP_GENES = 30

# Official InSTAnT run_ProximalPairs uses min_genecount = minimum total transcripts
# in a cell. The previous manual test required at least 3 molecules per gene, so a
# minimal two-gene testable cell has at least 6 transcripts.
MIN_GENCOUNT_FOR_INSTANT = MIN_MOLS_PER_GENE * 2

# Official InSTAnT global colocalization settings.
ALPHA_CELLWISE = 0.05
ALPHA_GLOBAL = 0.01
HIGH_PRECISION = False

THREADS = 2
PRECISION_MODE = "low"

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# If coordinates are pixels, distance_threshold=2.0 means 2 pixels, not 2 microns.
# Set this conversion only if you know the pixel-to-micron scale.
CONVERT_PIXELS_TO_MICRONS = False
PIXEL_SIZE_MICRONS = None

print("\nConfiguration:")
print(f"  DISTANCE_THRESHOLD       : {DISTANCE_THRESHOLD}")
print(f"  MIN_MOLS_PER_GENE        : {MIN_MOLS_PER_GENE}")
print(f"  MIN_GENCOUNT_FOR_INSTANT : {MIN_GENCOUNT_FOR_INSTANT}")
print(f"  SAMPLE_CELLS             : {SAMPLE_CELLS}")
print(f"  TOP_GENES                : {TOP_GENES}")
print(f"  ALPHA_CELLWISE           : {ALPHA_CELLWISE}")
print(f"  ALPHA_GLOBAL             : {ALPHA_GLOBAL}")
print(f"  HIGH_PRECISION           : {HIGH_PRECISION}")
print(f"  THREADS                  : {THREADS}")
print(f"  PRECISION_MODE           : {PRECISION_MODE}")

print(
    "\nCoordinate warning:\n"
    "  Official InSTAnT uses the coordinate units you provide.\n"
    "  If CosMx x/y are global pixels, then DISTANCE_THRESHOLD=2.0 means 2 pixels.\n"
    "  If you want 2 microns, set CONVERT_PIXELS_TO_MICRONS=True and provide PIXEL_SIZE_MICRONS."
)

# ------------------------------------------------------------------------------
# 3. Install and import official InSTAnT
# ------------------------------------------------------------------------------

def pip_install(args):
    print(f"Running: pip {args}")
    subprocess.check_call([sys.executable, "-m", "pip"] + args.split())


def import_official_instant():
    """
    Import official InSTAnT package.

    Official README uses:
        pip install sc-instant
        from InSTAnT import Instant
    """
    try:
        from InSTAnT import Instant
        print("Imported official InSTAnT successfully.")
        return Instant

    except Exception as e1:
        print(f"Initial import failed: {e1}")
        print("Installing official package with: pip install sc-instant")

        try:
            pip_install("install -q sc-instant")
            from InSTAnT import Instant
            print("Imported official InSTAnT after pip install.")
            return Instant

        except Exception as e2:
            raise ImportError(
                "Could not import official InSTAnT.\n\n"
                f"Initial import error: {e1}\n"
                f"pip install/import error: {e2}\n\n"
                "Try running in a fresh runtime. The official README recommends a clean "
                "environment because of package dependency issues."
            )

Instant = import_official_instant()

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def clean_molecule_table_for_instant(df, label):
    """
    Convert CosMx molecule table to minimal molecule table:
      cell_id, gene_id, x, y
    """
    required_cols = ["cell_id", "gene_id", "x", "y"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise KeyError(f"{label} missing required columns: {missing_cols}")

    d = df[required_cols].copy()

    d["cell_id"] = d["cell_id"].astype(str)
    d["gene_id"] = d["gene_id"].astype(str)

    d["x"] = pd.to_numeric(d["x"], errors="coerce").astype(np.float32)
    d["y"] = pd.to_numeric(d["y"], errors="coerce").astype(np.float32)

    d = d.dropna(subset=["cell_id", "gene_id", "x", "y"]).copy()

    return d


def build_official_instant_input(
    mol_df,
    selected_cell_ids,
    selected_genes,
    cell_id_to_uid,
    out_csv,
    dataset_label,
):
    """
    Build official InSTAnT input CSV with columns:
      gene, uID, absX, absY
    """

    selected_cell_ids = set(str(c) for c in selected_cell_ids)
    selected_genes = set(str(g) for g in selected_genes)

    d = mol_df[
        mol_df["cell_id"].astype(str).isin(selected_cell_ids)
        & mol_df["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    d["uID"] = d["cell_id"].astype(str).map(cell_id_to_uid)

    d = d.dropna(subset=["uID"]).copy()
    d["uID"] = d["uID"].astype(int)

    if CONVERT_PIXELS_TO_MICRONS:
        if PIXEL_SIZE_MICRONS is None:
            raise ValueError(
                "CONVERT_PIXELS_TO_MICRONS=True but PIXEL_SIZE_MICRONS is None."
            )

        abs_x = d["x"].astype(float) * float(PIXEL_SIZE_MICRONS)
        abs_y = d["y"].astype(float) * float(PIXEL_SIZE_MICRONS)
        coordinate_unit = "microns"
    else:
        abs_x = d["x"].astype(float)
        abs_y = d["y"].astype(float)
        coordinate_unit = "original molecule-table x/y unit"

    instant_df = pd.DataFrame({
        "gene": d["gene_id"].astype(str).values,
        "uID": d["uID"].astype(int).values,
        "absX": abs_x.astype(np.float32).values,
        "absY": abs_y.astype(np.float32).values,
    })

    instant_df.to_csv(out_csv, index=False)

    print(f"\nBuilt official InSTAnT input for {dataset_label}:")
    print(f"  rows          : {len(instant_df):,}")
    print(f"  cells         : {instant_df['uID'].nunique():,}")
    print(f"  genes         : {instant_df['gene'].nunique():,}")
    print(f"  coordinate unit: {coordinate_unit}")
    print(f"  saved         : {out_csv}")

    return instant_df


def instantiate_instant():
    """
    Make official InSTAnT object robustly across constructor versions.
    """
    try:
        obj = Instant(
            threads=THREADS,
            precision_mode=PRECISION_MODE,
            min_intensity=0,
            min_area=0,
        )
        return obj
    except TypeError:
        try:
            obj = Instant(
                threads=THREADS,
                precision_mode=PRECISION_MODE,
            )
            return obj
        except TypeError:
            obj = Instant()
            return obj


def run_official_instant_for_dataset(input_csv, prefix, dataset_label):
    """
    Run official InSTAnT:
      load_preprocessed_data
      run_ProximalPairs
      run_GlobalColocalization
    """

    print("\n" + "=" * 90)
    print(f"Running official InSTAnT for: {dataset_label}")
    print("=" * 90)

    obj = instantiate_instant()

    print("Loading preprocessed InSTAnT CSV...")
    obj.load_preprocessed_data(data=str(input_csv))

    # Output files.
    pval_matrix_path = str(prefix) + "_pp_test_pval_matrix.pkl"
    gene_count_path = str(prefix) + "_gene_count_matrix.pkl"

    global_coloc_path = str(prefix) + "_global_colocalization.csv"
    expected_coloc_path = str(prefix) + "_expected_colocalization.csv"
    unstacked_path = str(prefix) + "_unstacked_global_pvals.csv"

    print("Running official InSTAnT Proximal Pairs test...")
    obj.run_ProximalPairs(
        distance_threshold=DISTANCE_THRESHOLD,
        min_genecount=MIN_GENCOUNT_FOR_INSTANT,
        pval_matrix_name=pval_matrix_path,
        gene_count_name=gene_count_path,
    )

    print("Running official InSTAnT Global Colocalization / CPB analysis...")
    obj.run_GlobalColocalization(
        high_precision=HIGH_PRECISION,
        alpha_cellwise=ALPHA_CELLWISE,
        glob_coloc_name=global_coloc_path,
        exp_coloc_name=expected_coloc_path,
        unstacked_pvals_name=unstacked_path,
    )

    output_paths = {
        "pval_matrix_path": pval_matrix_path,
        "gene_count_path": gene_count_path,
        "global_coloc_path": global_coloc_path,
        "expected_coloc_path": expected_coloc_path,
        "unstacked_path": unstacked_path,
    }

    print("\nOfficial InSTAnT outputs:")
    for k, p in output_paths.items():
        print(f"  {k:22s}: {p} | exists={os.path.exists(p)}")

    return obj, output_paths


def read_csv_or_excel(path):
    """
    Read CSV/Excel if it exists.
    """
    path = Path(path)

    if not path.exists():
        return None

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    return pd.read_csv(path)


def matrix_to_gene_pair_table(df, dataset_label):
    """
    Convert a square gene x gene p-value matrix to long table.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()

    mat = df.copy()

    # If first column is unnamed/index-like, use it as index.
    first_col = mat.columns[0]
    if first_col.lower().startswith("unnamed") or first_col in ["gene", "genes", "gene_id"]:
        mat = mat.set_index(first_col)

    # Keep only numeric columns.
    numeric_cols = []
    for c in mat.columns:
        vals = pd.to_numeric(mat[c], errors="coerce")
        if vals.notna().sum() > 0:
            numeric_cols.append(c)
            mat[c] = vals

    mat = mat[numeric_cols]

    genes_row = mat.index.astype(str).tolist()
    genes_col = mat.columns.astype(str).tolist()

    rows = []

    for i, g1 in enumerate(genes_row):
        for j, g2 in enumerate(genes_col):
            if g1 >= g2:
                continue

            val = mat.iloc[i, j]

            if pd.isna(val):
                continue

            rows.append({
                "dataset": dataset_label,
                "gene_a": str(g1),
                "gene_b": str(g2),
                "p_value": float(val),
            })

    return pd.DataFrame(rows)


def normalize_official_instant_result_table(table, matrix_table, dataset_label):
    """
    Try to produce a gene-pair p-value table from official InSTAnT outputs.

    Priority:
      1. unstacked table, if it contains gene pair columns
      2. global matrix table
    """

    if table is not None and len(table) > 0:
        t = table.copy()

        # Common pair columns.
        col_lower = {c.lower(): c for c in t.columns}

        gene1_col = None
        gene2_col = None

        for c in ["gene_id1", "gene1", "g1", "gene_a"]:
            if c in col_lower:
                gene1_col = col_lower[c]
                break

        for c in ["gene_id2", "gene2", "g2", "gene_b"]:
            if c in col_lower:
                gene2_col = col_lower[c]
                break

        # Sometimes a combined g1g2 column exists.
        combined_col = None
        for c in ["g1g2", "gene_pair", "pair"]:
            if c in col_lower:
                combined_col = col_lower[c]
                break

        # Pick a p-value-like column.
        p_cols = [
            c for c in t.columns
            if "p" in c.lower()
            and pd.to_numeric(t[c], errors="coerce").notna().sum() > 0
        ]

        if gene1_col is not None and gene2_col is not None and len(p_cols) > 0:
            p_col = p_cols[0]

            out = pd.DataFrame({
                "dataset": dataset_label,
                "gene_a": t[gene1_col].astype(str),
                "gene_b": t[gene2_col].astype(str),
                "p_value": pd.to_numeric(t[p_col], errors="coerce"),
                "source_p_col": p_col,
            })

            out = out.dropna(subset=["p_value"]).copy()
            return out

        if combined_col is not None and len(p_cols) > 0:
            p_col = p_cols[0]

            pairs = t[combined_col].astype(str).str.replace(" ", "", regex=False)

            gene_a = []
            gene_b = []

            for pair in pairs:
                if "," in pair:
                    a, b = pair.split(",", 1)
                elif "-" in pair:
                    a, b = pair.split("-", 1)
                elif "_" in pair:
                    a, b = pair.split("_", 1)
                else:
                    a, b = pair, ""

                gene_a.append(a)
                gene_b.append(b)

            out = pd.DataFrame({
                "dataset": dataset_label,
                "gene_a": gene_a,
                "gene_b": gene_b,
                "p_value": pd.to_numeric(t[p_col], errors="coerce"),
                "source_p_col": p_col,
            })

            out = out.dropna(subset=["p_value"]).copy()
            out = out[out["gene_b"] != ""].copy()
            return out

    # Fallback to global matrix.
    return matrix_to_gene_pair_table(matrix_table, dataset_label)


# ------------------------------------------------------------------------------
# 5. Prepare raw and completed CosMx molecule tables
# ------------------------------------------------------------------------------

print("\nPreparing CosMx raw/completed molecule tables...")

if "status" in mol_observed.columns:
    raw_input = mol_observed[
        mol_observed["status"].astype(str) == "observed"
    ].copy()
else:
    raw_input = mol_observed.copy()

raw_mol = clean_molecule_table_for_instant(raw_input, "mol_observed")
learned_imp_mol = clean_molecule_table_for_instant(mol_learned_9E, "mol_learned_9E")

completed_mol = pd.concat(
    [
        raw_mol.assign(source="raw_observed"),
        learned_imp_mol.assign(source="learned_9E_imputed"),
    ],
    ignore_index=True,
)

print(f"Raw observed molecules       : {len(raw_mol):,}")
print(f"Learned 9E imputed molecules : {len(learned_imp_mol):,}")
print(f"Completed molecules          : {len(completed_mol):,}")

# ------------------------------------------------------------------------------
# 6. Sample cells and choose top genes
# ------------------------------------------------------------------------------

print("\nSampling cells and selecting top genes...")

raw_cell_counts = raw_mol.groupby("cell_id", observed=True).size()
completed_cell_counts = completed_mol.groupby("cell_id", observed=True).size()

valid_cids = sorted(
    set(raw_cell_counts[raw_cell_counts >= MIN_GENCOUNT_FOR_INSTANT].index.astype(str))
    & set(completed_cell_counts[completed_cell_counts >= MIN_GENCOUNT_FOR_INSTANT].index.astype(str))
)

if len(valid_cids) == 0:
    raise RuntimeError("No valid cells found for official InSTAnT input.")

if len(valid_cids) > SAMPLE_CELLS:
    sampled_cids = sorted(
        rng.choice(valid_cids, size=SAMPLE_CELLS, replace=False).tolist()
    )
else:
    sampled_cids = valid_cids

sampled_cids_set = set(sampled_cids)

sampled_cells_df = pd.DataFrame({
    "cell_id": sampled_cids,
})
sampled_cells_df.to_csv(SAMPLED_CELLS_PATH, index=False)

# Select top genes from completed sampled molecules.
completed_sample_for_gene_count = completed_mol[
    completed_mol["cell_id"].astype(str).isin(sampled_cids_set)
].copy()

top_genes = (
    completed_sample_for_gene_count
    .groupby("gene_id", observed=True)
    .size()
    .sort_values(ascending=False)
    .head(TOP_GENES)
    .index
    .astype(str)
    .tolist()
)

top_genes_df = pd.DataFrame({
    "gene_id": top_genes,
    "rank": np.arange(1, len(top_genes) + 1),
})
top_genes_df.to_csv(TOP_GENES_PATH, index=False)

print(f"Valid cells available : {len(valid_cids):,}")
print(f"Sampled cells         : {len(sampled_cids):,}")
print(f"Top genes selected    : {len(top_genes)}")
print(top_genes)

# Map CosMx string cell IDs to integer uID for official InSTAnT.
cell_id_to_uid = {
    cid: i + 1
    for i, cid in enumerate(sampled_cids)
}

uid_to_cell_id = {
    uid: cid
    for cid, uid in cell_id_to_uid.items()
}

mapping_df = pd.DataFrame({
    "cell_id": list(cell_id_to_uid.keys()),
    "uID": list(cell_id_to_uid.values()),
})
mapping_df.to_csv(CELL_ID_MAPPING_PATH, index=False)

print(f"Saved cell ID mapping: {CELL_ID_MAPPING_PATH}")

# ------------------------------------------------------------------------------
# 7. Build official InSTAnT CSV inputs
# ------------------------------------------------------------------------------

raw_instant_df = build_official_instant_input(
    mol_df=raw_mol,
    selected_cell_ids=sampled_cids,
    selected_genes=top_genes,
    cell_id_to_uid=cell_id_to_uid,
    out_csv=RAW_INSTANT_CSV,
    dataset_label="Raw observed",
)

completed_instant_df = build_official_instant_input(
    mol_df=completed_mol,
    selected_cell_ids=sampled_cids,
    selected_genes=top_genes,
    cell_id_to_uid=cell_id_to_uid,
    out_csv=COMPLETED_INSTANT_CSV,
    dataset_label="Learned 9E completed",
)

if len(raw_instant_df) == 0:
    raise RuntimeError("Raw InSTAnT input is empty.")
if len(completed_instant_df) == 0:
    raise RuntimeError("Completed InSTAnT input is empty.")

# ------------------------------------------------------------------------------
# 8. Run official InSTAnT on raw and completed
# ------------------------------------------------------------------------------

raw_obj, raw_paths = run_official_instant_for_dataset(
    input_csv=RAW_INSTANT_CSV,
    prefix=RAW_PREFIX,
    dataset_label="Raw observed",
)

completed_obj, completed_paths = run_official_instant_for_dataset(
    input_csv=COMPLETED_INSTANT_CSV,
    prefix=COMPLETED_PREFIX,
    dataset_label="Learned 9E completed",
)

# ------------------------------------------------------------------------------
# 9. Read official InSTAnT result tables
# ------------------------------------------------------------------------------

print("\nReading official InSTAnT output tables...")

raw_unstacked = read_csv_or_excel(raw_paths["unstacked_path"])
raw_global_matrix = read_csv_or_excel(raw_paths["global_coloc_path"])

completed_unstacked = read_csv_or_excel(completed_paths["unstacked_path"])
completed_global_matrix = read_csv_or_excel(completed_paths["global_coloc_path"])

raw_pairs_df = normalize_official_instant_result_table(
    table=raw_unstacked,
    matrix_table=raw_global_matrix,
    dataset_label="Raw observed",
)

completed_pairs_df = normalize_official_instant_result_table(
    table=completed_unstacked,
    matrix_table=completed_global_matrix,
    dataset_label="Learned 9E completed",
)

for df in [raw_pairs_df, completed_pairs_df]:
    if len(df) > 0:
        df["gene_a"] = df["gene_a"].astype(str)
        df["gene_b"] = df["gene_b"].astype(str)
        df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")
        df["is_significant"] = df["p_value"] < ALPHA_GLOBAL

print(f"Raw official InSTAnT gene pairs       : {len(raw_pairs_df):,}")
print(f"Completed official InSTAnT gene pairs : {len(completed_pairs_df):,}")

print("\nRaw official InSTAnT preview:")
display(raw_pairs_df.head(20))

print("\nCompleted official InSTAnT preview:")
display(completed_pairs_df.head(20))

# ------------------------------------------------------------------------------
# 10. Build summary and gain table
# ------------------------------------------------------------------------------

raw_sig = int(raw_pairs_df["is_significant"].sum()) if len(raw_pairs_df) else 0
completed_sig = int(completed_pairs_df["is_significant"].sum()) if len(completed_pairs_df) else 0

summary_df = pd.DataFrame([{
    "distance_threshold": float(DISTANCE_THRESHOLD),
    "min_mols_per_gene_previous_config": int(MIN_MOLS_PER_GENE),
    "min_genecount_for_instant": int(MIN_GENCOUNT_FOR_INSTANT),
    "sample_cells_requested": int(SAMPLE_CELLS),
    "sample_cells_used": int(len(sampled_cids)),
    "top_genes": int(TOP_GENES),
    "alpha_cellwise": float(ALPHA_CELLWISE),
    "alpha_global": float(ALPHA_GLOBAL),
    "high_precision": bool(HIGH_PRECISION),
    "threads": int(THREADS),
    "precision_mode": str(PRECISION_MODE),
    "convert_pixels_to_microns": bool(CONVERT_PIXELS_TO_MICRONS),
    "pixel_size_microns": PIXEL_SIZE_MICRONS,

    "raw_input_molecules": int(len(raw_instant_df)),
    "completed_input_molecules": int(len(completed_instant_df)),
    "gain_input_molecules": int(len(completed_instant_df) - len(raw_instant_df)),

    "raw_gene_pairs_reported": int(len(raw_pairs_df)),
    "completed_gene_pairs_reported": int(len(completed_pairs_df)),
    "gain_gene_pairs_reported": int(len(completed_pairs_df) - len(raw_pairs_df)),

    "raw_significant_gene_pairs": int(raw_sig),
    "completed_significant_gene_pairs": int(completed_sig),
    "gain_significant_gene_pairs": int(completed_sig - raw_sig),

    "method": "official InSTAnT package: run_ProximalPairs + run_GlobalColocalization",
    "coordinate_unit": "microns if converted, otherwise original x/y molecule-table unit",
}])

print("\nOfficial InSTAnT summary:")
display(summary_df)

# Gene-pair gain table.
raw_pairs_renamed = raw_pairs_df.rename(
    columns={
        "p_value": "raw_p_value",
        "is_significant": "raw_is_significant",
    }
)

completed_pairs_renamed = completed_pairs_df.rename(
    columns={
        "p_value": "completed_p_value",
        "is_significant": "completed_is_significant",
    }
)

keep_raw_cols = [
    c for c in raw_pairs_renamed.columns
    if c in ["gene_a", "gene_b", "raw_p_value", "raw_is_significant"]
]

keep_completed_cols = [
    c for c in completed_pairs_renamed.columns
    if c in ["gene_a", "gene_b", "completed_p_value", "completed_is_significant"]
]

gene_pair_gain_df = raw_pairs_renamed[keep_raw_cols].merge(
    completed_pairs_renamed[keep_completed_cols],
    on=["gene_a", "gene_b"],
    how="outer",
)

gene_pair_gain_df["raw_is_significant"] = (
    gene_pair_gain_df["raw_is_significant"].fillna(False).astype(bool)
)

gene_pair_gain_df["completed_is_significant"] = (
    gene_pair_gain_df["completed_is_significant"].fillna(False).astype(bool)
)

gene_pair_gain_df["became_significant_after_completion"] = (
    (~gene_pair_gain_df["raw_is_significant"])
    & (gene_pair_gain_df["completed_is_significant"])
)

gene_pair_gain_df["lost_significance_after_completion"] = (
    (gene_pair_gain_df["raw_is_significant"])
    & (~gene_pair_gain_df["completed_is_significant"])
)

gene_pair_gain_df["delta_minus_log10_p_completed_minus_raw"] = (
    -np.log10(gene_pair_gain_df["completed_p_value"].replace(0, np.nan))
    - -np.log10(gene_pair_gain_df["raw_p_value"].replace(0, np.nan))
)

gene_pair_gain_df = gene_pair_gain_df.sort_values(
    [
        "became_significant_after_completion",
        "completed_is_significant",
        "delta_minus_log10_p_completed_minus_raw",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

print("\nOfficial InSTAnT gene-pair gain table:")
display(gene_pair_gain_df.head(30))

# ------------------------------------------------------------------------------
# 11. Visualization
# ------------------------------------------------------------------------------

print("\nGenerating official InSTAnT summary figure...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: input molecules and reported pairs.
cats = ["Input\nmolecules", "Reported\ngene pairs", f"Significant\np<{ALPHA_GLOBAL}"]
raw_vals = [len(raw_instant_df), len(raw_pairs_df), raw_sig]
completed_vals = [len(completed_instant_df), len(completed_pairs_df), completed_sig]

x = np.arange(len(cats))
w = 0.35

axes[0].bar(x - w / 2, raw_vals, w, label="Raw observed")
axes[0].bar(x + w / 2, completed_vals, w, label="Learned 9E completed")
axes[0].set_xticks(x)
axes[0].set_xticklabels(cats)
axes[0].set_ylabel("Count")
axes[0].set_title("Official InSTAnT: testability and significance")
axes[0].legend()

if max(raw_vals + completed_vals) > 10000:
    axes[0].ticklabel_format(axis="y", style="sci", scilimits=(4, 4))

# Panel 2: p-value distribution.
if len(raw_pairs_df) > 0:
    raw_p = raw_pairs_df["p_value"].replace(0, np.nan).dropna()
    axes[1].hist(
        -np.log10(raw_p),
        bins=40,
        alpha=0.55,
        density=True,
        label="Raw observed",
    )

if len(completed_pairs_df) > 0:
    comp_p = completed_pairs_df["p_value"].replace(0, np.nan).dropna()
    axes[1].hist(
        -np.log10(comp_p),
        bins=40,
        alpha=0.55,
        density=True,
        label="Learned 9E completed",
    )

axes[1].axvline(-np.log10(ALPHA_GLOBAL), linestyle="--", label=f"p={ALPHA_GLOBAL}")
axes[1].set_xlabel("-log10(global InSTAnT p-value)")
axes[1].set_ylabel("Density")
axes[1].set_title("Official InSTAnT gene-pair p-value distribution")
axes[1].legend()

# Panel 3: top completed significant pairs.
if len(completed_pairs_df) > 0:
    top_completed = completed_pairs_df.copy()
    top_completed = top_completed.sort_values("p_value", ascending=True).head(15)

    labels = [
        f"{a}–{b}"
        for a, b in zip(top_completed["gene_a"], top_completed["gene_b"])
    ]

    vals = -np.log10(top_completed["p_value"].replace(0, np.nan))

    axes[2].barh(labels, vals)
    axes[2].axvline(-np.log10(ALPHA_GLOBAL), linestyle="--", alpha=0.7)
    axes[2].set_xlabel("-log10(p-value)")
    axes[2].set_title("Top official InSTAnT pairs in completed data")
    axes[2].invert_yaxis()
else:
    axes[2].text(0.5, 0.5, "No completed InSTAnT pairs", ha="center", va="center")
    axes[2].axis("off")

plt.tight_layout()
plt.savefig(FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure: {FIG_PATH}")

# ------------------------------------------------------------------------------
# 12. Save normalized outputs and config
# ------------------------------------------------------------------------------

raw_pairs_out = INSTANT_DIR / f"{RUN_NAME}_official_instant_raw_gene_pairs.csv"
completed_pairs_out = INSTANT_DIR / f"{RUN_NAME}_official_instant_completed_gene_pairs.csv"

raw_pairs_df.to_csv(raw_pairs_out, index=False)
completed_pairs_df.to_csv(completed_pairs_out, index=False)

summary_df.to_csv(SUMMARY_PATH, index=False)
gene_pair_gain_df.to_csv(GENE_PAIR_GAIN_PATH, index=False)

run_config = {
    "DISTANCE_THRESHOLD": DISTANCE_THRESHOLD,
    "MIN_MOLS_PER_GENE": MIN_MOLS_PER_GENE,
    "MIN_GENCOUNT_FOR_INSTANT": MIN_GENCOUNT_FOR_INSTANT,
    "SAMPLE_CELLS": SAMPLE_CELLS,
    "TOP_GENES": TOP_GENES,
    "ALPHA_CELLWISE": ALPHA_CELLWISE,
    "ALPHA_GLOBAL": ALPHA_GLOBAL,
    "HIGH_PRECISION": HIGH_PRECISION,
    "THREADS": THREADS,
    "PRECISION_MODE": PRECISION_MODE,
    "RANDOM_STATE": RANDOM_STATE,
    "CONVERT_PIXELS_TO_MICRONS": CONVERT_PIXELS_TO_MICRONS,
    "PIXEL_SIZE_MICRONS": PIXEL_SIZE_MICRONS,
    "top_genes": top_genes,
    "raw_paths": raw_paths,
    "completed_paths": completed_paths,
}

with open(RUN_CONFIG_PATH, "w") as f:
    json.dump(run_config, f, indent=2)

print("\nSaved official InSTAnT outputs:")
print(f"  Raw input CSV             : {RAW_INSTANT_CSV}")
print(f"  Completed input CSV       : {COMPLETED_INSTANT_CSV}")
print(f"  Cell ID mapping           : {CELL_ID_MAPPING_PATH}")
print(f"  Sampled cells             : {SAMPLED_CELLS_PATH}")
print(f"  Top genes                 : {TOP_GENES_PATH}")
print(f"  Raw normalized pairs      : {raw_pairs_out}")
print(f"  Completed normalized pairs: {completed_pairs_out}")
print(f"  Summary                   : {SUMMARY_PATH}")
print(f"  Gene-pair gain            : {GENE_PAIR_GAIN_PATH}")
print(f"  Figure                    : {FIG_PATH}")
print(f"  Run config                : {RUN_CONFIG_PATH}")
print(f"  InSTAnT raw prefix        : {RAW_PREFIX}")
print(f"  InSTAnT completed prefix  : {COMPLETED_PREFIX}")

print("\nInterpretation guide:")
print("  Good sign 1: completed data has more input molecules and more reported/testable gene pairs.")
print("  Good sign 2: completed data reveals additional significant InSTAnT d-colocalized gene pairs.")
print("  Good sign 3: top completed pairs are biologically plausible, not all random housekeeping-only pairs.")
print("  Good sign 4: completed data should not make every pair significant.")
print("  Caution 1: DISTANCE_THRESHOLD uses the coordinate unit supplied to InSTAnT.")
print("  Caution 2: if CosMx x/y are pixels, convert to microns before claiming d=2 µm.")
print("  Caution 3: this is true official InSTAnT, but run time can be high; increase SAMPLE_CELLS/TOP_GENES only after this works.")

print("\n" + "=" * 100)
print("COSMX OFFICIAL InSTAnT COMPLETE")
print("=" * 100)

gc.collect()

In [ ]:
###############################################################################
# FIXED CONTINUATION CELL — Official InSTAnT after constructor error
#
# Use this after the previous official InSTAnT cell failed at:
#   TypeError: Instant.__init__() missing 1 required positional argument: 'distance_threshold'
#
# This cell reuses the already-created raw/completed InSTAnT input CSVs and runs:
#   1. load_preprocessed_data()
#   2. run_ProximalPairs()
#   3. run_GlobalColocalization()
#
# It is CosMx-safe and saves outputs to the same official InSTAnT folder.
###############################################################################

import os
import sys
import gc
import json
import inspect
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("FIXED CONTINUATION — Official InSTAnT run after constructor error")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables from the previous failed cell / Cell 1
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Reconstruct paths from the previous official InSTAnT cell
# ------------------------------------------------------------------------------

INSTANT_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_official_instant_colocalization"
INSTANT_INPUT_DIR = INSTANT_DIR / "instant_inputs"
INSTANT_OUTPUT_DIR = INSTANT_DIR / "instant_outputs"

INSTANT_DIR.mkdir(parents=True, exist_ok=True)
INSTANT_INPUT_DIR.mkdir(parents=True, exist_ok=True)
INSTANT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_INSTANT_CSV = INSTANT_INPUT_DIR / f"{RUN_NAME}_raw_observed_instant_input.csv"
COMPLETED_INSTANT_CSV = INSTANT_INPUT_DIR / f"{RUN_NAME}_completed_9E_instant_input.csv"

CELL_ID_MAPPING_PATH = INSTANT_INPUT_DIR / f"{RUN_NAME}_instant_cell_id_mapping.csv"
SAMPLED_CELLS_PATH = INSTANT_INPUT_DIR / f"{RUN_NAME}_instant_sampled_cells.csv"
TOP_GENES_PATH = INSTANT_INPUT_DIR / f"{RUN_NAME}_instant_top_genes.csv"

RAW_PREFIX = INSTANT_OUTPUT_DIR / f"{RUN_NAME}_raw_observed"
COMPLETED_PREFIX = INSTANT_OUTPUT_DIR / f"{RUN_NAME}_completed_9E"

SUMMARY_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_summary.csv"
GENE_PAIR_GAIN_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_gene_pair_gain.csv"
FIG_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_summary.png"
RUN_CONFIG_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_run_config_fixed.json"

RAW_PAIRS_OUT = INSTANT_DIR / f"{RUN_NAME}_official_instant_raw_gene_pairs.csv"
COMPLETED_PAIRS_OUT = INSTANT_DIR / f"{RUN_NAME}_official_instant_completed_gene_pairs.csv"

print(f"INSTANT_DIR       : {INSTANT_DIR}")
print(f"RAW_INSTANT_CSV   : {RAW_INSTANT_CSV}")
print(f"COMPLETED_CSV     : {COMPLETED_INSTANT_CSV}")

if not RAW_INSTANT_CSV.exists():
    raise FileNotFoundError(
        f"Raw InSTAnT input CSV not found:\n{RAW_INSTANT_CSV}\n\n"
        "Rerun the official InSTAnT preparation cell up to input CSV creation."
    )

if not COMPLETED_INSTANT_CSV.exists():
    raise FileNotFoundError(
        f"Completed InSTAnT input CSV not found:\n{COMPLETED_INSTANT_CSV}\n\n"
        "Rerun the official InSTAnT preparation cell up to input CSV creation."
    )

# ------------------------------------------------------------------------------
# 2. Configuration — same as previous cell
# ------------------------------------------------------------------------------

DISTANCE_THRESHOLD = 2.0
MIN_MOLS_PER_GENE = 3
MIN_GENCOUNT_FOR_INSTANT = MIN_MOLS_PER_GENE * 2

SAMPLE_CELLS = 3000
TOP_GENES = 30

ALPHA_CELLWISE = 0.05
ALPHA_GLOBAL = 0.01
HIGH_PRECISION = False

THREADS = 2
PRECISION_MODE = "low"

RANDOM_STATE = 42

print("\nConfiguration:")
print(f"  DISTANCE_THRESHOLD       : {DISTANCE_THRESHOLD}")
print(f"  MIN_GENCOUNT_FOR_INSTANT : {MIN_GENCOUNT_FOR_INSTANT}")
print(f"  ALPHA_CELLWISE           : {ALPHA_CELLWISE}")
print(f"  ALPHA_GLOBAL             : {ALPHA_GLOBAL}")
print(f"  HIGH_PRECISION           : {HIGH_PRECISION}")
print(f"  THREADS                  : {THREADS}")
print(f"  PRECISION_MODE           : {PRECISION_MODE}")

# ------------------------------------------------------------------------------
# 3. Import official InSTAnT
# ------------------------------------------------------------------------------

def pip_install(args):
    print(f"Running: pip {args}")
    subprocess.check_call([sys.executable, "-m", "pip"] + args.split())


try:
    from InSTAnT import Instant
    print("\nImported official InSTAnT successfully.")
except Exception as e:
    print(f"\nImport failed: {e}")
    print("Installing sc-instant...")
    pip_install("install -q sc-instant")
    from InSTAnT import Instant
    print("Imported official InSTAnT after install.")

print("\nInstant constructor signature:")
try:
    print(inspect.signature(Instant))
except Exception as e:
    print(f"Could not inspect Instant signature: {e}")

# ------------------------------------------------------------------------------
# 4. Robust Instant object constructor
# ------------------------------------------------------------------------------

def instantiate_instant_fixed():
    """
    Robustly instantiate official InSTAnT across package versions.

    Your installed version requires:
        Instant(distance_threshold=...)
    Some README versions show:
        Instant(threads=..., precision_mode=..., min_intensity=..., min_area=...)
    This function tries both styles.
    """

    constructor_attempts = [
        {
            "distance_threshold": DISTANCE_THRESHOLD,
            "threads": THREADS,
            "precision_mode": PRECISION_MODE,
            "min_intensity": 0,
            "min_area": 0,
        },
        {
            "distance_threshold": DISTANCE_THRESHOLD,
            "threads": THREADS,
            "precision_mode": PRECISION_MODE,
        },
        {
            "distance_threshold": DISTANCE_THRESHOLD,
        },
    ]

    last_error = None

    for kwargs in constructor_attempts:
        try:
            print(f"Trying Instant constructor with kwargs: {kwargs}")
            obj = Instant(**kwargs)
            print("Instant object created successfully.")
            return obj
        except TypeError as e:
            print(f"  Constructor attempt failed: {e}")
            last_error = e

    raise TypeError(
        "Could not instantiate official InSTAnT with any known constructor pattern.\n"
        f"Last error: {last_error}"
    )

# ------------------------------------------------------------------------------
# 5. Robust wrappers for official InSTAnT methods
# ------------------------------------------------------------------------------

def call_load_preprocessed_data(obj, input_csv):
    """
    Load preprocessed InSTAnT transcript CSV.
    """
    print(f"Loading preprocessed data from:\n  {input_csv}")

    try:
        return obj.load_preprocessed_data(data=str(input_csv))
    except TypeError as e1:
        print(f"load_preprocessed_data(data=...) failed: {e1}")
        try:
            return obj.load_preprocessed_data(str(input_csv))
        except TypeError as e2:
            raise TypeError(
                "Could not call load_preprocessed_data with either supported pattern.\n"
                f"data= error: {e1}\n"
                f"positional error: {e2}"
            )


def call_run_proximal_pairs(obj, pval_matrix_path, gene_count_path):
    """
    Run official InSTAnT proximal-pair test.

    Some versions take distance_threshold in run_ProximalPairs.
    Your installed version likely already stores distance_threshold in __init__.
    This wrapper handles both.
    """

    attempts = [
        {
            "distance_threshold": DISTANCE_THRESHOLD,
            "min_genecount": MIN_GENCOUNT_FOR_INSTANT,
            "pval_matrix_name": str(pval_matrix_path),
            "gene_count_name": str(gene_count_path),
        },
        {
            "min_genecount": MIN_GENCOUNT_FOR_INSTANT,
            "pval_matrix_name": str(pval_matrix_path),
            "gene_count_name": str(gene_count_path),
        },
    ]

    last_error = None

    for kwargs in attempts:
        try:
            print(f"Running run_ProximalPairs with kwargs: {kwargs}")
            return obj.run_ProximalPairs(**kwargs)
        except TypeError as e:
            print(f"  run_ProximalPairs attempt failed: {e}")
            last_error = e

    raise TypeError(
        "Could not call run_ProximalPairs with any known argument pattern.\n"
        f"Last error: {last_error}"
    )


def call_run_global_colocalization(obj, global_coloc_path, expected_coloc_path, unstacked_path):
    """
    Run official InSTAnT global colocalization.
    """
    attempts = [
        {
            "high_precision": HIGH_PRECISION,
            "alpha_cellwise": ALPHA_CELLWISE,
            "glob_coloc_name": str(global_coloc_path),
            "exp_coloc_name": str(expected_coloc_path),
            "unstacked_pvals_name": str(unstacked_path),
        },
        {
            "high_precision": HIGH_PRECISION,
            "alpha_cellwise": ALPHA_CELLWISE,
            "glob_coloc_name": str(global_coloc_path),
            "exp_coloc_name": str(expected_coloc_path),
            "unstacked_pvals_name": str(unstacked_path),
        },
    ]

    last_error = None

    for kwargs in attempts:
        try:
            print(f"Running run_GlobalColocalization with kwargs: {kwargs}")
            return obj.run_GlobalColocalization(**kwargs)
        except TypeError as e:
            print(f"  run_GlobalColocalization attempt failed: {e}")
            last_error = e

    raise TypeError(
        "Could not call run_GlobalColocalization with known argument pattern.\n"
        f"Last error: {last_error}"
    )


def run_official_instant_for_dataset_fixed(input_csv, prefix, dataset_label):
    """
    Full official InSTAnT run for one dataset.
    """

    print("\n" + "=" * 90)
    print(f"Running official InSTAnT for: {dataset_label}")
    print("=" * 90)

    obj = instantiate_instant_fixed()

    call_load_preprocessed_data(obj, input_csv)

    pval_matrix_path = str(prefix) + "_pp_test_pval_matrix.pkl"
    gene_count_path = str(prefix) + "_gene_count_matrix.pkl"

    global_coloc_path = str(prefix) + "_global_colocalization.csv"
    expected_coloc_path = str(prefix) + "_expected_colocalization.csv"
    unstacked_path = str(prefix) + "_unstacked_global_pvals.csv"

    call_run_proximal_pairs(
        obj=obj,
        pval_matrix_path=pval_matrix_path,
        gene_count_path=gene_count_path,
    )

    call_run_global_colocalization(
        obj=obj,
        global_coloc_path=global_coloc_path,
        expected_coloc_path=expected_coloc_path,
        unstacked_path=unstacked_path,
    )

    output_paths = {
        "pval_matrix_path": pval_matrix_path,
        "gene_count_path": gene_count_path,
        "global_coloc_path": global_coloc_path,
        "expected_coloc_path": expected_coloc_path,
        "unstacked_path": unstacked_path,
    }

    print("\nOfficial InSTAnT output paths:")
    for k, p in output_paths.items():
        print(f"  {k:22s}: {p} | exists={os.path.exists(p)}")

    return obj, output_paths

# ------------------------------------------------------------------------------
# 6. Functions to read/normalize official InSTAnT outputs
# ------------------------------------------------------------------------------

def read_csv_or_excel(path):
    path = Path(path)

    if not path.exists():
        print(f"WARNING: output file not found: {path}")
        return None

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    return pd.read_csv(path)


def matrix_to_gene_pair_table(df, dataset_label):
    """
    Convert a square gene x gene p-value matrix to a long gene-pair table.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()

    mat = df.copy()

    first_col = str(mat.columns[0])

    if (
        first_col.lower().startswith("unnamed")
        or first_col.lower() in ["gene", "genes", "gene_id", "index"]
    ):
        mat = mat.set_index(mat.columns[0])

    numeric_cols = []

    for c in mat.columns:
        vals = pd.to_numeric(mat[c], errors="coerce")

        if vals.notna().sum() > 0:
            numeric_cols.append(c)
            mat[c] = vals

    mat = mat[numeric_cols]

    rows = []

    row_genes = mat.index.astype(str).tolist()
    col_genes = mat.columns.astype(str).tolist()

    for i, g1 in enumerate(row_genes):
        for j, g2 in enumerate(col_genes):
            if str(g1) >= str(g2):
                continue

            val = mat.iloc[i, j]

            if pd.isna(val):
                continue

            rows.append({
                "dataset": dataset_label,
                "gene_a": str(g1),
                "gene_b": str(g2),
                "p_value": float(val),
                "source_format": "matrix",
            })

    return pd.DataFrame(rows)


def normalize_official_instant_result_table(table, matrix_table, dataset_label):
    """
    Normalize official InSTAnT output to:
      dataset, gene_a, gene_b, p_value
    """

    # Preferred: unstacked output table.
    if table is not None and len(table) > 0:
        t = table.copy()
        lower = {str(c).lower(): c for c in t.columns}

        gene1_col = None
        gene2_col = None
        combined_col = None

        for c in ["gene_id1", "gene1", "g1", "gene_a"]:
            if c in lower:
                gene1_col = lower[c]
                break

        for c in ["gene_id2", "gene2", "g2", "gene_b"]:
            if c in lower:
                gene2_col = lower[c]
                break

        for c in ["g1g2", "gene_pair", "pair"]:
            if c in lower:
                combined_col = lower[c]
                break

        # p-value-like columns.
        preferred_p_names = [
            "pvalue",
            "p_value",
            "pval",
            "global_pvalue",
            "global_p_value",
            "p_uncond",
            "p",
        ]

        p_col = None

        for name in preferred_p_names:
            if name in lower:
                p_col = lower[name]
                break

        if p_col is None:
            p_candidates = [
                c for c in t.columns
                if "p" in str(c).lower()
                and pd.to_numeric(t[c], errors="coerce").notna().sum() > 0
            ]

            if len(p_candidates) > 0:
                p_col = p_candidates[0]

        if gene1_col is not None and gene2_col is not None and p_col is not None:
            out = pd.DataFrame({
                "dataset": dataset_label,
                "gene_a": t[gene1_col].astype(str),
                "gene_b": t[gene2_col].astype(str),
                "p_value": pd.to_numeric(t[p_col], errors="coerce"),
                "source_format": "unstacked",
                "source_p_col": str(p_col),
            })

            out = out.dropna(subset=["p_value"]).copy()
            return out

        if combined_col is not None and p_col is not None:
            gene_a = []
            gene_b = []

            for pair in t[combined_col].astype(str):
                s = pair.replace(" ", "")

                if "," in s:
                    a, b = s.split(",", 1)
                elif "–" in s:
                    a, b = s.split("–", 1)
                elif "-" in s:
                    a, b = s.split("-", 1)
                elif "_" in s:
                    a, b = s.split("_", 1)
                else:
                    a, b = s, ""

                gene_a.append(a)
                gene_b.append(b)

            out = pd.DataFrame({
                "dataset": dataset_label,
                "gene_a": gene_a,
                "gene_b": gene_b,
                "p_value": pd.to_numeric(t[p_col], errors="coerce"),
                "source_format": "unstacked_combined",
                "source_p_col": str(p_col),
            })

            out = out.dropna(subset=["p_value"]).copy()
            out = out[out["gene_b"] != ""].copy()
            return out

    # Fallback: global matrix.
    return matrix_to_gene_pair_table(matrix_table, dataset_label)

# ------------------------------------------------------------------------------
# 7. Run official InSTAnT on raw and completed
# ------------------------------------------------------------------------------

raw_obj, raw_paths = run_official_instant_for_dataset_fixed(
    input_csv=RAW_INSTANT_CSV,
    prefix=RAW_PREFIX,
    dataset_label="Raw observed",
)

completed_obj, completed_paths = run_official_instant_for_dataset_fixed(
    input_csv=COMPLETED_INSTANT_CSV,
    prefix=COMPLETED_PREFIX,
    dataset_label="Learned 9E completed",
)

# ------------------------------------------------------------------------------
# 8. Read official InSTAnT outputs
# ------------------------------------------------------------------------------

print("\nReading official InSTAnT output tables...")

raw_unstacked = read_csv_or_excel(raw_paths["unstacked_path"])
raw_global_matrix = read_csv_or_excel(raw_paths["global_coloc_path"])

completed_unstacked = read_csv_or_excel(completed_paths["unstacked_path"])
completed_global_matrix = read_csv_or_excel(completed_paths["global_coloc_path"])

raw_pairs_df = normalize_official_instant_result_table(
    table=raw_unstacked,
    matrix_table=raw_global_matrix,
    dataset_label="Raw observed",
)

completed_pairs_df = normalize_official_instant_result_table(
    table=completed_unstacked,
    matrix_table=completed_global_matrix,
    dataset_label="Learned 9E completed",
)

for df in [raw_pairs_df, completed_pairs_df]:
    if len(df) > 0:
        df["gene_a"] = df["gene_a"].astype(str)
        df["gene_b"] = df["gene_b"].astype(str)
        df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")
        df["is_significant"] = df["p_value"] < ALPHA_GLOBAL

print(f"\nRaw official InSTAnT gene pairs       : {len(raw_pairs_df):,}")
print(f"Completed official InSTAnT gene pairs : {len(completed_pairs_df):,}")

print("\nRaw official InSTAnT preview:")
display(raw_pairs_df.head(20))

print("\nCompleted official InSTAnT preview:")
display(completed_pairs_df.head(20))

# ------------------------------------------------------------------------------
# 9. Summary and gain table
# ------------------------------------------------------------------------------

raw_input_rows = pd.read_csv(RAW_INSTANT_CSV, usecols=["gene"]).shape[0]
completed_input_rows = pd.read_csv(COMPLETED_INSTANT_CSV, usecols=["gene"]).shape[0]

raw_sig = int(raw_pairs_df["is_significant"].sum()) if len(raw_pairs_df) else 0
completed_sig = int(completed_pairs_df["is_significant"].sum()) if len(completed_pairs_df) else 0

summary_df = pd.DataFrame([{
    "distance_threshold": float(DISTANCE_THRESHOLD),
    "min_mols_per_gene_previous_config": int(MIN_MOLS_PER_GENE),
    "min_genecount_for_instant": int(MIN_GENCOUNT_FOR_INSTANT),
    "alpha_cellwise": float(ALPHA_CELLWISE),
    "alpha_global": float(ALPHA_GLOBAL),
    "high_precision": bool(HIGH_PRECISION),
    "threads": int(THREADS),
    "precision_mode": str(PRECISION_MODE),

    "raw_input_molecules": int(raw_input_rows),
    "completed_input_molecules": int(completed_input_rows),
    "gain_input_molecules": int(completed_input_rows - raw_input_rows),

    "raw_gene_pairs_reported": int(len(raw_pairs_df)),
    "completed_gene_pairs_reported": int(len(completed_pairs_df)),
    "gain_gene_pairs_reported": int(len(completed_pairs_df) - len(raw_pairs_df)),

    "raw_significant_gene_pairs": int(raw_sig),
    "completed_significant_gene_pairs": int(completed_sig),
    "gain_significant_gene_pairs": int(completed_sig - raw_sig),

    "method": "official InSTAnT package",
    "constructor_fix": "Instant(distance_threshold=DISTANCE_THRESHOLD, ...)",
}])

print("\nOfficial InSTAnT summary:")
display(summary_df)

raw_pairs_renamed = raw_pairs_df.rename(
    columns={
        "p_value": "raw_p_value",
        "is_significant": "raw_is_significant",
    }
)

completed_pairs_renamed = completed_pairs_df.rename(
    columns={
        "p_value": "completed_p_value",
        "is_significant": "completed_is_significant",
    }
)

keep_raw_cols = [
    c for c in raw_pairs_renamed.columns
    if c in ["gene_a", "gene_b", "raw_p_value", "raw_is_significant"]
]

keep_completed_cols = [
    c for c in completed_pairs_renamed.columns
    if c in ["gene_a", "gene_b", "completed_p_value", "completed_is_significant"]
]

gene_pair_gain_df = raw_pairs_renamed[keep_raw_cols].merge(
    completed_pairs_renamed[keep_completed_cols],
    on=["gene_a", "gene_b"],
    how="outer",
)

if len(gene_pair_gain_df) > 0:
    gene_pair_gain_df["raw_is_significant"] = (
        gene_pair_gain_df["raw_is_significant"].fillna(False).astype(bool)
    )

    gene_pair_gain_df["completed_is_significant"] = (
        gene_pair_gain_df["completed_is_significant"].fillna(False).astype(bool)
    )

    gene_pair_gain_df["became_significant_after_completion"] = (
        (~gene_pair_gain_df["raw_is_significant"])
        & (gene_pair_gain_df["completed_is_significant"])
    )

    gene_pair_gain_df["lost_significance_after_completion"] = (
        (gene_pair_gain_df["raw_is_significant"])
        & (~gene_pair_gain_df["completed_is_significant"])
    )

    gene_pair_gain_df["delta_minus_log10_p_completed_minus_raw"] = (
        -np.log10(gene_pair_gain_df["completed_p_value"].replace(0, np.nan))
        - -np.log10(gene_pair_gain_df["raw_p_value"].replace(0, np.nan))
    )

    gene_pair_gain_df = gene_pair_gain_df.sort_values(
        [
            "became_significant_after_completion",
            "completed_is_significant",
            "delta_minus_log10_p_completed_minus_raw",
        ],
        ascending=[False, False, False],
    ).reset_index(drop=True)

print("\nOfficial InSTAnT gene-pair gain table:")
display(gene_pair_gain_df.head(30))

# ------------------------------------------------------------------------------
# 10. Visualization
# ------------------------------------------------------------------------------

print("\nGenerating official InSTAnT summary figure...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cats = [
    "Input\nmolecules",
    "Reported\ngene pairs",
    f"Significant\np<{ALPHA_GLOBAL}",
]

raw_vals = [
    raw_input_rows,
    len(raw_pairs_df),
    raw_sig,
]

completed_vals = [
    completed_input_rows,
    len(completed_pairs_df),
    completed_sig,
]

x = np.arange(len(cats))
w = 0.35

axes[0].bar(x - w / 2, raw_vals, w, label="Raw observed")
axes[0].bar(x + w / 2, completed_vals, w, label="Learned 9E completed")

axes[0].set_xticks(x)
axes[0].set_xticklabels(cats)
axes[0].set_ylabel("Count")
axes[0].set_title("Official InSTAnT: testability and significance")
axes[0].legend()

if max(raw_vals + completed_vals) > 10000:
    axes[0].ticklabel_format(axis="y", style="sci", scilimits=(4, 4))

# p-value distribution.
if len(raw_pairs_df) > 0:
    raw_p = raw_pairs_df["p_value"].replace(0, np.nan).dropna()
    axes[1].hist(
        -np.log10(raw_p),
        bins=40,
        alpha=0.55,
        density=True,
        label="Raw observed",
    )

if len(completed_pairs_df) > 0:
    comp_p = completed_pairs_df["p_value"].replace(0, np.nan).dropna()
    axes[1].hist(
        -np.log10(comp_p),
        bins=40,
        alpha=0.55,
        density=True,
        label="Learned 9E completed",
    )

axes[1].axvline(-np.log10(ALPHA_GLOBAL), linestyle="--", label=f"p={ALPHA_GLOBAL}")
axes[1].set_xlabel("-log10(global InSTAnT p-value)")
axes[1].set_ylabel("Density")
axes[1].set_title("Official InSTAnT gene-pair p-value distribution")
axes[1].legend()

# Top completed pairs.
if len(completed_pairs_df) > 0:
    top_completed = completed_pairs_df.sort_values("p_value", ascending=True).head(15)

    labels = [
        f"{a}–{b}"
        for a, b in zip(top_completed["gene_a"], top_completed["gene_b"])
    ]

    vals = -np.log10(top_completed["p_value"].replace(0, np.nan))

    axes[2].barh(labels, vals)
    axes[2].axvline(-np.log10(ALPHA_GLOBAL), linestyle="--", alpha=0.7)
    axes[2].set_xlabel("-log10(p-value)")
    axes[2].set_title("Top official InSTAnT pairs in completed data")
    axes[2].invert_yaxis()
else:
    axes[2].text(0.5, 0.5, "No completed InSTAnT pairs", ha="center", va="center")
    axes[2].axis("off")

plt.tight_layout()
plt.savefig(FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure: {FIG_PATH}")

# ------------------------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------------------------

raw_pairs_df.to_csv(RAW_PAIRS_OUT, index=False)
completed_pairs_df.to_csv(COMPLETED_PAIRS_OUT, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)
gene_pair_gain_df.to_csv(GENE_PAIR_GAIN_PATH, index=False)

run_config = {
    "DISTANCE_THRESHOLD": DISTANCE_THRESHOLD,
    "MIN_MOLS_PER_GENE": MIN_MOLS_PER_GENE,
    "MIN_GENCOUNT_FOR_INSTANT": MIN_GENCOUNT_FOR_INSTANT,
    "ALPHA_CELLWISE": ALPHA_CELLWISE,
    "ALPHA_GLOBAL": ALPHA_GLOBAL,
    "HIGH_PRECISION": HIGH_PRECISION,
    "THREADS": THREADS,
    "PRECISION_MODE": PRECISION_MODE,
    "RANDOM_STATE": RANDOM_STATE,
    "constructor_fix": "Instant(distance_threshold=DISTANCE_THRESHOLD, ...)",
    "raw_paths": raw_paths,
    "completed_paths": completed_paths,
}

with open(RUN_CONFIG_PATH, "w") as f:
    json.dump(run_config, f, indent=2)

print("\nSaved fixed official InSTAnT outputs:")
print(f"  Raw normalized pairs      : {RAW_PAIRS_OUT}")
print(f"  Completed normalized pairs: {COMPLETED_PAIRS_OUT}")
print(f"  Summary                   : {SUMMARY_PATH}")
print(f"  Gene-pair gain            : {GENE_PAIR_GAIN_PATH}")
print(f"  Figure                    : {FIG_PATH}")
print(f"  Run config                : {RUN_CONFIG_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: completed data reports more gene pairs or more significant gene pairs than raw.")
print("  Good sign 2: top completed pairs are biologically plausible.")
print("  Good sign 3: completed data should not make every gene pair significant.")
print("  Caution: DISTANCE_THRESHOLD is in the unit of absX/absY in your input CSV.")
print("  Your current input uses original CosMx x/y units, so d=2.0 may mean 2 pixels unless x/y are microns.")

print("\n" + "=" * 100)
print("FIXED OFFICIAL InSTAnT CONTINUATION COMPLETE")
print("=" * 100)

gc.collect()

In [ ]:
###############################################################################
# FIXED CONTINUATION CELL v2 — Official InSTAnT with .xlsx output files
#
# Reason for fix:
#   Official InSTAnT internally writes global colocalization outputs using
#   pandas ExcelWriter. Therefore filenames must end in .xlsx, not .csv.
#
# Use this after the error:
#   ValueError: No engine for filetype: 'csv'
###############################################################################

import os
import sys
import gc
import json
import inspect
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("FIXED CONTINUATION v2 — Official InSTAnT using .xlsx global output files")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "RUN_NAME",
    "DOWNSTREAM_DIR",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun CosMx downstream Cell 1 first."
    )

DOWNSTREAM_DIR = Path(DOWNSTREAM_DIR)
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------------------------

INSTANT_DIR = DOWNSTREAM_DIR / f"{RUN_NAME}_official_instant_colocalization"
INSTANT_INPUT_DIR = INSTANT_DIR / "instant_inputs"
INSTANT_OUTPUT_DIR = INSTANT_DIR / "instant_outputs"

INSTANT_DIR.mkdir(parents=True, exist_ok=True)
INSTANT_INPUT_DIR.mkdir(parents=True, exist_ok=True)
INSTANT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_INSTANT_CSV = INSTANT_INPUT_DIR / f"{RUN_NAME}_raw_observed_instant_input.csv"
COMPLETED_INSTANT_CSV = INSTANT_INPUT_DIR / f"{RUN_NAME}_completed_9E_instant_input.csv"

RAW_PREFIX = INSTANT_OUTPUT_DIR / f"{RUN_NAME}_raw_observed"
COMPLETED_PREFIX = INSTANT_OUTPUT_DIR / f"{RUN_NAME}_completed_9E"

SUMMARY_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_summary.csv"
GENE_PAIR_GAIN_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_gene_pair_gain.csv"
FIG_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_summary.png"
RUN_CONFIG_PATH = INSTANT_DIR / f"{RUN_NAME}_official_instant_run_config_fixed_v2.json"

RAW_PAIRS_OUT = INSTANT_DIR / f"{RUN_NAME}_official_instant_raw_gene_pairs.csv"
COMPLETED_PAIRS_OUT = INSTANT_DIR / f"{RUN_NAME}_official_instant_completed_gene_pairs.csv"

print(f"INSTANT_DIR     : {INSTANT_DIR}")
print(f"RAW_INPUT       : {RAW_INSTANT_CSV}")
print(f"COMPLETED_INPUT : {COMPLETED_INSTANT_CSV}")

if not RAW_INSTANT_CSV.exists():
    raise FileNotFoundError(f"Raw InSTAnT input CSV not found:\n{RAW_INSTANT_CSV}")

if not COMPLETED_INSTANT_CSV.exists():
    raise FileNotFoundError(f"Completed InSTAnT input CSV not found:\n{COMPLETED_INSTANT_CSV}")

# ------------------------------------------------------------------------------
# 2. Config
# ------------------------------------------------------------------------------

DISTANCE_THRESHOLD = 2.0
MIN_MOLS_PER_GENE = 3
MIN_GENCOUNT_FOR_INSTANT = MIN_MOLS_PER_GENE * 2

ALPHA_CELLWISE = 0.05
ALPHA_GLOBAL = 0.01
HIGH_PRECISION = False

THREADS = 2
PRECISION_MODE = "low"
RANDOM_STATE = 42

print("\nConfiguration:")
print(f"  DISTANCE_THRESHOLD       : {DISTANCE_THRESHOLD}")
print(f"  MIN_GENCOUNT_FOR_INSTANT : {MIN_GENCOUNT_FOR_INSTANT}")
print(f"  ALPHA_CELLWISE           : {ALPHA_CELLWISE}")
print(f"  ALPHA_GLOBAL             : {ALPHA_GLOBAL}")
print(f"  HIGH_PRECISION           : {HIGH_PRECISION}")
print(f"  THREADS                  : {THREADS}")
print(f"  PRECISION_MODE           : {PRECISION_MODE}")

# ------------------------------------------------------------------------------
# 3. Import official InSTAnT
# ------------------------------------------------------------------------------

def pip_install(args):
    print(f"Running: pip {args}")
    subprocess.check_call([sys.executable, "-m", "pip"] + args.split())

try:
    from InSTAnT import Instant
    print("\nImported official InSTAnT successfully.")
except Exception as e:
    print(f"\nImport failed: {e}")
    print("Installing sc-instant...")
    pip_install("install -q sc-instant")
    from InSTAnT import Instant
    print("Imported official InSTAnT after install.")

print("\nInstant constructor signature:")
try:
    print(inspect.signature(Instant))
except Exception as e:
    print(f"Could not inspect Instant signature: {e}")

# ------------------------------------------------------------------------------
# 4. Robust wrappers
# ------------------------------------------------------------------------------

def instantiate_instant_fixed():
    constructor_attempts = [
        {
            "distance_threshold": DISTANCE_THRESHOLD,
            "threads": THREADS,
            "precision_mode": PRECISION_MODE,
            "min_intensity": 0,
            "min_area": 0,
        },
        {
            "distance_threshold": DISTANCE_THRESHOLD,
            "threads": THREADS,
            "precision_mode": PRECISION_MODE,
        },
        {
            "distance_threshold": DISTANCE_THRESHOLD,
        },
    ]

    last_error = None

    for kwargs in constructor_attempts:
        try:
            print(f"Trying Instant constructor with kwargs: {kwargs}")
            obj = Instant(**kwargs)
            print("Instant object created successfully.")
            return obj
        except TypeError as e:
            print(f"  Constructor attempt failed: {e}")
            last_error = e

    raise TypeError(f"Could not instantiate InSTAnT. Last error: {last_error}")


def call_load_preprocessed_data(obj, input_csv):
    print(f"Loading preprocessed data from:\n  {input_csv}")

    try:
        return obj.load_preprocessed_data(data=str(input_csv))
    except TypeError as e1:
        print(f"load_preprocessed_data(data=...) failed: {e1}")
        return obj.load_preprocessed_data(str(input_csv))


def call_run_proximal_pairs(obj, pval_matrix_path, gene_count_path):
    attempts = [
        {
            "distance_threshold": DISTANCE_THRESHOLD,
            "min_genecount": MIN_GENCOUNT_FOR_INSTANT,
            "pval_matrix_name": str(pval_matrix_path),
            "gene_count_name": str(gene_count_path),
        },
        {
            "min_genecount": MIN_GENCOUNT_FOR_INSTANT,
            "pval_matrix_name": str(pval_matrix_path),
            "gene_count_name": str(gene_count_path),
        },
    ]

    last_error = None

    for kwargs in attempts:
        try:
            print(f"Running run_ProximalPairs with kwargs: {kwargs}")
            return obj.run_ProximalPairs(**kwargs)
        except TypeError as e:
            print(f"  run_ProximalPairs attempt failed: {e}")
            last_error = e

    raise TypeError(f"Could not call run_ProximalPairs. Last error: {last_error}")


def call_run_global_colocalization(obj, global_coloc_path, expected_coloc_path, unstacked_path):
    """
    Important:
      These paths must be .xlsx, because official InSTAnT writes Excel output.
    """
    attempts = [
        {
            "high_precision": HIGH_PRECISION,
            "alpha_cellwise": ALPHA_CELLWISE,
            "glob_coloc_name": str(global_coloc_path),
            "exp_coloc_name": str(expected_coloc_path),
            "unstacked_pvals_name": str(unstacked_path),
        },
        {
            "high_precision": HIGH_PRECISION,
            "glob_coloc_name": str(global_coloc_path),
            "exp_coloc_name": str(expected_coloc_path),
            "unstacked_pvals_name": str(unstacked_path),
        },
    ]

    last_error = None

    for kwargs in attempts:
        try:
            print(f"Running run_GlobalColocalization with kwargs: {kwargs}")
            return obj.run_GlobalColocalization(**kwargs)
        except TypeError as e:
            print(f"  run_GlobalColocalization attempt failed: {e}")
            last_error = e

    raise TypeError(f"Could not call run_GlobalColocalization. Last error: {last_error}")


def run_official_instant_for_dataset_fixed_v2(input_csv, prefix, dataset_label):
    print("\n" + "=" * 90)
    print(f"Running official InSTAnT for: {dataset_label}")
    print("=" * 90)

    obj = instantiate_instant_fixed()
    call_load_preprocessed_data(obj, input_csv)

    pval_matrix_path = str(prefix) + "_pp_test_pval_matrix.pkl"
    gene_count_path = str(prefix) + "_gene_count_matrix.pkl"

    # CRITICAL FIX:
    # InSTAnT global output paths must be .xlsx, not .csv.
    global_coloc_path = str(prefix) + "_global_colocalization.xlsx"
    expected_coloc_path = str(prefix) + "_expected_colocalization.xlsx"
    unstacked_path = str(prefix) + "_unstacked_global_pvals.xlsx"

    call_run_proximal_pairs(
        obj=obj,
        pval_matrix_path=pval_matrix_path,
        gene_count_path=gene_count_path,
    )

    call_run_global_colocalization(
        obj=obj,
        global_coloc_path=global_coloc_path,
        expected_coloc_path=expected_coloc_path,
        unstacked_path=unstacked_path,
    )

    output_paths = {
        "pval_matrix_path": pval_matrix_path,
        "gene_count_path": gene_count_path,
        "global_coloc_path": global_coloc_path,
        "expected_coloc_path": expected_coloc_path,
        "unstacked_path": unstacked_path,
    }

    print("\nOfficial InSTAnT output paths:")
    for k, p in output_paths.items():
        print(f"  {k:22s}: {p} | exists={os.path.exists(p)}")

    return obj, output_paths


# ------------------------------------------------------------------------------
# 5. Output readers
# ------------------------------------------------------------------------------

def read_table_any(path):
    path = Path(path)

    if not path.exists():
        print(f"WARNING: output file not found: {path}")
        return None

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    return pd.read_pickle(path)


def matrix_to_gene_pair_table(df, dataset_label):
    if df is None or len(df) == 0:
        return pd.DataFrame()

    mat = df.copy()

    first_col = str(mat.columns[0])

    if (
        first_col.lower().startswith("unnamed")
        or first_col.lower() in ["gene", "genes", "gene_id", "index"]
    ):
        mat = mat.set_index(mat.columns[0])

    numeric_cols = []

    for c in mat.columns:
        vals = pd.to_numeric(mat[c], errors="coerce")

        if vals.notna().sum() > 0:
            numeric_cols.append(c)
            mat[c] = vals

    mat = mat[numeric_cols]

    rows = []

    row_genes = mat.index.astype(str).tolist()
    col_genes = mat.columns.astype(str).tolist()

    for i, g1 in enumerate(row_genes):
        for j, g2 in enumerate(col_genes):
            if str(g1) >= str(g2):
                continue

            val = mat.iloc[i, j]

            if pd.isna(val):
                continue

            rows.append({
                "dataset": dataset_label,
                "gene_a": str(g1),
                "gene_b": str(g2),
                "p_value": float(val),
                "source_format": "matrix",
            })

    return pd.DataFrame(rows)


def normalize_official_instant_result_table(table, matrix_table, dataset_label):
    """
    Normalize official InSTAnT output to:
      dataset, gene_a, gene_b, p_value
    """

    if table is not None and len(table) > 0:
        t = table.copy()
        lower = {str(c).lower(): c for c in t.columns}

        gene1_col = None
        gene2_col = None
        combined_col = None

        for c in ["gene_id1", "gene1", "g1", "gene_a"]:
            if c in lower:
                gene1_col = lower[c]
                break

        for c in ["gene_id2", "gene2", "g2", "gene_b"]:
            if c in lower:
                gene2_col = lower[c]
                break

        for c in ["g1g2", "gene_pair", "pair"]:
            if c in lower:
                combined_col = lower[c]
                break

        preferred_p_names = [
            "pvalue",
            "p_value",
            "pval",
            "global_pvalue",
            "global_p_value",
            "p_uncond",
            "p",
        ]

        p_col = None

        for name in preferred_p_names:
            if name in lower:
                p_col = lower[name]
                break

        if p_col is None:
            p_candidates = [
                c for c in t.columns
                if "p" in str(c).lower()
                and pd.to_numeric(t[c], errors="coerce").notna().sum() > 0
            ]

            if len(p_candidates) > 0:
                p_col = p_candidates[0]

        if gene1_col is not None and gene2_col is not None and p_col is not None:
            out = pd.DataFrame({
                "dataset": dataset_label,
                "gene_a": t[gene1_col].astype(str),
                "gene_b": t[gene2_col].astype(str),
                "p_value": pd.to_numeric(t[p_col], errors="coerce"),
                "source_format": "unstacked",
                "source_p_col": str(p_col),
            })

            out = out.dropna(subset=["p_value"]).copy()
            return out

        if combined_col is not None and p_col is not None:
            gene_a = []
            gene_b = []

            for pair in t[combined_col].astype(str):
                s = pair.replace(" ", "")

                if "," in s:
                    a, b = s.split(",", 1)
                elif "–" in s:
                    a, b = s.split("–", 1)
                elif "-" in s:
                    a, b = s.split("-", 1)
                elif "_" in s:
                    a, b = s.split("_", 1)
                else:
                    a, b = s, ""

                gene_a.append(a)
                gene_b.append(b)

            out = pd.DataFrame({
                "dataset": dataset_label,
                "gene_a": gene_a,
                "gene_b": gene_b,
                "p_value": pd.to_numeric(t[p_col], errors="coerce"),
                "source_format": "unstacked_combined",
                "source_p_col": str(p_col),
            })

            out = out.dropna(subset=["p_value"]).copy()
            out = out[out["gene_b"] != ""].copy()
            return out

    return matrix_to_gene_pair_table(matrix_table, dataset_label)


# ------------------------------------------------------------------------------
# 6. Run official InSTAnT for raw and completed
# ------------------------------------------------------------------------------

raw_obj, raw_paths = run_official_instant_for_dataset_fixed_v2(
    input_csv=RAW_INSTANT_CSV,
    prefix=RAW_PREFIX,
    dataset_label="Raw observed",
)

completed_obj, completed_paths = run_official_instant_for_dataset_fixed_v2(
    input_csv=COMPLETED_INSTANT_CSV,
    prefix=COMPLETED_PREFIX,
    dataset_label="Learned 9E completed",
)

# ------------------------------------------------------------------------------
# 7. Read official outputs
# ------------------------------------------------------------------------------

print("\nReading official InSTAnT output tables...")

raw_unstacked = read_table_any(raw_paths["unstacked_path"])
raw_global_matrix = read_table_any(raw_paths["global_coloc_path"])

completed_unstacked = read_table_any(completed_paths["unstacked_path"])
completed_global_matrix = read_table_any(completed_paths["global_coloc_path"])

raw_pairs_df = normalize_official_instant_result_table(
    table=raw_unstacked,
    matrix_table=raw_global_matrix,
    dataset_label="Raw observed",
)

completed_pairs_df = normalize_official_instant_result_table(
    table=completed_unstacked,
    matrix_table=completed_global_matrix,
    dataset_label="Learned 9E completed",
)

for df in [raw_pairs_df, completed_pairs_df]:
    if len(df) > 0:
        df["gene_a"] = df["gene_a"].astype(str)
        df["gene_b"] = df["gene_b"].astype(str)
        df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")
        df["is_significant"] = df["p_value"] < ALPHA_GLOBAL

print(f"\nRaw official InSTAnT gene pairs       : {len(raw_pairs_df):,}")
print(f"Completed official InSTAnT gene pairs : {len(completed_pairs_df):,}")

print("\nRaw official InSTAnT preview:")
display(raw_pairs_df.head(20))

print("\nCompleted official InSTAnT preview:")
display(completed_pairs_df.head(20))

# ------------------------------------------------------------------------------
# 8. Summary and gain table
# ------------------------------------------------------------------------------

raw_input_rows = pd.read_csv(RAW_INSTANT_CSV, usecols=["gene"]).shape[0]
completed_input_rows = pd.read_csv(COMPLETED_INSTANT_CSV, usecols=["gene"]).shape[0]

raw_sig = int(raw_pairs_df["is_significant"].sum()) if len(raw_pairs_df) else 0
completed_sig = int(completed_pairs_df["is_significant"].sum()) if len(completed_pairs_df) else 0

summary_df = pd.DataFrame([{
    "distance_threshold": float(DISTANCE_THRESHOLD),
    "min_mols_per_gene_previous_config": int(MIN_MOLS_PER_GENE),
    "min_genecount_for_instant": int(MIN_GENCOUNT_FOR_INSTANT),
    "alpha_cellwise": float(ALPHA_CELLWISE),
    "alpha_global": float(ALPHA_GLOBAL),
    "high_precision": bool(HIGH_PRECISION),
    "threads": int(THREADS),
    "precision_mode": str(PRECISION_MODE),

    "raw_input_molecules": int(raw_input_rows),
    "completed_input_molecules": int(completed_input_rows),
    "gain_input_molecules": int(completed_input_rows - raw_input_rows),

    "raw_gene_pairs_reported": int(len(raw_pairs_df)),
    "completed_gene_pairs_reported": int(len(completed_pairs_df)),
    "gain_gene_pairs_reported": int(len(completed_pairs_df) - len(raw_pairs_df)),

    "raw_significant_gene_pairs": int(raw_sig),
    "completed_significant_gene_pairs": int(completed_sig),
    "gain_significant_gene_pairs": int(completed_sig - raw_sig),

    "method": "official InSTAnT package",
    "output_fix": "global colocalization outputs saved as .xlsx because InSTAnT uses ExcelWriter",
}])

print("\nOfficial InSTAnT summary:")
display(summary_df)

raw_pairs_renamed = raw_pairs_df.rename(
    columns={
        "p_value": "raw_p_value",
        "is_significant": "raw_is_significant",
    }
)

completed_pairs_renamed = completed_pairs_df.rename(
    columns={
        "p_value": "completed_p_value",
        "is_significant": "completed_is_significant",
    }
)

keep_raw_cols = [
    c for c in raw_pairs_renamed.columns
    if c in ["gene_a", "gene_b", "raw_p_value", "raw_is_significant"]
]

keep_completed_cols = [
    c for c in completed_pairs_renamed.columns
    if c in ["gene_a", "gene_b", "completed_p_value", "completed_is_significant"]
]

gene_pair_gain_df = raw_pairs_renamed[keep_raw_cols].merge(
    completed_pairs_renamed[keep_completed_cols],
    on=["gene_a", "gene_b"],
    how="outer",
)

if len(gene_pair_gain_df) > 0:
    gene_pair_gain_df["raw_is_significant"] = (
        gene_pair_gain_df["raw_is_significant"].fillna(False).astype(bool)
    )

    gene_pair_gain_df["completed_is_significant"] = (
        gene_pair_gain_df["completed_is_significant"].fillna(False).astype(bool)
    )

    gene_pair_gain_df["became_significant_after_completion"] = (
        (~gene_pair_gain_df["raw_is_significant"])
        & (gene_pair_gain_df["completed_is_significant"])
    )

    gene_pair_gain_df["lost_significance_after_completion"] = (
        (gene_pair_gain_df["raw_is_significant"])
        & (~gene_pair_gain_df["completed_is_significant"])
    )

    gene_pair_gain_df["delta_minus_log10_p_completed_minus_raw"] = (
        -np.log10(gene_pair_gain_df["completed_p_value"].replace(0, np.nan))
        - -np.log10(gene_pair_gain_df["raw_p_value"].replace(0, np.nan))
    )

    gene_pair_gain_df = gene_pair_gain_df.sort_values(
        [
            "became_significant_after_completion",
            "completed_is_significant",
            "delta_minus_log10_p_completed_minus_raw",
        ],
        ascending=[False, False, False],
    ).reset_index(drop=True)

print("\nOfficial InSTAnT gene-pair gain table:")
display(gene_pair_gain_df.head(30))

# ------------------------------------------------------------------------------
# 9. Visualization
# ------------------------------------------------------------------------------

print("\nGenerating official InSTAnT summary figure...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cats = [
    "Input\nmolecules",
    "Reported\ngene pairs",
    f"Significant\np<{ALPHA_GLOBAL}",
]

raw_vals = [
    raw_input_rows,
    len(raw_pairs_df),
    raw_sig,
]

completed_vals = [
    completed_input_rows,
    len(completed_pairs_df),
    completed_sig,
]

x = np.arange(len(cats))
w = 0.35

axes[0].bar(x - w / 2, raw_vals, w, label="Raw observed")
axes[0].bar(x + w / 2, completed_vals, w, label="Learned 9E completed")

axes[0].set_xticks(x)
axes[0].set_xticklabels(cats)
axes[0].set_ylabel("Count")
axes[0].set_title("Official InSTAnT: testability and significance")
axes[0].legend()

if max(raw_vals + completed_vals) > 10000:
    axes[0].ticklabel_format(axis="y", style="sci", scilimits=(4, 4))

if len(raw_pairs_df) > 0:
    raw_p = raw_pairs_df["p_value"].replace(0, np.nan).dropna()
    axes[1].hist(
        -np.log10(raw_p),
        bins=40,
        alpha=0.55,
        density=True,
        label="Raw observed",
    )

if len(completed_pairs_df) > 0:
    comp_p = completed_pairs_df["p_value"].replace(0, np.nan).dropna()
    axes[1].hist(
        -np.log10(comp_p),
        bins=40,
        alpha=0.55,
        density=True,
        label="Learned 9E completed",
    )

axes[1].axvline(-np.log10(ALPHA_GLOBAL), linestyle="--", label=f"p={ALPHA_GLOBAL}")
axes[1].set_xlabel("-log10(global InSTAnT p-value)")
axes[1].set_ylabel("Density")
axes[1].set_title("Official InSTAnT gene-pair p-value distribution")
axes[1].legend()

if len(completed_pairs_df) > 0:
    top_completed = completed_pairs_df.sort_values("p_value", ascending=True).head(15)

    labels = [
        f"{a}–{b}"
        for a, b in zip(top_completed["gene_a"], top_completed["gene_b"])
    ]

    vals = -np.log10(top_completed["p_value"].replace(0, np.nan))

    axes[2].barh(labels, vals)
    axes[2].axvline(-np.log10(ALPHA_GLOBAL), linestyle="--", alpha=0.7)
    axes[2].set_xlabel("-log10(p-value)")
    axes[2].set_title("Top official InSTAnT pairs in completed data")
    axes[2].invert_yaxis()
else:
    axes[2].text(0.5, 0.5, "No completed InSTAnT pairs", ha="center", va="center")
    axes[2].axis("off")

plt.tight_layout()
plt.savefig(FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure: {FIG_PATH}")

# ------------------------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------------------------

raw_pairs_df.to_csv(RAW_PAIRS_OUT, index=False)
completed_pairs_df.to_csv(COMPLETED_PAIRS_OUT, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)
gene_pair_gain_df.to_csv(GENE_PAIR_GAIN_PATH, index=False)

run_config = {
    "DISTANCE_THRESHOLD": DISTANCE_THRESHOLD,
    "MIN_MOLS_PER_GENE": MIN_MOLS_PER_GENE,
    "MIN_GENCOUNT_FOR_INSTANT": MIN_GENCOUNT_FOR_INSTANT,
    "ALPHA_CELLWISE": ALPHA_CELLWISE,
    "ALPHA_GLOBAL": ALPHA_GLOBAL,
    "HIGH_PRECISION": HIGH_PRECISION,
    "THREADS": THREADS,
    "PRECISION_MODE": PRECISION_MODE,
    "RANDOM_STATE": RANDOM_STATE,
    "output_fix": "Use .xlsx for InSTAnT global outputs because package uses pandas ExcelWriter.",
    "raw_paths": raw_paths,
    "completed_paths": completed_paths,
}

with open(RUN_CONFIG_PATH, "w") as f:
    json.dump(run_config, f, indent=2)

print("\nSaved fixed official InSTAnT outputs:")
print(f"  Raw normalized pairs      : {RAW_PAIRS_OUT}")
print(f"  Completed normalized pairs: {COMPLETED_PAIRS_OUT}")
print(f"  Summary                   : {SUMMARY_PATH}")
print(f"  Gene-pair gain            : {GENE_PAIR_GAIN_PATH}")
print(f"  Figure                    : {FIG_PATH}")
print(f"  Run config                : {RUN_CONFIG_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: completed data reports more gene pairs or more significant gene pairs than raw.")
print("  Good sign 2: top completed pairs are biologically plausible.")
print("  Good sign 3: completed data should not make every gene pair significant.")
print("  Caution: DISTANCE_THRESHOLD is in the unit of absX/absY in your input CSV.")
print("  Your current input uses original CosMx x/y units, so d=2.0 may mean 2 pixels unless x/y are microns.")

print("\n" + "=" * 100)
print("FIXED OFFICIAL InSTAnT v2 COMPLETE")
print("=" * 100)

gc.collect()